# Alpha Auditor — Executable Calculation & Verification Notebook

**Experiment Title:** KO / PEP Statistical Arbitrage Spread Convergence  
**Original Hypothesis:** `I think Coca-Cola and Pepsi prices tend to converge after unusually large divergences.`  
**Audit Run ID:** `audit-e7eae85a2e` (Schema `2.0`)  
**Market Data As-Of Date:** `2025-12-31` | **Chronological Split Date:** `2021-12-30`  
**Spec SHA-256:** `3d7d0cbb6bcd819f8b28ee4ee7b28001f6d05bab774872a9715f61f9946cfb2c`  
**Aligned Data Slice SHA-256:** `986ebea8eacb9fa8b8baed521fc8179fa58ef06a1a7a13990aeedc04d0ce83fa`

---
This notebook is self-contained and executes end-to-end offline without external network access or API keys. Every signal, trade fill, fee deduction, high-water mark drawdown, econometric test, adversarial stress test, and scorecard verdict is recomputed from the aligned daily OHLCV dataset and verified against the AuditReport.

In [1]:
# Cell 1: Environment Setup, Library Versions & Deterministic Seeds
import hashlib
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels
import statsmodels.api as sm

REPO_ROOT = Path('/Users/simbashakib/Desktop/Alpha Audit')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from engine.schemas import StrategySpec
from engine.backtester import (
    compute_causal_signals,
    compute_pair_leg_weights,
    compute_parameter_sensitivity_surface,
    compute_standard_bracket_profiles,
    simulate_strategy_core,
    _compute_period_metrics,
)
from engine.econometrics import (
    TRADING_DAYS_PER_YEAR,
    compute_pairs_econometrics,
    deflated_sharpe_ratio_bailey_lopez,
    exact_sign_test_p_value,
    lo_mertens_sharpe_se_and_psr,
    newey_west_factor_regression,
    stationary_bootstrap_cis,
)
from engine.stress_tests import (
    build_hypothesis_scorecard_and_ledger,
    run_adversarial_stress_suite,
)

BOOTSTRAP_SEED = 20261107
MONTE_CARLO_SEED = 42
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | pandas: {pd.__version__} | SciPy: {scipy.__version__} | statsmodels: {statsmodels.__version__}')
print(f'Audit Run ID: audit-e7eae85a2e | Spec SHA-256: 3d7d0cbb6bcd819f...')

Python: 3.14.6 | NumPy: 2.4.6 | pandas: 3.0.3 | SciPy: 1.18.0 | statsmodels: 0.14.6
Audit Run ID: audit-e7eae85a2e | Spec SHA-256: 3d7d0cbb6bcd819f...


## Cell 2 — Confirmed `StrategySpec` & Explicit vs. Software-Default Assumptions
Separates parameters explicitly stated in the user's natural-language prompt from literature defaults filled by the compiler.

In [2]:
# Cell 2: Reconstruct Confirmed StrategySpec & Inspect Assumptions
SPEC_JSON = '{\n  "hypothesis_text": "I think Coca-Cola and Pepsi prices tend to converge after unusually large divergences.",\n  "title": "KO / PEP Statistical Arbitrage Spread Convergence",\n  "strategy_family": "pairs_trading",\n  "template_id": "pairs_cointegration_reversion",\n  "signal_variant": "ols_log",\n  "asset_a": "KO",\n  "asset_b": "PEP",\n  "benchmark_ticker": "SPY",\n  "start_date": "2018-01-01",\n  "end_date": "2025-12-31",\n  "threshold_mode": "user_specified",\n  "parameters": {\n    "lookback_window": 20,\n    "secondary_window": 60,\n    "streak_length": 3,\n    "entry_threshold": 2.0,\n    "exit_threshold": 0.25,\n    "stop_z_threshold": 4.0,\n    "holding_period": 20,\n    "exit_mode": "threshold_reversion",\n    "direction": "both"\n  },\n  "execution": {\n    "initial_capital": 100000.0,\n    "execution_timing": "next_open",\n    "commission_bps": 5.0,\n    "slippage_bps": 5.0,\n    "train_split_ratio": 0.5\n  },\n  "risk_controls": {\n    "max_drawdown_circuit_breaker_pct": 25.0,\n    "per_trade_stop_loss_pct": 12.0,\n    "max_holding_days": 30\n  },\n  "selected_diagnostics": [\n    "pairs_cointegration",\n    "stationary_bootstrap",\n    "newey_west_alpha",\n    "cost_sensitivity",\n    "subperiod_regimes",\n    "parameter_sensitivity_grid"\n  ],\n  "human_confirmed": true,\n  "user_specified_fields": [\n    "asset_a",\n    "asset_b"\n  ],\n  "software_inferred_defaults": {\n    "parameters.entry_threshold": "±2.0σ",\n    "parameters.lookback_window": "20d",\n    "parameters.holding_period": "20d max",\n    "parameters.exit_threshold": "±0.25σ",\n    "execution.execution_timing": "next_open (t+1)",\n    "execution.cost_bps": "10 bps (5 comm + 5 slip)",\n    "risk_controls.max_drawdown_circuit_breaker_pct": "25.0%"\n  },\n  "evaluation_stage": "confirmatory",\n  "prior_holdout_touches": 0\n}'
EXPECTED_SNAPSHOT = json.loads('{\n  "run_id": "audit-e7eae85a2e",\n  "schema_version": "2.0",\n  "spec_sha256": "3d7d0cbb6bcd819f8b28ee4ee7b28001f6d05bab774872a9715f61f9946cfb2c",\n  "split_date": "2021-12-30",\n  "market_as_of_date": "2025-12-31",\n  "aligned_slice_sha256": "986ebea8eacb9fa8b8baed521fc8179fa58ef06a1a7a13990aeedc04d0ce83fa",\n  "trade_count": 66,\n  "full_sample": {\n    "period_label": "Full Sample",\n    "start_date": "2018-01-02",\n    "end_date": "2025-12-31",\n    "trading_days": 2011,\n    "trade_count": 66,\n    "exposure_pct": 40.28,\n    "total_gross_return_pct": -13.18,\n    "total_net_return_pct": -23.95,\n    "cagr_net_pct": -3.37,\n    "benchmark_return_pct": 187.48,\n    "benchmark_cagr_pct": 14.15,\n    "excess_return_pct": -211.43,\n    "annualized_volatility_pct": 5.36,\n    "benchmark_volatility_pct": 19.45,\n    "max_drawdown_pct": 25.83,\n    "benchmark_max_drawdown_pct": 33.72,\n    "sharpe_ratio": -0.614,\n    "benchmark_sharpe_ratio": 0.778,\n    "sharpe_se_lo2002": 0.351,\n    "probabilistic_sharpe_psr_pct": 4.02,\n    "sortino_ratio": -0.819,\n    "calmar_ratio": -0.131,\n    "win_rate_pct": 53.03,\n    "profit_factor": 0.536,\n    "mean_trade_net_pct": -0.398,\n    "median_trade_net_pct": 0.163,\n    "std_trade_net_pct": 1.778,\n    "best_trade_net_pct": 3.79,\n    "worst_trade_net_pct": -7.238,\n    "skewness_daily": -0.69,\n    "kurtosis_daily": 26.441,\n    "t_stat_mean_return": -1.734,\n    "p_value_two_sided": 0.0831,\n    "sign_test_p_value": 0.7122,\n    "circuit_breaker_triggered": true,\n    "circuit_breaker_date": "2025-07-17",\n    "is_zero_exposure": false,\n    "halted_prior_to_period": false,\n    "status_note": null\n  },\n  "in_sample": {\n    "period_label": "In-Sample Formation",\n    "start_date": "2018-01-02",\n    "end_date": "2021-12-29",\n    "trading_days": 1006,\n    "trade_count": 36,\n    "exposure_pct": 40.46,\n    "total_gross_return_pct": 6.96,\n    "total_net_return_pct": -0.47,\n    "cagr_net_pct": -0.12,\n    "benchmark_return_pct": 90.41,\n    "benchmark_cagr_pct": 17.51,\n    "excess_return_pct": -90.87,\n    "annualized_volatility_pct": 6.01,\n    "benchmark_volatility_pct": 20.82,\n    "max_drawdown_pct": 9.15,\n    "benchmark_max_drawdown_pct": 33.72,\n    "sharpe_ratio": 0.011,\n    "benchmark_sharpe_ratio": 0.88,\n    "sharpe_se_lo2002": 0.501,\n    "probabilistic_sharpe_psr_pct": 50.85,\n    "sortino_ratio": 0.015,\n    "calmar_ratio": -0.013,\n    "win_rate_pct": 63.89,\n    "profit_factor": 1.005,\n    "mean_trade_net_pct": 0.003,\n    "median_trade_net_pct": 0.512,\n    "std_trade_net_pct": 1.799,\n    "best_trade_net_pct": 3.79,\n    "worst_trade_net_pct": -7.238,\n    "skewness_daily": -0.755,\n    "kurtosis_daily": 28.677,\n    "t_stat_mean_return": 0.021,\n    "p_value_two_sided": 0.983,\n    "sign_test_p_value": 0.1325,\n    "circuit_breaker_triggered": false,\n    "circuit_breaker_date": null,\n    "is_zero_exposure": false,\n    "halted_prior_to_period": false,\n    "status_note": null\n  },\n  "out_of_sample": {\n    "period_label": "Out-of-Sample (Held-Out)",\n    "start_date": "2021-12-30",\n    "end_date": "2025-12-31",\n    "trading_days": 1005,\n    "trade_count": 30,\n    "exposure_pct": 40.1,\n    "total_gross_return_pct": -18.83,\n    "total_net_return_pct": -23.59,\n    "cagr_net_pct": -6.53,\n    "benchmark_return_pct": 50.98,\n    "benchmark_cagr_pct": 10.88,\n    "excess_return_pct": -74.58,\n    "annualized_volatility_pct": 4.6,\n    "benchmark_volatility_pct": 17.99,\n    "max_drawdown_pct": 23.96,\n    "benchmark_max_drawdown_pct": 24.5,\n    "sharpe_ratio": -1.445,\n    "benchmark_sharpe_ratio": 0.664,\n    "sharpe_se_lo2002": 0.494,\n    "probabilistic_sharpe_psr_pct": 0.17,\n    "sortino_ratio": -1.856,\n    "calmar_ratio": -0.272,\n    "win_rate_pct": 40.0,\n    "profit_factor": 0.242,\n    "mean_trade_net_pct": -0.88,\n    "median_trade_net_pct": -0.665,\n    "std_trade_net_pct": 1.657,\n    "best_trade_net_pct": 1.908,\n    "worst_trade_net_pct": -4.366,\n    "skewness_daily": -0.592,\n    "kurtosis_daily": 13.865,\n    "t_stat_mean_return": -2.886,\n    "p_value_two_sided": 0.004,\n    "sign_test_p_value": 0.3616,\n    "circuit_breaker_triggered": true,\n    "circuit_breaker_date": "2025-07-17",\n    "is_zero_exposure": false,\n    "halted_prior_to_period": false,\n    "status_note": null\n  },\n  "factor_regression": {\n    "annualized_alpha_pct": -3.15,\n    "alpha_t_stat_hac": -1.777,\n    "alpha_p_value_hac": 0.0755,\n    "beta": -0.009,\n    "beta_t_stat_hac": -0.465,\n    "r_squared": 0.0011,\n    "information_ratio": -0.905,\n    "newey_west_lags": 7\n  },\n  "bootstrap_cis": [\n    {\n      "metric_name": "Annualized Return (Mean)",\n      "point_estimate": -3.29,\n      "ci_lower_95": -6.96,\n      "ci_upper_95": 0.09,\n      "bootstrap_std_error": 1.81,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "%"\n    },\n    {\n      "metric_name": "Annualized Sharpe Ratio",\n      "point_estimate": -0.614,\n      "ci_lower_95": -1.246,\n      "ci_upper_95": 0.02,\n      "bootstrap_std_error": 0.32,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "ratio"\n    },\n    {\n      "metric_name": "Annualized Excess Return vs. Benchmark",\n      "point_estimate": -18.42,\n      "ci_lower_95": -29.33,\n      "ci_upper_95": -7.25,\n      "bootstrap_std_error": 5.64,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "%"\n    },\n    {\n      "metric_name": "Mean Net Return per Trade",\n      "point_estimate": -0.398,\n      "ci_lower_95": -0.865,\n      "ci_upper_95": 0.026,\n      "bootstrap_std_error": 0.227,\n      "block_length": 1,\n      "replications": 1000,\n      "unit": "%"\n    }\n  ],\n  "stress_test_outcomes": [\n    {\n      "test_id": "outlier_removal",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "-23.95% (N=66)",\n      "stressed_value": "-29.53% (N=63)"\n    },\n    {\n      "test_id": "execution_latency_shock",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "-23.95% (SR -0.61)",\n      "stressed_value": "-24.82% (SR -0.67)"\n    },\n    {\n      "test_id": "regime_vulnerability",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "0 / 5 Regimes",\n      "stressed_value": "-0.45 vs -0.77"\n    },\n    {\n      "test_id": "monte_carlo_random_entry",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "-23.95% (20th pct)",\n      "stressed_value": "-14.44% (p=0.800\\u00b10.018)"\n    },\n    {\n      "test_id": "pairs_structural_break",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "\\u03b2=0.687 (p=0.031)",\n      "stressed_value": "\\u0394\\u03b2=181.8% (p=0.897)"\n    }\n  ],\n  "pairs_diagnostics": {\n    "formation_start": "2018-01-02",\n    "formation_end": "2021-12-29",\n    "ols_alpha": 0.5088,\n    "ols_hedge_ratio_beta": 0.6866,\n    "ols_r_squared": 0.8438,\n    "return_correlation": 0.7392,\n    "level_correlation": 0.9169,\n    "adf_statistic": -3.039,\n    "adf_p_value": 0.0314,\n    "adf_critical_values": {\n      "1%": -3.437,\n      "5%": -2.864,\n      "10%": -2.568\n    },\n    "engle_granger_stat": -3.04,\n    "engle_granger_p_value": 0.1011,\n    "ou_lambda": -0.02056,\n    "half_life_days": 33.4,\n    "hurst_exponent": 0.467,\n    "variance_ratio_q5": 0.9957,\n    "variance_ratio_p_value": 0.9688,\n    "cointegration_status_note": "Formation spread does NOT reject the unit-root null at standard significance levels (Engle-Granger p = 0.1011, return correlation = 0.74). High return correlation does not establish cointegration; out-of-sample divergence risk is elevated."\n  },\n  "verdict_label": "CONTRADICTED BY HISTORICAL DATA",\n  "supported_checks_count": 0,\n  "evaluated_checks_count": 5,\n  "dimension_statuses": {\n    "post_signal_direction": "CONTRADICTED",\n    "statistical_significance": "CONTRADICTED",\n    "out_of_sample_persistence": "CONTRADICTED",\n    "transaction_cost_survival": "CONTRADICTED",\n    "stress_test_resilience": "CONTRADICTED"\n  }\n}')
spec = StrategySpec.model_validate_json(SPEC_JSON)
computed_spec_sha256 = hashlib.sha256(spec.model_dump_json().encode('utf-8')).hexdigest()
assert computed_spec_sha256 == EXPECTED_SNAPSHOT['spec_sha256'], 'Spec SHA-256 mismatch!'

assumptions_rows = [
    ('Strategy Family / Template', f'{spec.strategy_family} / {spec.template_id} ({spec.signal_variant})', 'Core Mapping'),
    ('Target Asset(s)', f'{spec.asset_a}' + (f' / {spec.asset_b}' if spec.asset_b else ''), 'User Stated' if 'asset_a' in spec.user_specified_fields else 'Default'),
    ('Benchmark Ticker', spec.benchmark_ticker, 'Default'),
    ('Primary Lookback Window', f'{spec.parameters.lookback_window} trading days', 'User Stated' if 'parameters.lookback_window' in spec.user_specified_fields else 'Software Default'),
    ('Entry Threshold', f'{spec.parameters.entry_threshold}', 'User Stated' if 'parameters.entry_threshold' in spec.user_specified_fields else 'Software Default'),
    ('Exit Threshold / Mode', f'{spec.parameters.exit_threshold} ({spec.parameters.exit_mode})', 'Software Default'),
    ('Holding Period / Max Hold', f'{spec.parameters.holding_period}d (Max {spec.risk_controls.max_holding_days}d)', 'User Stated' if 'parameters.holding_period' in spec.user_specified_fields else 'Software Default'),
    ('Execution Timing & Costs', f'{spec.execution.execution_timing}, {spec.execution.commission_bps} bps comm + {spec.execution.slippage_bps} bps slip', 'Software Default'),
    ('Risk Controls', f'Circuit Breaker: {spec.risk_controls.max_drawdown_circuit_breaker_pct}%, Stop-Loss: {spec.risk_controls.per_trade_stop_loss_pct}%', 'Software Default'),
]
assumptions_df = pd.DataFrame(assumptions_rows, columns=['Parameter', 'Configured Value', 'Provenance'])
display(assumptions_df)

,Parameter,Configured Value,Provenance
0,Strategy Family / Template,pairs_trading / pairs_cointegration_reversion ...,Core Mapping
1,Target Asset(s),KO / PEP,User Stated
2,Benchmark Ticker,SPY,Default
3,Primary Lookback Window,20 trading days,Software Default
4,Entry Threshold,2.0,Software Default
5,Exit Threshold / Mode,0.25 (threshold_reversion),Software Default
6,Holding Period / Max Hold,20d (Max 30d),Software Default
7,Execution Timing & Costs,"next_open, 5.0 bps comm + 5.0 bps slip",Software Default
8,Risk Controls,"Circuit Breaker: 25.0%, Stop-Loss: 12.0%",Software Default


## Cell 3 — Immutable Aligned Market Data Snapshot & Integrity Verification
Loads the exact inner-joined daily OHLCV market slice embedded at audit time and verifies its cryptographic SHA-256 digest, date monotonicity, and chronological In-Sample / Out-of-Sample split boundary.

In [3]:
# Cell 3: Load Embedded Aligned Market Data & Verify Integrity
DF_RECORDS = json.loads('[{"date":"2018-01-02","open_a":35.099345,"high_a":35.122279,"low_a":34.793533,"close_a":34.816471,"volume_a":10872200,"open_b":91.111547,"high_b":91.301474,"low_b":89.592135,"close_b":89.690895,"volume_b":4165800,"open_bench":234.55539,"close_bench":235.369812},{"date":"2018-01-03","open_a":34.778238,"high_a":34.931141,"low_a":34.663558,"close_a":34.740009,"volume_a":12635600,"open_b":89.576947,"high_b":90.00238,"low_b":89.371823,"close_b":89.455391,"volume_b":4224000,"open_bench":235.536252,"close_bench":236.858612},{"date":"2018-01-04","open_a":34.831763,"high_a":35.33635,"low_a":34.747665,"close_a":35.229317,"volume_a":12709400,"open_b":89.493391,"high_b":90.488604,"low_b":89.341446,"close_b":89.896034,"volume_b":3147000,"open_bench":237.497832,"close_bench":237.856857},{"date":"2018-01-05","open_a":35.183443,"high_a":35.321057,"low_a":35.007602,"close_a":35.221668,"volume_a":13113100,"open_b":89.956831,"high_b":90.473426,"low_b":89.835275,"close_b":90.15435,"volume_b":2714200,"open_bench":238.645069,"close_bench":239.441986},{"date":"2018-01-08","open_a":35.129919,"high_a":35.244596,"low_a":35.076402,"close_a":35.168144,"volume_a":7068600,"open_b":89.80487,"high_b":90.207513,"low_b":89.394628,"close_b":89.637733,"volume_b":5730400,"open_bench":239.345582,"close_bench":239.879791},{"date":"2018-01-09","open_a":35.168154,"high_a":35.443383,"low_a":35.122281,"close_a":35.343994,"volume_a":9962400,"open_b":89.842863,"high_b":90.389853,"low_b":89.60735,"close_b":89.706116,"volume_b":4573600,"open_bench":240.300206,"close_bench":240.422821},{"date":"2018-01-10","open_a":35.343992,"high_a":35.366927,"low_a":35.114635,"close_a":35.221668,"volume_a":10700700,"open_b":89.744094,"high_b":89.918829,"low_b":88.657714,"close_b":89.25029,"volume_b":3363900,"open_bench":239.669639,"close_bench":240.054962},{"date":"2018-01-11","open_a":35.343997,"high_a":35.343997,"low_a":35.061123,"close_a":35.198738,"volume_a":7583200,"open_b":89.371842,"high_b":89.53138,"low_b":88.414611,"close_b":88.794464,"volume_b":3685200,"open_bench":240.606667,"close_bench":241.806412},{"date":"2018-01-12","open_a":35.382223,"high_a":35.466321,"low_a":35.191092,"close_a":35.282837,"volume_a":14570100,"open_b":88.847635,"high_b":89.652927,"low_b":88.771665,"close_b":89.174309,"volume_b":4335500,"open_bench":242.069233,"close_bench":243.382828},{"date":"2018-01-16","open_a":35.28282,"high_a":35.642145,"low_a":35.275173,"close_a":35.573338,"volume_a":16459300,"open_b":89.03757,"high_b":89.911233,"low_b":88.908421,"close_b":89.538979,"volume_b":4936500,"open_bench":244.635068,"close_bench":242.550827},{"date":"2018-01-17","open_a":35.611575,"high_a":35.932674,"low_a":35.588637,"close_a":35.795059,"volume_a":12143100,"open_b":89.865656,"high_b":91.293901,"low_b":89.804878,"close_b":90.541794,"volume_b":4554600,"open_bench":243.479076,"close_bench":244.862717},{"date":"2018-01-18","open_a":35.741538,"high_a":35.986186,"low_a":35.634505,"close_a":35.840927,"volume_a":11952900,"open_b":90.473422,"high_b":90.876071,"low_b":90.139155,"close_b":90.443039,"volume_b":6221800,"open_bench":244.748901,"close_bench":244.451157},{"date":"2018-01-19","open_a":35.925036,"high_a":36.123811,"low_a":35.833291,"close_a":36.055004,"volume_a":16187400,"open_b":90.769699,"high_b":91.681344,"low_b":90.131542,"close_b":91.293892,"volume_b":6002600,"open_bench":245.029115,"close_bench":245.563324},{"date":"2018-01-22","open_a":36.039711,"high_a":36.31494,"low_a":36.001485,"close_a":36.223198,"volume_a":9577400,"open_b":91.240716,"high_b":92.570202,"low_b":91.164747,"close_b":92.311905,"volume_b":4308600,"open_bench":245.353093,"close_bench":247.559921},{"date":"2018-01-23","open_a":36.093235,"high_a":36.467853,"low_a":36.032074,"close_a":36.276722,"volume_a":11463700,"open_b":91.977624,"high_b":93.071603,"low_b":91.810488,"close_b":92.152359,"volume_b":3536900,"open_bench":247.603797,"close_bench":248.085464},{"date":"2018-01-24","open_a":36.337874,"high_a":36.658976,"low_a":36.299649,"close_a":36.567234,"volume_a":15964500,"open_b":92.106762,"high_b":92.289095,"low_b":91.787687,"close_b":92.030792,"volume_b":3445200,"open_bench":248.72473,"close_bench":247.98912},{"date":"2018-01-25","open_a":36.467849,"high_a":36.697206,"low_a":36.467849,"close_a":36.574883,"volume_a":11755000,"open_b":91.62818,"high_b":92.26633,"low_b":91.438253,"close_b":91.848495,"volume_b":2712900,"open_bench":248.84738,"close_bench":248.094238},{"date":"2018-01-26","open_a":36.529015,"high_a":37.171217,"low_a":36.498437,"close_a":37.102409,"volume_a":16671200,"open_b":91.628183,"high_b":92.562618,"low_b":91.628183,"close_b":92.501846,"volume_b":3993900,"open_bench":248.92613,"close_bench":250.966568},{"date":"2018-01-29","open_a":36.857752,"high_a":37.048884,"low_a":36.376104,"close_a":36.467846,"volume_a":11662600,"open_b":92.213129,"high_b":92.805704,"low_b":90.89884,"close_b":90.914032,"volume_b":3854700,"open_bench":250.397351,"close_bench":249.302689},{"date":"2018-01-30","open_a":36.490772,"high_a":36.819518,"low_a":36.23848,"close_a":36.246124,"volume_a":12404800,"open_b":90.921647,"high_b":92.076398,"low_b":90.784899,"close_b":91.081184,"volume_b":4945200,"open_bench":247.481215,"close_bench":246.745605},{"date":"2018-01-31","open_a":36.276714,"high_a":36.620751,"low_a":36.215551,"close_a":36.383747,"volume_a":10960100,"open_b":91.043192,"high_b":91.734528,"low_b":90.792493,"close_b":91.392662,"volume_b":4018100,"open_bench":247.595035,"close_bench":246.868164},{"date":"2018-02-01","open_a":36.24614,"high_a":36.490788,"low_a":36.016783,"close_a":36.276722,"volume_a":10279400,"open_b":91.10397,"high_b":91.932053,"low_b":90.617758,"close_b":91.392662,"volume_b":3485600,"open_bench":246.141256,"close_bench":246.58786},{"date":"2018-02-02","open_a":36.200262,"high_a":36.246135,"low_a":35.649806,"close_a":35.726257,"volume_a":12912300,"open_b":91.09639,"high_b":91.377478,"low_b":89.949232,"close_b":90.15435,"volume_b":3348900,"open_bench":245.274363,"close_bench":241.219757},{"date":"2018-02-05","open_a":35.6651,"high_a":35.871519,"low_a":33.952563,"close_a":34.319534,"volume_a":23261500,"open_b":90.078372,"high_b":90.906454,"low_b":86.917997,"close_b":86.955978,"volume_b":5347600,"open_bench":239.468296,"close_bench":231.131332},{"date":"2018-02-06","open_a":33.539698,"high_a":34.204837,"low_a":33.210953,"close_a":34.151318,"volume_a":25184300,"open_b":85.42138,"high_b":86.568532,"low_b":83.681652,"close_b":86.272247,"volume_b":6642600,"open_bench":227.637162,"close_bench":235.68512},{"date":"2018-02-07","open_a":34.082533,"high_a":34.785894,"low_a":33.853173,"close_a":34.067242,"volume_a":18041600,"open_b":86.583709,"high_b":87.951177,"low_b":85.656867,"close_b":86.226646,"volume_b":4931400,"open_bench":235.13343,"close_bench":234.406586},{"date":"2018-02-08","open_a":34.113113,"high_a":34.113113,"low_a":32.9281,"close_a":32.951035,"volume_a":23295300,"open_b":86.378594,"high_b":86.393792,"low_b":83.636054,"close_b":83.681641,"volume_b":4907900,"open_bench":234.704257,"close_bench":225.614182},{"date":"2018-02-09","open_a":33.149817,"high_a":33.226268,"low_a":32.255321,"close_a":32.973976,"volume_a":22853100,"open_b":84.183048,"high_b":85.079502,"low_b":82.716815,"close_b":84.464142,"volume_b":5301700,"open_bench":228.390274,"close_bench":229.003296},{"date":"2018-02-12","open_a":33.302706,"high_a":33.814937,"low_a":33.157445,"close_a":33.616161,"volume_a":14689000,"open_b":85.01113,"high_b":85.656879,"low_b":84.372974,"close_b":85.03392,"volume_b":4759200,"open_bench":231.043698,"close_bench":232.366058},{"date":"2018-02-13","open_a":33.577937,"high_a":33.952555,"low_a":33.402096,"close_a":33.784359,"volume_a":13163200,"open_b":85.087085,"high_b":85.596087,"low_b":84.084273,"close_b":85.193443,"volume_b":5274700,"open_bench":231.166301,"close_bench":232.944031},{"date":"2018-02-14","open_a":33.654393,"high_a":33.784361,"low_a":33.356228,"close_a":33.715553,"volume_a":13457900,"open_b":84.836394,"high_b":85.011129,"low_b":82.625652,"close_b":82.891548,"volume_b":7702700,"open_bench":231.46413,"close_bench":236.087982},{"date":"2018-02-15","open_a":33.830246,"high_a":34.273672,"low_a":33.730857,"close_a":34.235443,"volume_a":13467300,"open_b":83.210617,"high_b":84.357775,"low_b":82.595257,"close_b":84.304596,"volume_b":5125200,"open_bench":237.821845,"close_bench":239.100403},{"date":"2018-02-16","open_a":34.839406,"high_a":34.892923,"low_a":34.189561,"close_a":34.388336,"volume_a":17202300,"open_b":84.372963,"high_b":84.859174,"low_b":83.385348,"close_b":84.372963,"volume_b":5578500,"open_bench":238.478664,"close_bench":239.170471},{"date":"2018-02-20","open_a":34.212496,"high_a":34.273659,"low_a":33.59323,"close_a":33.631458,"volume_a":14137300,"open_b":84.297,"high_b":84.312197,"low_b":83.066274,"close_b":83.172638,"volume_b":3835500,"open_bench":238.224687,"close_bench":237.672974},{"date":"2018-02-21","open_a":33.723198,"high_a":33.837875,"low_a":33.12687,"close_a":33.134514,"volume_a":12792900,"open_b":83.073897,"high_b":83.446151,"low_b":82.002714,"close_b":82.048294,"volume_b":4501900,"open_bench":238.11095,"close_bench":236.490845},{"date":"2018-02-22","open_a":33.203322,"high_a":33.501486,"low_a":33.042772,"close_a":33.272129,"volume_a":12377800,"open_b":82.298989,"high_b":83.544907,"low_b":82.063476,"close_b":83.005516,"volume_b":5394700,"open_bench":237.410308,"close_bench":236.797287},{"date":"2018-02-23","open_a":33.318023,"high_a":33.684995,"low_a":33.195697,"close_a":33.669704,"volume_a":8277200,"open_b":83.081502,"high_b":83.605701,"low_b":82.481333,"close_b":83.324608,"volume_b":4760400,"open_bench":238.01454,"close_bench":240.571655},{"date":"2018-02-26","open_a":33.814946,"high_a":33.93727,"low_a":33.493847,"close_a":33.662041,"volume_a":12713300,"open_b":83.658848,"high_b":84.456536,"low_b":83.590472,"close_b":84.281807,"volume_b":5355400,"open_bench":241.640061,"close_bench":243.36525},{"date":"2018-02-27","open_a":33.75378,"high_a":33.876104,"low_a":33.348579,"close_a":33.348579,"volume_a":12774400,"open_b":84.471719,"high_b":85.147857,"low_b":84.107058,"close_b":84.243805,"volume_b":6385800,"open_bench":243.549187,"close_bench":240.326508},{"date":"2018-02-28","open_a":33.509139,"high_a":33.570299,"low_a":33.042778,"close_a":33.042778,"volume_a":11929400,"open_b":84.327388,"high_b":84.821199,"low_b":83.324576,"close_b":83.362564,"volume_b":4638900,"open_bench":241.421126,"close_bench":237.891937},{"date":"2018-03-01","open_a":32.951021,"high_a":33.363865,"low_a":32.736955,"close_a":33.203316,"volume_a":17746800,"open_b":83.580672,"high_b":84.093435,"low_b":82.072989,"close_b":82.746475,"volume_b":6896300,"open_bench":237.681749,"close_bench":234.4328},{"date":"2018-03-02","open_a":33.06571,"high_a":33.486198,"low_a":33.058066,"close_a":33.425037,"volume_a":14050100,"open_b":82.585751,"high_b":83.66485,"low_b":82.578096,"close_b":83.450562,"volume_b":4903600,"open_bench":232.76892,"close_bench":235.641312},{"date":"2018-03-05","open_a":33.310364,"high_a":33.593237,"low_a":33.256847,"close_a":33.555012,"volume_a":11992700,"open_b":83.006699,"high_b":83.879165,"low_b":82.669956,"close_b":83.749062,"volume_b":3781700,"open_bench":234.459061,"close_bench":238.364807},{"date":"2018-03-06","open_a":33.600874,"high_a":33.677325,"low_a":33.325644,"close_a":33.585583,"volume_a":10010500,"open_b":83.695474,"high_b":83.894459,"low_b":82.991381,"close_b":83.863846,"volume_b":3380800,"open_bench":239.336847,"close_bench":238.969055},{"date":"2018-03-07","open_a":33.325647,"high_a":33.577939,"low_a":33.233905,"close_a":33.501488,"volume_a":10881700,"open_b":83.335768,"high_b":84.231192,"low_b":83.121479,"close_b":84.06282,"volume_b":4612600,"open_bench":236.814776,"close_bench":238.881485},{"date":"2018-03-08","open_a":33.562658,"high_a":33.990791,"low_a":33.486204,"close_a":33.983147,"volume_a":12361100,"open_b":84.208266,"high_b":85.815438,"low_b":83.863873,"close_b":85.777176,"volume_b":5437700,"open_bench":239.555778,"close_bench":240.037445},{"date":"2018-03-09","open_a":34.074883,"high_a":34.266014,"low_a":33.937268,"close_a":34.266014,"volume_a":9778600,"open_b":85.922579,"high_b":86.152178,"low_b":85.440429,"close_b":86.129219,"volume_b":3680600,"open_bench":241.438613,"close_bench":244.214661},{"date":"2018-03-12","open_a":34.365418,"high_a":34.472451,"low_a":33.891413,"close_a":34.044315,"volume_a":11857700,"open_b":86.098572,"high_b":86.550108,"low_b":85.777138,"close_b":86.289902,"volume_b":4509600,"open_bench":244.503643,"close_bench":243.908127},{"date":"2018-03-13","open_a":34.151326,"high_a":34.273653,"low_a":33.944907,"close_a":34.074875,"volume_a":11342100,"open_b":86.68788,"high_b":87.170036,"low_b":86.266957,"close_b":86.940437,"volume_b":4590000,"open_bench":245.064166,"close_bench":242.331894},{"date":"2018-03-14","open_a":34.229127,"high_a":34.26769,"low_a":33.727798,"close_a":33.766361,"volume_a":15321200,"open_b":86.917477,"high_b":87.070539,"low_b":85.89195,"close_b":86.18277,"volume_b":4055700,"open_bench":243.286407,"close_bench":241.088318},{"date":"2018-03-15","open_a":33.8435,"high_a":33.936052,"low_a":33.588978,"close_a":33.68153,"volume_a":12456700,"open_b":86.022067,"high_b":86.672588,"low_b":85.455731,"close_b":85.708282,"volume_b":3906800,"open_bench":241.596299,"close_bench":240.825653},{"date":"2018-03-16","open_a":33.704673,"high_a":33.95148,"low_a":33.511854,"close_a":33.519566,"volume_a":26603300,"open_b":85.968472,"high_b":86.634298,"low_b":85.394482,"close_b":85.440399,"volume_b":7730800,"open_bench":241.350537,"close_bench":241.086777},{"date":"2018-03-19","open_a":33.519566,"high_a":33.712385,"low_a":33.30361,"close_a":33.365311,"volume_a":12380300,"open_b":85.448057,"high_b":85.708263,"low_b":84.223545,"close_b":84.460793,"volume_b":4216700,"open_bench":240.33943,"close_bench":237.824799},{"date":"2018-03-20","open_a":33.411585,"high_a":33.534989,"low_a":33.172489,"close_a":33.288181,"volume_a":13559800,"open_b":84.759286,"high_b":85.180215,"low_b":84.055192,"close_b":84.108765,"volume_b":3319000,"open_bench":238.220493,"close_bench":238.229294},{"date":"2018-03-21","open_a":33.265046,"high_a":33.511854,"low_a":33.002813,"close_a":33.16478,"volume_a":13054000,"open_b":84.101106,"high_b":84.246518,"low_b":83.060269,"close_b":83.220985,"volume_b":3867600,"open_bench":238.185383,"close_bench":237.772141},{"date":"2018-03-22","open_a":33.072234,"high_a":33.450157,"low_a":32.933405,"close_a":32.979679,"volume_a":12274100,"open_b":82.999051,"high_b":83.932744,"low_b":82.501597,"close_b":82.608742,"volume_b":4325500,"open_bench":235.556371,"close_bench":231.828415},{"date":"2018-03-23","open_a":33.025944,"high_a":33.288177,"low_a":32.61717,"close_a":32.648022,"volume_a":13475200,"open_b":82.723495,"high_b":83.220955,"low_b":81.078061,"close_b":81.238777,"volume_b":4873300,"open_bench":232.267975,"close_bench":226.887024},{"date":"2018-03-26","open_a":32.863989,"high_a":33.018245,"low_a":32.58633,"close_a":32.92569,"volume_a":11276100,"open_b":81.644407,"high_b":82.249011,"low_b":81.192865,"close_b":81.743896,"volume_b":4119400,"open_bench":230.474328,"close_bench":233.094437},{"date":"2018-03-27","open_a":32.933388,"high_a":33.403866,"low_a":32.80227,"close_a":33.079929,"volume_a":11627000,"open_b":82.065364,"high_b":83.067933,"low_b":81.514332,"close_b":82.379143,"volume_b":5036300,"open_bench":234.026491,"close_bench":229.129135},{"date":"2018-03-28","open_a":33.164779,"high_a":33.704672,"low_a":33.164779,"close_a":33.411587,"volume_a":16277400,"open_b":82.761805,"high_b":84.269488,"low_b":82.65466,"close_b":83.557739,"volume_b":6413000,"open_bench":229.260997,"close_bench":228.452087},{"date":"2018-03-29","open_a":33.581259,"high_a":33.87434,"low_a":33.38844,"close_a":33.496418,"volume_a":10450500,"open_b":84.277129,"high_b":84.690403,"low_b":83.397009,"close_b":83.534767,"volume_b":3829500,"open_bench":229.58629,"close_bench":231.37114},{"date":"2018-04-02","open_a":33.504125,"high_a":33.550403,"low_a":32.755992,"close_a":32.910244,"volume_a":14207900,"open_b":83.129151,"high_b":83.588342,"low_b":81.460751,"close_b":81.927597,"volume_b":4325500,"open_bench":230.843617,"close_bench":226.377106},{"date":"2018-04-03","open_a":33.149352,"high_a":33.581267,"low_a":33.041374,"close_a":33.457863,"volume_a":15484600,"open_b":82.187775,"high_b":83.06024,"low_b":81.575516,"close_b":82.800034,"volume_b":4699000,"open_bench":227.60802,"close_bench":229.278564},{"date":"2018-04-04","open_a":33.272759,"high_a":34.244566,"low_a":33.249622,"close_a":34.121162,"volume_a":13563300,"open_b":82.486273,"high_b":84.743973,"low_b":82.072999,"close_b":84.323044,"volume_b":4694200,"open_bench":225.744103,"close_bench":231.731705},{"date":"2018-04-05","open_a":34.152009,"high_a":34.406531,"low_a":33.889776,"close_a":34.244564,"volume_a":11707800,"open_b":84.514377,"high_b":85.088367,"low_b":83.993958,"close_b":84.621521,"volume_b":3275700,"open_bench":233.481331,"close_bench":233.560486},{"date":"2018-04-06","open_a":34.152007,"high_a":34.337114,"low_a":33.712381,"close_a":33.874348,"volume_a":11495100,"open_b":84.231222,"high_b":84.935321,"low_b":83.243957,"close_b":83.649582,"volume_b":3579100,"open_bench":231.608595,"close_bench":228.355408},{"date":"2018-04-09","open_a":33.959196,"high_a":34.175154,"low_a":33.766377,"close_a":33.804943,"volume_a":11913400,"open_b":83.833254,"high_b":84.529693,"low_b":83.251609,"close_b":83.7108,"volume_b":2620000,"open_bench":229.806164,"close_bench":229.48085},{"date":"2018-04-10","open_a":33.990041,"high_a":34.074882,"low_a":33.720097,"close_a":33.928341,"volume_a":10379500,"open_b":83.986303,"high_b":84.33835,"low_b":83.358741,"close_b":83.726097,"volume_b":2817000,"open_bench":232.355936,"close_bench":233.129669},{"date":"2018-04-11","open_a":33.789511,"high_a":34.04403,"low_a":33.750945,"close_a":33.928341,"volume_a":8424300,"open_b":83.090901,"high_b":83.779691,"low_b":82.976102,"close_b":83.175087,"volume_b":4000200,"open_bench":231.65253,"close_bench":231.907516},{"date":"2018-04-12","open_a":33.99776,"high_a":34.252279,"low_a":33.897494,"close_a":33.959194,"volume_a":8476400,"open_b":83.389367,"high_b":83.810295,"low_b":82.861293,"close_b":82.891907,"volume_b":2691700,"open_bench":233.226447,"close_bench":233.815521},{"date":"2018-04-13","open_a":34.082588,"high_a":34.468225,"low_a":34.044025,"close_a":34.329395,"volume_a":10774200,"open_b":83.305173,"high_b":83.772019,"low_b":83.052616,"close_b":83.618958,"volume_b":2828300,"open_bench":235.116753,"close_bench":233.129669},{"date":"2018-04-16","open_a":34.437381,"high_a":34.622485,"low_a":34.244562,"close_a":34.460518,"volume_a":8962100,"open_b":83.993981,"high_b":84.675117,"low_b":83.817955,"close_b":84.124084,"volume_b":2769800,"open_bench":234.756235,"close_bench":235.046371},{"date":"2018-04-17","open_a":34.66875,"high_a":34.730451,"low_a":34.437369,"close_a":34.614761,"volume_a":8406400,"open_b":83.182684,"high_b":83.481159,"low_b":82.639307,"close_b":83.320442,"volume_b":5022000,"open_bench":236.804817,"close_bench":237.560974},{"date":"2018-04-18","open_a":34.630195,"high_a":34.684183,"low_a":34.190569,"close_a":34.329395,"volume_a":7429000,"open_b":83.396995,"high_b":83.695469,"low_b":81.476038,"close_b":81.874008,"volume_b":6525500,"open_bench":238.000668,"close_bench":237.736908},{"date":"2018-04-19","open_a":34.198285,"high_a":34.298552,"low_a":33.951478,"close_a":34.175148,"volume_a":10370400,"open_b":81.667372,"high_b":81.843398,"low_b":80.419901,"close_b":80.794907,"volume_b":6320200,"open_bench":237.086293,"close_bench":236.418091},{"date":"2018-04-20","open_a":34.175145,"high_a":34.252271,"low_a":33.612116,"close_a":33.735519,"volume_a":14690700,"open_b":80.703067,"high_b":80.817866,"low_b":78.269346,"close_b":78.430069,"volume_b":8650000,"open_bench":236.347616,"close_bench":234.413284},{"date":"2018-04-23","open_a":33.73552,"high_a":33.928338,"low_a":33.59669,"close_a":33.920624,"volume_a":11878500,"open_b":78.483633,"high_b":78.674963,"low_b":77.94025,"close_b":78.506592,"volume_b":6670600,"open_bench":234.98488,"close_bench":234.378204},{"date":"2018-04-24","open_a":33.920628,"high_a":34.01318,"low_a":32.956536,"close_a":33.218769,"volume_a":18523900,"open_b":78.537219,"high_b":78.713239,"low_b":77.274438,"close_b":77.63414,"volume_b":5695500,"open_bench":235.398116,"close_bench":231.221741},{"date":"2018-04-25","open_a":32.964243,"high_a":33.079932,"low_a":32.324085,"close_a":32.725147,"volume_a":20774700,"open_b":77.458095,"high_b":78.338215,"low_b":77.18258,"close_b":77.412178,"volume_b":4828300,"open_bench":231.160146,"close_bench":231.793198},{"date":"2018-04-26","open_a":32.702017,"high_a":33.241907,"low_a":32.594036,"close_a":32.971962,"volume_a":13561000,"open_b":78.254048,"high_b":79.654586,"low_b":77.412196,"close_b":79.027023,"volume_b":5978800,"open_bench":232.813106,"close_bench":234.149536},{"date":"2018-04-27","open_a":32.956544,"high_a":33.427022,"low_a":32.941119,"close_a":33.403885,"volume_a":9777900,"open_b":78.23109,"high_b":78.820384,"low_b":77.794857,"close_b":77.840775,"volume_b":4765900,"open_bench":234.756159,"close_bench":234.369293},{"date":"2018-04-30","open_a":33.550414,"high_a":33.696955,"low_a":33.14164,"close_a":33.326744,"volume_a":16037700,"open_b":77.947904,"high_b":78.361177,"low_b":76.975949,"close_b":77.251465,"volume_b":5479900,"open_bench":234.984919,"close_bench":232.567017},{"date":"2018-05-01","open_a":33.133927,"high_a":33.157067,"low_a":32.532334,"close_a":32.848557,"volume_a":11514200,"open_b":76.807589,"high_b":77.243821,"low_b":75.437664,"close_b":75.866241,"volume_b":4543900,"open_bench":232.004212,"close_bench":232.980179},{"date":"2018-05-02","open_a":32.702014,"high_a":32.848555,"low_a":32.347226,"close_a":32.439781,"volume_a":13736400,"open_b":75.889209,"high_b":76.042271,"low_b":74.259072,"close_b":74.41214,"volume_b":5927700,"open_bench":232.786799,"close_bench":231.415192},{"date":"2018-05-03","open_a":32.563192,"high_a":32.702021,"low_a":32.023299,"close_a":32.30867,"volume_a":18352200,"open_b":74.603478,"high_b":75.070318,"low_b":74.006523,"close_b":74.695312,"volume_b":6668200,"open_bench":230.588654,"close_bench":230.905167},{"date":"2018-05-04","open_a":32.231536,"high_a":32.77914,"low_a":32.21611,"close_a":32.671162,"volume_a":10946200,"open_b":74.657059,"high_b":76.356067,"low_b":74.618791,"close_b":75.759117,"volume_b":5937000,"open_bench":229.938026,"close_bench":233.894592},{"date":"2018-05-07","open_a":32.655728,"high_a":32.77142,"low_a":32.362643,"close_a":32.501472,"volume_a":9336000,"open_b":75.751481,"high_b":76.004033,"low_b":75.055036,"close_b":75.17749,"volume_b":3475400,"open_bench":234.659522,"close_bench":234.685898},{"date":"2018-05-08","open_a":32.370367,"high_a":32.462919,"low_a":32.154409,"close_a":32.246964,"volume_a":15062200,"open_b":74.93257,"high_b":74.993792,"low_b":73.799893,"close_b":74.075409,"volume_b":5333700,"open_bench":234.316607,"close_bench":234.685898},{"date":"2018-05-09","open_a":32.370367,"high_a":32.570897,"low_a":32.138983,"close_a":32.223824,"volume_a":9295200,"open_b":74.381526,"high_b":74.634077,"low_b":73.424875,"close_b":73.593246,"volume_b":6201500,"open_bench":235.354098,"close_bench":236.954315},{"date":"2018-05-10","open_a":32.385785,"high_a":32.671155,"low_a":32.347219,"close_a":32.532326,"volume_a":7924200,"open_b":73.799867,"high_b":74.802435,"low_b":73.776909,"close_b":74.442734,"volume_b":3100600,"open_bench":237.692841,"close_bench":239.169952},{"date":"2018-05-11","open_a":32.578602,"high_a":32.755994,"low_a":32.424346,"close_a":32.501472,"volume_a":6331900,"open_b":74.680008,"high_b":75.200426,"low_b":74.14428,"close_b":74.565208,"volume_b":3507300,"open_bench":239.293089,"close_bench":239.899765},{"date":"2018-05-14","open_a":32.586318,"high_a":32.640306,"low_a":32.316373,"close_a":32.424351,"volume_a":10715500,"open_b":74.764188,"high_b":74.794801,"low_b":74.029481,"close_b":74.350914,"volume_b":4307900,"open_bench":240.330566,"close_bench":240.014053},{"date":"2018-05-15","open_a":32.223822,"high_a":32.3318,"low_a":31.969303,"close_a":32.177547,"volume_a":16449700,"open_b":73.937641,"high_b":74.419791,"low_b":73.746311,"close_b":74.083054,"volume_b":6555700,"open_bench":238.791978,"close_bench":238.36116},{"date":"2018-05-16","open_a":32.192969,"high_a":32.347221,"low_a":32.038713,"close_a":32.046425,"volume_a":11590500,"open_b":73.960594,"high_b":74.657039,"low_b":73.891718,"close_b":74.381523,"volume_b":5544700,"open_bench":238.396371,"close_bench":239.36351},{"date":"2018-05-17","open_a":32.401223,"high_a":32.663456,"low_a":32.293245,"close_a":32.624893,"volume_a":14857400,"open_b":74.657028,"high_b":74.993765,"low_b":74.350899,"close_b":74.940193,"volume_b":6531100,"open_bench":239.099671,"close_bench":239.161224},{"date":"2018-05-18","open_a":32.624878,"high_a":32.686581,"low_a":32.4552,"close_a":32.532326,"volume_a":11950200,"open_b":75.238672,"high_b":75.345817,"low_b":74.481006,"close_b":74.626419,"volume_b":6894000,"open_bench":238.818371,"close_bench":238.563385},{"date":"2018-05-21","open_a":32.632602,"high_a":32.794569,"low_a":32.60175,"close_a":32.709728,"volume_a":9110200,"open_b":75.169825,"high_b":76.516791,"low_b":75.04737,"close_b":76.157089,"volume_b":6678100,"open_bench":240.040482,"close_bench":240.356995},{"date":"2018-05-22","open_a":32.779141,"high_a":32.856267,"low_a":32.555471,"close_a":32.586323,"volume_a":8222900,"open_b":76.363724,"high_b":76.884142,"low_b":76.095863,"close_b":76.478523,"volume_b":6807600,"open_bench":240.875834,"close_bench":239.688858},{"date":"2018-05-23","open_a":32.586326,"high_a":32.740582,"low_a":32.339519,"close_a":32.702019,"volume_a":8971400,"open_b":76.478501,"high_b":76.991264,"low_b":76.233598,"close_b":76.876465,"volume_b":6249900,"open_bench":238.422656,"close_bench":240.34816},{"date":"2018-05-24","open_a":32.702016,"high_a":32.756004,"low_a":32.540046,"close_a":32.640312,"volume_a":9269600,"open_b":76.769335,"high_b":76.975975,"low_b":76.348412,"close_b":76.669846,"volume_b":4696700,"open_bench":239.952565,"close_bench":239.855835},{"date":"2018-05-25","open_a":32.655741,"high_a":32.840848,"low_a":32.547763,"close_a":32.702019,"volume_a":8806500,"open_b":76.669828,"high_b":77.381577,"low_b":76.455539,"close_b":76.769318,"volume_b":5324300,"open_bench":239.284271,"close_bench":239.284271},{"date":"2018-05-29","open_a":32.486054,"high_a":33.049084,"low_a":32.470628,"close_a":32.917969,"volume_a":14826900,"open_b":76.53208,"high_b":77.519345,"low_b":76.53208,"close_b":77.373932,"volume_b":8302600,"open_bench":237.666557,"close_bench":236.532333},{"date":"2018-05-30","open_a":32.987394,"high_a":33.380742,"low_a":32.933405,"close_a":33.265053,"volume_a":10114200,"open_b":77.802529,"high_b":78.246411,"low_b":77.366291,"close_b":77.932632,"volume_b":6256600,"open_bench":237.83368,"close_bench":239.688858},{"date":"2018-05-31","open_a":33.319035,"high_a":33.326746,"low_a":32.917972,"close_a":33.16478,"volume_a":15571300,"open_b":78.124171,"high_b":78.240027,"low_b":76.973357,"close_b":77.429047,"volume_b":8270100,"open_bench":239.284282,"close_bench":238.220413},{"date":"2018-06-01","open_a":33.241906,"high_a":33.411587,"low_a":33.195631,"close_a":33.257332,"volume_a":10664700,"open_b":77.653032,"high_b":77.954251,"low_b":76.965632,"close_b":77.429047,"volume_b":4414200,"open_bench":239.512925,"close_bench":240.559219},{"date":"2018-06-04","open_a":33.442428,"high_a":33.61982,"low_a":33.388439,"close_a":33.542694,"volume_a":11727600,"open_b":77.830642,"high_b":78.077797,"low_b":77.259098,"close_b":77.328613,"volume_b":3932000,"open_bench":241.376896,"close_bench":241.702209},{"date":"2018-06-05","open_a":33.550411,"high_a":33.581263,"low_a":33.2419,"close_a":33.249615,"volume_a":9975900,"open_b":77.591237,"high_b":77.977417,"low_b":77.143268,"close_b":77.398148,"volume_b":4851900,"open_bench":241.83412,"close_bench":241.878098},{"date":"2018-06-06","open_a":33.211058,"high_a":33.519566,"low_a":33.195632,"close_a":33.519566,"volume_a":9202900,"open_b":77.544877,"high_b":77.900161,"low_b":77.096907,"close_b":77.876991,"volume_b":3946700,"open_bench":242.484741,"close_bench":243.900299},{"date":"2018-06-07","open_a":33.419301,"high_a":33.727809,"low_a":33.419301,"close_a":33.504139,"volume_a":11948100,"open_b":77.629854,"high_b":78.402214,"low_b":77.49083,"close_b":78.131889,"volume_b":5052900,"open_bench":244.383874,"close_bench":243.873901},{"date":"2018-06-08","open_a":33.604399,"high_a":33.936047,"low_a":33.596688,"close_a":33.897484,"volume_a":12078800,"open_b":78.147337,"high_b":79.251812,"low_b":78.147337,"close_b":79.159126,"volume_b":4795500,"open_bench":243.416735,"close_bench":244.59491},{"date":"2018-06-11","open_a":33.897486,"high_a":34.128868,"low_a":33.851209,"close_a":34.036316,"volume_a":7526100,"open_b":79.251818,"high_b":79.869708,"low_b":78.989213,"close_b":79.715233,"volume_b":3984500,"open_bench":244.814738,"close_bench":244.920242},{"date":"2018-06-12","open_a":34.036318,"high_a":34.259985,"low_a":33.997755,"close_a":34.252274,"volume_a":8243300,"open_b":79.730679,"high_b":80.750194,"low_b":79.414009,"close_b":80.696129,"volume_b":5075900,"open_bench":245.333413,"close_bench":245.23671},{"date":"2018-06-13","open_a":34.205994,"high_a":34.337112,"low_a":34.051741,"close_a":34.074879,"volume_a":16745000,"open_b":80.448946,"high_b":81.561139,"low_b":80.448946,"close_b":81.306259,"volume_b":7553500,"open_bench":245.474126,"close_bench":244.454208},{"date":"2018-06-14","open_a":34.043755,"high_a":34.121568,"low_a":33.864783,"close_a":33.911472,"volume_a":12696200,"open_b":81.298569,"high_b":81.375804,"low_b":80.657509,"close_b":81.198158,"volume_b":4812600,"open_bench":245.315796,"close_bench":245.069611},{"date":"2018-06-15","open_a":33.895911,"high_a":34.355015,"low_a":33.810314,"close_a":34.331669,"volume_a":21562500,"open_b":81.174993,"high_b":83.221747,"low_b":80.958733,"close_b":83.113617,"volume_b":10573000,"open_bench":244.288932,"close_bench":244.757019},{"date":"2018-06-18","open_a":34.1449,"high_a":34.207149,"low_a":33.771392,"close_a":33.888111,"volume_a":12161900,"open_b":82.650193,"high_b":82.804662,"low_b":80.966449,"close_b":81.391243,"volume_b":5964400,"open_bench":243.308602,"close_bench":244.253616},{"date":"2018-06-19","open_a":33.693598,"high_a":34.043763,"low_a":33.615785,"close_a":33.92704,"volume_a":12936400,"open_b":81.476208,"high_b":82.395318,"low_b":81.267674,"close_b":81.839218,"volume_b":5504400,"open_bench":241.992666,"close_bench":243.317444},{"date":"2018-06-20","open_a":33.9426,"high_a":34.067103,"low_a":33.639125,"close_a":33.662468,"volume_a":10893700,"open_b":81.661566,"high_b":81.955061,"low_b":81.375791,"close_b":81.723351,"volume_b":4601400,"open_bench":243.99755,"close_bench":243.732605},{"date":"2018-06-21","open_a":33.670252,"high_a":33.693595,"low_a":33.429026,"close_a":33.530186,"volume_a":10465200,"open_b":82.310333,"high_b":82.974562,"low_b":82.101798,"close_b":82.827812,"volume_b":6550700,"open_bench":243.723776,"close_bench":242.204697},{"date":"2018-06-22","open_a":33.576873,"high_a":33.779189,"low_a":33.522403,"close_a":33.654686,"volume_a":14924400,"open_b":82.827799,"high_b":83.932273,"low_b":82.750565,"close_b":83.700569,"volume_b":5709400,"open_bench":243.458782,"close_bench":242.64624},{"date":"2018-06-25","open_a":33.584651,"high_a":34.043755,"low_a":33.584651,"close_a":33.911472,"volume_a":12092400,"open_b":83.190824,"high_b":84.642858,"low_b":83.190824,"close_b":84.179443,"volume_b":5183000,"open_bench":241.498129,"close_bench":239.343155},{"date":"2018-06-26","open_a":33.895907,"high_a":34.113786,"low_a":33.678025,"close_a":33.716934,"volume_a":13974700,"open_b":84.225805,"high_b":84.318484,"low_b":83.005476,"close_b":83.453445,"volume_b":5034100,"open_bench":239.908382,"close_bench":239.873047},{"date":"2018-06-27","open_a":33.716944,"high_a":33.934824,"low_a":33.452376,"close_a":33.514626,"volume_a":15332600,"open_b":83.700585,"high_b":84.156274,"low_b":83.121316,"close_b":83.623344,"volume_b":6229200,"open_bench":240.455951,"close_bench":237.88588},{"date":"2018-06-28","open_a":33.646909,"high_a":33.989291,"low_a":33.592437,"close_a":33.825882,"volume_a":12668900,"open_b":83.739218,"high_b":84.418898,"low_b":83.646538,"close_b":83.940033,"volume_b":4297400,"open_bench":237.832916,"close_bench":239.246017},{"date":"2018-06-29","open_a":33.903694,"high_a":34.316109,"low_a":33.701378,"close_a":34.129356,"volume_a":11485600,"open_b":83.970884,"high_b":84.573323,"low_b":83.260315,"close_b":84.086739,"volume_b":4721400,"open_bench":240.332326,"close_bench":239.590454},{"date":"2018-07-02","open_a":34.067102,"high_a":34.362797,"low_a":33.911475,"close_a":34.043758,"volume_a":9232500,"open_b":83.95546,"high_b":84.959529,"low_b":83.65424,"close_b":84.596519,"volume_b":4711900,"open_bench":238.027168,"close_bench":240.102631},{"date":"2018-07-03","open_a":34.28498,"high_a":34.495079,"low_a":34.230511,"close_a":34.230511,"volume_a":8214300,"open_b":83.862787,"high_b":84.627421,"low_b":83.461157,"close_b":84.032707,"volume_b":3769100,"open_bench":240.994682,"close_bench":239.254807},{"date":"2018-07-05","open_a":34.362783,"high_a":34.759634,"low_a":34.339439,"close_a":34.751854,"volume_a":11653000,"open_b":84.11769,"high_b":84.666064,"low_b":83.41484,"close_b":84.612,"volume_b":4870400,"open_bench":240.376496,"close_bench":241.206665},{"date":"2018-07-06","open_a":34.759649,"high_a":34.969748,"low_a":34.689619,"close_a":34.736305,"volume_a":8733900,"open_b":84.565648,"high_b":84.974998,"low_b":84.164024,"close_b":84.619713,"volume_b":4387300,"open_bench":241.233165,"close_bench":243.246826},{"date":"2018-07-09","open_a":34.658489,"high_a":34.806335,"low_a":34.502859,"close_a":34.549549,"volume_a":7373100,"open_b":84.287598,"high_b":84.418904,"low_b":83.044099,"close_b":83.229469,"volume_b":7866400,"open_bench":244.244874,"close_bench":245.43718},{"date":"2018-07-10","open_a":34.635144,"high_a":35.016435,"low_a":34.627363,"close_a":34.993092,"volume_a":10606100,"open_b":85.438397,"high_b":87.27661,"low_b":84.712377,"close_b":87.19165,"volume_b":16107200,"open_bench":245.88756,"close_bench":246.320312},{"date":"2018-07-11","open_a":35.008657,"high_a":35.016437,"low_a":34.736305,"close_a":34.853027,"volume_a":7715600,"open_b":86.998576,"high_b":87.230286,"low_b":86.581501,"close_b":86.921341,"volume_b":5521400,"open_bench":244.774655,"close_bench":244.518524},{"date":"2018-07-12","open_a":34.884157,"high_a":34.985313,"low_a":34.619588,"close_a":34.759651,"volume_a":6331600,"open_b":86.921328,"high_b":87.052627,"low_b":85.708724,"close_b":86.141243,"volume_b":5500100,"open_bench":245.772683,"close_bench":246.735352},{"date":"2018-07-13","open_a":34.782983,"high_a":34.845233,"low_a":34.604011,"close_a":34.81411,"volume_a":5585900,"open_b":86.257101,"high_b":87.11442,"low_b":86.210761,"close_b":87.037186,"volume_b":4417100,"open_bench":246.55875,"close_bench":246.929672},{"date":"2018-07-16","open_a":34.806339,"high_a":34.860809,"low_a":34.65071,"close_a":34.806339,"volume_a":5831400,"open_b":86.936775,"high_b":87.3693,"low_b":86.311166,"close_b":87.24572,"volume_b":5413600,"open_bench":246.973864,"close_bench":246.708893},{"date":"2018-07-17","open_a":34.821897,"high_a":35.319908,"low_a":34.767427,"close_a":35.210968,"volume_a":11592200,"open_b":87.639629,"high_b":88.844508,"low_b":87.639629,"close_b":88.728653,"volume_b":6783800,"open_bench":245.940494,"close_bench":247.706863},{"date":"2018-07-18","open_a":35.164272,"high_a":35.242088,"low_a":34.891923,"close_a":35.109802,"volume_a":7351600,"open_b":88.759535,"high_b":88.821325,"low_b":87.964006,"close_b":88.690025,"volume_b":5379000,"open_bench":247.786359,"close_bench":248.227951},{"date":"2018-07-19","open_a":35.008648,"high_a":35.35103,"low_a":34.954175,"close_a":35.102024,"volume_a":10097500,"open_b":88.643694,"high_b":90.049388,"low_b":88.62825,"close_b":89.416054,"volume_b":9241300,"open_bench":247.565594,"close_bench":247.291809},{"date":"2018-07-20","open_a":35.031991,"high_a":35.234306,"low_a":34.938614,"close_a":35.234306,"volume_a":8588400,"open_b":89.400602,"high_b":90.203857,"low_b":88.991253,"close_b":89.601418,"volume_b":27559700,"open_bench":247.088623,"close_bench":247.00914},{"date":"2018-07-23","open_a":35.234313,"high_a":35.296566,"low_a":34.837462,"close_a":34.977528,"volume_a":9055700,"open_b":89.222953,"high_b":89.578244,"low_b":87.72458,"close_b":88.350189,"volume_b":7769300,"open_bench":246.806071,"close_bench":247.46846},{"date":"2018-07-24","open_a":34.845238,"high_a":35.226529,"low_a":34.712955,"close_a":35.218746,"volume_a":13786100,"open_b":88.157099,"high_b":88.759539,"low_b":87.933115,"close_b":88.620514,"volume_b":5095100,"open_bench":248.872678,"close_bench":248.713684},{"date":"2018-07-25","open_a":35.79458,"high_a":36.245904,"low_a":35.600044,"close_a":35.864613,"volume_a":21330600,"open_b":88.774994,"high_b":89.338813,"low_b":88.435154,"close_b":88.983528,"volume_b":4762300,"open_bench":248.466403,"close_bench":250.833359},{"date":"2018-07-26","open_a":36.074715,"high_a":36.253687,"low_a":35.950212,"close_a":35.981339,"volume_a":12673100,"open_b":89.701824,"high_b":90.288819,"low_b":88.72865,"close_b":88.821335,"volume_b":5096300,"open_bench":250.117991,"close_bench":250.241623},{"date":"2018-07-27","open_a":36.043589,"high_a":36.183655,"low_a":35.841273,"close_a":35.957993,"volume_a":10516900,"open_b":88.643675,"high_b":89.091645,"low_b":88.072131,"close_b":88.265221,"volume_b":4563700,"open_bench":250.568372,"close_bench":248.545898},{"date":"2018-07-30","open_a":35.872389,"high_a":36.136958,"low_a":35.833483,"close_a":35.973549,"volume_a":8967900,"open_b":87.879043,"high_b":88.504653,"low_b":87.794083,"close_b":88.187988,"volume_b":3621800,"open_bench":248.625478,"close_bench":247.247711},{"date":"2018-07-31","open_a":36.105821,"high_a":36.315917,"low_a":36.035788,"close_a":36.284794,"volume_a":12438800,"open_b":88.558735,"high_b":89.199789,"low_b":88.133935,"close_b":88.821335,"volume_b":4892100,"open_bench":248.007215,"close_bench":248.466461},{"date":"2018-08-01","open_a":36.144739,"high_a":36.175865,"low_a":35.926857,"close_a":36.098049,"volume_a":9904100,"open_b":88.373359,"high_b":88.458319,"low_b":87.454255,"close_b":87.747749,"volume_b":4169000,"open_bench":248.669556,"close_bench":248.051315},{"date":"2018-08-02","open_a":36.168077,"high_a":36.2848,"low_a":35.934635,"close_a":35.934635,"volume_a":12009700,"open_b":87.740019,"high_b":88.767259,"low_b":87.346114,"close_b":88.581894,"volume_b":4167200,"open_bench":246.753079,"close_bench":249.402634},{"date":"2018-08-03","open_a":35.973544,"high_a":36.385959,"low_a":35.903511,"close_a":36.27702,"volume_a":11462800,"open_b":88.643686,"high_b":90.13434,"low_b":88.334747,"close_b":89.825401,"volume_b":3934500,"open_bench":249.526238,"close_bench":250.471252},{"date":"2018-08-06","open_a":36.261456,"high_a":36.471555,"low_a":36.222549,"close_a":36.300365,"volume_a":10056100,"open_b":90.559134,"high_b":91.478243,"low_b":90.288809,"close_b":90.659538,"volume_b":6704800,"open_bench":250.506586,"close_bench":251.389771},{"date":"2018-08-07","open_a":36.199207,"high_a":36.26924,"low_a":35.895731,"close_a":36.183643,"volume_a":8632900,"open_b":89.941253,"high_b":89.941253,"low_b":87.685965,"close_b":88.218895,"volume_b":6919600,"open_bench":252.05219,"close_bench":252.219971},{"date":"2018-08-08","open_a":36.175873,"high_a":36.230343,"low_a":35.887958,"close_a":35.911304,"volume_a":6810000,"open_b":88.45062,"high_b":88.7055,"low_b":87.639646,"close_b":87.65509,"volume_b":5172900,"open_bench":252.052165,"close_bench":252.113968},{"date":"2018-08-09","open_a":35.989116,"high_a":36.121399,"low_a":35.856833,"close_a":36.066929,"volume_a":8924100,"open_b":88.141656,"high_b":88.141656,"low_b":87.570106,"close_b":87.817261,"volume_b":4115400,"open_bench":252.175819,"close_bench":251.769562},{"date":"2018-08-10","open_a":35.957987,"high_a":36.121396,"low_a":35.771234,"close_a":35.856831,"volume_a":7535400,"open_b":87.732325,"high_b":88.025826,"low_b":86.944521,"close_b":87.176231,"volume_b":3711900,"open_bench":250.33888,"close_bench":250.082748},{"date":"2018-08-13","open_a":35.833486,"high_a":35.895739,"low_a":35.561137,"close_a":35.662296,"volume_a":8143400,"open_b":87.129858,"high_b":87.384738,"low_b":86.511975,"close_b":87.292053,"volume_b":3170100,"open_bench":250.35652,"close_bench":249.146561},{"date":"2018-08-14","open_a":35.72454,"high_a":35.872386,"low_a":35.693414,"close_a":35.708977,"volume_a":6846100,"open_b":87.044914,"high_b":87.655079,"low_b":86.589225,"close_b":87.353859,"volume_b":3836700,"open_bench":249.870748,"close_bench":250.736252},{"date":"2018-08-15","open_a":35.615605,"high_a":35.9113,"low_a":35.522229,"close_a":35.856831,"volume_a":9686400,"open_b":87.091237,"high_b":87.848146,"low_b":86.697332,"close_b":87.624161,"volume_b":6513300,"open_bench":249.393777,"close_bench":248.863861},{"date":"2018-08-16","open_a":35.880163,"high_a":36.090262,"low_a":35.771224,"close_a":35.965759,"volume_a":10600700,"open_b":87.848176,"high_b":88.612816,"low_b":87.562406,"close_b":88.242081,"volume_b":4504600,"open_bench":250.294612,"close_bench":250.877518},{"date":"2018-08-17","open_a":36.020237,"high_a":36.362619,"low_a":35.98133,"close_a":36.261459,"volume_a":8885700,"open_b":88.048987,"high_b":88.890856,"low_b":88.048987,"close_b":88.790451,"volume_b":3391600,"open_bench":250.674445,"close_bench":251.760773},{"date":"2018-08-20","open_a":36.323711,"high_a":36.471557,"low_a":36.253678,"close_a":36.362617,"volume_a":6392000,"open_b":88.813601,"high_b":89.477831,"low_b":88.512382,"close_b":88.697746,"volume_b":3676000,"open_bench":252.211145,"close_bench":252.299469},{"date":"2018-08-21","open_a":36.362611,"high_a":36.378174,"low_a":35.957976,"close_a":35.965759,"volume_a":13986300,"open_b":89.014435,"high_b":89.014435,"low_b":87.778661,"close_b":87.832726,"volume_b":4667700,"open_bench":252.811706,"close_bench":252.89119},{"date":"2018-08-22","open_a":36.059147,"high_a":36.090273,"low_a":35.732325,"close_a":35.740108,"volume_a":7159100,"open_b":87.817288,"high_b":87.909973,"low_b":86.473384,"close_b":86.689644,"volume_b":3860900,"open_bench":252.484897,"close_bench":252.741028},{"date":"2018-08-23","open_a":35.708992,"high_a":35.856841,"low_a":35.553363,"close_a":35.561146,"volume_a":13144900,"open_b":86.63556,"high_b":86.681906,"low_b":86.125806,"close_b":86.450195,"volume_b":4508800,"open_bench":252.564454,"close_bench":252.405487},{"date":"2018-08-24","open_a":35.522231,"high_a":35.638951,"low_a":35.467762,"close_a":35.506668,"volume_a":10952900,"open_b":86.403857,"high_b":86.913612,"low_b":86.349793,"close_b":86.620117,"volume_b":3677700,"open_bench":252.9795,"close_bench":253.924515},{"date":"2018-08-27","open_a":35.83348,"high_a":35.83348,"low_a":35.452192,"close_a":35.553349,"volume_a":9581200,"open_b":86.95223,"high_b":87.098974,"low_b":86.372961,"close_b":86.82093,"volume_b":3155700,"open_bench":255.11684,"close_bench":255.929382},{"date":"2018-08-28","open_a":35.522219,"high_a":35.584468,"low_a":35.234304,"close_a":35.304337,"volume_a":8476800,"open_b":86.674177,"high_b":86.851816,"low_b":85.646938,"close_b":85.863197,"volume_b":5229400,"open_bench":256.388597,"close_bench":256.053009},{"date":"2018-08-29","open_a":35.21096,"high_a":35.343242,"low_a":35.187616,"close_a":35.273212,"volume_a":12410900,"open_b":85.963608,"high_b":86.596942,"low_b":85.901817,"close_b":86.542877,"volume_b":2403200,"open_bench":256.264945,"close_bench":257.430756},{"date":"2018-08-30","open_a":35.234313,"high_a":35.26544,"low_a":34.930838,"close_a":34.977528,"volume_a":9716500,"open_b":86.403864,"high_b":86.681914,"low_b":86.156709,"close_b":86.473373,"volume_b":2119100,"open_bench":256.953893,"close_bench":256.388641},{"date":"2018-08-31","open_a":35.016437,"high_a":35.070907,"low_a":34.541769,"close_a":34.681835,"volume_a":19834100,"open_b":86.326615,"high_b":86.95223,"low_b":85.932716,"close_b":86.511986,"volume_b":4248800,"open_bench":255.982363,"close_bench":256.397461},{"date":"2018-09-04","open_a":34.705171,"high_a":34.938616,"low_a":34.635138,"close_a":34.829674,"volume_a":11138500,"open_b":86.365245,"high_b":86.57378,"low_b":85.307111,"close_b":85.731911,"volume_b":4576300,"open_bench":255.982366,"close_bench":255.955872},{"date":"2018-09-05","open_a":34.751866,"high_a":35.343253,"low_a":34.751866,"close_a":35.31213,"volume_a":10554900,"open_b":85.546547,"high_b":87.407931,"low_b":85.500201,"close_b":87.369316,"volume_b":4392100,"open_bench":255.602582,"close_bench":255.266968},{"date":"2018-09-06","open_a":35.327696,"high_a":35.794581,"low_a":35.24988,"close_a":35.615608,"volume_a":11439400,"open_b":87.219779,"high_b":88.411263,"low_b":86.783682,"close_b":88.387901,"volume_b":4808300,"open_bench":255.37291,"close_bench":254.498566},{"date":"2018-09-07","open_a":35.483323,"high_a":35.654515,"low_a":35.234317,"close_a":35.576702,"volume_a":10208100,"open_b":87.881721,"high_b":88.387909,"low_b":86.838202,"close_b":87.796059,"volume_b":3833800,"open_bench":253.456504,"close_bench":254.004074},{"date":"2018-09-10","open_a":35.72454,"high_a":36.035799,"low_a":35.638944,"close_a":35.841263,"volume_a":8568900,"open_b":87.936256,"high_b":88.769517,"low_b":87.764931,"close_b":88.442444,"volume_b":3289700,"open_bench":255.010903,"close_bench":254.445679},{"date":"2018-09-11","open_a":35.872389,"high_a":35.911298,"low_a":35.701199,"close_a":35.810139,"volume_a":9278300,"open_b":87.102966,"high_b":88.582589,"low_b":87.102966,"close_b":88.255516,"volume_b":2807300,"open_bench":253.800847,"close_bench":255.284592},{"date":"2018-09-12","open_a":35.919086,"high_a":36.028025,"low_a":35.810146,"close_a":35.981339,"volume_a":13216400,"open_b":88.239929,"high_b":88.956382,"low_b":88.02967,"close_b":88.660454,"volume_b":3651400,"open_bench":255.293401,"close_bench":255.34639},{"date":"2018-09-13","open_a":36.106895,"high_a":36.19322,"low_a":35.871467,"close_a":35.965641,"volume_a":8294800,"open_b":89.158882,"high_b":89.563834,"low_b":88.535887,"close_b":89.548256,"volume_b":3808100,"open_bench":256.406222,"close_bench":256.856628},{"date":"2018-09-14","open_a":36.036262,"high_a":36.122586,"low_a":35.792988,"close_a":36.091198,"volume_a":7612900,"open_b":89.470352,"high_b":89.470352,"low_b":88.559219,"close_b":89.221153,"volume_b":2698300,"open_bench":257.059815,"close_bench":256.900848},{"date":"2018-09-17","open_a":36.099047,"high_a":36.358019,"low_a":36.028418,"close_a":36.35017,"volume_a":8527300,"open_b":89.244538,"high_b":89.906472,"low_b":88.948615,"close_b":89.867538,"volume_b":3092600,"open_bench":256.84788,"close_bench":255.540756},{"date":"2018-09-18","open_a":36.405111,"high_a":36.514977,"low_a":36.044121,"close_a":36.044121,"volume_a":9847400,"open_b":89.929833,"high_b":89.929833,"low_b":88.535873,"close_b":88.761711,"volume_b":4081200,"open_bench":255.752717,"close_bench":256.927368},{"date":"2018-09-19","open_a":36.099041,"high_a":36.185365,"low_a":35.997021,"close_a":36.06765,"volume_a":7119000,"open_b":88.738358,"high_b":88.964195,"low_b":88.356773,"close_b":88.598183,"volume_b":2844900,"open_bench":256.980376,"close_bench":257.201172},{"date":"2018-09-20","open_a":36.271693,"high_a":36.632683,"low_a":36.21676,"close_a":36.601292,"volume_a":9736700,"open_b":88.956397,"high_b":89.953194,"low_b":88.862945,"close_b":89.727356,"volume_b":3375900,"open_bench":258.455292,"close_bench":259.285461},{"date":"2018-09-21","open_a":36.601294,"high_a":36.640532,"low_a":36.397258,"close_a":36.554211,"volume_a":19240300,"open_b":89.758507,"high_b":89.960978,"low_b":89.431429,"close_b":89.485947,"volume_b":6989500,"open_bench":260.02443,"close_bench":259.048523},{"date":"2018-09-24","open_a":36.483582,"high_a":36.51497,"low_a":35.973485,"close_a":36.15398,"volume_a":10778900,"open_b":88.855149,"high_b":89.337975,"low_b":87.375526,"close_b":87.430038,"volume_b":3387400,"open_bench":258.471802,"close_bench":258.187897},{"date":"2018-09-25","open_a":36.14613,"high_a":36.2403,"low_a":35.792986,"close_a":35.847919,"volume_a":7919100,"open_b":87.718194,"high_b":87.803857,"low_b":87.032898,"close_b":87.196434,"volume_b":3538700,"open_bench":258.640549,"close_bench":257.948547},{"date":"2018-09-26","open_a":35.918555,"high_a":36.153983,"low_a":35.761602,"close_a":35.981335,"volume_a":10162600,"open_b":87.430055,"high_b":87.889519,"low_b":86.682457,"close_b":86.830421,"volume_b":4053500,"open_bench":258.090433,"close_bench":257.176636},{"date":"2018-09-27","open_a":35.887167,"high_a":36.24031,"low_a":35.847929,"close_a":36.106899,"volume_a":10826900,"open_b":86.77591,"high_b":87.531297,"low_b":86.370964,"close_b":86.479988,"volume_b":3841500,"open_bench":257.646853,"close_bench":257.895264},{"date":"2018-09-28","open_a":36.099046,"high_a":36.24815,"low_a":36.075504,"close_a":36.24815,"volume_a":9071900,"open_b":86.861565,"high_b":87.258728,"low_b":86.370956,"close_b":87.064041,"volume_b":5945300,"open_bench":257.274191,"close_bench":257.921844},{"date":"2018-10-01","open_a":36.232447,"high_a":36.373704,"low_a":36.138276,"close_a":36.279533,"volume_a":9364700,"open_b":87.336571,"high_b":87.593559,"low_b":86.184021,"close_b":86.215172,"volume_b":5064700,"open_bench":259.154917,"close_bench":258.81781},{"date":"2018-10-02","open_a":36.318777,"high_a":36.538512,"low_a":36.146131,"close_a":36.499271,"volume_a":7462700,"open_b":86.012704,"high_b":86.526681,"low_b":83.871148,"close_b":84.665474,"volume_b":6702200,"open_bench":258.667023,"close_bench":258.667023},{"date":"2018-10-03","open_a":36.460026,"high_a":36.530655,"low_a":35.942084,"close_a":35.997017,"volume_a":10542300,"open_b":85.047074,"high_b":85.436448,"low_b":83.458427,"close_b":83.730988,"volume_b":6107500,"open_bench":259.713877,"close_bench":258.80896},{"date":"2018-10-04","open_a":35.832231,"high_a":36.020575,"low_a":35.651736,"close_a":35.981335,"volume_a":9582300,"open_b":83.209221,"high_b":83.637535,"low_b":82.39154,"close_b":83.022324,"volume_b":4930300,"open_bench":258.329945,"close_bench":256.786255},{"date":"2018-10-05","open_a":36.051951,"high_a":36.2246,"low_a":35.855761,"close_a":36.004868,"volume_a":8734100,"open_b":82.741989,"high_b":83.333839,"low_b":82.531729,"close_b":82.928886,"volume_b":3438100,"open_bench":257.008057,"close_bench":255.34903},{"date":"2018-10-08","open_a":36.091203,"high_a":36.554213,"low_a":36.075507,"close_a":36.475735,"volume_a":9808100,"open_b":82.827651,"high_b":84.369569,"low_b":82.788712,"close_b":84.252762,"volume_b":3994100,"open_bench":254.665881,"close_bench":255.34903},{"date":"2018-10-09","open_a":36.397248,"high_a":36.530656,"low_a":36.271686,"close_a":36.405094,"volume_a":10824500,"open_b":84.634339,"high_b":84.836815,"low_b":83.730989,"close_b":84.198242,"volume_b":4105600,"open_bench":254.967541,"close_bench":254.976395},{"date":"2018-10-10","open_a":36.334471,"high_a":36.648377,"low_a":35.84007,"close_a":35.847919,"volume_a":13917000,"open_b":83.7855,"high_b":85.156098,"low_b":83.481788,"close_b":83.590813,"volume_b":5796200,"open_bench":254.470732,"close_bench":246.903061},{"date":"2018-10-11","open_a":35.942089,"high_a":36.0755,"low_a":34.921899,"close_a":35.031765,"volume_a":17677900,"open_b":84.268335,"high_b":84.4786,"low_b":81.69068,"close_b":81.815277,"volume_b":5755100,"open_bench":245.82066,"close_bench":241.464615},{"date":"2018-10-12","open_a":34.906207,"high_a":35.220113,"low_a":34.725712,"close_a":35.06316,"volume_a":12420100,"open_b":82.002181,"high_b":82.360405,"low_b":81.402542,"close_b":81.986603,"volume_b":4543000,"open_bench":245.545692,"close_bench":244.818222},{"date":"2018-10-15","open_a":35.031774,"high_a":35.526175,"low_a":34.992536,"close_a":35.24366,"volume_a":11608500,"open_b":81.924309,"high_b":83.053497,"low_b":81.900947,"close_b":82.259171,"volume_b":3805700,"open_bench":244.463263,"close_bench":243.443008},{"date":"2018-10-16","open_a":35.329983,"high_a":35.785143,"low_a":35.204421,"close_a":35.698822,"volume_a":10442500,"open_b":82.788719,"high_b":83.863395,"low_b":82.422706,"close_b":83.629768,"volume_b":4061600,"open_bench":245.394783,"close_bench":248.766068},{"date":"2018-10-17","open_a":35.636034,"high_a":36.091198,"low_a":35.596797,"close_a":35.989178,"volume_a":11055800,"open_b":83.4896,"high_b":84.712235,"low_b":83.474028,"close_b":84.346222,"volume_b":4495900,"open_bench":248.801575,"close_bench":248.810455},{"date":"2018-10-18","open_a":36.020566,"high_a":36.161823,"low_a":35.675269,"close_a":35.792984,"volume_a":12716900,"open_b":84.478584,"high_b":85.156091,"low_b":83.754348,"close_b":84.05027,"volume_b":4093700,"open_bench":247.878895,"close_bench":245.217346},{"date":"2018-10-19","open_a":35.965636,"high_a":36.483579,"low_a":35.965636,"close_a":36.358017,"volume_a":13989200,"open_b":84.564247,"high_b":86.129533,"low_b":84.486374,"close_b":85.888123,"volume_b":5109000,"open_bench":245.865122,"close_bench":245.084396},{"date":"2018-10-22","open_a":36.389392,"high_a":36.460021,"low_a":35.949927,"close_a":36.075489,"volume_a":14283600,"open_b":85.732393,"high_b":86.355394,"low_b":85.070458,"close_b":85.195061,"volume_b":5477700,"open_bench":245.749723,"close_bench":243.984238},{"date":"2018-10-23","open_a":35.800825,"high_a":36.460025,"low_a":35.667417,"close_a":36.38155,"volume_a":17483900,"open_b":84.829032,"high_b":86.075027,"low_b":84.353995,"close_b":85.763527,"volume_b":4484100,"open_bench":240.382228,"close_bench":242.742111},{"date":"2018-10-24","open_a":36.358018,"high_a":37.189863,"low_a":36.287389,"close_a":36.671921,"volume_a":21626000,"open_b":85.818016,"high_b":88.901862,"low_b":85.623329,"close_b":87.967361,"volume_b":8222900,"open_bench":242.493692,"close_bench":235.387375},{"date":"2018-10-25","open_a":36.609141,"high_a":36.679769,"low_a":36.193215,"close_a":36.499271,"volume_a":14220200,"open_b":87.967369,"high_b":88.208779,"low_b":86.768096,"close_b":87.881706,"volume_b":6712900,"open_bench":237.215004,"close_bench":239.610382},{"date":"2018-10-26","open_a":36.389406,"high_a":36.47573,"low_a":35.651731,"close_a":36.036263,"volume_a":21458000,"open_b":86.923873,"high_b":87.266518,"low_b":85.420889,"close_b":86.012733,"volume_b":5872800,"open_bench":235.919702,"close_bench":235.39624},{"date":"2018-10-29","open_a":36.216759,"high_a":36.538511,"low_a":36.138281,"close_a":36.460033,"volume_a":20932200,"open_b":86.666884,"high_b":88.239953,"low_b":86.550071,"close_b":87.91288,"volume_b":5706000,"open_bench":238.474804,"close_bench":234.092117},{"date":"2018-10-30","open_a":36.585593,"high_a":37.464529,"low_a":36.130433,"close_a":37.378204,"volume_a":20895400,"open_b":88.58261,"high_b":88.956412,"low_b":87.687049,"close_b":88.598183,"volume_b":4900400,"open_bench":233.923561,"close_bench":237.560989},{"date":"2018-10-31","open_a":37.276188,"high_a":37.731351,"low_a":37.056455,"close_a":37.574398,"volume_a":25309400,"open_b":88.380112,"high_b":88.380112,"low_b":86.588995,"close_b":87.515701,"volume_b":6449500,"open_bench":240.116062,"close_bench":240.098328},{"date":"2018-11-01","open_a":37.519464,"high_a":37.786283,"low_a":37.119234,"close_a":37.464531,"volume_a":16494500,"open_b":87.219796,"high_b":88.107568,"low_b":86.425475,"close_b":86.838211,"volume_b":5736700,"open_bench":240.958904,"close_bench":242.653427},{"date":"2018-11-02","open_a":37.527314,"high_a":37.668571,"low_a":37.229104,"close_a":37.668571,"volume_a":17584900,"open_b":86.822617,"high_b":87.297654,"low_b":85.374145,"close_b":86.573418,"volume_b":6705700,"open_bench":243.753563,"close_bench":241.216232},{"date":"2018-11-05","open_a":37.668572,"high_a":38.280685,"low_a":37.605789,"close_a":38.210056,"volume_a":16543500,"open_b":86.978369,"high_b":89.119926,"low_b":86.659079,"close_b":89.080986,"volume_b":5270400,"open_bench":241.704132,"close_bench":242.546967},{"date":"2018-11-06","open_a":38.178664,"high_a":38.555349,"low_a":38.139424,"close_a":38.539654,"volume_a":14373400,"open_b":88.800643,"high_b":89.828591,"low_b":88.668258,"close_b":89.719566,"volume_b":3913400,"open_bench":242.484899,"close_bench":244.081818},{"date":"2018-11-07","open_a":38.759394,"high_a":38.884956,"low_a":38.476879,"close_a":38.743698,"volume_a":12239700,"open_b":89.906464,"high_b":90.155663,"low_b":89.119933,"close_b":90.031067,"volume_b":4219600,"open_bench":246.246595,"close_bench":249.307388},{"date":"2018-11-08","open_a":38.547512,"high_a":38.743703,"low_a":38.531817,"close_a":38.712315,"volume_a":9762800,"open_b":89.851936,"high_b":90.724136,"low_b":89.750701,"close_b":90.381485,"volume_b":2925100,"open_bench":248.508768,"close_bench":248.854782},{"date":"2018-11-09","open_a":38.759387,"high_a":39.151768,"low_a":38.735845,"close_a":38.986969,"volume_a":12398100,"open_b":90.397087,"high_b":91.666444,"low_b":90.397087,"close_b":91.487335,"volume_b":5026300,"open_bench":247.55077,"close_bench":246.424057},{"date":"2018-11-12","open_a":38.932034,"high_a":39.426434,"low_a":38.8771,"close_a":39.13607,"volume_a":13918300,"open_b":91.113518,"high_b":92.141465,"low_b":90.88768,"close_b":91.090157,"volume_b":4781900,"open_bench":245.918239,"close_bench":241.819458},{"date":"2018-11-13","open_a":39.08899,"high_a":39.238093,"low_a":38.908495,"close_a":39.128227,"volume_a":13872900,"open_b":91.697594,"high_b":91.697594,"low_b":90.498321,"close_b":91.222557,"volume_b":3609900,"open_bench":242.280786,"close_bench":241.366989},{"date":"2018-11-14","open_a":39.120369,"high_a":39.190997,"low_a":38.916332,"close_a":39.04974,"volume_a":12106800,"open_b":91.456185,"high_b":91.487335,"low_b":90.545045,"close_b":90.840973,"volume_b":4423800,"open_bench":243.230141,"close_bench":239.716904},{"date":"2018-11-15","open_a":38.971271,"high_a":39.0419,"low_a":38.665215,"close_a":39.034054,"volume_a":12080100,"open_b":90.66964,"high_b":91.105737,"low_b":90.023278,"close_b":90.957779,"volume_b":4045700,"open_bench":238.456952,"close_bench":242.218597},{"date":"2018-11-16","open_a":38.971271,"high_a":39.43428,"low_a":38.947729,"close_a":39.371498,"volume_a":13405800,"open_b":91.152479,"high_b":92.266089,"low_b":90.988937,"close_b":92.164848,"volume_b":4484300,"open_bench":241.12745,"close_bench":242.848587},{"date":"2018-11-19","open_a":39.481371,"high_a":39.834511,"low_a":39.434284,"close_a":39.638321,"volume_a":13575800,"open_b":91.954551,"high_b":93.075944,"low_b":91.868889,"close_b":92.678787,"volume_b":4673900,"open_bench":242.245339,"close_bench":238.740982},{"date":"2018-11-20","open_a":39.881597,"high_a":39.897293,"low_a":38.61813,"close_a":38.751541,"volume_a":16255300,"open_b":93.294023,"high_b":93.667819,"low_b":90.163452,"close_b":90.334778,"volume_b":5087800,"open_bench":235.422913,"close_bench":234.322815},{"date":"2018-11-21","open_a":38.633832,"high_a":38.798632,"low_a":38.131585,"close_a":38.241451,"volume_a":13559200,"open_b":90.249115,"high_b":91.027863,"low_b":89.493729,"close_b":89.774078,"volume_b":3245000,"open_bench":235.8664,"close_bench":235.12117},{"date":"2018-11-23","open_a":38.257142,"high_a":38.633828,"low_a":38.100189,"close_a":38.469028,"volume_a":4792500,"open_b":89.384705,"high_b":90.778665,"low_b":89.384705,"close_b":89.87532,"volume_b":1444700,"open_bench":233.488825,"close_bench":233.550934},{"date":"2018-11-26","open_a":38.461181,"high_a":38.633829,"low_a":38.194362,"close_a":38.351315,"volume_a":9341900,"open_b":90.241334,"high_b":90.669648,"low_b":89.548248,"close_b":90.225761,"volume_b":2972500,"open_bench":235.795486,"close_bench":237.321442},{"date":"2018-11-27","open_a":38.359166,"high_a":38.775089,"low_a":38.241451,"close_a":38.743698,"volume_a":9180700,"open_b":89.844184,"high_b":90.747529,"low_b":89.478171,"close_b":90.677444,"volume_b":2884700,"open_bench":236.292294,"close_bench":238.119888},{"date":"2018-11-28","open_a":38.618129,"high_a":39.018359,"low_a":38.618129,"close_a":39.010509,"volume_a":11368700,"open_b":90.319219,"high_b":92.297234,"low_b":89.99993,"close_b":92.281662,"volume_b":3989400,"open_bench":239.184568,"close_bench":243.602722},{"date":"2018-11-29","open_a":38.978871,"high_a":39.097517,"low_a":38.733673,"close_a":38.741581,"volume_a":11564300,"open_b":92.20378,"high_b":92.569793,"low_b":91.456183,"close_b":92.102539,"volume_b":2878700,"open_bench":242.830832,"close_bench":243.070389},{"date":"2018-11-30","open_a":38.757406,"high_a":39.935949,"low_a":38.749497,"close_a":39.864761,"volume_a":25538200,"open_b":91.993529,"high_b":95.007287,"low_b":91.993529,"close_b":94.960564,"volume_b":8051100,"open_bench":242.919588,"close_bench":244.552002},{"date":"2018-12-03","open_a":39.445555,"high_a":39.516742,"low_a":38.670407,"close_a":39.303181,"volume_a":14877200,"open_b":93.823586,"high_b":94.158448,"low_b":91.510703,"close_b":92.655464,"volume_b":6112600,"open_bench":248.659625,"close_bench":247.790176},{"date":"2018-12-04","open_a":39.524637,"high_a":39.951757,"low_a":39.152881,"close_a":39.21616,"volume_a":19059900,"open_b":92.959153,"high_b":93.924799,"low_b":91.463952,"close_b":91.736519,"volume_b":7018200,"open_bench":246.965153,"close_bench":239.76123},{"date":"2018-12-06","open_a":39.160796,"high_a":39.255714,"low_a":38.488474,"close_a":39.057972,"volume_a":17674900,"open_b":91.295383,"high_b":92.135259,"low_b":89.560685,"close_b":91.711395,"volume_b":5191700,"open_bench":235.919719,"close_bench":239.397461},{"date":"2018-12-07","open_a":39.208254,"high_a":39.295261,"low_a":38.662488,"close_a":38.82859,"volume_a":15565700,"open_b":91.342481,"high_b":91.82129,"low_b":90.565395,"close_b":90.910767,"volume_b":4615500,"open_bench":239.060348,"close_bench":233.834854},{"date":"2018-12-10","open_a":39.113338,"high_a":39.113338,"low_a":38.33819,"close_a":38.947235,"volume_a":12331100,"open_b":91.366022,"high_b":91.538708,"low_b":89.843253,"close_b":91.201187,"volume_b":4150600,"open_bench":233.657358,"close_bench":234.278397},{"date":"2018-12-11","open_a":39.224065,"high_a":39.556269,"low_a":38.899768,"close_a":39.184517,"volume_a":18083600,"open_b":91.758498,"high_b":92.771061,"low_b":91.585812,"close_b":92.064621,"volume_b":5545100,"open_bench":237.463389,"close_bench":234.331635},{"date":"2018-12-12","open_a":39.508818,"high_a":39.564185,"low_a":38.907681,"close_a":38.931412,"volume_a":12316900,"open_b":92.888803,"high_b":93.085036,"low_b":91.373891,"close_b":91.836998,"volume_b":4889700,"open_bench":237.294797,"close_bench":235.511551},{"date":"2018-12-13","open_a":38.535929,"high_a":39.239889,"low_a":38.417282,"close_a":39.129154,"volume_a":13766200,"open_b":91.946886,"high_b":93.320516,"low_b":91.923339,"close_b":92.896652,"volume_b":6181200,"open_bench":236.452033,"close_bench":235.431778},{"date":"2018-12-14","open_a":38.939322,"high_a":39.168703,"low_a":38.77322,"close_a":39.026329,"volume_a":13835900,"open_b":92.088169,"high_b":92.268706,"low_b":89.223168,"close_b":89.442947,"volume_b":7404200,"open_bench":233.29361,"close_bench":231.084534},{"date":"2018-12-17","open_a":39.026331,"high_a":39.176614,"low_a":37.982254,"close_a":38.227455,"volume_a":18482700,"open_b":89.772651,"high_b":90.322101,"low_b":88.069344,"close_b":88.595253,"volume_b":6015300,"open_bench":230.135269,"close_bench":226.551056},{"date":"2018-12-18","open_a":38.448913,"high_a":38.622927,"low_a":37.847776,"close_a":38.219532,"volume_a":14765700,"open_b":88.869937,"high_b":89.113264,"low_b":86.931156,"close_b":87.786728,"volume_b":5846200,"open_bench":228.183465,"close_bench":226.302628},{"date":"2018-12-19","open_a":38.156266,"high_a":38.567566,"low_a":37.665864,"close_a":37.887337,"volume_a":17903800,"open_b":88.406849,"high_b":88.846414,"low_b":86.052054,"close_b":86.373878,"volume_b":6915400,"open_bench":226.382509,"close_bench":222.91362},{"date":"2018-12-20","open_a":37.816158,"high_a":38.069267,"low_a":37.254569,"close_a":37.602596,"volume_a":21130000,"open_b":86.499443,"high_b":87.857377,"low_b":85.714511,"close_b":86.507294,"volume_b":8468300,"open_bench":221.671559,"close_bench":219.285034},{"date":"2018-12-21","open_a":37.381119,"high_a":38.559662,"low_a":37.357391,"close_a":37.62632,"volume_a":32477700,"open_b":87.096026,"high_b":89.607811,"low_b":85.408421,"close_b":85.88723,"volume_b":13263600,"open_bench":220.181862,"close_bench":214.791977},{"date":"2018-12-24","open_a":37.665869,"high_a":37.863611,"low_a":36.25004,"close_a":36.352863,"volume_a":10733700,"open_b":86.444514,"high_b":87.198047,"low_b":83.17135,"close_b":83.226295,"volume_b":3354000,"open_bench":213.310623,"close_bench":209.116516},{"date":"2018-12-26","open_a":36.289583,"high_a":37.143826,"low_a":36.115569,"close_a":37.128006,"volume_a":14342600,"open_b":83.375426,"high_b":85.117974,"low_b":82.441355,"close_b":85.031631,"volume_b":4537000,"open_bench":210.571102,"close_bench":219.682129},{"date":"2018-12-27","open_a":36.969804,"high_a":37.610485,"low_a":36.408215,"close_a":37.594666,"volume_a":16966500,"open_b":84.592091,"high_b":85.926479,"low_b":83.226312,"close_b":85.88723,"volume_b":5175900,"open_bench":216.460697,"close_bench":221.368698},{"date":"2018-12-28","open_a":37.879427,"high_a":37.974342,"low_a":37.199194,"close_a":37.33366,"volume_a":13218200,"open_b":86.413103,"high_b":87.205886,"low_b":85.730217,"close_b":86.625038,"volume_b":5410300,"open_bench":222.716173,"close_bench":221.083145},{"date":"2018-12-31","open_a":37.563047,"high_a":37.602594,"low_a":37.143832,"close_a":37.452309,"volume_a":10576300,"open_b":86.797738,"high_b":87.198049,"low_b":85.808722,"close_b":86.719246,"volume_b":5019100,"open_bench":222.698341,"close_bench":223.019592},{"date":"2019-01-02","open_a":37.12801,"high_a":37.349483,"low_a":36.827445,"close_a":37.120102,"volume_a":11603700,"open_b":85.675297,"high_b":86.358184,"low_b":84.953161,"close_b":85.777336,"volume_b":4835400,"open_bench":219.503661,"close_bench":223.251587},{"date":"2019-01-03","open_a":37.033091,"high_a":37.468122,"low_a":36.80371,"close_a":36.890717,"volume_a":14714400,"open_b":85.486891,"high_b":86.460205,"low_b":84.81185,"close_b":84.976685,"volume_b":5975900,"open_bench":221.511472,"close_bench":217.924179},{"date":"2019-01-04","open_a":36.977727,"high_a":37.62632,"low_a":36.89072,"close_a":37.62632,"volume_a":13013700,"open_b":85.981408,"high_b":86.813434,"low_b":84.960995,"close_b":86.719246,"volume_b":6293900,"open_bench":220.94036,"close_bench":225.223709},{"date":"2019-01-07","open_a":37.62632,"high_a":37.768695,"low_a":37.096374,"close_a":37.135921,"volume_a":13135500,"open_b":86.350311,"high_b":86.640732,"low_b":85.282804,"close_b":85.973541,"volume_b":5220300,"open_bench":225.491384,"close_bench":226.999481},{"date":"2019-01-08","open_a":37.373204,"high_a":37.626313,"low_a":37.207102,"close_a":37.555126,"volume_a":15420700,"open_b":86.452365,"high_b":86.970423,"low_b":85.887214,"close_b":86.797737,"volume_b":6407700,"open_bench":229.176928,"close_bench":229.132294},{"date":"2019-01-09","open_a":37.238749,"high_a":37.39694,"low_a":36.724619,"close_a":36.835354,"volume_a":18032500,"open_b":86.601508,"high_b":87.025372,"low_b":84.372299,"close_b":84.372299,"volume_b":8521300,"open_bench":229.837222,"close_bench":230.203094},{"date":"2019-01-10","open_a":36.906546,"high_a":37.246662,"low_a":36.653437,"close_a":37.230843,"volume_a":15976600,"open_b":84.309503,"high_b":85.039483,"low_b":84.270254,"close_b":84.89035,"volume_b":7905000,"open_bench":228.677239,"close_bench":231.015228},{"date":"2019-01-11","open_a":37.404849,"high_a":37.460217,"low_a":37.15174,"close_a":37.444397,"volume_a":11088600,"open_b":84.662719,"high_b":85.165075,"low_b":84.490033,"close_b":84.898201,"volume_b":5939000,"open_bench":229.944342,"close_bench":231.104431},{"date":"2019-01-14","open_a":37.373209,"high_a":37.491855,"low_a":37.167558,"close_a":37.294113,"volume_a":11514300,"open_b":84.654889,"high_b":84.741232,"low_b":83.9563,"close_b":84.238876,"volume_b":5804600,"open_bench":229.212559,"close_bench":229.694443},{"date":"2019-01-15","open_a":37.088462,"high_a":37.650048,"low_a":37.025186,"close_a":37.62632,"volume_a":11342900,"open_b":84.348738,"high_b":85.612479,"low_b":83.775742,"close_b":85.549683,"volume_b":4508000,"open_bench":230.069195,"close_bench":232.326874},{"date":"2019-01-16","open_a":37.491857,"high_a":37.547225,"low_a":37.064734,"close_a":37.11219,"volume_a":16907800,"open_b":85.549689,"high_b":85.596789,"low_b":84.537126,"close_b":85.039482,"volume_b":4953500,"open_bench":232.755223,"close_bench":232.889099},{"date":"2019-01-17","open_a":37.112189,"high_a":37.286203,"low_a":37.080553,"close_a":37.222927,"volume_a":12044600,"open_b":84.88249,"high_b":85.92645,"low_b":84.843241,"close_b":85.730217,"volume_b":4612200,"open_bench":232.023549,"close_bench":234.656006},{"date":"2019-01-18","open_a":37.373206,"high_a":37.721231,"low_a":37.27038,"close_a":37.657955,"volume_a":12760700,"open_b":86.318931,"high_b":86.57796,"low_b":85.690983,"close_b":86.397423,"volume_b":5142700,"open_bench":236.458555,"close_bench":237.779236},{"date":"2019-01-22","open_a":37.729141,"high_a":37.800328,"low_a":37.341566,"close_a":37.744961,"volume_a":11700400,"open_b":86.232563,"high_b":86.52299,"low_b":85.384836,"close_b":86.271812,"volume_b":5298200,"open_bench":236.315779,"close_bench":234.566727},{"date":"2019-01-23","open_a":37.824064,"high_a":38.19582,"low_a":37.689601,"close_a":38.18,"volume_a":12058300,"open_b":86.091324,"high_b":86.962598,"low_b":85.957887,"close_b":86.915504,"volume_b":5197400,"open_bench":235.593026,"close_bench":235.057602},{"date":"2019-01-24","open_a":38.053437,"high_a":38.053437,"low_a":37.476032,"close_a":37.72123,"volume_a":10272000,"open_b":86.585797,"high_b":86.585797,"low_b":85.094432,"close_b":85.950005,"volume_b":4478000,"open_bench":234.879095,"close_bench":235.182495},{"date":"2019-01-25","open_a":37.871508,"high_a":37.990151,"low_a":37.365289,"close_a":37.468113,"volume_a":12066500,"open_b":86.460223,"high_b":86.805589,"low_b":85.588948,"close_b":85.832275,"volume_b":5074800,"open_bench":237.020756,"close_bench":237.17247},{"date":"2019-01-28","open_a":37.349483,"high_a":37.412759,"low_a":37.128011,"close_a":37.309933,"volume_a":9966900,"open_b":85.526157,"high_b":86.028513,"low_b":84.772624,"close_b":85.416267,"volume_b":4055900,"open_bench":235.039733,"close_bench":235.369904},{"date":"2019-01-29","open_a":37.333662,"high_a":37.673778,"low_a":37.278294,"close_a":37.491856,"volume_a":11998600,"open_b":85.502604,"high_b":86.044208,"low_b":85.172934,"close_b":85.557549,"volume_b":4301500,"open_bench":235.512716,"close_bench":235.057602},{"date":"2019-01-30","open_a":37.373211,"high_a":37.87943,"low_a":37.325752,"close_a":37.855701,"volume_a":12952500,"open_b":85.510453,"high_b":87.315792,"low_b":85.196479,"close_b":86.986122,"volume_b":5023800,"open_bench":236.565596,"close_bench":238.778641},{"date":"2019-01-31","open_a":37.847787,"high_a":38.583387,"low_a":37.547221,"close_a":38.06926,"volume_a":32181500,"open_b":87.103859,"high_b":88.618777,"low_b":86.507309,"close_b":88.43824,"volume_b":6183400,"open_bench":238.716258,"close_bench":240.875763},{"date":"2019-02-01","open_a":38.346097,"high_a":38.583387,"low_a":38.282821,"close_a":38.520111,"volume_a":19130100,"open_b":88.658044,"high_b":88.917073,"low_b":87.480647,"close_b":88.061501,"volume_b":5010500,"open_bench":241.072054,"close_bench":240.991745},{"date":"2019-02-04","open_a":38.520115,"high_a":39.113341,"low_a":38.393561,"close_a":38.955147,"volume_a":15105700,"open_b":88.077176,"high_b":89.011247,"low_b":87.7946,"close_b":88.767914,"volume_b":3665000,"open_bench":241.036393,"close_bench":242.687271},{"date":"2019-02-05","open_a":39.018418,"high_a":39.129152,"low_a":38.836492,"close_a":38.963047,"volume_a":12651900,"open_b":88.720806,"high_b":89.121122,"low_b":88.296942,"close_b":88.618767,"volume_b":4593400,"open_bench":243.115566,"close_bench":243.704529},{"date":"2019-02-06","open_a":38.923499,"high_a":39.042146,"low_a":38.773217,"close_a":38.963047,"volume_a":8689100,"open_b":89.019102,"high_b":89.152539,"low_b":88.147828,"close_b":88.736526,"volume_b":4209400,"open_bench":243.427951,"close_bench":243.383316},{"date":"2019-02-07","open_a":38.891858,"high_a":39.121239,"low_a":38.757395,"close_a":39.0896,"volume_a":10047800,"open_b":88.555985,"high_b":88.822865,"low_b":88.037933,"close_b":88.767914,"volume_b":4227700,"open_bench":241.777082,"close_bench":241.063202},{"date":"2019-02-08","open_a":39.050051,"high_a":39.152878,"low_a":38.844401,"close_a":39.152878,"volume_a":9813300,"open_b":88.642311,"high_b":88.728654,"low_b":87.857379,"close_b":88.712952,"volume_b":3868300,"open_bench":239.822769,"close_bench":241.357635},{"date":"2019-02-11","open_a":39.208252,"high_a":39.453453,"low_a":39.137064,"close_a":39.239891,"volume_a":9369300,"open_b":88.822876,"high_b":89.152545,"low_b":88.414708,"close_b":88.673737,"volume_b":4851100,"open_bench":242.009148,"close_bench":241.491562},{"date":"2019-02-12","open_a":39.303166,"high_a":39.390173,"low_a":39.121244,"close_a":39.279438,"volume_a":10983300,"open_b":88.877793,"high_b":89.874653,"low_b":88.08501,"close_b":89.317352,"volume_b":6765300,"open_bench":243.097714,"close_bench":244.596878},{"date":"2019-02-13","open_a":39.2399,"high_a":39.500918,"low_a":39.200352,"close_a":39.382275,"volume_a":12748800,"open_b":89.568535,"high_b":90.055194,"low_b":89.191765,"close_b":89.576385,"volume_b":5528000,"open_bench":245.426892,"close_bench":245.39119},{"date":"2019-02-14","open_a":36.93817,"high_a":37.14382,"low_a":35.989009,"close_a":36.060196,"volume_a":58905400,"open_b":88.697267,"high_b":88.697267,"low_b":87.818142,"close_b":88.375443,"volume_b":6359200,"open_bench":244.311408,"close_bench":244.846832},{"date":"2019-02-15","open_a":36.305409,"high_a":36.590155,"low_a":35.751732,"close_a":35.783371,"volume_a":35701600,"open_b":90.934307,"high_b":91.271828,"low_b":89.568528,"close_b":90.981407,"volume_b":8889500,"open_bench":246.613627,"close_bench":247.514923},{"date":"2019-02-19","open_a":35.846652,"high_a":36.060214,"low_a":35.411621,"close_a":35.45908,"volume_a":25511300,"open_b":91.680016,"high_b":91.813453,"low_b":90.424121,"close_b":90.997124,"volume_b":5596100,"open_bench":246.720825,"close_bench":247.943359},{"date":"2019-02-20","open_a":35.522344,"high_a":35.783364,"low_a":35.435337,"close_a":35.672626,"volume_a":27464000,"open_b":90.855837,"high_b":91.554425,"low_b":90.588963,"close_b":90.918633,"volume_b":6038900,"open_bench":247.907569,"close_bench":248.442993},{"date":"2019-02-21","open_a":35.593533,"high_a":36.313313,"low_a":35.561894,"close_a":36.273766,"volume_a":23121200,"open_b":90.659619,"high_b":91.782072,"low_b":90.636072,"close_b":91.130577,"volume_b":4825900,"open_bench":247.809324,"close_bench":247.559464},{"date":"2019-02-22","open_a":36.194655,"high_a":36.344941,"low_a":35.751715,"close_a":35.814991,"volume_a":24728400,"open_b":91.138421,"high_b":91.656473,"low_b":90.730254,"close_b":91.648628,"volume_b":4865400,"open_bench":248.175291,"close_bench":249.094452},{"date":"2019-02-25","open_a":35.94155,"high_a":36.044376,"low_a":35.466971,"close_a":35.546066,"volume_a":23774400,"open_b":91.970426,"high_b":92.072471,"low_b":90.824427,"close_b":91.099152,"volume_b":3755900,"open_bench":250.513313,"close_bench":249.433533},{"date":"2019-02-26","open_a":35.67262,"high_a":35.70426,"low_a":35.324596,"close_a":35.348324,"volume_a":20463200,"open_b":91.342493,"high_b":91.444531,"low_b":90.604655,"close_b":91.028519,"volume_b":4074600,"open_bench":249.085515,"close_bench":249.255066},{"date":"2019-02-27","open_a":35.277137,"high_a":35.609345,"low_a":35.134763,"close_a":35.546066,"volume_a":17827500,"open_b":90.691008,"high_b":90.816594,"low_b":89.804038,"close_b":90.557571,"volume_b":4175100,"open_bench":248.541121,"close_bench":249.147949},{"date":"2019-02-28","open_a":35.68844,"high_a":36.076015,"low_a":35.680531,"close_a":35.862453,"volume_a":22533900,"open_b":90.793363,"high_b":91.964485,"low_b":90.373977,"close_b":91.505531,"volume_b":5586300,"open_bench":248.933836,"close_bench":248.683975},{"date":"2019-03-01","open_a":35.965287,"high_a":36.09975,"low_a":35.751726,"close_a":35.8941,"volume_a":16291600,"open_b":91.679631,"high_b":92.083194,"low_b":91.181117,"close_b":91.932846,"volume_b":3629100,"open_bench":250.254517,"close_bench":250.236679},{"date":"2019-03-04","open_a":36.012752,"high_a":36.123487,"low_a":35.830829,"close_a":36.10767,"volume_a":15973800,"open_b":92.225623,"high_b":92.526314,"low_b":91.394759,"close_b":91.924927,"volume_b":4383100,"open_bench":251.289627,"close_bench":249.326416},{"date":"2019-03-05","open_a":36.107666,"high_a":36.147214,"low_a":35.909924,"close_a":36.068115,"volume_a":13062300,"open_b":91.774587,"high_b":92.439273,"low_b":91.521372,"close_b":91.814148,"volume_b":3785400,"open_bench":249.451351,"close_bench":248.987305},{"date":"2019-03-06","open_a":36.123481,"high_a":36.2184,"low_a":35.941559,"close_a":35.949471,"volume_a":14232100,"open_b":91.6163,"high_b":92.447164,"low_b":91.458043,"close_b":92.312645,"volume_b":4770700,"open_bench":249.103295,"close_bench":247.479187},{"date":"2019-03-07","open_a":35.981096,"high_a":36.012733,"low_a":35.696348,"close_a":35.814991,"volume_a":12084000,"open_b":91.956588,"high_b":92.842843,"low_b":91.553025,"close_b":91.869545,"volume_b":4858300,"open_bench":247.033093,"close_bench":245.409012},{"date":"2019-03-08","open_a":35.712173,"high_a":35.791269,"low_a":35.28505,"close_a":35.466972,"volume_a":15579100,"open_b":91.655868,"high_b":91.996126,"low_b":90.413533,"close_b":91.181091,"volume_b":5003300,"open_bench":243.561799,"close_bench":244.918182},{"date":"2019-03-11","open_a":35.704264,"high_a":36.57433,"low_a":35.672625,"close_a":36.526871,"volume_a":22126900,"open_b":91.236496,"high_b":92.352221,"low_b":90.983281,"close_b":92.257263,"volume_b":4821600,"open_bench":245.632032,"close_bench":248.469742},{"date":"2019-03-12","open_a":36.344946,"high_a":36.582236,"low_a":36.305398,"close_a":36.424042,"volume_a":19348500,"open_b":92.383866,"high_b":92.747861,"low_b":91.798308,"close_b":92.241432,"volume_b":4356400,"open_bench":249.023028,"close_bench":249.406738},{"date":"2019-03-13","open_a":36.487319,"high_a":36.653421,"low_a":36.313305,"close_a":36.558506,"volume_a":16222500,"open_b":92.486729,"high_b":93.024811,"low_b":92.154386,"close_b":92.249344,"volume_b":5968600,"open_bench":250.290243,"close_bench":251.057663},{"date":"2019-03-14","open_a":36.574474,"high_a":36.7899,"low_a":36.406922,"close_a":36.462772,"volume_a":18504100,"open_b":92.470916,"high_b":92.565874,"low_b":91.355185,"close_b":91.394753,"volume_b":8986600,"open_bench":251.084422,"close_bench":250.897034},{"date":"2019-03-15","open_a":36.478728,"high_a":36.622345,"low_a":36.01596,"close_a":36.14362,"volume_a":47254500,"open_b":91.537193,"high_b":92.209794,"low_b":91.291893,"close_b":91.52137,"volume_b":18194200,"open_bench":251.446399,"close_bench":252.136536},{"date":"2019-03-18","open_a":36.247338,"high_a":36.446806,"low_a":36.191487,"close_a":36.23138,"volume_a":24006400,"open_b":91.924917,"high_b":92.890299,"low_b":91.829965,"close_b":92.795341,"volume_b":9476000,"open_bench":252.351718,"close_bench":253.050827},{"date":"2019-03-19","open_a":36.343082,"high_a":36.454784,"low_a":36.17553,"close_a":36.351063,"volume_a":14987000,"open_b":92.866586,"high_b":93.966488,"low_b":92.779543,"close_b":93.444229,"volume_b":6751700,"open_bench":254.108472,"close_bench":253.113571},{"date":"2019-03-20","open_a":36.223408,"high_a":36.622344,"low_a":36.047875,"close_a":36.327129,"volume_a":13376100,"open_b":93.428389,"high_b":94.54412,"low_b":92.858654,"close_b":93.626213,"volume_b":5733500,"open_bench":252.898438,"close_bench":252.351685},{"date":"2019-03-21","open_a":36.327119,"high_a":36.510629,"low_a":36.25531,"close_a":36.311161,"volume_a":20043000,"open_b":93.380897,"high_b":94.694446,"low_b":93.380897,"close_b":94.512451,"volume_b":4305600,"open_bench":251.536094,"close_bench":255.201935},{"date":"2019-03-22","open_a":36.271277,"high_a":36.789896,"low_a":36.271277,"close_a":36.646278,"volume_a":19055900,"open_b":94.559939,"high_b":95.976361,"low_b":94.520372,"close_b":95.525322,"volume_b":6668100,"open_bench":253.848481,"close_bench":250.290192},{"date":"2019-03-25","open_a":36.590434,"high_a":36.86171,"low_a":36.534583,"close_a":36.72607,"volume_a":11668200,"open_b":95.161319,"high_b":95.976354,"low_b":95.018886,"close_b":95.628181,"volume_b":4275400,"open_bench":249.949638,"close_bench":250.10202},{"date":"2019-03-26","open_a":36.837764,"high_a":37.332447,"low_a":36.837764,"close_a":37.212765,"volume_a":11434900,"open_b":96.1109,"high_b":96.664811,"low_b":95.952643,"close_b":96.554024,"volume_b":4583000,"open_bench":251.849771,"close_bench":251.966293},{"date":"2019-03-27","open_a":37.220743,"high_a":37.428189,"low_a":36.965424,"close_a":37.188828,"volume_a":11543600,"open_b":96.902188,"high_b":97.479837,"low_b":95.897244,"close_b":96.451149,"volume_b":5318500,"open_bench":251.957325,"close_bench":250.648743},{"date":"2019-03-28","open_a":37.244684,"high_a":37.436172,"low_a":37.085109,"close_a":37.164898,"volume_a":11121100,"open_b":96.45115,"high_b":96.910104,"low_b":96.213759,"close_b":96.411583,"volume_b":3694700,"open_bench":251.276148,"close_bench":251.598801},{"date":"2019-03-29","open_a":37.2367,"high_a":37.428191,"low_a":37.085106,"close_a":37.388298,"volume_a":16045600,"open_b":96.767658,"high_b":97.139568,"low_b":95.960532,"close_b":96.973396,"volume_b":5758800,"open_bench":253.104578,"close_bench":253.185242},{"date":"2019-04-01","open_a":37.492018,"high_a":37.635632,"low_a":37.172868,"close_a":37.276592,"volume_a":11522700,"open_b":97.00504,"high_b":97.242431,"low_b":96.213743,"close_b":96.538177,"volume_b":5025700,"open_bench":255.175016,"close_bench":256.187805},{"date":"2019-04-02","open_a":37.300527,"high_a":37.444144,"low_a":37.15691,"close_a":37.15691,"volume_a":8495000,"open_b":96.459075,"high_b":96.743942,"low_b":96.190031,"close_b":96.284988,"volume_b":3242600,"open_bench":256.376041,"close_bench":256.313293},{"date":"2019-04-03","open_a":37.03724,"high_a":37.17288,"low_a":36.734051,"close_a":36.845753,"volume_a":12854500,"open_b":96.284953,"high_b":96.720168,"low_b":95.541132,"close_b":96.189995,"volume_b":4534400,"open_bench":257.523273,"close_bench":256.716614},{"date":"2019-04-04","open_a":36.9734,"high_a":37.132976,"low_a":36.78989,"close_a":37.085102,"volume_a":7814900,"open_b":96.38782,"high_b":96.641035,"low_b":95.881391,"close_b":96.118782,"volume_b":3022500,"open_bench":257.039284,"close_bench":257.397797},{"date":"2019-04-05","open_a":37.093083,"high_a":37.164892,"low_a":36.8617,"close_a":37.077126,"volume_a":7677600,"open_b":96.023865,"high_b":96.27708,"low_b":95.604478,"close_b":96.237518,"volume_b":3971500,"open_bench":258.06106,"close_bench":258.643646},{"date":"2019-04-08","open_a":37.132976,"high_a":37.156911,"low_a":36.91755,"close_a":37.140953,"volume_a":9421300,"open_b":96.316615,"high_b":96.56983,"low_b":95.913053,"close_b":96.538177,"volume_b":3140100,"open_bench":258.222466,"close_bench":258.840912},{"date":"2019-04-09","open_a":37.061168,"high_a":37.276594,"low_a":36.917551,"close_a":37.236698,"volume_a":8641800,"open_b":96.332449,"high_b":96.696445,"low_b":95.628195,"close_b":96.672707,"volume_b":3864400,"open_bench":257.881766,"close_bench":257.514282},{"date":"2019-04-10","open_a":37.37234,"high_a":37.499999,"low_a":37.148937,"close_a":37.212765,"volume_a":8059900,"open_b":96.815132,"high_b":97.092084,"low_b":96.269141,"close_b":96.569832,"volume_b":2681100,"open_bench":257.926731,"close_bench":258.392822},{"date":"2019-04-11","open_a":37.2766,"high_a":37.300535,"low_a":37.132983,"close_a":37.26862,"volume_a":7352400,"open_b":96.823043,"high_b":97.131648,"low_b":95.881398,"close_b":96.277046,"volume_b":3399700,"open_bench":258.876713,"close_bench":258.321014},{"date":"2019-04-12","open_a":37.236707,"high_a":37.444155,"low_a":37.156921,"close_a":37.292561,"volume_a":10690000,"open_b":96.166258,"high_b":96.799298,"low_b":95.865567,"close_b":96.720169,"volume_b":3501100,"open_bench":259.925285,"close_bench":260.068695},{"date":"2019-04-15","open_a":37.404249,"high_a":37.531908,"low_a":37.260631,"close_a":37.499992,"volume_a":10261300,"open_b":96.886348,"high_b":98.120768,"low_b":96.823042,"close_b":97.85173,"volume_b":4419800,"open_bench":260.140458,"close_bench":259.898468},{"date":"2019-04-16","open_a":37.539884,"high_a":37.579777,"low_a":37.26063,"close_a":37.460098,"volume_a":10804600,"open_b":97.503543,"high_b":97.978319,"low_b":96.451117,"close_b":96.862595,"volume_b":5263700,"open_bench":260.776775,"close_bench":260.068695},{"date":"2019-04-17","open_a":37.659587,"high_a":37.938841,"low_a":37.619694,"close_a":37.723415,"volume_a":11849300,"open_b":98.912071,"high_b":100.581707,"low_b":97.994164,"close_b":100.502579,"volume_b":7318200,"open_bench":261.1801,"close_bench":259.432343},{"date":"2019-04-18","open_a":37.763301,"high_a":37.962769,"low_a":37.715428,"close_a":37.88298,"volume_a":12167000,"open_b":101.467957,"high_b":101.491695,"low_b":100.281019,"close_b":100.56588,"volume_b":6331200,"open_bench":260.015048,"close_bench":259.943329},{"date":"2019-04-22","open_a":37.819149,"high_a":37.986701,"low_a":37.515957,"close_a":37.819149,"volume_a":9834700,"open_b":100.312652,"high_b":100.684562,"low_b":99.608399,"close_b":100.051529,"volume_b":3707000,"open_bench":259.181487,"close_bench":260.167389},{"date":"2019-04-23","open_a":38.36968,"high_a":38.87234,"low_a":37.938828,"close_a":38.465424,"volume_a":25327800,"open_b":100.241451,"high_b":100.921967,"low_b":99.798327,"close_b":100.56588,"volume_b":4464100,"open_bench":260.534855,"close_bench":262.506714},{"date":"2019-04-24","open_a":38.321799,"high_a":38.481374,"low_a":38.234033,"close_a":38.281906,"volume_a":15997300,"open_b":100.819098,"high_b":101.270137,"low_b":100.233534,"close_b":100.431358,"volume_b":3517200,"open_bench":262.426087,"close_bench":261.924164},{"date":"2019-04-25","open_a":38.090441,"high_a":38.21012,"low_a":37.875015,"close_a":38.170227,"volume_a":11444400,"open_b":99.426425,"high_b":100.019897,"low_b":99.054514,"close_b":99.806244,"volume_b":4433700,"open_bench":261.825504,"close_bench":261.762756},{"date":"2019-04-26","open_a":38.289895,"high_a":38.656915,"low_a":38.281914,"close_a":38.505318,"volume_a":10154300,"open_b":100.114872,"high_b":100.692521,"low_b":99.798358,"close_b":100.01992,"volume_b":3033800,"open_bench":261.807637,"close_bench":262.981781},{"date":"2019-04-29","open_a":38.433505,"high_a":38.656911,"low_a":38.385634,"close_a":38.632973,"volume_a":9731500,"open_b":99.814144,"high_b":100.463007,"low_b":99.719186,"close_b":100.249359,"volume_b":2793400,"open_bench":263.071299,"close_bench":263.393951},{"date":"2019-04-30","open_a":38.720745,"high_a":39.239363,"low_a":38.609043,"close_a":39.14362,"volume_a":17912900,"open_b":100.53425,"high_b":101.594584,"low_b":99.972424,"close_b":101.325546,"volume_b":4963600,"open_bench":263.053468,"close_bench":263.528503},{"date":"2019-05-01","open_a":39.05585,"high_a":39.159574,"low_a":38.7367,"close_a":38.768616,"volume_a":10551200,"open_b":101.167277,"high_b":101.285973,"low_b":100.502591,"close_b":100.684586,"volume_b":3810800,"open_bench":264.155765,"close_bench":261.547546},{"date":"2019-05-02","open_a":38.720741,"high_a":38.912232,"low_a":38.513295,"close_a":38.609039,"volume_a":8724600,"open_b":100.684566,"high_b":100.929872,"low_b":99.837879,"close_b":100.360138,"volume_b":2875900,"open_bench":261.431149,"close_bench":260.983002},{"date":"2019-05-03","open_a":38.776606,"high_a":39.06384,"low_a":38.712777,"close_a":38.872353,"volume_a":7076900,"open_b":100.573772,"high_b":101.428368,"low_b":100.431338,"close_b":101.024811,"volume_b":2854900,"open_bench":262.452967,"close_bench":263.537476},{"date":"2019-05-06","open_a":38.561177,"high_a":38.78458,"low_a":38.505323,"close_a":38.672878,"volume_a":8657800,"open_b":100.732042,"high_b":100.993165,"low_b":100.304741,"close_b":100.708298,"volume_b":2883900,"open_bench":259.253164,"close_bench":262.452942},{"date":"2019-05-07","open_a":38.481396,"high_a":38.52927,"low_a":38.114376,"close_a":38.297886,"volume_a":12487500,"open_b":100.209806,"high_b":100.455106,"low_b":99.054514,"close_b":99.687553,"volume_b":3558900,"open_bench":260.059843,"close_bench":258.070068},{"date":"2019-05-08","open_a":38.337767,"high_a":38.481384,"low_a":38.154257,"close_a":38.178192,"volume_a":9281400,"open_b":99.545112,"high_b":100.288927,"low_b":98.872505,"close_b":99.719193,"volume_b":3844700,"open_bench":257.711609,"close_bench":257.711609},{"date":"2019-05-09","open_a":38.138296,"high_a":38.281913,"low_a":37.55585,"close_a":37.819149,"volume_a":11450500,"open_b":99.521359,"high_b":100.130661,"low_b":98.793368,"close_b":99.742928,"volume_b":3127200,"open_bench":255.650099,"close_bench":256.931793},{"date":"2019-05-10","open_a":37.827135,"high_a":38.537241,"low_a":37.819157,"close_a":38.449474,"volume_a":9653800,"open_b":99.814141,"high_b":101.507515,"low_b":99.402669,"close_b":101.293861,"volume_b":3720400,"open_bench":255.999592,"close_bench":258.222412},{"date":"2019-05-13","open_a":38.226063,"high_a":38.401596,"low_a":38.082446,"close_a":38.337765,"volume_a":11911500,"open_b":100.51049,"high_b":101.238487,"low_b":100.201885,"close_b":100.621277,"volume_b":4297200,"open_bench":253.131414,"close_bench":251.73317},{"date":"2019-05-14","open_a":38.720741,"high_a":39.351061,"low_a":38.577123,"close_a":38.8484,"volume_a":16844200,"open_b":100.937781,"high_b":101.689511,"low_b":100.724134,"close_b":100.819092,"volume_b":5150300,"open_bench":252.746079,"close_bench":254.009857},{"date":"2019-05-15","open_a":38.832444,"high_a":39.398936,"low_a":38.776593,"close_a":39.239361,"volume_a":11422500,"open_b":100.573794,"high_b":102.765683,"low_b":100.573794,"close_b":102.354218,"volume_b":3751900,"open_bench":252.387508,"close_bench":255.49765},{"date":"2019-05-16","open_a":39.311168,"high_a":39.622338,"low_a":39.263295,"close_a":39.55851,"volume_a":17771300,"open_b":102.370048,"high_b":103.462041,"low_b":101.887356,"close_b":103.082222,"volume_b":4812900,"open_bench":256.1968,"close_bench":257.863922},{"date":"2019-05-17","open_a":39.31118,"high_a":39.534583,"low_a":39.231391,"close_a":39.255329,"volume_a":11257000,"open_b":102.805248,"high_b":103.556977,"low_b":102.591594,"close_b":103.27211,"volume_b":4011700,"open_bench":255.569327,"close_bench":256.196716},{"date":"2019-05-20","open_a":39.127657,"high_a":39.207443,"low_a":38.90425,"close_a":38.976059,"volume_a":13850500,"open_b":102.900197,"high_b":103.224631,"low_b":102.512469,"close_b":102.860641,"volume_b":3965600,"open_bench":254.601372,"close_bench":254.502792},{"date":"2019-05-21","open_a":39.095754,"high_a":39.119689,"low_a":38.656925,"close_a":38.776604,"volume_a":10587400,"open_b":103.327504,"high_b":103.327504,"low_b":101.958564,"close_b":102.116821,"volume_b":4080800,"open_bench":256.187832,"close_bench":256.797333},{"date":"2019-05-22","open_a":38.920206,"high_a":39.662227,"low_a":38.872336,"close_a":39.614357,"volume_a":16050400,"open_b":102.156384,"high_b":103.374975,"low_b":102.037682,"close_b":102.923943,"volume_b":3588000,"open_bench":255.84728,"close_bench":256.008606},{"date":"2019-05-23","open_a":39.494683,"high_a":39.813833,"low_a":39.462767,"close_a":39.773937,"volume_a":12199400,"open_b":102.552027,"high_b":103.580703,"low_b":102.480801,"close_b":102.844803,"volume_b":3901400,"open_bench":253.794734,"close_bench":252.880524},{"date":"2019-05-24","open_a":39.750007,"high_a":39.805858,"low_a":39.518623,"close_a":39.582455,"volume_a":6887600,"open_b":103.129701,"high_b":103.248391,"low_b":102.045628,"close_b":102.472923,"volume_b":2702700,"open_bench":254.314582,"close_bench":253.454147},{"date":"2019-05-28","open_a":39.574466,"high_a":39.630316,"low_a":39.119679,"close_a":39.175529,"volume_a":18614000,"open_b":102.472906,"high_b":102.78942,"low_b":100.779532,"close_b":100.993179,"volume_b":8806000,"open_bench":253.73197,"close_bench":251.096863},{"date":"2019-05-29","open_a":38.888297,"high_a":39.119678,"low_a":38.624998,"close_a":38.784573,"volume_a":12534600,"open_b":100.953602,"high_b":101.531245,"low_b":100.368038,"close_b":101.420464,"volume_b":5502700,"open_bench":249.985537,"close_bench":249.411896},{"date":"2019-05-30","open_a":38.872337,"high_a":39.343081,"low_a":38.832441,"close_a":39.295208,"volume_a":9087200,"open_b":101.404645,"high_b":102.006038,"low_b":101.111869,"close_b":101.768646,"volume_b":2710300,"open_bench":250.164755,"close_bench":250.093063},{"date":"2019-05-31","open_a":39.095746,"high_a":39.343088,"low_a":38.920213,"close_a":39.199471,"volume_a":11368600,"open_b":101.285942,"high_b":101.642023,"low_b":100.486731,"close_b":101.285942,"volume_b":4191200,"open_bench":247.556536,"close_bench":246.722961},{"date":"2019-06-03","open_a":39.223404,"high_a":39.877659,"low_a":39.111702,"close_a":39.877659,"volume_a":14167200,"open_b":101.175168,"high_b":102.203856,"low_b":100.882387,"close_b":102.061417,"volume_b":3588700,"open_bench":246.758784,"close_bench":246.095535},{"date":"2019-06-04","open_a":39.893616,"high_a":40.117019,"low_a":39.590423,"close_a":39.893616,"volume_a":12010600,"open_b":102.67072,"high_b":102.718196,"low_b":101.301781,"close_b":102.132645,"volume_b":4242400,"open_bench":248.381117,"close_bench":251.437485},{"date":"2019-06-05","open_a":39.901595,"high_a":40.803191,"low_a":39.646276,"close_a":40.515957,"volume_a":15837600,"open_b":102.25134,"high_b":104.13463,"low_b":101.966473,"close_b":103.968452,"volume_b":5084200,"open_bench":253.050763,"close_bench":253.615433},{"date":"2019-06-06","open_a":40.61968,"high_a":41.122341,"low_a":40.364362,"close_a":41.010639,"volume_a":15190400,"open_b":104.482583,"high_b":105.431122,"low_b":104.068092,"close_b":105.199959,"volume_b":5528000,"open_bench":253.911217,"close_bench":255.264603},{"date":"2019-06-07","open_a":41.066493,"high_a":41.640959,"low_a":41.050535,"close_a":41.082451,"volume_a":15293600,"open_b":105.885433,"high_b":107.376,"low_b":105.686161,"close_b":106.387604,"volume_b":6332700,"open_bench":256.277495,"close_bench":257.819122},{"date":"2019-06-10","open_a":41.122349,"high_a":41.122349,"low_a":40.763306,"close_a":40.99469,"volume_a":9737400,"open_b":106.156451,"high_b":106.554996,"low_b":105.502844,"close_b":106.483261,"volume_b":2957900,"open_bench":259.36074,"close_bench":259.002228},{"date":"2019-06-11","open_a":41.05054,"high_a":41.162241,"low_a":40.835114,"close_a":40.954796,"volume_a":9915600,"open_b":106.857915,"high_b":107.081099,"low_b":105.765905,"close_b":106.172417,"volume_b":3553600,"open_bench":260.812767,"close_bench":258.939514},{"date":"2019-06-12","open_a":41.090424,"high_a":41.601062,"low_a":41.066489,"close_a":41.202126,"volume_a":11088100,"open_b":106.818045,"high_b":107.1608,"low_b":105.741981,"close_b":106.12458,"volume_b":3302800,"open_bench":258.706465,"close_bench":258.482391},{"date":"2019-06-13","open_a":41.37903,"high_a":41.572015,"low_a":40.832241,"close_a":41.113678,"volume_a":10353600,"open_b":106.371683,"high_b":106.642691,"low_b":105.096346,"close_b":105.965172,"volume_b":2851400,"open_bench":259.387563,"close_bench":259.548889},{"date":"2019-06-14","open_a":41.049343,"high_a":41.435311,"low_a":41.049343,"close_a":41.258411,"volume_a":8694300,"open_b":106.020944,"high_b":106.443401,"low_b":105.662256,"close_b":105.79776,"volume_b":3291800,"open_bench":259.262207,"close_bench":259.262207},{"date":"2019-06-17","open_a":41.210169,"high_a":41.395112,"low_a":40.695545,"close_a":40.90461,"volume_a":12322400,"open_b":105.78981,"high_b":106.2043,"low_b":105.128223,"close_b":105.630394,"volume_b":2725800,"open_bench":259.495126,"close_bench":259.360687},{"date":"2019-06-18","open_a":41.21016,"high_a":41.250364,"low_a":40.631206,"close_a":40.719658,"volume_a":9221100,"open_b":107.144866,"high_b":107.144866,"low_b":105.104317,"close_b":105.263733,"volume_b":4378200,"open_bench":261.171299,"close_bench":262.076538},{"date":"2019-06-19","open_a":40.639256,"high_a":41.202127,"low_a":40.582969,"close_a":41.105633,"volume_a":9414100,"open_b":105.263725,"high_b":106.244142,"low_b":104.681852,"close_b":105.893433,"volume_b":4349900,"open_bench":262.210972,"close_bench":262.668091},{"date":"2019-06-20","open_a":41.411196,"high_a":41.644386,"low_a":41.29058,"close_a":41.539852,"volume_a":10656600,"open_b":106.706444,"high_b":107.168758,"low_b":106.283987,"close_b":106.937607,"volume_b":3847500,"open_bench":265.338966,"close_bench":265.177612},{"date":"2019-06-21","open_a":41.435311,"high_a":41.724788,"low_a":41.33882,"close_a":41.451393,"volume_a":21264400,"open_b":106.937637,"high_b":107.256468,"low_b":106.51518,"close_b":106.778221,"volume_b":7696300,"open_bench":264.909326,"close_bench":264.792236},{"date":"2019-06-24","open_a":41.604177,"high_a":41.974063,"low_a":41.572013,"close_a":41.748913,"volume_a":10780000,"open_b":107.328181,"high_b":107.798473,"low_b":106.762253,"close_b":106.921669,"volume_b":3685800,"open_bench":264.999431,"close_bench":264.468048},{"date":"2019-06-25","open_a":41.813248,"high_a":41.893657,"low_a":41.523771,"close_a":41.620262,"volume_a":11519900,"open_b":106.794134,"high_b":107.192679,"low_b":106.443413,"close_b":106.65863,"volume_b":4176300,"open_bench":264.521978,"close_bench":261.874054},{"date":"2019-06-26","open_a":41.427274,"high_a":41.523766,"low_a":41.081511,"close_a":41.105633,"volume_a":12405100,"open_b":106.060809,"high_b":106.626748,"low_b":105.455025,"close_b":105.47097,"volume_b":6024100,"open_bench":262.765659,"close_bench":261.612823},{"date":"2019-06-27","open_a":41.242328,"high_a":41.330779,"low_a":40.985016,"close_a":41.073467,"volume_a":12374900,"open_b":105.415134,"high_b":105.566583,"low_b":104.721669,"close_b":104.968765,"volume_b":4750900,"open_bench":262.369431,"close_bench":262.540558},{"date":"2019-06-28","open_a":41.226254,"high_a":41.330788,"low_a":40.623178,"close_a":40.944817,"volume_a":16339900,"open_b":105.43911,"high_b":105.542736,"low_b":104.227529,"close_b":104.522461,"volume_b":6318500,"open_bench":263.513223,"close_bench":263.89151},{"date":"2019-07-01","open_a":41.065424,"high_a":41.515721,"low_a":40.832234,"close_a":41.491596,"volume_a":14976400,"open_b":105.207933,"high_b":105.447051,"low_b":104.474612,"close_b":105.112274,"volume_b":3544800,"open_bench":267.20598,"close_bench":266.287323},{"date":"2019-07-02","open_a":41.63634,"high_a":41.853448,"low_a":41.451397,"close_a":41.797161,"volume_a":8569200,"open_b":105.287632,"high_b":105.734,"low_b":104.697793,"close_b":105.55864,"volume_b":3233300,"open_bench":266.242201,"close_bench":266.980743},{"date":"2019-07-03","open_a":41.813249,"high_a":42.078604,"low_a":41.813249,"close_a":41.893658,"volume_a":7253700,"open_b":105.997055,"high_b":106.881826,"low_b":105.829673,"close_b":106.802124,"volume_b":2859200,"open_bench":267.656332,"close_bench":269.115387},{"date":"2019-07-05","open_a":41.7248,"high_a":41.909743,"low_a":41.395118,"close_a":41.901703,"volume_a":8916300,"open_b":105.925305,"high_b":106.403564,"low_b":104.857207,"close_b":106.028931,"volume_b":3356000,"open_bench":267.890425,"close_bench":268.809082},{"date":"2019-07-08","open_a":41.974057,"high_a":42.078591,"low_a":41.724785,"close_a":41.813236,"volume_a":9495100,"open_b":106.092705,"high_b":106.323856,"low_b":105.247779,"close_b":105.66227,"volume_b":5854500,"open_bench":267.503158,"close_bench":267.332031},{"date":"2019-07-09","open_a":41.588096,"high_a":41.789122,"low_a":41.354906,"close_a":41.483562,"volume_a":10584900,"open_b":106.012981,"high_b":106.331812,"low_b":104.195616,"close_b":105.008652,"volume_b":8233200,"open_bench":266.179271,"close_bench":267.665344},{"date":"2019-07-10","open_a":41.65242,"high_a":41.789119,"low_a":41.539847,"close_a":41.668503,"volume_a":7620900,"open_b":105.534743,"high_b":107.176759,"low_b":105.112286,"close_b":107.136902,"volume_b":6141500,"open_bench":268.727974,"close_bench":268.944122},{"date":"2019-07-11","open_a":41.805209,"high_a":41.861496,"low_a":41.539854,"close_a":41.845413,"volume_a":10376500,"open_b":106.977473,"high_b":107.39993,"low_b":106.291975,"close_b":107.089066,"volume_b":3944200,"open_bench":269.583722,"close_bench":269.574707},{"date":"2019-07-12","open_a":42.014263,"high_a":42.046428,"low_a":41.740869,"close_a":41.909729,"volume_a":7347400,"open_b":106.810089,"high_b":107.152832,"low_b":105.702134,"close_b":106.236183,"volume_b":4481200,"open_bench":270.060982,"close_bench":270.781494},{"date":"2019-07-15","open_a":42.038384,"high_a":42.046426,"low_a":41.708702,"close_a":41.91777,"volume_a":5876100,"open_b":106.491221,"high_b":106.93759,"low_b":105.654275,"close_b":106.929611,"volume_b":3077900,"open_bench":271.213896,"close_bench":270.871643},{"date":"2019-07-16","open_a":41.781081,"high_a":42.191174,"low_a":41.660468,"close_a":41.925819,"volume_a":7132500,"open_b":106.873857,"high_b":107.798485,"low_b":106.052855,"close_b":106.650673,"volume_b":4019700,"open_bench":270.781458,"close_bench":269.997894},{"date":"2019-07-17","open_a":41.990154,"high_a":42.022315,"low_a":41.82129,"close_a":41.957989,"volume_a":8264200,"open_b":106.746314,"high_b":107.033266,"low_b":105.590535,"close_b":105.837631,"volume_b":4105100,"open_bench":269.970937,"close_bench":268.160614},{"date":"2019-07-18","open_a":41.845411,"high_a":41.949942,"low_a":41.547891,"close_a":41.837368,"volume_a":11911800,"open_b":105.702125,"high_b":106.172406,"low_b":104.8333,"close_b":105.917343,"volume_b":3698600,"open_bench":267.665246,"close_bench":269.142303},{"date":"2019-07-19","open_a":41.925822,"high_a":41.933865,"low_a":41.306664,"close_a":41.322746,"volume_a":10355500,"open_b":106.148521,"high_b":106.164466,"low_b":103.629713,"close_b":103.693481,"volume_b":5022000,"open_bench":270.23215,"close_bench":267.647278},{"date":"2019-07-22","open_a":41.42728,"high_a":41.42728,"low_a":41.113681,"close_a":41.18605,"volume_a":12090700,"open_b":104.012273,"high_b":104.267336,"low_b":103.326776,"close_b":103.844879,"volume_b":4411700,"open_bench":268.043582,"close_bench":268.304779},{"date":"2019-07-23","open_a":42.818368,"high_a":43.936067,"low_a":42.424357,"close_a":43.686798,"volume_a":33417900,"open_b":104.522412,"high_b":105.407183,"low_b":103.948507,"close_b":104.211548,"volume_b":4145800,"open_bench":269.421607,"close_bench":270.223175},{"date":"2019-07-24","open_a":43.517943,"high_a":44.080815,"low_a":43.115893,"close_a":43.244549,"volume_a":15633400,"open_b":104.578221,"high_b":104.578221,"low_b":102.521727,"close_b":102.99202,"volume_b":4669700,"open_bench":269.466602,"close_bench":271.493073},{"date":"2019-07-25","open_a":43.123916,"high_a":43.260611,"low_a":42.392184,"close_a":42.673618,"volume_a":19135000,"open_b":102.52972,"high_b":103.247097,"low_b":102.338414,"close_b":102.816673,"volume_b":3335200,"open_bench":271.042723,"close_bench":270.196106},{"date":"2019-07-26","open_a":42.890741,"high_a":43.67072,"low_a":42.770125,"close_a":43.558144,"volume_a":11451900,"open_b":103.17535,"high_b":104.753585,"low_b":102.593478,"close_b":104.59417,"volume_b":4580800,"open_bench":270.880561,"close_bench":272.006378},{"date":"2019-07-29","open_a":43.558142,"high_a":43.662676,"low_a":43.341037,"close_a":43.413406,"volume_a":12283900,"open_b":104.817351,"high_b":105.26372,"low_b":104.562276,"close_b":104.841263,"volume_b":4533400,"open_bench":271.889395,"close_bench":271.511108},{"date":"2019-07-30","open_a":43.389291,"high_a":43.662685,"low_a":43.075692,"close_a":43.196308,"volume_a":8127100,"open_b":104.689839,"high_b":105.27969,"low_b":103.805069,"close_b":104.004341,"volume_b":3367900,"open_bench":270.114986,"close_bench":270.844513},{"date":"2019-07-31","open_a":43.027431,"high_a":43.027431,"low_a":41.845405,"close_a":42.319824,"volume_a":16865500,"open_b":104.004312,"high_b":104.004312,"low_b":100.624652,"close_b":101.876083,"volume_b":5757600,"open_bench":271.087733,"close_bench":267.881409},{"date":"2019-08-01","open_a":42.440444,"high_a":42.778168,"low_a":41.821286,"close_a":41.837368,"volume_a":19527200,"open_b":102.792759,"high_b":103.143468,"low_b":101.047123,"close_b":101.342049,"volume_b":6199400,"open_bench":268.03458,"close_bench":265.548767},{"date":"2019-08-02","open_a":41.740878,"high_a":42.40024,"low_a":41.740878,"close_a":42.078602,"volume_a":13102300,"open_b":101.310147,"high_b":102.681149,"low_b":100.831894,"close_b":101.96376,"volume_b":5573300,"open_bench":264.657069,"close_bench":263.549255},{"date":"2019-08-05","open_a":42.126836,"high_a":42.343944,"low_a":41.330778,"close_a":41.531803,"volume_a":19986500,"open_b":102.306522,"high_b":102.944184,"low_b":98.743522,"close_b":99.158012,"volume_b":6192800,"open_bench":259.469378,"close_bench":255.623596},{"date":"2019-08-06","open_a":41.708701,"high_a":42.42435,"low_a":41.499636,"close_a":42.030342,"volume_a":19903100,"open_b":99.460907,"high_b":101.230448,"low_b":99.142069,"close_b":100.792046,"volume_b":6258600,"open_bench":257.50603,"close_bench":259.208252},{"date":"2019-08-07","open_a":42.030346,"high_a":42.930938,"low_a":41.628295,"close_a":42.762077,"volume_a":17937000,"open_b":100.752176,"high_b":102.298526,"low_b":99.341323,"close_b":101.852158,"volume_b":6149000,"open_bench":256.145927,"close_bench":259.361267},{"date":"2019-08-08","open_a":42.786208,"high_a":43.244546,"low_a":42.496731,"close_a":43.172176,"volume_a":10799000,"open_b":102.107247,"high_b":103.422441,"low_b":101.230443,"close_b":103.199257,"volume_b":3702100,"open_bench":260.847336,"close_bench":264.449951},{"date":"2019-08-09","open_a":43.252586,"high_a":43.341038,"low_a":42.681675,"close_a":42.955067,"volume_a":9260900,"open_b":103.390561,"high_b":103.414473,"low_b":101.48551,"close_b":102.505791,"volume_b":2929500,"open_bench":263.51324,"close_bench":262.648621},{"date":"2019-08-12","open_a":42.842487,"high_a":42.971143,"low_a":42.6093,"close_a":42.77816,"volume_a":6588600,"open_b":102.386243,"high_b":102.944204,"low_b":101.923935,"close_b":102.306541,"volume_b":2920800,"open_bench":261.153515,"close_bench":259.451294},{"date":"2019-08-13","open_a":42.536942,"high_a":43.107853,"low_a":42.512817,"close_a":43.019402,"volume_a":8581200,"open_b":102.250724,"high_b":103.924618,"low_b":101.979716,"close_b":103.565918,"volume_b":4088200,"open_bench":259.154154,"close_bench":263.486298},{"date":"2019-08-14","open_a":42.89074,"high_a":43.397325,"low_a":42.593224,"close_a":42.609306,"volume_a":17088900,"open_b":103.239131,"high_b":104.044188,"low_b":102.665226,"close_b":102.920288,"volume_b":6232700,"open_bench":259.451325,"close_bench":255.695587},{"date":"2019-08-15","open_a":42.963114,"high_a":43.421451,"low_a":42.802293,"close_a":43.316917,"volume_a":12393200,"open_b":103.350704,"high_b":104.594163,"low_b":103.143453,"close_b":104.195618,"volume_b":3701800,"open_bench":256.578254,"close_bench":256.371094},{"date":"2019-08-16","open_a":43.405362,"high_a":43.815455,"low_a":43.38124,"close_a":43.751125,"volume_a":14288700,"open_b":104.753567,"high_b":105.470955,"low_b":104.131837,"close_b":105.024574,"volume_b":3405000,"open_bench":258.019234,"close_bench":260.153778},{"date":"2019-08-19","open_a":43.775247,"high_a":44.048642,"low_a":43.670716,"close_a":43.976273,"volume_a":10936200,"open_b":105.27169,"high_b":105.893419,"low_b":105.168064,"close_b":105.670235,"volume_b":3101200,"open_bench":263.162009,"close_bench":263.288086},{"date":"2019-08-20","open_a":43.919992,"high_a":43.968239,"low_a":43.292794,"close_a":43.324959,"volume_a":8176700,"open_b":105.853584,"high_b":106.092714,"low_b":105.160107,"close_b":105.263733,"volume_b":3160300,"open_bench":262.783691,"close_bench":261.270599},{"date":"2019-08-21","open_a":43.62247,"high_a":43.751126,"low_a":43.300829,"close_a":43.501854,"volume_a":10886500,"open_b":105.941248,"high_b":106.52312,"low_b":104.912994,"close_b":105.4869,"volume_b":2936900,"open_bench":263.423228,"close_bench":263.39621},{"date":"2019-08-22","open_a":43.517941,"high_a":43.936074,"low_a":43.501858,"close_a":43.81546,"volume_a":7467900,"open_b":105.016604,"high_b":105.957164,"low_b":104.673861,"close_b":105.741959,"volume_b":2484200,"open_bench":264.098686,"close_bench":263.315094},{"date":"2019-08-23","open_a":43.759173,"high_a":43.992363,"low_a":42.93095,"close_a":43.212387,"volume_a":14160400,"open_b":105.574603,"high_b":106.291992,"low_b":103.255068,"close_b":103.836952,"volume_b":4239500,"open_bench":262.018152,"close_bench":256.551178},{"date":"2019-08-26","open_a":43.421444,"high_a":43.855659,"low_a":43.204337,"close_a":43.855659,"volume_a":9226900,"open_b":104.275327,"high_b":106.610796,"low_b":104.091988,"close_b":106.602829,"volume_b":3954100,"open_bench":258.730818,"close_bench":259.388306},{"date":"2019-08-27","open_a":43.984311,"high_a":44.185336,"low_a":43.751121,"close_a":44.000393,"volume_a":13537000,"open_b":106.969502,"high_b":107.89413,"low_b":106.555012,"close_b":107.415871,"volume_b":7187800,"open_bench":260.775333,"close_bench":258.370575},{"date":"2019-08-28","open_a":43.952153,"high_a":44.346164,"low_a":43.759168,"close_a":44.313999,"volume_a":10046400,"open_b":107.208627,"high_b":107.87818,"low_b":107.033266,"close_b":107.758621,"volume_b":3972600,"open_bench":257.713113,"close_bench":260.189911},{"date":"2019-08-29","open_a":44.434607,"high_a":44.498937,"low_a":43.903901,"close_a":44.265747,"volume_a":9262800,"open_b":108.093387,"high_b":108.754973,"low_b":107.224561,"close_b":108.420197,"volume_b":3045200,"open_bench":262.738787,"close_bench":263.513336},{"date":"2019-08-30","open_a":44.354193,"high_a":44.40244,"low_a":44.072759,"close_a":44.257702,"volume_a":8746700,"open_b":108.491961,"high_b":109.289051,"low_b":108.212974,"close_b":108.986153,"volume_b":3742800,"open_bench":264.990357,"close_bench":263.39621},{"date":"2019-09-03","open_a":44.209458,"high_a":44.490895,"low_a":44.121007,"close_a":44.46677,"volume_a":12991400,"open_b":108.68323,"high_b":109.735383,"low_b":108.308597,"close_b":109.568001,"volume_b":2989800,"open_bench":261.703014,"close_bench":261.85611},{"date":"2019-09-04","open_a":44.50698,"high_a":44.965317,"low_a":44.466775,"close_a":44.844704,"volume_a":10718100,"open_b":109.846987,"high_b":110.938997,"low_b":109.400618,"close_b":110.915085,"volume_b":3201700,"open_bench":264.017573,"close_bench":264.828156},{"date":"2019-09-05","open_a":44.828625,"high_a":44.844707,"low_a":44.249671,"close_a":44.322041,"volume_a":13677000,"open_b":111.40067,"high_b":111.40067,"low_b":108.792225,"close_b":109.137337,"volume_b":4994600,"open_bench":267.305024,"close_bench":268.232697},{"date":"2019-09-06","open_a":44.579345,"high_a":44.579345,"low_a":44.032556,"close_a":44.410484,"volume_a":9093500,"open_b":109.899784,"high_b":110.293062,"low_b":109.169429,"close_b":110.25293,"volume_b":3051600,"open_bench":268.54786,"close_bench":268.439758},{"date":"2019-09-09","open_a":44.249671,"high_a":44.289878,"low_a":43.710925,"close_a":43.839581,"volume_a":10814900,"open_b":109.474403,"high_b":109.69913,"low_b":108.695882,"close_b":109.474403,"volume_b":4062100,"open_bench":269.421568,"close_bench":268.574951},{"date":"2019-09-10","open_a":43.606389,"high_a":43.775249,"low_a":43.30083,"close_a":43.743088,"volume_a":12038200,"open_b":108.944723,"high_b":109.562715,"low_b":108.093958,"close_b":109.041031,"volume_b":4175000,"open_bench":267.81832,"close_bench":268.511841},{"date":"2019-09-11","open_a":43.654639,"high_a":44.064729,"low_a":43.397327,"close_a":44.040607,"volume_a":9740400,"open_b":108.567508,"high_b":109.787449,"low_b":108.254495,"close_b":109.442337,"volume_b":5005100,"open_bench":268.818163,"close_bench":270.421326},{"date":"2019-09-12","open_a":44.305956,"high_a":44.47482,"low_a":44.209465,"close_a":44.313999,"volume_a":12357900,"open_b":109.747324,"high_b":110.991342,"low_b":109.747324,"close_b":110.188744,"volume_b":4146900,"open_bench":271.321998,"close_bench":271.358032},{"date":"2019-09-13","open_a":44.127698,"high_a":44.378792,"low_a":43.795606,"close_a":43.949501,"volume_a":11486100,"open_b":109.21761,"high_b":109.915867,"low_b":108.896563,"close_b":109.506546,"volume_b":4008200,"open_bench":271.799279,"close_bench":271.177826},{"date":"2019-09-16","open_a":43.819905,"high_a":44.014302,"low_a":43.439217,"close_a":43.706509,"volume_a":11172000,"open_b":109.466398,"high_b":109.554684,"low_b":107.853179,"close_b":108.254478,"volume_b":2953600,"open_bench":270.051964,"close_bench":270.340179},{"date":"2019-09-17","open_a":43.819908,"high_a":44.305898,"low_a":43.706511,"close_a":43.933308,"volume_a":11347400,"open_b":108.318721,"high_b":109.402221,"low_b":107.957541,"close_b":108.912636,"volume_b":4013300,"open_bench":270.142099,"close_bench":271.02475},{"date":"2019-09-18","open_a":43.900914,"high_a":43.973813,"low_a":43.593121,"close_a":43.925213,"volume_a":9700400,"open_b":109.000894,"high_b":109.15339,"low_b":107.628458,"close_b":108.75209,"volume_b":2647800,"open_bench":270.637385,"close_bench":271.186798},{"date":"2019-09-19","open_a":43.925219,"high_a":44.224911,"low_a":43.844223,"close_a":44.160114,"volume_a":7560300,"open_b":108.67984,"high_b":109.024951,"low_b":108.414981,"close_b":108.736015,"volume_b":2804500,"open_bench":271.574098,"close_bench":271.168793},{"date":"2019-09-20","open_a":44.297795,"high_a":44.297795,"low_a":43.657913,"close_a":43.666012,"volume_a":19211500,"open_b":108.350775,"high_b":108.960745,"low_b":107.93342,"close_b":108.118027,"volume_b":6671100,"open_bench":271.769603,"close_bench":269.887604},{"date":"2019-09-23","open_a":43.666019,"high_a":44.030511,"low_a":43.641721,"close_a":43.852314,"volume_a":6485300,"open_b":108.318705,"high_b":108.952753,"low_b":108.318705,"close_b":108.703949,"volume_b":3346500,"open_bench":269.227069,"close_bench":269.824249},{"date":"2019-09-24","open_a":43.941405,"high_a":44.232998,"low_a":43.77131,"close_a":43.981903,"volume_a":10093000,"open_b":109.354045,"high_b":109.514562,"low_b":108.013707,"close_b":108.639732,"volume_b":4798500,"open_bench":270.910077,"close_bench":267.707031},{"date":"2019-09-25","open_a":44.095293,"high_a":44.127693,"low_a":43.738903,"close_a":43.876598,"volume_a":10248400,"open_b":108.519355,"high_b":108.880522,"low_b":107.957526,"close_b":108.423035,"volume_b":3579500,"open_bench":267.788443,"close_bench":269.290436},{"date":"2019-09-26","open_a":44.119609,"high_a":44.208708,"low_a":43.844216,"close_a":44.05481,"volume_a":10031200,"open_b":109.049027,"high_b":109.418228,"low_b":108.358792,"close_b":108.928642,"volume_b":4309600,"open_bench":269.299468,"close_bench":268.729431},{"date":"2019-09-27","open_a":44.1925,"high_a":44.257297,"low_a":43.738911,"close_a":43.990005,"volume_a":7939900,"open_b":109.34603,"high_b":109.34603,"low_b":108.085944,"close_b":108.832367,"volume_b":3559400,"open_bench":269.480396,"close_bench":267.281708},{"date":"2019-09-30","open_a":43.860399,"high_a":44.289688,"low_a":43.860399,"close_a":44.095291,"volume_a":9808700,"open_b":109.209564,"high_b":110.541868,"low_b":109.049035,"close_b":110.03624,"volume_b":3705200,"open_bench":267.797492,"close_bench":268.521332},{"date":"2019-10-01","open_a":44.168208,"high_a":44.370703,"low_a":43.747018,"close_a":44.265408,"volume_a":14261500,"open_b":109.899784,"high_b":110.710417,"low_b":109.225605,"close_b":110.25293,"volume_b":3638900,"open_bench":269.399021,"close_bench":265.327362},{"date":"2019-10-02","open_a":43.909015,"high_a":44.09531,"low_a":42.847943,"close_a":42.99374,"volume_a":14343600,"open_b":109.867701,"high_b":110.260979,"low_b":107.138871,"close_b":107.500038,"volume_b":5728600,"open_bench":263.752908,"close_bench":260.64035},{"date":"2019-10-03","open_a":43.333924,"high_a":43.722714,"low_a":42.783138,"close_a":43.609318,"volume_a":13817500,"open_b":111.111733,"high_b":112.163135,"low_b":109.55469,"close_b":110.7024,"volume_b":8829700,"open_bench":260.414181,"close_bench":262.775757},{"date":"2019-10-04","open_a":43.738914,"high_a":44.176304,"low_a":43.649816,"close_a":44.176304,"volume_a":9295400,"open_b":110.630159,"high_b":112.724941,"low_b":110.453586,"close_b":112.588501,"volume_b":5278700,"open_bench":263.427193,"close_bench":266.331635},{"date":"2019-10-07","open_a":44.006211,"high_a":44.030509,"low_a":43.552621,"close_a":43.633617,"volume_a":5964900,"open_b":111.79393,"high_b":112.050767,"low_b":110.734506,"close_b":111.031464,"volume_b":4273300,"open_bench":265.535416,"close_bench":265.182526},{"date":"2019-10-08","open_a":43.641717,"high_a":43.795612,"low_a":43.293425,"close_a":43.398724,"volume_a":13301900,"open_b":110.477677,"high_b":111.119758,"low_b":110.020202,"close_b":110.148621,"volume_b":4074900,"open_bench":263.336763,"close_bench":261.065674},{"date":"2019-10-09","open_a":43.787518,"high_a":43.795616,"low_a":43.366328,"close_a":43.601223,"volume_a":7135000,"open_b":110.212823,"high_b":110.895024,"low_b":109.819545,"close_b":110.638199,"volume_b":3065300,"open_bench":263.074399,"close_bench":263.544891},{"date":"2019-10-10","open_a":43.374422,"high_a":43.641717,"low_a":43.220527,"close_a":43.46352,"volume_a":8942900,"open_b":110.04427,"high_b":110.959232,"low_b":109.667048,"close_b":110.670296,"volume_b":2191000,"open_bench":263.463449,"close_bench":265.327362},{"date":"2019-10-11","open_a":43.455423,"high_a":43.495922,"low_a":43.090932,"close_a":43.171928,"volume_a":8981700,"open_b":110.822773,"high_b":111.32038,"low_b":110.028196,"close_b":110.469627,"volume_b":2761400,"open_bench":268.069007,"close_bench":268.078064},{"date":"2019-10-14","open_a":43.325827,"high_a":43.479722,"low_a":43.107132,"close_a":43.171928,"volume_a":7306300,"open_b":110.630174,"high_b":110.999375,"low_b":110.148623,"close_b":110.317162,"volume_b":2371500,"open_bench":267.761336,"close_bench":267.779449},{"date":"2019-10-15","open_a":43.277225,"high_a":43.47972,"low_a":43.147629,"close_a":43.342022,"volume_a":9564300,"open_b":110.341234,"high_b":110.718457,"low_b":109.281811,"close_b":109.418251,"volume_b":3369300,"open_bench":268.820006,"close_bench":270.430573},{"date":"2019-10-16","open_a":43.301527,"high_a":43.390625,"low_a":43.034235,"close_a":43.325829,"volume_a":10454800,"open_b":109.265752,"high_b":109.546666,"low_b":108.816299,"close_b":109.490479,"volume_b":2812000,"open_bench":269.969042,"close_bench":269.996185},{"date":"2019-10-17","open_a":43.487829,"high_a":43.83612,"low_a":43.398734,"close_a":43.568829,"volume_a":13190300,"open_b":110.012191,"high_b":110.45361,"low_b":109.634968,"close_b":109.923904,"volume_b":2322500,"open_bench":271.154278,"close_bench":270.792358},{"date":"2019-10-18","open_a":44.548886,"high_a":44.864778,"low_a":44.103398,"close_a":44.370689,"volume_a":18122400,"open_b":110.172696,"high_b":110.252948,"low_b":109.337987,"close_b":109.354042,"volume_b":3630400,"open_bench":270.258522,"close_bench":269.607056},{"date":"2019-10-21","open_a":44.103409,"high_a":44.192508,"low_a":43.479725,"close_a":43.925213,"volume_a":12778900,"open_b":109.42626,"high_b":109.747307,"low_b":108.880499,"close_b":109.249687,"volume_b":2829400,"open_bench":270.919123,"close_bench":271.434845},{"date":"2019-10-22","open_a":43.925221,"high_a":44.079116,"low_a":43.528333,"close_a":43.617428,"volume_a":9533400,"open_b":109.675079,"high_b":110.236908,"low_b":109.402199,"close_b":109.683113,"volume_b":3248100,"open_bench":271.968662,"close_bench":270.548126},{"date":"2019-10-23","open_a":43.787511,"high_a":44.273497,"low_a":43.747009,"close_a":44.257298,"volume_a":8556600,"open_b":109.642982,"high_b":110.357294,"low_b":109.642982,"close_b":110.317162,"volume_b":2323600,"open_bench":270.294828,"close_bench":271.335358},{"date":"2019-10-24","open_a":44.200598,"high_a":44.403093,"low_a":44.135801,"close_a":44.232998,"volume_a":11051400,"open_b":110.5178,"high_b":110.846868,"low_b":109.787433,"close_b":110.670296,"volume_b":2292500,"open_bench":272.267263,"close_bench":271.778656},{"date":"2019-10-25","open_a":44.18441,"high_a":44.192511,"low_a":43.471627,"close_a":43.536427,"volume_a":9782400,"open_b":110.581992,"high_b":110.694355,"low_b":109.337962,"close_b":109.66703,"volume_b":2520000,"open_bench":271.208665,"close_bench":272.891632},{"date":"2019-10-28","open_a":43.528316,"high_a":43.698411,"low_a":43.228621,"close_a":43.390617,"volume_a":8866100,"open_b":109.594823,"high_b":110.397421,"low_b":109.273776,"close_b":109.610878,"volume_b":3212600,"open_bench":274.10409,"close_bench":274.42981},{"date":"2019-10-29","open_a":43.244821,"high_a":43.463516,"low_a":43.139522,"close_a":43.261021,"volume_a":7999700,"open_b":109.819562,"high_b":110.05231,"low_b":109.10525,"close_b":109.217613,"volume_b":2550300,"open_bench":274.158294,"close_bench":274.348297},{"date":"2019-10-30","open_a":43.350125,"high_a":43.779414,"low_a":43.058532,"close_a":43.690315,"volume_a":8620900,"open_b":109.041028,"high_b":110.188738,"low_b":108.679861,"close_b":109.875725,"volume_b":2653700,"open_bench":274.547382,"close_bench":275.189819},{"date":"2019-10-31","open_a":43.706509,"high_a":44.1196,"low_a":43.61741,"close_a":44.0872,"volume_a":13982000,"open_b":109.795465,"high_b":110.381358,"low_b":109.233649,"close_b":110.092422,"volume_b":4492300,"open_bench":275.180715,"close_bench":274.456848},{"date":"2019-11-01","open_a":44.249202,"high_a":44.3383,"low_a":43.617416,"close_a":43.657917,"volume_a":9889500,"open_b":110.39743,"high_b":111.031478,"low_b":109.715217,"close_b":109.899811,"volume_b":3054900,"open_bench":275.895578,"close_bench":276.999451},{"date":"2019-11-04","open_a":43.706515,"high_a":43.738915,"low_a":42.904637,"close_a":43.042332,"volume_a":17515100,"open_b":109.795479,"high_b":110.036261,"low_b":108.134094,"close_b":108.479218,"volume_b":3202400,"open_bench":278.546594,"close_bench":278.112274},{"date":"2019-11-05","open_a":42.839836,"high_a":42.839836,"low_a":42.216151,"close_a":42.459145,"volume_a":19284000,"open_b":108.214335,"high_b":108.43104,"low_b":106.424544,"close_b":107.556198,"volume_b":4300800,"open_bench":278.311503,"close_bench":277.80481},{"date":"2019-11-06","open_a":42.548242,"high_a":42.766937,"low_a":42.451046,"close_a":42.766937,"volume_a":13288900,"open_b":107.949498,"high_b":108.25449,"low_b":107.187032,"close_b":107.901344,"volume_b":3645600,"open_bench":277.804698,"close_bench":277.868042},{"date":"2019-11-07","open_a":42.831743,"high_a":42.92894,"low_a":42.08656,"close_a":42.353855,"volume_a":11725500,"open_b":107.877248,"high_b":107.949479,"low_b":106.841889,"close_b":107.002419,"volume_b":3338700,"open_bench":279.198105,"close_bench":278.845215},{"date":"2019-11-08","open_a":42.491546,"high_a":42.604943,"low_a":42.191854,"close_a":42.289051,"volume_a":7141800,"open_b":106.978352,"high_b":107.652544,"low_b":106.400492,"close_b":106.849945,"volume_b":2604000,"open_bench":278.501368,"close_bench":279.532867},{"date":"2019-11-11","open_a":42.386248,"high_a":42.418645,"low_a":41.940757,"close_a":41.989357,"volume_a":8198300,"open_b":106.769667,"high_b":106.769667,"low_b":105.918914,"close_b":106.256004,"volume_b":2227200,"open_bench":278.157585,"close_bench":278.999054},{"date":"2019-11-12","open_a":42.046045,"high_a":42.046045,"low_a":41.778753,"close_a":41.884048,"volume_a":12656900,"open_b":106.488757,"high_b":106.488757,"low_b":105.613928,"close_b":105.926941,"volume_b":3246500,"open_bench":279.361016,"close_bench":279.587219},{"date":"2019-11-13","open_a":42.264743,"high_a":42.483438,"low_a":42.086547,"close_a":42.451038,"volume_a":12257900,"open_b":106.448642,"high_b":107.203087,"low_b":105.951023,"close_b":106.978348,"volume_b":3166600,"open_bench":278.600883,"close_bench":279.677612},{"date":"2019-11-14","open_a":42.548242,"high_a":42.661639,"low_a":42.402445,"close_a":42.629242,"volume_a":8660300,"open_b":107.227165,"high_b":107.475969,"low_b":106.528896,"close_b":107.01046,"volume_b":2141900,"open_bench":279.3972,"close_bench":280.084839},{"date":"2019-11-15","open_a":42.637346,"high_a":42.669746,"low_a":42.426752,"close_a":42.661644,"volume_a":8982400,"open_b":107.275291,"high_b":107.483962,"low_b":106.585056,"close_b":107.395676,"volume_b":3648700,"open_bench":281.414971,"close_bench":282.111694},{"date":"2019-11-18","open_a":42.66974,"high_a":43.285323,"low_a":42.661639,"close_a":42.953232,"volume_a":13782000,"open_b":107.54819,"high_b":108.904583,"low_b":107.475959,"close_b":107.596344,"volume_b":3619700,"open_bench":281.876444,"close_bench":282.319794},{"date":"2019-11-19","open_a":43.050438,"high_a":43.180035,"low_a":42.823642,"close_a":42.99374,"volume_a":12679500,"open_b":108.069881,"high_b":108.398949,"low_b":107.604385,"close_b":108.005684,"volume_b":3019100,"open_bench":282.916952,"close_bench":282.238342},{"date":"2019-11-20","open_a":43.009928,"high_a":43.252925,"low_a":42.93703,"close_a":43.107128,"volume_a":11970300,"open_b":107.796998,"high_b":108.51131,"low_b":107.066631,"close_b":107.339523,"volume_b":3047700,"open_bench":281.650154,"close_bench":281.18869},{"date":"2019-11-21","open_a":43.018028,"high_a":43.204323,"low_a":42.750734,"close_a":42.89653,"volume_a":7856100,"open_b":107.090696,"high_b":107.716734,"low_b":106.817816,"close_b":107.475952,"volume_b":3391200,"open_bench":281.297305,"close_bench":280.736298},{"date":"2019-11-22","open_a":42.896533,"high_a":43.082828,"low_a":42.726438,"close_a":42.953232,"volume_a":9113300,"open_b":107.484003,"high_b":107.973589,"low_b":106.76167,"close_b":107.604401,"volume_b":3014900,"open_bench":281.478195,"close_bench":281.360565},{"date":"2019-11-25","open_a":43.155725,"high_a":43.244823,"low_a":42.742637,"close_a":43.107128,"volume_a":10963500,"open_b":108.077901,"high_b":108.174209,"low_b":106.914136,"close_b":107.411743,"volume_b":3612400,"open_bench":282.283614,"close_bench":283.54129},{"date":"2019-11-26","open_a":43.180029,"high_a":43.690314,"low_a":43.10713,"close_a":43.657917,"volume_a":13219800,"open_b":107.780969,"high_b":109.049076,"low_b":107.443879,"close_b":108.912636,"volume_b":4947400,"open_bench":283.577385,"close_bench":284.183594},{"date":"2019-11-27","open_a":43.682214,"high_a":43.819909,"low_a":43.414919,"close_a":43.698414,"volume_a":9595200,"open_b":108.912618,"high_b":109.570755,"low_b":108.615661,"close_b":109.081169,"volume_b":2476100,"open_bench":284.663168,"close_bench":285.450378},{"date":"2019-11-29","open_a":43.82082,"high_a":43.8453,"low_a":43.429126,"close_a":43.576012,"volume_a":7222300,"open_b":109.201562,"high_b":109.490498,"low_b":108.559481,"close_b":109.016968,"volume_b":2978400,"open_bench":284.88939,"close_bench":284.391754},{"date":"2019-12-02","open_a":43.510732,"high_a":43.861626,"low_a":43.127197,"close_a":43.861626,"volume_a":13173800,"open_b":109.675083,"high_b":109.675083,"low_b":107.901347,"close_b":109.057091,"volume_b":3866300,"open_bench":284.645165,"close_bench":281.975983},{"date":"2019-12-03","open_a":43.828977,"high_a":43.935062,"low_a":43.57601,"close_a":43.90242,"volume_a":14498300,"open_b":109.683094,"high_b":109.683094,"low_b":107.805017,"close_b":108.719978,"volume_b":5000000,"open_bench":279.270513,"close_bench":280.084839},{"date":"2019-12-04","open_a":43.902428,"high_a":44.318605,"low_a":43.837147,"close_a":44.302284,"volume_a":13624000,"open_b":108.310636,"high_b":110.252922,"low_b":107.981568,"close_b":110.196747,"volume_b":4228200,"open_bench":281.12541,"close_bench":281.813049},{"date":"2019-12-05","open_a":44.237001,"high_a":44.302285,"low_a":43.926912,"close_a":44.22068,"volume_a":14068100,"open_b":110.07149,"high_b":110.35436,"low_b":109.327929,"close_b":110.192719,"volume_b":3351800,"open_bench":282.509824,"close_bench":282.319794},{"date":"2019-12-06","open_a":44.318595,"high_a":44.47364,"low_a":44.204348,"close_a":44.408356,"volume_a":7843500,"open_b":110.459424,"high_b":111.283812,"low_b":110.19271,"close_b":111.017097,"volume_b":3216900,"open_bench":284.219889,"close_bench":284.898499},{"date":"2019-12-09","open_a":44.310421,"high_a":44.481786,"low_a":44.081933,"close_a":44.122734,"volume_a":13887200,"open_b":110.847356,"high_b":111.566671,"low_b":110.726128,"close_b":110.855446,"volume_b":3882900,"open_bench":284.50944,"close_bench":284.002747},{"date":"2019-12-10","open_a":44.09827,"high_a":44.12275,"low_a":43.845299,"close_a":43.877941,"volume_a":8993900,"open_b":111.219182,"high_b":111.219182,"low_b":110.055342,"close_b":110.136169,"volume_b":2940200,"open_bench":283.948439,"close_bench":283.686035},{"date":"2019-12-11","open_a":44.016656,"high_a":44.188025,"low_a":43.788169,"close_a":44.024818,"volume_a":8879100,"open_b":110.766548,"high_b":111.162567,"low_b":110.17655,"close_b":110.806961,"volume_b":3339100,"open_bench":284.138443,"close_bench":284.491333},{"date":"2019-12-12","open_a":44.073786,"high_a":44.432841,"low_a":44.016664,"close_a":44.179871,"volume_a":11233300,"open_b":110.564528,"high_b":111.453563,"low_b":110.524114,"close_b":110.871643,"volume_b":3502100,"open_bench":284.500262,"close_bench":286.943268},{"date":"2019-12-13","open_a":44.016662,"high_a":44.51444,"low_a":43.93506,"close_a":44.408356,"volume_a":13072200,"open_b":110.669575,"high_b":111.736432,"low_b":110.0715,"close_b":111.485886,"volume_b":3557900,"open_bench":286.708149,"close_bench":287.115326},{"date":"2019-12-16","open_a":44.628684,"high_a":44.742931,"low_a":44.367555,"close_a":44.408356,"volume_a":9538100,"open_b":110.685746,"high_b":111.146437,"low_b":110.047257,"close_b":110.774651,"volume_b":4701000,"open_bench":288.834483,"close_bench":289.08783},{"date":"2019-12-17","open_a":44.392038,"high_a":44.538924,"low_a":44.302274,"close_a":44.408356,"volume_a":9230000,"open_b":110.766566,"high_b":111.081771,"low_b":109.667386,"close_b":110.055328,"volume_b":5491800,"open_bench":289.467813,"close_bench":289.151123},{"date":"2019-12-18","open_a":44.498118,"high_a":44.677645,"low_a":43.975859,"close_a":43.99218,"volume_a":15132100,"open_b":110.766571,"high_b":110.968626,"low_b":109.6189,"close_b":109.893692,"volume_b":5370500,"open_bench":289.540197,"close_bench":289.16922},{"date":"2019-12-19","open_a":44.098272,"high_a":44.34308,"low_a":44.000346,"close_a":44.326759,"volume_a":13385900,"open_b":110.095751,"high_b":110.516029,"low_b":109.877528,"close_b":110.297806,"volume_b":3641700,"open_bench":289.359192,"close_bench":290.354492},{"date":"2019-12-20","open_a":44.832699,"high_a":45.044868,"low_a":44.449166,"close_a":44.857182,"volume_a":23832800,"open_b":111.057539,"high_b":111.639458,"low_b":109.239043,"close_b":111.469727,"volume_b":10311300,"open_bench":291.381954,"close_bench":291.627472},{"date":"2019-12-23","open_a":44.881655,"high_a":45.077504,"low_a":44.75109,"close_a":44.808212,"volume_a":9300800,"open_b":111.283811,"high_b":111.63943,"low_b":110.596819,"close_b":110.758461,"volume_b":2992900,"open_bench":292.409329,"close_bench":292.072906},{"date":"2019-12-24","open_a":44.718455,"high_a":44.881661,"low_a":44.58789,"close_a":44.645012,"volume_a":3359300,"open_b":110.79891,"high_b":111.0333,"low_b":110.136175,"close_b":110.637268,"volume_b":1220600,"open_bench":292.300306,"close_bench":292.082092},{"date":"2019-12-26","open_a":44.816375,"high_a":44.89798,"low_a":44.710293,"close_a":44.89798,"volume_a":6228500,"open_b":110.612989,"high_b":111.000932,"low_b":109.998747,"close_b":110.475594,"volume_b":2934700,"open_bench":292.463886,"close_bench":293.636841},{"date":"2019-12-27","open_a":44.889824,"high_a":45.248879,"low_a":44.881665,"close_a":45.167274,"volume_a":6895500,"open_b":110.629174,"high_b":111.194926,"low_b":110.427119,"close_b":111.16259,"volume_b":2554000,"open_bench":294.364333,"close_bench":293.564178},{"date":"2019-12-30","open_a":45.028543,"high_a":45.199909,"low_a":44.93062,"close_a":45.101986,"volume_a":6431700,"open_b":110.483674,"high_b":111.510117,"low_b":110.305864,"close_b":110.572578,"volume_b":3024700,"open_bench":293.646021,"close_bench":291.945679},{"date":"2019-12-31","open_a":45.044872,"high_a":45.191758,"low_a":44.865344,"close_a":45.167274,"volume_a":7982600,"open_b":110.475595,"high_b":110.936274,"low_b":109.942167,"close_b":110.459427,"volume_b":3229400,"open_bench":291.445571,"close_bench":292.654877},{"date":"2020-01-02","open_a":45.142778,"high_a":45.232541,"low_a":44.6858,"close_a":44.873489,"volume_a":11867700,"open_b":110.621059,"high_b":110.912019,"low_b":109.222842,"close_b":109.772438,"volume_b":3784100,"open_bench":294.182418,"close_bench":295.391724},{"date":"2020-01-03","open_a":44.326755,"high_a":44.873497,"low_a":44.139068,"close_a":44.628685,"volume_a":11354500,"open_b":109.481516,"high_b":110.556438,"low_b":109.214802,"close_b":109.618912,"volume_b":4000100,"open_bench":292.018422,"close_bench":293.154999},{"date":"2020-01-06","open_a":44.596048,"high_a":44.808214,"low_a":44.489963,"close_a":44.612366,"volume_a":14698300,"open_b":109.35219,"high_b":110.176578,"low_b":109.271363,"close_b":110.039169,"volume_b":4085100,"open_bench":291.409241,"close_bench":294.273438},{"date":"2020-01-07","open_a":44.432845,"high_a":44.555247,"low_a":44.188036,"close_a":44.269638,"volume_a":9973900,"open_b":109.91794,"high_b":109.974522,"low_b":108.261087,"close_b":108.309578,"volume_b":5718100,"open_bench":293.70968,"close_bench":293.446014},{"date":"2020-01-08","open_a":44.285954,"high_a":44.587884,"low_a":44.188031,"close_a":44.351234,"volume_a":10676000,"open_b":108.673284,"high_b":109.400677,"low_b":108.358078,"close_b":108.867249,"volume_b":3681400,"open_bench":293.636924,"close_bench":295.009918},{"date":"2020-01-09","open_a":44.881661,"high_a":45.208074,"low_a":44.751096,"close_a":45.159111,"volume_a":15398000,"open_b":109.344074,"high_b":109.416823,"low_b":108.729832,"close_b":108.939964,"volume_b":3954700,"open_bench":296.564847,"close_bench":297.010376},{"date":"2020-01-10","open_a":45.273351,"high_a":45.640565,"low_a":45.248871,"close_a":45.314152,"volume_a":13040100,"open_b":109.247103,"high_b":109.449158,"low_b":108.616693,"close_b":108.729843,"volume_b":3099500,"open_bench":297.592197,"close_bench":296.155548},{"date":"2020-01-13","open_a":45.314141,"high_a":45.885362,"low_a":45.224377,"close_a":45.803761,"volume_a":14302900,"open_b":108.826825,"high_b":110.394774,"low_b":108.81067,"close_b":110.346283,"volume_b":5127200,"open_bench":296.773937,"close_bench":298.192383},{"date":"2020-01-14","open_a":45.893541,"high_a":45.934342,"low_a":45.558969,"close_a":45.697693,"volume_a":16345300,"open_b":110.063409,"high_b":110.677664,"low_b":109.675467,"close_b":110.604927,"volume_b":4159700,"open_bench":297.755846,"close_bench":297.737671},{"date":"2020-01-15","open_a":45.697695,"high_a":46.497405,"low_a":45.697695,"close_a":46.268917,"volume_a":17735400,"open_b":110.758476,"high_b":112.544648,"low_b":110.564511,"close_b":112.504234,"volume_b":5002600,"open_bench":297.646777,"close_bench":298.410553},{"date":"2020-01-16","open_a":46.391313,"high_a":46.472918,"low_a":46.195467,"close_a":46.366833,"volume_a":10234100,"open_b":112.625461,"high_b":113.280118,"low_b":112.019296,"close_b":112.835594,"volume_b":4977900,"open_bench":299.783522,"close_bench":300.892822},{"date":"2020-01-17","open_a":46.407634,"high_a":46.636125,"low_a":46.350512,"close_a":46.464756,"volume_a":14264300,"open_b":113.231656,"high_b":114.217685,"low_b":113.070002,"close_b":114.169182,"volume_b":6908700,"open_bench":301.602061,"close_bench":301.829376},{"date":"2020-01-21","open_a":46.39131,"high_a":46.717723,"low_a":46.162822,"close_a":46.652439,"volume_a":11968900,"open_b":113.643839,"high_b":114.864247,"low_b":113.425616,"close_b":114.654114,"volume_b":4440000,"open_bench":300.874582,"close_bench":301.238281},{"date":"2020-01-22","open_a":46.717719,"high_a":47.03597,"low_a":46.660597,"close_a":47.01149,"volume_a":12869600,"open_b":114.61368,"high_b":116.036155,"low_b":114.540943,"close_b":115.882591,"volume_b":5366100,"open_bench":302.093049,"close_bench":301.274719},{"date":"2020-01-23","open_a":46.823814,"high_a":47.207346,"low_a":46.709571,"close_a":47.191029,"volume_a":12858900,"open_b":115.648176,"high_b":116.262431,"low_b":115.066256,"close_b":116.100777,"volume_b":3915700,"open_bench":300.629148,"close_bench":301.620239},{"date":"2020-01-24","open_a":47.207339,"high_a":47.22366,"low_a":46.889087,"close_a":47.068615,"volume_a":8594300,"open_b":116.246309,"high_b":116.504933,"low_b":115.130962,"close_b":115.510826,"volume_b":2791500,"open_bench":302.274806,"close_bench":298.937805},{"date":"2020-01-27","open_a":46.685086,"high_a":47.207345,"low_a":46.619805,"close_a":46.905415,"volume_a":14359900,"open_b":114.484377,"high_b":115.866439,"low_b":113.999452,"close_b":114.880409,"volume_b":3895300,"open_bench":293.718734,"close_bench":294.146088},{"date":"2020-01-28","open_a":46.880933,"high_a":47.125741,"low_a":46.489238,"close_a":46.521877,"volume_a":15240200,"open_b":114.87234,"high_b":115.510829,"low_b":114.573291,"close_b":115.122887,"volume_b":2908300,"open_bench":295.564491,"close_bench":297.228455},{"date":"2020-01-29","open_a":46.513718,"high_a":46.725885,"low_a":46.236268,"close_a":46.521877,"volume_a":13163600,"open_b":114.767234,"high_b":115.130931,"low_b":114.09641,"close_b":114.549011,"volume_b":3263200,"open_bench":298.58328,"close_bench":296.982971},{"date":"2020-01-30","open_a":47.27262,"high_a":48.129451,"low_a":47.133894,"close_a":48.031528,"volume_a":20225600,"open_b":115.025877,"high_b":116.351357,"low_b":114.379298,"close_b":116.246284,"volume_b":4058500,"open_bench":294.928053,"close_bench":297.946808},{"date":"2020-01-31","open_a":48.121295,"high_a":48.211059,"low_a":47.452151,"close_a":47.656158,"volume_a":17685200,"open_b":116.230141,"high_b":116.909043,"low_b":114.419735,"close_b":114.783432,"volume_b":4753100,"open_bench":297.328511,"close_bench":292.536713},{"date":"2020-02-03","open_a":47.941771,"high_a":48.121298,"low_a":47.754084,"close_a":47.803047,"volume_a":12333600,"open_b":115.389596,"high_b":115.632065,"low_b":114.637945,"close_b":115.179451,"volume_b":3977800,"open_bench":294.009717,"close_bench":294.709839},{"date":"2020-02-04","open_a":47.94177,"high_a":48.512992,"low_a":47.835685,"close_a":48.015213,"volume_a":15417900,"open_b":115.73712,"high_b":116.43219,"low_b":114.848086,"close_b":115.753288,"volume_b":3494300,"open_bench":298.30147,"close_bench":299.20163},{"date":"2020-02-05","open_a":48.06418,"high_a":48.211066,"low_a":47.770409,"close_a":48.023376,"volume_a":11414900,"open_b":116.12504,"high_b":116.860523,"low_b":115.922985,"close_b":116.626144,"volume_b":3504900,"open_bench":302.120338,"close_bench":302.656799},{"date":"2020-02-06","open_a":48.129457,"high_a":48.423228,"low_a":48.015213,"close_a":48.137619,"volume_a":9347000,"open_b":116.900922,"high_b":117.43435,"low_b":116.440243,"close_b":116.650375,"volume_b":3571400,"open_bench":303.611485,"close_bench":303.67514},{"date":"2020-02-07","open_a":48.047863,"high_a":48.406916,"low_a":48.02338,"close_a":48.300831,"volume_a":8757800,"open_b":116.593834,"high_b":117.773841,"low_b":116.246306,"close_b":117.490959,"volume_b":3138500,"open_bench":302.620439,"close_bench":302.056702},{"date":"2020-02-10","open_a":48.406908,"high_a":48.831244,"low_a":48.382428,"close_a":48.757801,"volume_a":7984900,"open_b":117.943552,"high_b":118.339585,"low_b":117.296986,"close_b":117.725342,"volume_b":3775600,"open_bench":301.17471,"close_bench":304.311646},{"date":"2020-02-11","open_a":48.733316,"high_a":48.798597,"low_a":48.406903,"close_a":48.627232,"volume_a":7431200,"open_b":118.169846,"high_b":118.291074,"low_b":117.086822,"close_b":118.064774,"volume_b":3870500,"open_bench":305.657319,"close_bench":304.838989},{"date":"2020-02-12","open_a":48.537472,"high_a":48.774122,"low_a":48.406908,"close_a":48.48035,"volume_a":9179100,"open_b":118.549698,"high_b":118.598202,"low_b":117.474776,"close_b":118.064774,"volume_b":4217400,"open_bench":306.26652,"close_bench":306.803009},{"date":"2020-02-13","open_a":48.455872,"high_a":48.831245,"low_a":48.317145,"close_a":48.643559,"volume_a":8173200,"open_b":117.39396,"high_b":118.913418,"low_b":115.995743,"close_b":118.37999,"volume_b":5041400,"open_bench":305.384461,"close_bench":306.475586},{"date":"2020-02-14","open_a":48.659883,"high_a":49.018936,"low_a":48.651721,"close_a":48.921013,"volume_a":9965500,"open_b":118.460796,"high_b":118.929553,"low_b":117.951614,"close_b":118.800247,"volume_b":3555600,"open_bench":306.884814,"close_bench":306.966644},{"date":"2020-02-18","open_a":48.806776,"high_a":48.904699,"low_a":48.513004,"close_a":48.578285,"volume_a":10075700,"open_b":118.897273,"high_b":118.97001,"low_b":117.515211,"close_b":117.676865,"volume_b":4938700,"open_bench":305.975621,"close_bench":306.175659},{"date":"2020-02-19","open_a":48.561962,"high_a":48.961818,"low_a":48.529323,"close_a":48.774132,"volume_a":6265700,"open_b":117.692991,"high_b":118.388061,"low_b":117.587931,"close_b":117.620255,"volume_b":2978800,"open_bench":307.139383,"close_bench":307.639465},{"date":"2020-02-20","open_a":48.700684,"high_a":48.84757,"low_a":48.235547,"close_a":48.733326,"volume_a":7597200,"open_b":117.814235,"high_b":117.838481,"low_b":116.343269,"close_b":117.32122,"volume_b":2805900,"open_bench":307.093999,"close_bench":306.375702},{"date":"2020-02-21","open_a":48.561953,"high_a":49.067894,"low_a":48.496672,"close_a":49.067894,"volume_a":10842400,"open_b":117.240405,"high_b":117.935475,"low_b":117.03835,"close_b":117.878906,"volume_b":4271800,"open_bench":305.029941,"close_bench":303.22052},{"date":"2020-02-24","open_a":48.235549,"high_a":48.798609,"low_a":47.794892,"close_a":47.860176,"volume_a":14415500,"open_b":117.886977,"high_b":118.929576,"low_b":115.219849,"close_b":115.284508,"volume_b":5027700,"open_bench":293.818762,"close_bench":293.164093},{"date":"2020-02-25","open_a":47.900973,"high_a":48.308988,"low_a":47.019658,"close_a":47.182865,"volume_a":15915100,"open_b":115.244111,"high_b":116.690832,"low_b":113.611516,"close_b":114.039871,"volume_b":7585800,"open_bench":294.546148,"close_bench":284.280579},{"date":"2020-02-26","open_a":47.31343,"high_a":47.713286,"low_a":46.905415,"close_a":47.003338,"volume_a":15837800,"open_b":115.122908,"high_b":116.181675,"low_b":114.088387,"close_b":114.751122,"volume_b":6157200,"open_bench":285.671769,"close_bench":283.234955},{"date":"2020-02-27","open_a":46.636135,"high_a":47.239996,"low_a":44.783746,"close_a":44.824547,"volume_a":23754000,"open_b":114.290385,"high_b":115.688602,"low_b":110.168458,"close_b":110.216949,"volume_b":8605000,"open_bench":277.742985,"close_bench":270.514374},{"date":"2020-02-28","open_a":43.926903,"high_a":43.975862,"low_a":42.090835,"close_a":43.649452,"volume_a":40890600,"open_b":107.687257,"high_b":108.099444,"low_b":103.735061,"close_b":106.709305,"volume_b":14164000,"open_bench":262.503793,"close_bench":269.377808},{"date":"2020-03-02","open_a":44.00851,"high_a":45.66505,"low_a":43.820823,"close_a":45.632408,"volume_a":25698600,"open_b":106.685054,"high_b":111.348487,"low_b":106.458753,"close_b":111.194923,"volume_b":8504500,"open_bench":271.150841,"close_bench":281.04361},{"date":"2020-03-03","open_a":46.317869,"high_a":46.913574,"low_a":45.403916,"close_a":45.746651,"volume_a":27099200,"open_b":110.879711,"high_b":112.884093,"low_b":108.899574,"close_b":109.578476,"volume_b":8728300,"open_bench":281.416381,"close_bench":272.996613},{"date":"2020-03-04","open_a":46.032263,"high_a":48.145782,"low_a":46.032263,"close_a":48.080498,"volume_a":20526100,"open_b":110.071474,"high_b":115.341072,"low_b":109.925988,"close_b":115.082436,"volume_b":7635700,"open_bench":278.343114,"close_bench":284.471527},{"date":"2020-03-05","open_a":47.16655,"high_a":47.46848,"low_a":45.85274,"close_a":46.301559,"volume_a":17948500,"open_b":112.946539,"high_b":115.070238,"low_b":111.205271,"close_b":112.368835,"volume_b":6579100,"open_bench":277.306523,"close_bench":275.015167},{"date":"2020-03-06","open_a":44.832701,"high_a":45.624251,"low_a":44.049312,"close_a":45.09383,"volume_a":22353900,"open_b":107.047397,"high_b":112.425806,"low_b":106.957892,"close_b":111.685356,"volume_b":6754900,"open_bench":266.549988,"close_bench":270.468903},{"date":"2020-03-09","open_a":42.474369,"high_a":43.722897,"low_a":42.009232,"close_a":42.319324,"volume_a":27898300,"open_b":105.924529,"high_b":108.503897,"low_b":104.297174,"close_b":105.908264,"volume_b":8716300,"open_bench":250.319736,"close_bench":249.346848},{"date":"2020-03-10","open_a":43.429122,"high_a":43.902417,"low_a":41.756257,"close_a":43.788174,"volume_a":27457400,"open_b":107.446108,"high_b":109.496578,"low_b":103.22312,"close_b":109.14669,"volume_b":8258600,"open_bench":258.812139,"close_bench":262.249146},{"date":"2020-03-11","open_a":42.719182,"high_a":42.963994,"low_a":41.731786,"close_a":42.604939,"volume_a":22653900,"open_b":106.25,"high_b":107.844801,"low_b":103.849648,"close_b":105.574646,"volume_b":7465100,"open_bench":255.229708,"close_bench":249.464966},{"date":"2020-03-12","open_a":39.610109,"high_a":40.74439,"low_a":38.100451,"close_a":38.483986,"volume_a":31913100,"open_b":99.350017,"high_b":101.709683,"low_b":92.42562,"close_b":93.849556,"volume_b":12808300,"open_bench":232.770919,"close_bench":225.596848},{"date":"2020-03-13","open_a":40.755976,"high_a":40.797138,"low_a":38.368736,"close_a":39.899864,"volume_a":33697200,"open_b":98.194607,"high_b":103.817116,"low_b":93.629875,"close_b":103.703201,"volume_b":12007700,"open_bench":239.2176,"close_bench":244.882309},{"date":"2020-03-16","open_a":34.77141,"high_a":39.414183,"low_a":33.890597,"close_a":37.257431,"volume_a":28682300,"open_b":93.320665,"high_b":100.749537,"low_b":91.131871,"close_b":92.018776,"volume_b":10733300,"open_bench":219.295686,"close_bench":218.08638},{"date":"2020-03-17","open_a":37.767817,"high_a":39.101379,"low_a":37.34799,"close_a":38.837959,"volume_a":32106700,"open_b":94.956146,"high_b":104.036789,"low_b":94.703908,"close_b":103.922874,"volume_b":14242300,"open_bench":222.805435,"close_bench":229.861313},{"date":"2020-03-18","open_a":36.829378,"high_a":37.166884,"low_a":34.236338,"close_a":36.919926,"volume_a":35408300,"open_b":98.324751,"high_b":103.922854,"low_b":92.572054,"close_b":98.389847,"volume_b":15250800,"open_bench":214.813047,"close_bench":218.222778},{"date":"2020-03-19","open_a":36.516584,"high_a":36.76354,"low_a":33.783603,"close_a":34.433922,"volume_a":41956900,"open_b":98.804858,"high_b":99.341882,"low_b":94.435407,"close_b":95.476913,"volume_b":9393500,"open_bench":217.540813,"close_bench":218.686478},{"date":"2020-03-20","open_a":34.532686,"high_a":34.631468,"low_a":30.877735,"close_a":31.528053,"volume_a":51513500,"open_b":96.209221,"high_b":96.380092,"low_b":82.523166,"close_b":84.565498,"volume_b":13093200,"open_bench":221.81991,"close_bench":209.262344},{"date":"2020-03-23","open_a":31.906717,"high_a":32.013732,"low_a":29.856984,"close_a":30.918896,"volume_a":44978500,"open_b":86.249811,"high_b":91.839778,"low_b":83.703003,"close_b":85.582596,"volume_b":13926800,"open_bench":208.704432,"close_bench":203.91188},{"date":"2020-03-24","open_a":33.931757,"high_a":34.079931,"low_a":31.964341,"close_a":32.474716,"volume_a":49096200,"open_b":89.447575,"high_b":93.109124,"low_b":88.991918,"close_b":92.629051,"volume_b":11447000,"open_bench":214.402452,"close_bench":222.386978},{"date":"2020-03-25","open_a":33.446077,"high_a":35.767464,"low_a":32.853382,"close_a":34.2528,"volume_a":42511700,"open_b":91.733985,"high_b":94.679499,"low_b":90.350736,"close_b":91.538704,"volume_b":9485500,"open_bench":223.960049,"close_bench":225.716095},{"date":"2020-03-26","open_a":35.265328,"high_a":37.265673,"low_a":34.944286,"close_a":36.45895,"volume_a":37821700,"open_b":91.709585,"high_b":99.105916,"low_b":91.603809,"close_b":97.852852,"volume_b":10376800,"open_bench":228.213059,"close_bench":238.895691},{"date":"2020-03-27","open_a":35.940339,"high_a":36.244917,"low_a":34.903125,"close_a":35.240631,"volume_a":23297900,"open_b":96.477722,"high_b":101.424882,"low_b":96.005788,"close_b":98.015572,"volume_b":9852500,"open_bench":231.642844,"close_bench":231.780029},{"date":"2020-03-30","open_a":35.429962,"high_a":37.504393,"low_a":35.429962,"close_a":37.043407,"volume_a":23446000,"open_b":98.951287,"high_b":102.393138,"low_b":98.86992,"close_b":102.10022,"volume_b":6916800,"open_bench":233.865305,"close_bench":239.30722},{"date":"2020-03-31","open_a":37.101031,"high_a":37.463232,"low_a":36.269613,"close_a":36.426018,"volume_a":25373900,"open_b":101.660866,"high_b":102.279257,"low_b":97.258868,"close_b":97.722664,"volume_b":7091700,"open_bench":238.310296,"close_bench":235.74025},{"date":"2020-04-01","open_a":34.96897,"high_a":35.9321,"low_a":34.409203,"close_a":34.672623,"volume_a":21936400,"open_b":95.419942,"high_b":99.512744,"low_b":94.118059,"close_b":96.111572,"volume_b":6873000,"open_bench":226.804533,"close_bench":225.130798},{"date":"2020-04-02","open_a":34.754953,"high_a":36.376629,"low_a":34.458605,"close_a":36.179066,"volume_a":17621900,"open_b":95.379271,"high_b":101.856143,"low_b":94.777152,"close_b":100.782089,"volume_b":6181800,"open_bench":224.252778,"close_bench":230.325775},{"date":"2020-04-03","open_a":35.87448,"high_a":36.5001,"low_a":35.652219,"close_a":36.080276,"volume_a":14207100,"open_b":100.236932,"high_b":102.010749,"low_b":99.626674,"close_b":101.376076,"volume_b":5552000,"open_bench":229.347148,"close_bench":226.996613},{"date":"2020-04-06","open_a":37.364443,"high_a":38.698005,"low_a":36.788212,"close_a":38.418121,"volume_a":22850000,"open_b":104.687745,"high_b":107.511207,"low_b":101.94565,"close_b":106.721939,"volume_b":8242200,"open_bench":235.822564,"close_bench":242.243103},{"date":"2020-04-07","open_a":39.924554,"high_a":40.006872,"low_a":38.261717,"close_a":38.286411,"volume_a":21279600,"open_b":107.983119,"high_b":109.91153,"low_b":106.014009,"close_b":106.16861,"volume_b":6253900,"open_bench":250.794759,"close_bench":242.490128},{"date":"2020-04-08","open_a":38.393427,"high_a":39.587049,"low_a":38.138242,"close_a":39.364788,"volume_a":18937100,"open_b":105.485163,"high_b":108.455082,"low_b":105.070192,"close_b":107.901787,"volume_b":5241100,"open_bench":245.078403,"close_bench":250.630081},{"date":"2020-04-09","open_a":39.875165,"high_a":40.937077,"low_a":39.801078,"close_a":40.336151,"volume_a":19165300,"open_b":107.478635,"high_b":109.911534,"low_b":107.0067,"close_b":108.731705,"volume_b":5502400,"open_bench":253.876935,"close_bench":254.444016},{"date":"2020-04-13","open_a":39.801076,"high_a":40.039801,"low_a":38.278182,"close_a":38.632153,"volume_a":19586000,"open_b":107.153157,"high_b":108.503864,"low_b":105.314238,"close_b":106.502213,"volume_b":3785000,"open_bench":253.474556,"close_bench":252.120926},{"date":"2020-04-14","open_a":39.463583,"high_a":40.541959,"low_a":38.632165,"close_a":40.270306,"volume_a":21801900,"open_b":108.398117,"high_b":112.800116,"low_b":107.690222,"close_b":112.612961,"volume_b":6278700,"open_bench":256.986624,"close_bench":259.556671},{"date":"2020-04-15","open_a":39.274246,"high_a":39.842247,"low_a":39.027291,"close_a":39.191929,"volume_a":12893200,"open_b":111.587702,"high_b":111.929445,"low_b":108.78865,"close_b":109.870842,"volume_b":5773100,"open_bench":253.867757,"close_bench":254.041534},{"date":"2020-04-16","open_a":39.019053,"high_a":39.389485,"low_a":38.492211,"close_a":38.772095,"volume_a":17606800,"open_b":111.449395,"high_b":111.449395,"low_b":108.910726,"close_b":110.367203,"volume_b":4564000,"open_bench":255.312855,"close_bench":255.267136},{"date":"2020-04-17","open_a":39.759922,"high_a":39.759922,"low_a":38.994358,"close_a":39.562359,"volume_a":19844600,"open_b":112.067782,"high_b":112.287478,"low_b":109.773218,"close_b":111.921326,"volume_b":4866500,"open_bench":261.010945,"close_bench":262.163361},{"date":"2020-04-20","open_a":39.340096,"high_a":39.512965,"low_a":38.12178,"close_a":38.302879,"volume_a":18957900,"open_b":111.424989,"high_b":111.78301,"low_b":109.447747,"close_b":109.480301,"volume_b":3995400,"open_bench":258.477385,"close_bench":257.544495},{"date":"2020-04-21","open_a":37.767818,"high_a":38.607469,"low_a":36.434252,"close_a":37.356224,"volume_a":26863700,"open_b":107.519333,"high_b":108.170278,"low_b":106.258131,"close_b":106.542923,"volume_b":4359500,"open_bench":253.099547,"close_bench":249.72464},{"date":"2020-04-22","open_a":37.907754,"high_a":37.990075,"low_a":37.455002,"close_a":37.603176,"volume_a":15811600,"open_b":107.763441,"high_b":108.764261,"low_b":106.770753,"close_b":107.909897,"volume_b":4084600,"open_bench":254.58118,"close_bench":255.267136},{"date":"2020-04-23","open_a":37.825434,"high_a":37.94891,"low_a":37.018711,"close_a":37.101028,"volume_a":17668200,"open_b":107.812251,"high_b":109.146681,"low_b":105.802466,"close_b":105.989609,"volume_b":5527700,"open_bench":256.538441,"close_bench":255.24884},{"date":"2020-04-24","open_a":37.455004,"high_a":37.6608,"low_a":37.142192,"close_a":37.397381,"volume_a":15175000,"open_b":106.485938,"high_b":109.699973,"low_b":106.306928,"close_b":109.325676,"volume_b":4063100,"open_bench":256.757988,"close_bench":258.806702},{"date":"2020-04-27","open_a":37.86659,"high_a":38.673313,"low_a":37.627865,"close_a":38.508675,"volume_a":19175500,"open_b":109.374543,"high_b":110.017355,"low_b":108.943295,"close_b":109.407097,"volume_b":4009200,"open_bench":260.772991,"close_bench":262.538177},{"date":"2020-04-28","open_a":39.430654,"high_a":39.512972,"low_a":38.335814,"close_a":38.475758,"volume_a":17246700,"open_b":111.294794,"high_b":112.027099,"low_b":109.569802,"close_b":110.920509,"volume_b":6674200,"open_bench":266.169282,"close_bench":261.331024},{"date":"2020-04-29","open_a":38.912027,"high_a":39.397707,"low_a":38.590985,"close_a":38.788548,"volume_a":15780000,"open_b":111.286644,"high_b":111.457509,"low_b":108.137704,"close_b":108.512001,"volume_b":6706800,"open_bench":266.635706,"close_bench":268.172241},{"date":"2020-04-30","open_a":38.689781,"high_a":38.689781,"low_a":37.619639,"close_a":37.776043,"volume_a":20801100,"open_b":107.738999,"high_b":108.35739,"low_b":106.119777,"close_b":107.64135,"volume_b":6783000,"open_bench":266.800304,"close_bench":265.675354},{"date":"2020-05-01","open_a":37.553787,"high_a":38.187641,"low_a":37.216281,"close_a":37.537323,"volume_a":14290400,"open_b":106.689383,"high_b":107.592566,"low_b":105.387507,"close_b":105.891983,"volume_b":4488000,"open_bench":260.946924,"close_bench":258.64212},{"date":"2020-05-04","open_a":37.306831,"high_a":37.438541,"low_a":36.714135,"close_a":37.158657,"volume_a":22544200,"open_b":105.916369,"high_b":106.876514,"low_b":105.22475,"close_b":106.526627,"volume_b":5245300,"open_bench":256.767082,"close_bench":259.355438},{"date":"2020-05-05","open_a":37.33976,"high_a":37.948917,"low_a":37.166892,"close_a":37.372688,"volume_a":13370500,"open_b":107.112493,"high_b":108.487609,"low_b":106.404597,"close_b":107.136902,"volume_b":3580000,"open_bench":262.163386,"close_bench":261.751801},{"date":"2020-05-06","open_a":37.372685,"high_a":37.562017,"low_a":36.812918,"close_a":36.837612,"volume_a":11892900,"open_b":107.063679,"high_b":107.592565,"low_b":106.339507,"close_b":106.412735,"volume_b":4192200,"open_bench":263.443759,"close_bench":259.977386},{"date":"2020-05-07","open_a":37.166886,"high_a":37.240973,"low_a":36.631813,"close_a":36.71413,"volume_a":16466000,"open_b":107.039261,"high_b":107.405413,"low_b":105.875696,"close_b":107.039261,"volume_b":5528000,"open_bench":263.178501,"close_bench":263.114471},{"date":"2020-05-08","open_a":37.109257,"high_a":38.047689,"low_a":36.944619,"close_a":37.957138,"volume_a":13985100,"open_b":108.276053,"high_b":109.976636,"low_b":107.77971,"close_b":109.21991,"volume_b":4209100,"open_bench":266.233291,"close_bench":267.468018},{"date":"2020-05-11","open_a":37.611399,"high_a":37.957139,"low_a":37.471459,"close_a":37.487923,"volume_a":10874600,"open_b":108.568992,"high_b":110.822882,"low_b":108.35744,"close_b":109.341995,"volume_b":3591100,"open_bench":265.547422,"close_bench":267.52298},{"date":"2020-05-12","open_a":37.537318,"high_a":37.743115,"low_a":36.878769,"close_a":36.895233,"volume_a":13424400,"open_b":109.651159,"high_b":110.969325,"low_b":108.105177,"close_b":108.170273,"volume_b":3914600,"open_bench":268.702866,"close_bench":262.190857},{"date":"2020-05-13","open_a":36.804676,"high_a":36.862299,"low_a":35.825085,"close_a":36.170822,"volume_a":15854800,"open_b":108.365529,"high_b":109.415179,"low_b":106.95787,"close_b":108.186531,"volume_b":5168500,"open_bench":261.632763,"close_bench":257.553619},{"date":"2020-05-14","open_a":35.709843,"high_a":36.269611,"low_a":35.586364,"close_a":35.973263,"volume_a":18000800,"open_b":107.9831,"high_b":108.674718,"low_b":105.900076,"close_b":108.284157,"volume_b":5522200,"open_bench":255.130023,"close_bench":260.635956},{"date":"2020-05-15","open_a":36.187281,"high_a":36.788207,"low_a":35.56166,"close_a":35.61105,"volume_a":31579400,"open_b":108.84562,"high_b":112.010824,"low_b":108.048207,"close_b":110.619431,"volume_b":20338200,"open_bench":258.257864,"close_bench":261.833984},{"date":"2020-05-18","open_a":36.895232,"high_a":37.364448,"low_a":36.327234,"close_a":37.018711,"volume_a":26118500,"open_b":111.351739,"high_b":112.27933,"low_b":109.89525,"close_b":110.269547,"volume_b":8480300,"open_bench":268.025951,"close_bench":269.809448},{"date":"2020-05-19","open_a":37.051641,"high_a":37.315064,"low_a":36.42602,"close_a":36.664745,"volume_a":17839900,"open_b":109.382649,"high_b":109.651164,"low_b":106.917207,"close_b":107.185722,"volume_b":6640300,"open_bench":269.214857,"close_bench":267.038086},{"date":"2020-05-20","open_a":37.208045,"high_a":38.006534,"low_a":37.059871,"close_a":37.776043,"volume_a":22295900,"open_b":107.844797,"high_b":107.958712,"low_b":106.05471,"close_b":106.803291,"volume_b":8213100,"open_bench":270.559383,"close_bench":271.574585},{"date":"2020-05-21","open_a":37.932449,"high_a":37.940679,"low_a":37.0681,"close_a":37.183346,"volume_a":13193500,"open_b":106.575459,"high_b":106.949756,"low_b":105.460725,"close_b":105.948936,"volume_b":4886300,"open_bench":271.446551,"close_bench":269.699646},{"date":"2020-05-22","open_a":37.224511,"high_a":37.298598,"low_a":36.755295,"close_a":37.068104,"volume_a":13008400,"open_b":105.460714,"high_b":106.290669,"low_b":105.102694,"close_b":106.16861,"volume_b":3429200,"open_bench":269.416118,"close_bench":270.211823},{"date":"2020-05-26","open_a":38.623909,"high_a":38.632143,"low_a":37.808956,"close_a":37.940666,"volume_a":22223100,"open_b":107.356602,"high_b":107.446107,"low_b":105.289854,"close_b":105.574646,"volume_b":5579300,"open_bench":276.147716,"close_bench":273.541077},{"date":"2020-05-27","open_a":38.689786,"high_a":38.763873,"low_a":38.00654,"close_a":38.467525,"volume_a":18447600,"open_b":105.550213,"high_b":106.461528,"low_b":104.622622,"close_b":106.437119,"volume_b":5686500,"open_bench":276.321422,"close_bench":277.611023},{"date":"2020-05-28","open_a":38.936735,"high_a":39.068445,"low_a":38.541606,"close_a":38.763866,"volume_a":16230600,"open_b":106.762589,"high_b":107.99937,"low_b":105.867539,"close_b":107.64135,"volume_b":4974400,"open_bench":278.635474,"close_bench":277.098938},{"date":"2020-05-29","open_a":38.426365,"high_a":38.846189,"low_a":37.981843,"close_a":38.426365,"volume_a":51125000,"open_b":107.413546,"high_b":107.6007,"low_b":106.038441,"close_b":107.039261,"volume_b":13337900,"open_bench":276.632371,"close_bench":278.333557},{"date":"2020-06-01","open_a":38.409888,"high_a":38.879104,"low_a":38.352266,"close_a":38.681541,"volume_a":17010700,"open_b":107.079919,"high_b":108.992053,"low_b":106.746309,"close_b":108.21093,"volume_b":4049200,"open_bench":277.693339,"close_bench":279.458527},{"date":"2020-06-02","open_a":38.821483,"high_a":38.920265,"low_a":38.327573,"close_a":38.607456,"volume_a":15255100,"open_b":108.259773,"high_b":108.878163,"low_b":106.689369,"close_b":107.91803,"volume_b":5327500,"open_bench":280.373142,"close_bench":281.772491},{"date":"2020-06-03","open_a":38.912036,"high_a":39.611744,"low_a":38.788557,"close_a":39.430645,"volume_a":15508300,"open_b":108.414375,"high_b":109.260594,"low_b":107.893622,"close_b":108.072632,"volume_b":5421200,"open_bench":283.748118,"close_bench":285.522461},{"date":"2020-06-04","open_a":39.463573,"high_a":39.735226,"low_a":39.002591,"close_a":39.447109,"volume_a":13604700,"open_b":107.902905,"high_b":109.518292,"low_b":106.746711,"close_b":107.066521,"volume_b":5229700,"open_bench":284.543748,"close_bench":284.7724},{"date":"2020-06-05","open_a":39.957488,"high_a":40.739516,"low_a":39.891632,"close_a":40.41024,"volume_a":20504800,"open_b":107.4191,"high_b":109.485486,"low_b":107.230505,"close_b":108.599892,"volume_b":7075700,"open_bench":290.141204,"close_bench":292.071014},{"date":"2020-06-08","open_a":40.780676,"high_a":41.134647,"low_a":40.723053,"close_a":41.035862,"volume_a":19896700,"open_b":107.730712,"high_b":108.624502,"low_b":107.615914,"close_b":108.411308,"volume_b":5613000,"open_bench":292.875807,"close_bench":295.601349},{"date":"2020-06-09","open_a":40.821831,"high_a":40.821831,"low_a":40.072732,"close_a":40.336151,"volume_a":16713700,"open_b":108.099686,"high_b":109.18207,"low_b":107.460092,"close_b":109.034477,"volume_b":6446900,"open_bench":292.94904,"close_bench":293.397217},{"date":"2020-06-10","open_a":40.467864,"high_a":40.558415,"low_a":39.982183,"close_a":40.023342,"volume_a":15716600,"open_b":109.288696,"high_b":110.830287,"low_b":109.198496,"close_b":109.985695,"volume_b":7069900,"open_bench":293.973374,"close_bench":291.76001},{"date":"2020-06-11","open_a":39.356557,"high_a":39.447105,"low_a":37.422066,"close_a":37.487923,"volume_a":26070400,"open_b":110.059479,"high_b":110.813871,"low_b":104.516324,"close_b":104.827919,"volume_b":8541000,"open_bench":284.86381,"close_bench":274.940308},{"date":"2020-06-12","open_a":38.385052,"high_a":38.418279,"low_a":37.388253,"close_a":37.878345,"volume_a":21822700,"open_b":105.902115,"high_b":106.541709,"low_b":104.491732,"close_b":105.779121,"volume_b":7039500,"open_bench":281.91881,"close_bench":278.232941},{"date":"2020-06-15","open_a":37.022761,"high_a":38.584413,"low_a":36.939696,"close_a":38.459812,"volume_a":15622300,"open_b":105.196897,"high_b":107.386289,"low_b":103.753714,"close_b":106.992683,"volume_b":5481000,"open_bench":272.571563,"close_bench":280.830475},{"date":"2020-06-16","open_a":39.50645,"high_a":39.614434,"low_a":38.368436,"close_a":38.850224,"volume_a":18803500,"open_b":108.796687,"high_b":109.108288,"low_b":106.697506,"close_b":107.968498,"volume_b":5197700,"open_bench":288.54064,"close_bench":286.235809},{"date":"2020-06-17","open_a":38.858521,"high_a":39.091109,"low_a":38.576095,"close_a":38.69239,"volume_a":12602800,"open_b":108.148885,"high_b":109.091872,"low_b":107.820893,"close_b":108.042282,"volume_b":3309700,"open_bench":287.250992,"close_bench":285.046783},{"date":"2020-06-18","open_a":38.459809,"high_a":39.074503,"low_a":38.268757,"close_a":39.03297,"volume_a":9380800,"open_b":107.763479,"high_b":109.165672,"low_b":107.11569,"close_b":108.87867,"volume_b":3417200,"open_bench":283.537587,"close_bench":285.156433},{"date":"2020-06-19","open_a":39.680891,"high_a":39.697505,"low_a":38.227226,"close_a":38.227226,"volume_a":30464600,"open_b":110.215272,"high_b":110.551472,"low_b":107.599492,"close_b":107.648689,"volume_b":8310100,"open_bench":288.606956,"close_bench":283.526917},{"date":"2020-06-22","open_a":38.235519,"high_a":38.335202,"low_a":37.695589,"close_a":37.994629,"volume_a":20876200,"open_b":107.050094,"high_b":108.042279,"low_b":106.558093,"close_b":107.460091,"volume_b":3379900,"open_bench":282.929785,"close_bench":285.345795},{"date":"2020-06-23","open_a":38.210608,"high_a":38.451501,"low_a":37.86173,"close_a":37.911568,"volume_a":14675300,"open_b":107.927473,"high_b":108.796663,"low_b":107.402678,"close_b":107.714279,"volume_b":3625600,"open_bench":287.982196,"close_bench":286.659363},{"date":"2020-06-24","open_a":37.687284,"high_a":37.74543,"low_a":36.964606,"close_a":37.089207,"volume_a":17631800,"open_b":107.025495,"high_b":107.476494,"low_b":105.516699,"close_b":106.238297,"volume_b":4853500,"open_bench":284.629247,"close_bench":279.347107},{"date":"2020-06-25","open_a":36.939687,"high_a":37.363325,"low_a":36.574193,"close_a":37.321793,"volume_a":13829700,"open_b":105.893907,"high_b":108.017686,"low_b":105.000118,"close_b":107.755295,"volume_b":4442000,"open_bench":278.777557,"close_bench":282.341858},{"date":"2020-06-26","open_a":37.31349,"high_a":37.38825,"low_a":36.142252,"close_a":36.192093,"volume_a":21955100,"open_b":107.976707,"high_b":108.542504,"low_b":105.008335,"close_b":105.721725,"volume_b":7372700,"open_bench":281.248603,"close_bench":275.635742},{"date":"2020-06-29","open_a":36.54928,"high_a":37.022759,"low_a":36.17548,"close_a":36.84832,"volume_a":17603900,"open_b":106.746694,"high_b":107.747087,"low_b":106.131699,"close_b":107.484695,"volume_b":4173900,"open_bench":276.885162,"close_bench":279.686981},{"date":"2020-06-30","open_a":36.756942,"high_a":37.238729,"low_a":36.582503,"close_a":37.114128,"volume_a":23679800,"open_b":108.099687,"high_b":108.911485,"low_b":107.296097,"close_b":108.452278,"volume_b":4626700,"open_bench":279.255232,"close_bench":283.269653},{"date":"2020-07-01","open_a":37.321797,"high_a":37.820195,"low_a":37.139049,"close_a":37.230423,"volume_a":14316500,"open_b":108.673691,"high_b":109.567481,"low_b":108.189899,"close_b":108.534294,"volume_b":3633000,"open_bench":284.381222,"close_bench":285.253906},{"date":"2020-07-02","open_a":37.620831,"high_a":37.737123,"low_a":37.213803,"close_a":37.280258,"volume_a":15111900,"open_b":109.501889,"high_b":110.018488,"low_b":108.681895,"close_b":108.936104,"volume_b":2769200,"open_bench":288.671293,"close_bench":286.82486},{"date":"2020-07-06","open_a":37.496232,"high_a":37.604219,"low_a":37.05598,"close_a":37.570992,"volume_a":15146000,"open_b":109.12467,"high_b":109.838059,"low_b":108.493271,"close_b":109.305069,"volume_b":4221600,"open_bench":290.627991,"close_bench":291.252655},{"date":"2020-07-07","open_a":37.37994,"high_a":37.795272,"low_a":37.213806,"close_a":37.554379,"volume_a":13043600,"open_b":108.460496,"high_b":110.387472,"low_b":108.378491,"close_b":109.526489,"volume_b":4424500,"open_bench":289.718537,"close_bench":288.248718},{"date":"2020-07-08","open_a":37.496238,"high_a":37.529465,"low_a":36.939693,"close_a":37.438091,"volume_a":13851200,"open_b":110.010258,"high_b":110.444854,"low_b":109.190265,"close_b":110.059456,"volume_b":3798200,"open_bench":289.011109,"close_bench":290.453369},{"date":"2020-07-09","open_a":37.421472,"high_a":37.421472,"low_a":36.408062,"close_a":36.474514,"volume_a":16087100,"open_b":108.977093,"high_b":109.829892,"low_b":108.017702,"close_b":108.821304,"volume_b":4005200,"open_bench":291.059664,"close_bench":288.799835},{"date":"2020-07-10","open_a":36.449598,"high_a":37.587609,"low_a":36.432984,"close_a":37.504543,"volume_a":15627800,"open_b":108.542514,"high_b":110.608901,"low_b":107.648724,"close_b":110.25631,"volume_b":5932800,"open_bench":288.735572,"close_bench":291.748688},{"date":"2020-07-13","open_a":37.820186,"high_a":37.944787,"low_a":37.346707,"close_a":37.587601,"volume_a":16705300,"open_b":112.150455,"high_b":113.216448,"low_b":110.264468,"close_b":110.625267,"volume_b":8960000,"open_bench":294.081989,"close_bench":289.222412},{"date":"2020-07-14","open_a":37.305174,"high_a":38.160758,"low_a":37.105813,"close_a":38.102612,"volume_a":17080100,"open_b":110.36285,"high_b":112.298034,"low_b":110.084056,"close_b":111.12545,"volume_b":6766800,"open_bench":287.807748,"close_bench":292.97049},{"date":"2020-07-15","open_a":38.459805,"high_a":39.199097,"low_a":38.37674,"close_a":38.542873,"volume_a":23738000,"open_b":110.986062,"high_b":111.674853,"low_b":109.264071,"close_b":109.62487,"volume_b":5561300,"open_bench":296.176419,"close_bench":295.661987},{"date":"2020-07-16","open_a":38.426576,"high_a":38.783762,"low_a":38.160763,"close_a":38.335205,"volume_a":13306100,"open_b":109.928293,"high_b":110.174294,"low_b":109.100105,"close_b":109.780701,"volume_b":3788100,"open_bench":293.769691,"close_bench":294.688324},{"date":"2020-07-17","open_a":38.601026,"high_a":38.966518,"low_a":38.3186,"close_a":38.891758,"volume_a":13899800,"open_b":110.412088,"high_b":110.748288,"low_b":109.510103,"close_b":110.420296,"volume_b":3229300,"open_bench":295.689618,"close_bench":295.542633},{"date":"2020-07-20","open_a":38.808689,"high_a":38.900062,"low_a":38.086007,"close_a":38.310287,"volume_a":16633200,"open_b":110.379251,"high_b":110.379251,"low_b":108.386662,"close_b":109.149261,"volume_b":3620300,"open_bench":295.276168,"close_bench":297.93103},{"date":"2020-07-21","open_a":39.473218,"high_a":39.863631,"low_a":38.983125,"close_a":39.207405,"volume_a":25681500,"open_b":109.879083,"high_b":111.85527,"low_b":109.706879,"close_b":110.231674,"volume_b":5435400,"open_bench":299.887714,"close_bench":298.56488},{"date":"2020-07-22","open_a":39.813795,"high_a":40.303888,"low_a":39.066195,"close_a":40.27066,"volume_a":29831200,"open_b":111.158273,"high_b":111.72407,"low_b":109.616682,"close_b":111.527267,"volume_b":3522300,"open_bench":298.206736,"close_bench":300.264465},{"date":"2020-07-23","open_a":40.262359,"high_a":40.345424,"low_a":39.896864,"close_a":40.10453,"volume_a":15935000,"open_b":112.002832,"high_b":112.54403,"low_b":111.100847,"close_b":112.404633,"volume_b":5011400,"open_bench":299.906143,"close_bench":296.681732},{"date":"2020-07-24","open_a":40.204216,"high_a":40.536481,"low_a":39.930095,"close_a":40.278976,"volume_a":16196800,"open_b":112.33086,"high_b":113.683844,"low_b":111.018866,"close_b":111.56826,"volume_b":3300000,"open_bench":294.835246,"close_bench":294.770935},{"date":"2020-07-27","open_a":40.021461,"high_a":40.295579,"low_a":40.021461,"close_a":40.27066,"volume_a":17346500,"open_b":111.133658,"high_b":112.962239,"low_b":110.576069,"close_b":112.888443,"volume_b":3607700,"open_bench":295.459909,"close_bench":296.920532},{"date":"2020-07-28","open_a":40.154364,"high_a":40.935188,"low_a":39.955003,"close_a":40.021458,"volume_a":13872700,"open_b":113.142638,"high_b":113.642835,"low_b":112.412845,"close_b":112.65065,"volume_b":3787500,"open_bench":296.194904,"close_bench":295.037445},{"date":"2020-07-29","open_a":39.988225,"high_a":40.287265,"low_a":39.722412,"close_a":39.888546,"volume_a":13758100,"open_b":112.650637,"high_b":113.979022,"low_b":112.37183,"close_b":113.101624,"volume_b":3179000,"open_bench":295.910086,"close_bench":298.665985},{"date":"2020-07-30","open_a":39.597808,"high_a":40.062982,"low_a":39.207398,"close_a":39.614422,"volume_a":17276500,"open_b":112.535882,"high_b":113.339472,"low_b":111.912679,"close_b":112.904877,"volume_b":2817600,"open_bench":295.70793,"close_bench":297.600311},{"date":"2020-07-31","open_a":39.406765,"high_a":39.680887,"low_a":38.816994,"close_a":39.240635,"volume_a":14849200,"open_b":112.552252,"high_b":112.913051,"low_b":110.863068,"close_b":112.880257,"volume_b":4304100,"open_bench":299.382448,"close_bench":299.951996},{"date":"2020-08-03","open_a":39.157571,"high_a":39.340319,"low_a":38.401666,"close_a":38.459812,"volume_a":14137500,"open_b":112.429241,"high_b":112.749037,"low_b":111.519048,"close_b":112.09304,"volume_b":2819200,"open_bench":301.605595,"close_bench":302.037354},{"date":"2020-08-04","open_a":38.451492,"high_a":38.924971,"low_a":38.393346,"close_a":38.783756,"volume_a":11717200,"open_b":112.765451,"high_b":112.822844,"low_b":111.535462,"close_b":112.724449,"volume_b":2553500,"open_bench":301.183097,"close_bench":303.204102},{"date":"2020-08-05","open_a":38.958202,"high_a":39.265547,"low_a":38.883442,"close_a":39.224014,"volume_a":10498700,"open_b":113.282034,"high_b":113.282034,"low_b":111.330459,"close_b":111.724052,"volume_b":2924000,"open_bench":304.499309,"close_bench":305.087219},{"date":"2020-08-06","open_a":38.758846,"high_a":39.481524,"low_a":38.709005,"close_a":39.439991,"volume_a":10745000,"open_b":110.920458,"high_b":111.756842,"low_b":110.920458,"close_b":111.404251,"volume_b":2692900,"open_bench":304.508414,"close_bench":307.126495},{"date":"2020-08-07","open_a":39.431685,"high_a":39.938391,"low_a":39.21571,"close_a":39.705803,"volume_a":11669700,"open_b":111.469873,"high_b":112.142261,"low_b":111.002471,"close_b":112.12587,"volume_b":3692100,"open_bench":306.161971,"close_bench":307.347015},{"date":"2020-08-10","open_a":39.921777,"high_a":40.104521,"low_a":39.58951,"close_a":39.639351,"volume_a":11888000,"open_b":111.781477,"high_b":112.240672,"low_b":111.010682,"close_b":111.50267,"volume_b":3052000,"open_bench":307.797235,"close_bench":308.265747},{"date":"2020-08-11","open_a":40.254041,"high_a":40.411869,"low_a":39.622737,"close_a":39.813789,"volume_a":16531600,"open_b":112.314431,"high_b":112.708025,"low_b":110.387443,"close_b":110.79744,"volume_b":3301200,"open_bench":309.441488,"close_bench":305.721008},{"date":"2020-08-12","open_a":40.121131,"high_a":40.44509,"low_a":39.971611,"close_a":40.229118,"volume_a":9562200,"open_b":110.97786,"high_b":113.716634,"low_b":110.904064,"close_b":112.995049,"volume_b":4424600,"open_bench":308.146224,"close_bench":309.98349},{"date":"2020-08-13","open_a":40.087902,"high_a":40.478315,"low_a":39.921771,"close_a":40.187584,"volume_a":9882900,"open_b":112.42105,"high_b":113.659235,"low_b":112.42105,"close_b":113.241043,"volume_b":2393500,"open_bench":309.221087,"close_bench":309.423187},{"date":"2020-08-14","open_a":40.154361,"high_a":40.445093,"low_a":39.896854,"close_a":40.245735,"volume_a":10075800,"open_b":113.249239,"high_b":113.298437,"low_b":112.503043,"close_b":112.798241,"volume_b":2461700,"open_bench":309.037246,"close_bench":309.432251},{"date":"2020-08-17","open_a":40.137745,"high_a":40.494931,"low_a":40.038066,"close_a":40.046371,"volume_a":9102400,"open_b":112.749049,"high_b":113.888839,"low_b":112.380055,"close_b":113.093445,"volume_b":2862500,"open_bench":310.442933,"close_bench":310.415375},{"date":"2020-08-18","open_a":40.038058,"high_a":40.287257,"low_a":39.838697,"close_a":40.220802,"volume_a":9829000,"open_b":113.659257,"high_b":113.659257,"low_b":112.371861,"close_b":113.24926,"volume_b":2638600,"open_bench":310.81027,"close_bench":311.085876},{"date":"2020-08-19","open_a":39.971613,"high_a":40.187587,"low_a":39.224016,"close_a":39.348614,"volume_a":17371000,"open_b":113.724858,"high_b":113.80685,"low_b":111.601079,"close_b":111.888069,"volume_b":3682200,"open_bench":311.46254,"close_bench":309.790649},{"date":"2020-08-20","open_a":39.265549,"high_a":39.539667,"low_a":39.116029,"close_a":39.332001,"volume_a":11308700,"open_b":111.240299,"high_b":112.462081,"low_b":111.2157,"close_b":112.109489,"volume_b":2263000,"open_bench":308.072676,"close_bench":310.755096},{"date":"2020-08-21","open_a":39.323691,"high_a":39.456597,"low_a":39.016345,"close_a":39.273849,"volume_a":14734100,"open_b":111.822462,"high_b":112.060267,"low_b":110.904074,"close_b":111.896271,"volume_b":4295600,"open_bench":310.424479,"close_bench":311.857544},{"date":"2020-08-24","open_a":39.398463,"high_a":39.855328,"low_a":39.190797,"close_a":39.847023,"volume_a":9257700,"open_b":111.896268,"high_b":112.896648,"low_b":111.609265,"close_b":112.863853,"volume_b":3177800,"open_bench":314.282746,"close_bench":315.01767},{"date":"2020-08-25","open_a":39.871944,"high_a":39.913477,"low_a":39.581212,"close_a":39.797184,"volume_a":7827800,"open_b":113.372262,"high_b":113.47067,"low_b":112.093074,"close_b":112.585075,"volume_b":2597400,"open_bench":315.577975,"close_bench":316.119965},{"date":"2020-08-26","open_a":39.672581,"high_a":40.021462,"low_a":39.43169,"close_a":40.004848,"volume_a":10124200,"open_b":111.994662,"high_b":113.70025,"low_b":111.830666,"close_b":113.544449,"volume_b":6019700,"open_bench":316.70792,"close_bench":319.289276},{"date":"2020-08-27","open_a":40.079594,"high_a":40.553073,"low_a":40.038062,"close_a":40.062981,"volume_a":10931600,"open_b":113.741261,"high_b":114.741642,"low_b":113.085264,"close_b":113.487053,"volume_b":4159300,"open_bench":320.152771,"close_bench":319.987396},{"date":"2020-08-28","open_a":40.494943,"high_a":41.408674,"low_a":40.312195,"close_a":41.392063,"volume_a":16420300,"open_b":113.585436,"high_b":114.766228,"low_b":112.535846,"close_b":114.749825,"volume_b":3355400,"open_bench":321.007125,"close_bench":322.054352},{"date":"2020-08-31","open_a":41.367129,"high_a":41.591409,"low_a":40.993329,"close_a":41.142849,"volume_a":18510800,"open_b":114.241442,"high_b":115.635427,"low_b":113.979038,"close_b":114.848228,"volume_b":3828700,"open_bench":321.843081,"close_bench":320.887695},{"date":"2020-09-01","open_a":40.960109,"high_a":40.993333,"low_a":40.56139,"close_a":40.80228,"volume_a":12588400,"open_b":115.004025,"high_b":115.19262,"low_b":113.577233,"close_b":114.134834,"volume_b":3421700,"open_bench":321.714397,"close_bench":323.909943},{"date":"2020-09-02","open_a":41.001643,"high_a":42.662974,"low_a":41.001643,"close_a":42.521759,"volume_a":26332600,"open_b":114.405431,"high_b":117.931403,"low_b":114.266034,"close_b":117.496796,"volume_b":4918200,"open_bench":325.811514,"close_bench":328.594971},{"date":"2020-09-03","open_a":42.696202,"high_a":43.310892,"low_a":41.616334,"close_a":41.931988,"volume_a":22289400,"open_b":117.912265,"high_b":118.961136,"low_b":114.071871,"close_b":115.104233,"volume_b":6642400,"open_bench":326.913939,"close_bench":317.286682},{"date":"2020-09-04","open_a":42.314091,"high_a":42.754346,"low_a":41.765852,"close_a":42.39716,"volume_a":21019500,"open_b":115.343726,"high_b":115.880549,"low_b":113.097306,"close_b":114.600418,"volume_b":4836400,"open_bench":317.966375,"close_bench":314.696045},{"date":"2020-09-08","open_a":42.305787,"high_a":42.397161,"low_a":41.001642,"close_a":41.375443,"volume_a":17776700,"open_b":114.228761,"high_b":114.493051,"low_b":111.007797,"close_b":111.94931,"volume_b":4901500,"open_bench":309.312933,"close_bench":306.097717},{"date":"2020-09-09","open_a":41.200997,"high_a":42.164569,"low_a":41.159464,"close_a":41.69109,"volume_a":15252300,"open_b":111.908019,"high_b":114.352657,"low_b":111.825425,"close_b":113.237701,"volume_b":4282300,"open_bench":310.084516,"close_bench":312.142273},{"date":"2020-09-10","open_a":41.824,"high_a":42.098121,"low_a":41.491735,"close_a":41.533268,"volume_a":15439500,"open_b":112.816498,"high_b":113.601091,"low_b":110.570078,"close_b":111.296852,"volume_b":6050500,"open_bench":314.007134,"close_bench":306.722382},{"date":"2020-09-11","open_a":41.865545,"high_a":42.530077,"low_a":41.616343,"close_a":42.413784,"volume_a":18272300,"open_b":111.676775,"high_b":112.63481,"low_b":110.85914,"close_b":112.164047,"volume_b":3411900,"open_bench":308.495282,"close_bench":306.878479},{"date":"2020-09-14","open_a":42.681739,"high_a":42.899459,"low_a":42.346782,"close_a":42.464016,"volume_a":10911700,"open_b":112.758659,"high_b":113.559772,"low_b":112.155763,"close_b":112.874283,"volume_b":2779000,"open_bench":310.029427,"close_bench":310.920502},{"date":"2020-09-15","open_a":42.706861,"high_a":43.02507,"low_a":42.572879,"close_a":42.74873,"volume_a":12838400,"open_b":112.717391,"high_b":113.345075,"low_b":111.833696,"close_b":112.007126,"volume_b":4080800,"open_bench":313.364079,"close_bench":312.491394},{"date":"2020-09-16","open_a":42.664992,"high_a":43.083687,"low_a":42.489141,"close_a":42.53101,"volume_a":15353900,"open_b":112.552228,"high_b":112.816519,"low_b":111.313394,"close_b":111.478569,"volume_b":3353500,"open_bench":313.722346,"close_bench":311.251221},{"date":"2020-09-17","open_a":42.053692,"high_a":42.397021,"low_a":41.819221,"close_a":42.330029,"volume_a":13287100,"open_b":110.16539,"high_b":111.123424,"low_b":108.76138,"close_b":110.008469,"volume_b":5186400,"open_bench":306.419251,"close_bench":308.513733},{"date":"2020-09-18","open_a":42.497516,"high_a":42.840846,"low_a":42.045325,"close_a":42.2463,"volume_a":23816600,"open_b":110.429678,"high_b":111.321628,"low_b":107.373877,"close_b":108.579681,"volume_b":8473000,"open_bench":309.31522,"close_bench":304.961914},{"date":"2020-09-21","open_a":41.660122,"high_a":41.886217,"low_a":40.705498,"close_a":41.107445,"volume_a":17514800,"open_b":107.671215,"high_b":108.447553,"low_b":106.837071,"close_b":108.389748,"volume_b":4402700,"open_bench":300.396516,"close_bench":301.567841},{"date":"2020-09-22","open_a":41.124191,"high_a":41.902964,"low_a":41.090695,"close_a":41.584755,"volume_a":13034600,"open_b":108.191513,"high_b":109.752444,"low_b":107.737272,"close_b":109.141281,"volume_b":3951000,"open_bench":303.04349,"close_bench":304.639069},{"date":"2020-09-23","open_a":41.643365,"high_a":41.802468,"low_a":40.337037,"close_a":40.370533,"volume_a":17121200,"open_b":109.59553,"high_b":109.694646,"low_b":107.241753,"close_b":108.191521,"volume_b":5909900,"open_bench":305.192389,"close_bench":297.574127},{"date":"2020-09-24","open_a":40.370543,"high_a":41.115819,"low_a":39.826241,"close_a":40.780865,"volume_a":16788100,"open_b":108.554908,"high_b":109.273428,"low_b":107.299553,"close_b":108.670532,"volume_b":4316600,"open_bench":296.264535,"close_bench":298.367401},{"date":"2020-09-25","open_a":40.362157,"high_a":40.831093,"low_a":40.17793,"close_a":40.7976,"volume_a":12603400,"open_b":108.10067,"high_b":110.528787,"low_b":107.786828,"close_b":110.297539,"volume_b":4576000,"open_bench":297.51887,"close_bench":303.191101},{"date":"2020-09-28","open_a":41.291668,"high_a":41.626622,"low_a":41.14931,"close_a":41.266544,"volume_a":11215700,"open_b":112.354009,"high_b":114.451776,"low_b":112.048434,"close_b":113.947983,"volume_b":7487500,"open_bench":307.332347,"close_bench":308.22699},{"date":"2020-09-29","open_a":41.358662,"high_a":41.50102,"low_a":40.956715,"close_a":40.965088,"volume_a":12426600,"open_b":114.517833,"high_b":115.244608,"low_b":113.254224,"close_b":113.278999,"volume_b":4330500,"open_bench":308.023981,"close_bench":306.548279},{"date":"2020-09-30","open_a":41.25817,"high_a":41.551255,"low_a":40.939961,"close_a":41.341908,"volume_a":15755900,"open_b":113.964492,"high_b":114.955556,"low_b":113.064262,"close_b":114.468285,"volume_b":7060700,"open_bench":307.212353,"close_bench":308.872528},{"date":"2020-10-01","open_a":41.400528,"high_a":41.584754,"low_a":40.688748,"close_a":41.182808,"volume_a":17577300,"open_b":115.162038,"high_b":116.326545,"low_b":113.551555,"close_b":116.285248,"volume_b":6483900,"open_bench":311.454995,"close_bench":310.855499},{"date":"2020-10-02","open_a":40.831114,"high_a":41.517776,"low_a":40.655263,"close_a":41.333549,"volume_a":13610100,"open_b":115.029887,"high_b":115.550201,"low_b":112.486146,"close_b":114.022301,"volume_b":5130200,"open_bench":305.930293,"close_bench":307.904022},{"date":"2020-10-05","open_a":41.392161,"high_a":41.492647,"low_a":41.132569,"close_a":41.350292,"volume_a":10628700,"open_b":114.063568,"high_b":114.286562,"low_b":113.130309,"close_b":113.914902,"volume_b":4519800,"open_bench":309.951516,"close_bench":313.364075},{"date":"2020-10-06","open_a":41.49265,"high_a":41.643381,"low_a":40.889732,"close_a":40.981842,"volume_a":10077400,"open_b":113.972721,"high_b":114.649957,"low_b":111.618932,"close_b":112.073174,"volume_b":5031000,"open_bench":313.50254,"close_bench":308.909424},{"date":"2020-10-07","open_a":41.048827,"high_a":41.668494,"low_a":40.99021,"close_a":41.501019,"volume_a":10135600,"open_b":112.271395,"high_b":113.576302,"low_b":112.122729,"close_b":113.15509,"volume_b":4347300,"open_bench":311.851616,"close_bench":314.28653},{"date":"2020-10-08","open_a":41.626623,"high_a":42.321658,"low_a":41.484269,"close_a":42.254665,"volume_a":12143900,"open_b":113.378137,"high_b":114.080148,"low_b":112.94867,"close_b":113.8489,"volume_b":2977400,"open_bench":316.214036,"close_bench":317.071777},{"date":"2020-10-09","open_a":42.430519,"high_a":42.899458,"low_a":42.371902,"close_a":42.547756,"volume_a":11407800,"open_b":114.550847,"high_b":115.013343,"low_b":113.890133,"close_b":114.336121,"volume_b":3194000,"open_bench":318.713502,"close_bench":319.90329},{"date":"2020-10-12","open_a":42.572875,"high_a":43.150673,"low_a":42.564503,"close_a":42.782223,"volume_a":11386500,"open_b":116.549533,"high_b":118.5069,"low_b":115.442844,"close_b":117.38369,"volume_b":6167200,"open_bench":322.430447,"close_bench":325.049805},{"date":"2020-10-13","open_a":42.698494,"high_a":42.698494,"low_a":41.785739,"close_a":42.053703,"volume_a":14365300,"open_b":117.458004,"high_b":118.928087,"low_b":117.201981,"close_b":118.548172,"volume_b":4293200,"open_bench":324.911459,"close_bench":322.928497},{"date":"2020-10-14","open_a":41.911338,"high_a":42.279792,"low_a":41.576385,"close_a":41.969955,"volume_a":10582100,"open_b":118.556407,"high_b":119.018903,"low_b":116.970701,"close_b":117.705742,"volume_b":3641000,"open_bench":323.500336,"close_bench":320.899414},{"date":"2020-10-15","open_a":41.367039,"high_a":41.978333,"low_a":41.283298,"close_a":41.861099,"volume_a":9961700,"open_b":116.582553,"high_b":117.251522,"low_b":116.136566,"close_b":116.805534,"volume_b":3306800,"open_bench":317.007184,"close_bench":320.502747},{"date":"2020-10-16","open_a":41.861095,"high_a":42.162553,"low_a":41.760606,"close_a":41.894588,"volume_a":13569800,"open_b":117.069805,"high_b":117.953513,"low_b":116.557758,"close_b":117.053284,"volume_b":5076000,"open_bench":321.849324,"close_bench":320.309082},{"date":"2020-10-19","open_a":41.894595,"high_a":42.271421,"low_a":41.417284,"close_a":41.551266,"volume_a":9930400,"open_b":116.731209,"high_b":117.714019,"low_b":115.005103,"close_b":115.360229,"volume_b":3223200,"open_bench":321.563456,"close_bench":315.439331},{"date":"2020-10-20","open_a":41.777355,"high_a":42.388649,"low_a":41.693617,"close_a":42.078815,"volume_a":10970700,"open_b":116.450409,"high_b":116.632106,"low_b":115.137248,"close_b":115.385017,"volume_b":2750900,"open_bench":316.776684,"close_bench":316.702911},{"date":"2020-10-21","open_a":41.835975,"high_a":42.229549,"low_a":41.676873,"close_a":41.861099,"volume_a":12517400,"open_b":115.360227,"high_b":116.020941,"low_b":114.881222,"close_b":115.294167,"volume_b":2560100,"open_bench":316.656788,"close_bench":316.103424},{"date":"2020-10-22","open_a":42.849211,"high_a":42.874335,"low_a":42.187676,"close_a":42.438892,"volume_a":20189200,"open_b":115.459365,"high_b":115.789716,"low_b":114.806906,"close_b":115.302444,"volume_b":2909000,"open_bench":316.315535,"close_bench":317.837341},{"date":"2020-10-23","open_a":42.706859,"high_a":42.857589,"low_a":42.263044,"close_a":42.304913,"volume_a":13491300,"open_b":115.806191,"high_b":116.062214,"low_b":114.831635,"close_b":115.261101,"volume_b":2758400,"open_bench":319.054814,"close_bench":318.916473},{"date":"2020-10-26","open_a":42.154187,"high_a":42.296542,"low_a":41.199563,"close_a":41.701996,"volume_a":14696400,"open_b":114.583912,"high_b":114.864712,"low_b":112.659588,"close_b":114.022301,"volume_b":3431100,"open_bench":315.549948,"close_bench":313.022827},{"date":"2020-10-27","open_a":41.710359,"high_a":42.112306,"low_a":41.492636,"close_a":41.785725,"volume_a":13892100,"open_b":114.443522,"high_b":115.533689,"low_b":114.220528,"close_b":114.897751,"volume_b":3390800,"open_bench":313.364183,"close_bench":311.943817},{"date":"2020-10-28","open_a":41.032076,"high_a":41.157685,"low_a":40.01046,"close_a":40.16119,"volume_a":19886200,"open_b":113.27074,"high_b":113.733249,"low_b":110.619642,"close_b":110.809593,"volume_b":5196000,"open_bench":306.299261,"close_bench":301.281891},{"date":"2020-10-29","open_a":39.968601,"high_a":40.596643,"low_a":39.608523,"close_a":40.211445,"volume_a":15369900,"open_b":110.470978,"high_b":112.0154,"low_b":109.232144,"close_b":110.660942,"volume_b":3856500,"open_bench":301.512434,"close_bench":304.343933},{"date":"2020-10-30","open_a":40.035587,"high_a":40.295175,"low_a":39.667133,"close_a":40.244934,"volume_a":15804000,"open_b":109.785489,"high_b":110.537051,"low_b":108.777915,"close_b":110.082809,"volume_b":4303900,"open_bench":302.776107,"close_bench":301.171295},{"date":"2020-11-02","open_a":40.822733,"high_a":40.990211,"low_a":40.395665,"close_a":40.713871,"volume_a":15156100,"open_b":111.998849,"high_b":112.510897,"low_b":110.48747,"close_b":111.618935,"volume_b":3126400,"open_bench":304.546875,"close_bench":304.546875},{"date":"2020-11-03","open_a":41.216303,"high_a":41.835969,"low_a":41.032076,"close_a":41.325161,"volume_a":15285100,"open_b":112.832983,"high_b":114.468253,"low_b":112.552184,"close_b":113.179855,"volume_b":3293900,"open_bench":307.765687,"close_bench":309.923889},{"date":"2020-11-04","open_a":41.392162,"high_a":42.045324,"low_a":41.099076,"close_a":41.174438,"volume_a":12930800,"open_b":114.79863,"high_b":117.028529,"low_b":113.691941,"close_b":113.857117,"volume_b":4576300,"open_bench":314.378712,"close_bench":316.850525},{"date":"2020-11-05","open_a":41.668488,"high_a":42.045314,"low_a":41.316786,"close_a":41.400524,"volume_a":12452400,"open_b":115.087712,"high_b":115.847529,"low_b":113.535036,"close_b":114.014053,"volume_b":3667800,"open_bench":322.107567,"close_bench":323.029877},{"date":"2020-11-06","open_a":41.576387,"high_a":41.8611,"low_a":41.24143,"close_a":41.408909,"volume_a":9294900,"open_b":114.435245,"high_b":115.434576,"low_b":113.419404,"close_b":114.286591,"volume_b":2867000,"open_bench":322.744035,"close_bench":322.956177},{"date":"2020-11-09","open_a":43.962947,"high_a":44.599362,"low_a":43.60287,"close_a":44.021564,"volume_a":33442000,"open_b":117.276309,"high_b":118.639021,"low_b":114.11315,"close_b":114.344398,"volume_b":6111300,"open_bench":335.693287,"close_bench":327.014343},{"date":"2020-11-10","open_a":44.021563,"high_a":45.453498,"low_a":43.921077,"close_a":45.202282,"volume_a":29706700,"open_b":113.138605,"high_b":117.639701,"low_b":112.477891,"close_b":117.045059,"volume_b":5284500,"open_bench":326.027381,"close_bench":326.534668},{"date":"2020-11-11","open_a":45.160418,"high_a":45.470251,"low_a":44.657985,"close_a":44.867332,"volume_a":17399900,"open_b":117.986593,"high_b":119.126324,"low_b":116.954231,"close_b":118.845512,"volume_b":4389800,"open_bench":328.711377,"close_bench":328.960419},{"date":"2020-11-12","open_a":44.423507,"high_a":44.557489,"low_a":44.055057,"close_a":44.390011,"volume_a":15024400,"open_b":119.134531,"high_b":119.539222,"low_b":117.903952,"close_b":118.795914,"volume_b":2570100,"open_bench":327.955097,"close_bench":325.769226},{"date":"2020-11-13","open_a":44.590982,"high_a":44.80033,"low_a":44.289522,"close_a":44.758461,"volume_a":12447900,"open_b":118.73812,"high_b":119.663112,"low_b":118.325175,"close_b":119.514458,"volume_b":2580400,"open_bench":327.669143,"close_bench":330.279297},{"date":"2020-11-16","open_a":45.63773,"high_a":45.738216,"low_a":44.616114,"close_a":45.093426,"volume_a":14908200,"open_b":119.753959,"high_b":120.794575,"low_b":119.497935,"close_b":120.38163,"volume_b":3399000,"open_bench":332.935538,"close_bench":334.402008},{"date":"2020-11-17","open_a":44.934313,"high_a":45.009678,"low_a":44.406757,"close_a":44.951061,"volume_a":14248400,"open_b":119.15108,"high_b":120.66246,"low_b":119.018935,"close_b":119.952194,"volume_b":2924000,"open_bench":332.00402,"close_bench":332.603516},{"date":"2020-11-18","open_a":45.009681,"high_a":45.235776,"low_a":44.013189,"close_a":44.055058,"volume_a":15278200,"open_b":120.158674,"high_b":121.050636,"low_b":118.349973,"close_b":118.416046,"volume_b":4335200,"open_bench":332.87104,"close_bench":328.600739},{"date":"2020-11-19","open_a":43.954573,"high_a":44.574239,"low_a":43.502382,"close_a":44.490501,"volume_a":17189500,"open_b":118.0031,"high_b":118.680323,"low_b":117.557112,"close_b":118.366493,"volume_b":4263500,"open_bench":327.973471,"close_bench":329.9841},{"date":"2020-11-20","open_a":44.540755,"high_a":44.591,"low_a":44.038323,"close_a":44.105312,"volume_a":15645600,"open_b":118.639025,"high_b":119.45666,"low_b":118.102202,"close_b":118.449074,"volume_b":3469100,"open_bench":329.725882,"close_bench":327.724457},{"date":"2020-11-23","open_a":44.339758,"high_a":44.523982,"low_a":43.711716,"close_a":44.113663,"volume_a":13191900,"open_b":118.358231,"high_b":118.63903,"low_b":117.177215,"close_b":118.118729,"volume_b":3443300,"open_bench":329.522986,"close_bench":329.688995},{"date":"2020-11-24","open_a":44.498872,"high_a":44.875698,"low_a":44.398386,"close_a":44.565865,"volume_a":13827600,"open_b":118.259121,"high_b":119.225411,"low_b":117.953546,"close_b":118.952866,"volume_b":4472300,"open_bench":332.225303,"close_bench":335.001465},{"date":"2020-11-25","open_a":44.440251,"high_a":44.616102,"low_a":44.205783,"close_a":44.323017,"volume_a":10194800,"open_b":119.299752,"high_b":119.687921,"low_b":118.680335,"close_b":119.060249,"volume_b":2278300,"open_bench":334.918472,"close_bench":334.484985},{"date":"2020-11-27","open_a":44.473746,"high_a":44.540736,"low_a":44.055052,"close_a":44.130417,"volume_a":8499300,"open_b":119.440131,"high_b":119.985221,"low_b":119.076737,"close_b":119.423622,"volume_b":1596000,"open_bench":335.573403,"close_bench":335.416626},{"date":"2020-11-30","open_a":43.961631,"high_a":44.046026,"low_a":43.109238,"close_a":43.548092,"volume_a":43203600,"open_b":119.307998,"high_b":119.448398,"low_b":118.003091,"close_b":119.118034,"volume_b":4906900,"open_bench":334.641809,"close_bench":333.931641},{"date":"2020-12-01","open_a":44.003832,"high_a":44.164185,"low_a":43.708449,"close_a":43.919437,"volume_a":18968800,"open_b":119.778733,"high_b":121.100161,"low_b":119.341012,"close_b":120.637665,"volume_b":3999200,"open_bench":337.168992,"close_bench":337.584015},{"date":"2020-12-02","open_a":43.801274,"high_a":43.995384,"low_a":43.581844,"close_a":43.978504,"volume_a":14913100,"open_b":120.480753,"high_b":121.356194,"low_b":118.837229,"close_b":119.440125,"volume_b":4432900,"open_bench":336.477237,"close_bench":338.294189},{"date":"2020-12-03","open_a":43.801282,"high_a":44.678993,"low_a":43.801282,"close_a":44.543961,"volume_a":17318900,"open_b":118.960213,"high_b":120.573855,"low_b":118.960213,"close_b":120.149643,"volume_b":4757900,"open_bench":338.192703,"close_bench":338.201935},{"date":"2020-12-04","open_a":44.543959,"high_a":45.46387,"low_a":44.52708,"close_a":45.446991,"volume_a":18122400,"open_b":119.667167,"high_b":121.422209,"low_b":119.650541,"close_b":121.314087,"volume_b":3592600,"open_bench":338.783011,"close_bench":341.116455},{"date":"2020-12-07","open_a":45.371034,"high_a":45.387914,"low_a":44.476444,"close_a":44.721191,"volume_a":16210900,"open_b":121.172734,"high_b":121.904699,"low_b":120.291056,"close_b":120.914886,"volume_b":4217300,"open_bench":340.350928,"close_bench":340.415497},{"date":"2020-12-08","open_a":44.678999,"high_a":45.134736,"low_a":44.552408,"close_a":44.88155,"volume_a":12485800,"open_b":120.881634,"high_b":121.572007,"low_b":120.016582,"close_b":121.039673,"volume_b":3623800,"open_bench":339.151889,"close_bench":341.41156},{"date":"2020-12-09","open_a":45.041885,"high_a":45.235994,"low_a":44.729622,"close_a":45.008129,"volume_a":11508400,"open_b":121.072873,"high_b":121.522037,"low_b":120.008204,"close_b":121.181007,"volume_b":3842000,"open_bench":342.066458,"close_bench":338.349548},{"date":"2020-12-10","open_a":45.06721,"high_a":45.160043,"low_a":44.712747,"close_a":44.771824,"volume_a":12097300,"open_b":121.472159,"high_b":121.680103,"low_b":120.124691,"close_b":120.332634,"volume_b":3985000,"open_bench":336.984504,"close_bench":338.238861},{"date":"2020-12-11","open_a":44.560838,"high_a":45.185367,"low_a":44.543959,"close_a":45.025013,"volume_a":10638100,"open_b":119.467565,"high_b":120.690273,"low_b":119.284574,"close_b":120.582138,"volume_b":2975500,"open_bench":336.550965,"close_bench":337.842194},{"date":"2020-12-14","open_a":45.278197,"high_a":45.438547,"low_a":44.881537,"close_a":44.957493,"volume_a":22292000,"open_b":120.815049,"high_b":121.829813,"low_b":119.816924,"close_b":119.966637,"volume_b":3952400,"open_bench":340.000332,"close_bench":336.329529},{"date":"2020-12-15","open_a":45.39636,"high_a":45.759261,"low_a":45.236006,"close_a":45.438557,"volume_a":16193700,"open_b":120.241116,"high_b":120.831681,"low_b":119.68383,"close_b":120.415794,"volume_b":3183300,"open_bench":338.856695,"close_bench":340.876556},{"date":"2020-12-16","open_a":45.235999,"high_a":45.404791,"low_a":44.678989,"close_a":44.780266,"volume_a":15679600,"open_b":119.75867,"high_b":120.998017,"low_b":119.692127,"close_b":120.515587,"volume_b":4915800,"open_bench":341.088746,"close_bench":341.41156},{"date":"2020-12-17","open_a":44.98281,"high_a":45.328832,"low_a":44.856219,"close_a":44.957493,"volume_a":12677300,"open_b":120.815041,"high_b":121.605223,"low_b":120.756811,"close_b":121.197662,"volume_b":3312500,"open_bench":343.044119,"close_bench":343.320801},{"date":"2020-12-18","open_a":44.79715,"high_a":45.438555,"low_a":44.408928,"close_a":45.35416,"volume_a":35480100,"open_b":121.713361,"high_b":122.35383,"low_b":121.022988,"close_b":122.212418,"volume_b":8980400,"open_bench":343.607901,"close_bench":341.949921},{"date":"2020-12-21","open_a":44.459564,"high_a":44.729629,"low_a":43.868794,"close_a":44.569279,"volume_a":14926600,"open_b":120.607146,"high_b":121.156119,"low_b":119.334533,"close_b":120.93985,"volume_b":3946900,"open_bench":338.050471,"close_bench":340.727295},{"date":"2020-12-22","open_a":44.569285,"high_a":44.881547,"low_a":44.214822,"close_a":44.527084,"volume_a":9689500,"open_b":120.274392,"high_b":120.607096,"low_b":119.342797,"close_b":119.791962,"volume_b":2653700,"open_bench":341.05141,"close_bench":340.152954},{"date":"2020-12-23","open_a":44.687446,"high_a":45.058785,"low_a":44.679005,"close_a":44.797161,"volume_a":7040300,"open_b":120.182907,"high_b":121.106176,"low_b":119.833564,"close_b":120.116364,"volume_b":2645100,"open_bench":341.116333,"close_bench":340.45871},{"date":"2020-12-24","open_a":44.746512,"high_a":45.193807,"low_a":44.746512,"close_a":45.100971,"volume_a":3265500,"open_b":120.465693,"high_b":120.698589,"low_b":119.891768,"close_b":120.656998,"volume_b":883300,"open_bench":340.93117,"close_bench":341.783325},{"date":"2020-12-28","open_a":45.446991,"high_a":45.944925,"low_a":45.345717,"close_a":45.708618,"volume_a":9020500,"open_b":121.255901,"high_b":122.769734,"low_b":121.05627,"close_b":122.195808,"volume_b":2493500,"open_bench":344.321116,"close_bench":344.719421},{"date":"2020-12-29","open_a":45.953377,"high_a":45.987136,"low_a":45.590476,"close_a":45.683311,"volume_a":8320600,"open_b":122.836259,"high_b":123.742902,"low_b":122.137573,"close_b":122.620003,"volume_b":3186700,"open_bench":346.238441,"close_bench":344.061768},{"date":"2020-12-30","open_a":45.615785,"high_a":46.105281,"low_a":45.598906,"close_a":45.944927,"volume_a":8142700,"open_b":122.661552,"high_b":123.094077,"low_b":122.278931,"close_b":122.528465,"volume_b":2224900,"open_bench":344.876894,"close_bench":344.552704},{"date":"2020-12-31","open_a":45.953363,"high_a":46.358461,"low_a":45.801451,"close_a":46.282505,"volume_a":8495000,"open_b":122.669899,"high_b":123.485046,"low_b":121.971213,"close_b":123.351959,"volume_b":2076800,"open_bench":344.358231,"close_bench":346.303345},{"date":"2021-01-04","open_a":45.801459,"high_a":46.105283,"low_a":43.910997,"close_a":44.527084,"volume_a":25611100,"open_b":122.278951,"high_b":122.91942,"low_b":118.061868,"close_b":119.999901,"volume_b":7486900,"open_bench":347.62777,"close_bench":341.588684},{"date":"2021-01-05","open_a":44.164188,"high_a":44.408933,"low_a":43.910999,"close_a":44.037594,"volume_a":20323800,"open_b":119.833596,"high_b":120.756853,"low_b":118.86041,"close_b":120.357605,"volume_b":4126000,"open_bench":340.949542,"close_bench":343.941284},{"date":"2021-01-06","open_a":43.860355,"high_a":43.902552,"low_a":42.358113,"close_a":42.63662,"volume_a":38724500,"open_b":117.837323,"high_b":119.442652,"low_b":117.77078,"close_b":118.885353,"volume_b":4843300,"open_bench":342.440824,"close_bench":345.997589},{"date":"2021-01-07","open_a":42.27372,"high_a":42.417191,"low_a":41.792666,"close_a":42.164005,"volume_a":53225700,"open_b":118.785508,"high_b":119.068308,"low_b":117.404762,"close_b":118.502708,"volume_b":4473200,"open_bench":348.359489,"close_bench":351.138214},{"date":"2021-01-08","open_a":42.223075,"high_a":43.151427,"low_a":42.062724,"close_a":43.10923,"volume_a":29674000,"open_b":118.502701,"high_b":120.108017,"low_b":117.828967,"close_b":119.925026,"volume_b":4312000,"open_bench":352.518352,"close_bench":353.138947},{"date":"2021-01-11","open_a":42.721002,"high_a":42.923553,"low_a":42.155554,"close_a":42.366543,"volume_a":19304400,"open_b":119.800295,"high_b":120.033191,"low_b":117.853949,"close_b":118.186653,"volume_b":3902700,"open_bench":349.980414,"close_bench":350.758453},{"date":"2021-01-12","open_a":42.231519,"high_a":42.510023,"low_a":41.99521,"close_a":42.290596,"volume_a":18538700,"open_b":117.80403,"high_b":118.111782,"low_b":115.990758,"close_b":117.637665,"volume_b":5150000,"open_bench":350.943753,"close_bench":350.832581},{"date":"2021-01-13","open_a":42.180882,"high_a":42.501586,"low_a":42.155564,"close_a":42.332794,"volume_a":18864100,"open_b":117.379831,"high_b":119.143199,"low_b":117.280023,"close_b":118.602539,"volume_b":4361600,"open_bench":350.758564,"close_bench":351.777435},{"date":"2021-01-14","open_a":42.332795,"high_a":42.349675,"low_a":41.539478,"close_a":41.547916,"volume_a":34303600,"open_b":117.679275,"high_b":118.477783,"low_b":117.338258,"close_b":117.91217,"volume_b":4724700,"open_bench":352.518279,"close_bench":350.54538},{"date":"2021-01-15","open_a":41.488839,"high_a":41.505718,"low_a":40.957146,"close_a":41.10062,"volume_a":24366800,"open_b":117.654305,"high_b":118.610852,"low_b":116.889075,"close_b":117.604401,"volume_b":4309100,"open_bench":348.933831,"close_bench":347.989075},{"date":"2021-01-19","open_a":41.134378,"high_a":41.387564,"low_a":40.712401,"close_a":40.940266,"volume_a":29114500,"open_b":118.427863,"high_b":118.594215,"low_b":117.645994,"close_b":118.16169,"volume_b":5158600,"open_bench":350.434181,"close_bench":350.721313},{"date":"2021-01-20","open_a":40.965595,"high_a":41.201901,"low_a":40.771485,"close_a":41.083748,"volume_a":23115800,"open_b":117.313283,"high_b":117.9288,"low_b":116.381701,"close_b":117.554504,"volume_b":5019300,"open_bench":353.000002,"close_bench":355.574982},{"date":"2021-01-21","open_a":41.168145,"high_a":41.429772,"low_a":40.898079,"close_a":41.311619,"volume_a":19702500,"open_b":116.689471,"high_b":117.105357,"low_b":115.400232,"close_b":116.123871,"volume_b":5684100,"open_bench":356.130639,"close_bench":355.899078},{"date":"2021-01-22","open_a":41.193463,"high_a":41.303174,"low_a":40.923397,"close_a":40.923397,"volume_a":15400300,"open_b":116.032345,"high_b":116.065611,"low_b":114.97599,"close_b":115.275429,"volume_b":4014900,"open_bench":354.055898,"close_bench":354.639435},{"date":"2021-01-25","open_a":40.864312,"high_a":41.218775,"low_a":40.712403,"close_a":41.168137,"volume_a":16604200,"open_b":114.834614,"high_b":116.722743,"low_b":114.435366,"close_b":116.597969,"volume_b":3829000,"open_bench":355.371191,"close_bench":356.038086},{"date":"2021-01-26","open_a":41.277845,"high_a":41.691384,"low_a":41.024659,"close_a":41.598549,"volume_a":12922600,"open_b":117.537869,"high_b":118.186651,"low_b":116.240305,"close_b":117.945442,"volume_b":4242500,"open_bench":356.982837,"close_bench":355.48233},{"date":"2021-01-27","open_a":41.260978,"high_a":41.623879,"low_a":40.703967,"close_a":40.957153,"volume_a":21242400,"open_b":116.872436,"high_b":118.386269,"low_b":114.410394,"close_b":114.817955,"volume_b":6659800,"open_bench":352.175621,"close_bench":346.794159},{"date":"2021-01-28","open_a":41.092182,"high_a":41.775787,"low_a":41.024667,"close_a":41.480404,"volume_a":17498900,"open_b":115.475059,"high_b":117.105327,"low_b":114.585056,"close_b":115.774498,"volume_b":5514000,"open_bench":348.600358,"close_bench":349.776703},{"date":"2021-01-29","open_a":41.379134,"high_a":41.45509,"low_a":40.602697,"close_a":40.636456,"volume_a":18670300,"open_b":114.876193,"high_b":115.167319,"low_b":113.129465,"close_b":113.595268,"volume_b":6778100,"open_bench":347.924194,"close_bench":342.774292},{"date":"2021-02-01","open_a":40.737725,"high_a":41.092185,"low_a":40.661769,"close_a":40.914955,"volume_a":12183600,"open_b":113.936279,"high_b":114.751426,"low_b":113.037963,"close_b":113.936279,"volume_b":4185700,"open_bench":346.155104,"close_bench":348.47998},{"date":"2021-02-02","open_a":41.185017,"high_a":41.649191,"low_a":41.041546,"close_a":41.320049,"volume_a":13319800,"open_b":114.327213,"high_b":116.057303,"low_b":113.620214,"close_b":115.100769,"volume_b":3581600,"open_bench":351.647708,"close_bench":353.407562},{"date":"2021-02-03","open_a":41.311615,"high_a":41.429768,"low_a":41.075305,"close_a":41.159702,"volume_a":12274100,"open_b":114.576771,"high_b":115.574897,"low_b":114.011171,"close_b":114.801353,"volume_b":3609600,"open_bench":354.231843,"close_bench":353.685364},{"date":"2021-02-04","open_a":41.32005,"high_a":41.412886,"low_a":41.058426,"close_a":41.362247,"volume_a":20597400,"open_b":114.576737,"high_b":116.248596,"low_b":114.576737,"close_b":116.182053,"volume_b":3860300,"open_bench":354.713551,"close_bench":357.705322},{"date":"2021-02-05","open_a":41.775787,"high_a":42.012094,"low_a":41.581678,"close_a":41.902382,"volume_a":12742000,"open_b":116.855808,"high_b":118.211589,"low_b":116.223652,"close_b":117.246742,"volume_b":4437900,"open_bench":359.567037,"close_bench":359.113159},{"date":"2021-02-08","open_a":42.164004,"high_a":42.341234,"low_a":41.767347,"close_a":42.130245,"volume_a":17833200,"open_b":117.438053,"high_b":117.953748,"low_b":116.032354,"close_b":116.780945,"volume_b":4112200,"open_bench":360.558136,"close_bench":361.706696},{"date":"2021-02-09","open_a":42.071168,"high_a":42.214642,"low_a":41.8855,"close_a":41.944576,"volume_a":14592900,"open_b":117.388151,"high_b":117.404778,"low_b":116.024044,"close_b":116.11554,"volume_b":3179300,"open_bench":360.873073,"close_bench":361.465881},{"date":"2021-02-10","open_a":42.543781,"high_a":42.687255,"low_a":41.801099,"close_a":41.860176,"volume_a":22965400,"open_b":117.379834,"high_b":117.379834,"low_b":113.753303,"close_b":114.535172,"volume_b":6966600,"open_bench":363.197891,"close_bench":361.30835},{"date":"2021-02-11","open_a":42.037418,"high_a":42.695702,"low_a":42.028977,"close_a":42.450954,"volume_a":21928600,"open_b":114.659922,"high_b":114.959361,"low_b":111.740391,"close_b":112.264412,"volume_b":7528000,"open_bench":362.382844,"close_bench":361.891937},{"date":"2021-02-12","open_a":42.678832,"high_a":42.915138,"low_a":42.52692,"close_a":42.780106,"volume_a":13137100,"open_b":112.505629,"high_b":112.788441,"low_b":111.000121,"close_b":111.349464,"volume_b":5659300,"open_bench":361.095343,"close_bench":363.679565},{"date":"2021-02-16","open_a":42.721019,"high_a":42.805417,"low_a":42.332801,"close_a":42.425636,"volume_a":15093400,"open_b":111.923402,"high_b":112.131346,"low_b":110.958542,"close_b":111.773689,"volume_b":5180100,"open_bench":364.902189,"close_bench":363.364624},{"date":"2021-02-17","open_a":42.197757,"high_a":42.611297,"low_a":42.104922,"close_a":42.307472,"volume_a":12794300,"open_b":111.507531,"high_b":112.34763,"low_b":111.341179,"close_b":111.840248,"volume_b":3353000,"open_bench":361.623331,"close_bench":363.448029},{"date":"2021-02-18","open_a":42.332797,"high_a":42.915125,"low_a":42.223082,"close_a":42.84761,"volume_a":12747100,"open_b":111.515827,"high_b":112.871609,"low_b":111.382741,"close_b":112.597122,"volume_b":3935200,"open_bench":360.854466,"close_bench":361.901123},{"date":"2021-02-19","open_a":42.864486,"high_a":42.898245,"low_a":42.172443,"close_a":42.290596,"volume_a":15968800,"open_b":112.297679,"high_b":112.389175,"low_b":110.035255,"close_b":110.218246,"volume_b":5414900,"open_bench":363.151626,"close_bench":361.262085},{"date":"2021-02-22","open_a":42.071174,"high_a":42.940451,"low_a":41.919262,"close_a":42.729462,"volume_a":14370900,"open_b":109.827315,"high_b":110.293118,"low_b":108.762646,"close_b":109.785736,"volume_b":5792900,"open_bench":358.511093,"close_bench":358.483307},{"date":"2021-02-23","open_a":43.041718,"high_a":43.472134,"low_a":42.552223,"close_a":42.6535,"volume_a":16222300,"open_b":110.334706,"high_b":111.740405,"low_b":109.21182,"close_b":110.442841,"volume_b":6264300,"open_bench":356.288148,"close_bench":358.918671},{"date":"2021-02-24","open_a":42.526905,"high_a":42.957324,"low_a":42.307478,"close_a":42.79697,"volume_a":14442000,"open_b":109.669294,"high_b":110.035264,"low_b":109.128634,"close_b":109.868912,"volume_b":4944000,"open_bench":357.834991,"close_bench":362.873749},{"date":"2021-02-25","open_a":42.594425,"high_a":42.830732,"low_a":42.180886,"close_a":42.341236,"volume_a":14211100,"open_b":109.253407,"high_b":109.868911,"low_b":107.964156,"close_b":108.130508,"volume_b":5702300,"open_bench":361.614054,"close_bench":354.130005},{"date":"2021-02-26","open_a":42.366556,"high_a":42.374995,"low_a":41.328492,"close_a":41.345371,"volume_a":23638400,"open_b":108.604623,"high_b":108.854145,"low_b":106.941077,"close_b":107.456772,"volume_b":8329000,"open_bench":356.00096,"close_bench":352.305237},{"date":"2021-03-01","open_a":41.725151,"high_a":42.569106,"low_a":41.691395,"close_a":42.113373,"volume_a":13901200,"open_b":108.637889,"high_b":109.769101,"low_b":108.413307,"close_b":108.646202,"volume_b":5287300,"open_bench":357.14955,"close_bench":360.845245},{"date":"2021-03-02","open_a":42.062731,"high_a":42.67038,"low_a":41.944578,"close_a":42.282158,"volume_a":11755100,"open_b":108.546399,"high_b":109.819011,"low_b":108.44659,"close_b":109.020515,"volume_b":5614000,"open_bench":361.067521,"close_bench":358.029449},{"date":"2021-03-03","open_a":41.961451,"high_a":42.670373,"low_a":41.953009,"close_a":42.180878,"volume_a":15410700,"open_b":107.964157,"high_b":108.571348,"low_b":107.282097,"close_b":107.415184,"volume_b":7505800,"open_bench":357.334812,"close_bench":353.28714},{"date":"2021-03-04","open_a":42.459395,"high_a":43.117679,"low_a":41.877064,"close_a":42.147129,"volume_a":22036400,"open_b":108.36511,"high_b":110.461147,"low_b":107.585391,"close_b":108.012978,"volume_b":9102800,"open_bench":353.101886,"close_bench":348.915283},{"date":"2021-03-05","open_a":42.23997,"high_a":42.999531,"low_a":42.113378,"close_a":42.864498,"volume_a":21310800,"open_b":108.692106,"high_b":111.852934,"low_b":108.314822,"close_b":111.534332,"volume_b":10186500,"open_bench":352.397908,"close_bench":355.334106},{"date":"2021-03-08","open_a":43.438381,"high_a":44.299216,"low_a":43.176754,"close_a":43.581852,"volume_a":25084900,"open_b":111.701988,"high_b":112.498479,"low_b":110.662362,"close_b":110.779739,"volume_b":8634900,"open_bench":356.288186,"close_bench":353.565033},{"date":"2021-03-09","open_a":43.649363,"high_a":43.784396,"low_a":42.889802,"close_a":42.923561,"volume_a":23082700,"open_b":111.458864,"high_b":112.272127,"low_b":110.670752,"close_b":110.880356,"volume_b":8930200,"open_bench":357.390422,"close_bench":358.613068},{"date":"2021-03-10","open_a":43.193632,"high_a":43.573413,"low_a":42.805414,"close_a":43.413059,"volume_a":21331600,"open_b":111.10675,"high_b":112.741657,"low_b":109.44668,"close_b":111.995468,"volume_b":7791700,"open_bench":360.947147,"close_bench":360.845245},{"date":"2021-03-11","open_a":43.38774,"high_a":43.40462,"low_a":42.881368,"close_a":42.940445,"volume_a":17417700,"open_b":111.802632,"high_b":112.833879,"low_b":111.593028,"close_b":111.693634,"volume_b":4458900,"open_bench":363.299773,"close_bench":364.503876},{"date":"2021-03-12","open_a":43.051068,"high_a":43.127656,"low_a":42.719186,"close_a":42.855343,"volume_a":17598600,"open_b":111.609769,"high_b":112.171505,"low_b":111.207333,"close_b":111.542694,"volume_b":5275100,"open_bench":363.151661,"close_bench":364.994873},{"date":"2021-03-15","open_a":42.974488,"high_a":43.502095,"low_a":42.94896,"close_a":43.425507,"volume_a":13411900,"open_b":111.978691,"high_b":112.037386,"low_b":110.645615,"close_b":111.534332,"volume_b":5389200,"open_bench":365.24491,"close_bench":367.171509},{"date":"2021-03-16","open_a":43.570173,"high_a":43.842486,"low_a":43.434016,"close_a":43.587193,"volume_a":14214200,"open_b":111.576245,"high_b":112.833867,"low_b":111.299567,"close_b":112.355965,"volume_b":5544700,"open_bench":367.782848,"close_bench":366.708405},{"date":"2021-03-17","open_a":43.612722,"high_a":43.927584,"low_a":43.527625,"close_a":43.604214,"volume_a":17502700,"open_b":112.649423,"high_b":112.817104,"low_b":111.82778,"close_b":112.028992,"volume_b":6199500,"open_bench":365.430212,"close_bench":367.958862},{"date":"2021-03-18","open_a":43.451032,"high_a":43.561657,"low_a":42.965973,"close_a":43.03405,"volume_a":18007300,"open_b":110.972583,"high_b":111.576243,"low_b":110.176092,"close_b":111.115112,"volume_b":6476700,"open_bench":365.383828,"close_bench":362.605103},{"date":"2021-03-19","open_a":42.940448,"high_a":43.468058,"low_a":42.693663,"close_a":43.238293,"volume_a":67845700,"open_b":110.930662,"high_b":113.194379,"low_b":110.821664,"close_b":112.766792,"volume_b":16114400,"open_bench":362.305878,"close_bench":361.934174},{"date":"2021-03-22","open_a":42.821308,"high_a":43.468055,"low_a":42.821308,"close_a":43.399975,"volume_a":17911400,"open_b":113.714203,"high_b":115.868923,"low_b":113.261452,"close_b":115.533562,"volume_b":7813400,"open_bench":362.445185,"close_bench":364.824127},{"date":"2021-03-23","open_a":43.391463,"high_a":43.95311,"low_a":43.255307,"close_a":43.731853,"volume_a":16936700,"open_b":116.078554,"high_b":117.520629,"low_b":115.852179,"close_b":116.925362,"volume_b":6978400,"open_bench":364.192347,"close_bench":361.952789},{"date":"2021-03-24","open_a":43.544639,"high_a":44.157345,"low_a":43.527622,"close_a":43.842484,"volume_a":14997400,"open_b":116.548044,"high_b":117.470305,"low_b":116.355212,"close_b":116.380363,"volume_b":5082300,"open_bench":363.346712,"close_bench":360.112823},{"date":"2021-03-25","open_a":43.995662,"high_a":44.302015,"low_a":43.54464,"close_a":44.267975,"volume_a":17091900,"open_b":116.690551,"high_b":117.587661,"low_b":115.986286,"close_b":117.067848,"volume_b":5827600,"open_bench":358.681675,"close_bench":362.13858},{"date":"2021-03-26","open_a":44.370092,"high_a":45.195543,"low_a":44.046718,"close_a":45.135975,"volume_a":17126700,"open_b":117.059428,"high_b":119.893273,"low_b":116.648613,"close_b":119.641747,"volume_b":6785600,"open_bench":363.281569,"close_bench":367.974426},{"date":"2021-03-29","open_a":44.872175,"high_a":45.90186,"low_a":44.855155,"close_a":45.825272,"volume_a":17514100,"open_b":119.784309,"high_b":121.536605,"low_b":119.407025,"close_b":121.410835,"volume_b":6105800,"open_bench":366.506231,"close_bench":367.788635},{"date":"2021-03-30","open_a":45.74017,"high_a":45.77421,"low_a":45.042367,"close_a":45.229584,"volume_a":14871800,"open_b":120.463422,"high_b":121.360532,"low_b":119.155497,"close_b":119.507629,"volume_b":4372700,"open_bench":366.524824,"close_bench":366.812897},{"date":"2021-03-31","open_a":45.127465,"high_a":45.246602,"low_a":44.676447,"close_a":44.855152,"volume_a":15826500,"open_b":120.161586,"high_b":120.169966,"low_b":118.358987,"close_b":118.593742,"volume_b":6257900,"open_bench":367.37974,"close_bench":368.299713},{"date":"2021-04-01","open_a":45.067893,"high_a":45.229582,"low_a":44.642403,"close_a":44.684952,"volume_a":15834700,"open_b":118.375794,"high_b":119.306435,"low_b":117.470304,"close_b":118.451248,"volume_b":4823400,"open_bench":370.223317,"close_bench":372.277008},{"date":"2021-04-05","open_a":44.54879,"high_a":45.289144,"low_a":44.497733,"close_a":44.940243,"volume_a":16368700,"open_b":118.929122,"high_b":121.016766,"low_b":118.484763,"close_b":120.027443,"volume_b":5980700,"open_bench":374.925498,"close_bench":377.620392},{"date":"2021-04-06","open_a":45.13597,"high_a":45.655068,"low_a":45.016833,"close_a":45.263615,"volume_a":15614300,"open_b":119.926835,"high_b":120.899399,"low_b":119.524399,"close_b":120.45504,"volume_b":4751900,"open_bench":377.062844,"close_bench":377.397369},{"date":"2021-04-07","open_a":45.340202,"high_a":45.527419,"low_a":45.204046,"close_a":45.340202,"volume_a":10062700,"open_b":120.731737,"high_b":120.815584,"low_b":119.910095,"close_b":120.237076,"volume_b":4849200,"open_bench":377.230053,"close_bench":377.834076},{"date":"2021-04-08","open_a":45.2466,"high_a":45.425309,"low_a":45.076407,"close_a":45.204052,"volume_a":9695600,"open_b":120.463422,"high_b":120.86587,"low_b":119.096814,"close_b":119.507629,"volume_b":6725100,"open_bench":379.079294,"close_bench":379.627563},{"date":"2021-04-09","open_a":45.2466,"high_a":45.340208,"low_a":44.94025,"close_a":45.255112,"volume_a":10828200,"open_b":119.256105,"high_b":120.010673,"low_b":118.7279,"close_b":119.532784,"volume_b":4950800,"open_bench":379.506842,"close_bench":382.387573},{"date":"2021-04-12","open_a":45.382764,"high_a":45.569977,"low_a":45.187035,"close_a":45.39978,"volume_a":8565300,"open_b":119.222574,"high_b":120.287364,"low_b":119.096817,"close_b":119.91008,"volume_b":4386700,"open_bench":381.792842,"close_bench":382.526978},{"date":"2021-04-13","open_a":45.135976,"high_a":45.348721,"low_a":44.940251,"close_a":45.178524,"volume_a":11071700,"open_b":119.113599,"high_b":120.706582,"low_b":118.945906,"close_b":119.935242,"volume_b":5283300,"open_bench":382.424751,"close_bench":383.660675},{"date":"2021-04-14","open_a":45.084921,"high_a":45.263626,"low_a":44.804099,"close_a":45.170021,"volume_a":9787600,"open_b":119.532809,"high_b":119.658566,"low_b":118.417716,"close_b":119.147133,"volume_b":4685300,"open_bench":383.632687,"close_bench":382.350311},{"date":"2021-04-15","open_a":45.212562,"high_a":45.663581,"low_a":45.204051,"close_a":45.382759,"volume_a":13078100,"open_b":118.778196,"high_b":119.977136,"low_b":118.568593,"close_b":119.314781,"volume_b":4777100,"open_bench":384.478402,"close_bench":386.457764},{"date":"2021-04-16","open_a":45.731659,"high_a":45.782716,"low_a":45.425306,"close_a":45.680599,"volume_a":17974100,"open_b":120.278995,"high_b":121.000032,"low_b":119.44058,"close_b":120.865883,"volume_b":6481400,"open_bench":387.740088,"close_bench":387.74939},{"date":"2021-04-19","open_a":45.952915,"high_a":46.259268,"low_a":45.357229,"close_a":45.952915,"volume_a":19352900,"open_b":121.142559,"high_b":122.0732,"low_b":120.1113,"close_b":121.360542,"volume_b":6621200,"open_bench":386.820269,"close_bench":385.844513},{"date":"2021-04-20","open_a":45.816762,"high_a":46.454997,"low_a":45.791234,"close_a":46.097584,"volume_a":14419200,"open_b":120.840709,"high_b":122.953504,"low_b":120.673015,"close_b":122.165405,"volume_b":4999700,"open_bench":384.63637,"close_bench":383.01944},{"date":"2021-04-21","open_a":46.157142,"high_a":46.582632,"low_a":46.157142,"close_a":46.472004,"volume_a":13866500,"open_b":123.540447,"high_b":123.917731,"low_b":122.584654,"close_b":123.230225,"volume_b":6316000,"open_bench":382.406152,"close_bench":386.643646},{"date":"2021-04-22","open_a":46.352882,"high_a":46.693272,"low_a":46.225234,"close_a":46.327351,"volume_a":12558900,"open_b":122.609784,"high_b":122.953524,"low_b":121.888746,"close_b":122.467255,"volume_b":4904400,"open_bench":386.476367,"close_bench":383.112366},{"date":"2021-04-23","open_a":46.29331,"high_a":46.454995,"low_a":46.097582,"close_a":46.352879,"volume_a":9020500,"open_b":121.997723,"high_b":122.693609,"low_b":121.620439,"close_b":122.266022,"volume_b":4354700,"open_bench":383.669883,"close_bench":387.266174},{"date":"2021-04-26","open_a":46.352869,"high_a":46.386906,"low_a":45.595498,"close_a":45.663574,"volume_a":11684600,"open_b":122.031275,"high_b":122.215727,"low_b":119.876543,"close_b":120.195145,"volume_b":5817700,"open_bench":387.916685,"close_bench":388.074646},{"date":"2021-04-27","open_a":45.578485,"high_a":45.697621,"low_a":45.374248,"close_a":45.595505,"volume_a":9852400,"open_b":119.574741,"high_b":120.044252,"low_b":119.163926,"close_b":119.801117,"volume_b":4693700,"open_bench":388.372095,"close_bench":387.991089},{"date":"2021-04-28","open_a":45.935897,"high_a":45.995465,"low_a":45.476367,"close_a":45.604015,"volume_a":10868100,"open_b":120.30414,"high_b":120.471821,"low_b":118.828522,"close_b":118.912369,"volume_b":4962700,"open_bench":388.260553,"close_bench":387.879547},{"date":"2021-04-29","open_a":45.561468,"high_a":46.259271,"low_a":45.46786,"close_a":46.174171,"volume_a":15391000,"open_b":119.138724,"high_b":121.092219,"low_b":119.029726,"close_b":120.681404,"volume_b":4699900,"open_bench":390.593061,"close_bench":390.35144},{"date":"2021-04-30","open_a":46.165659,"high_a":46.182679,"low_a":45.689109,"close_a":45.935894,"volume_a":14912600,"open_b":120.161617,"high_b":120.932958,"low_b":119.474111,"close_b":120.865883,"volume_b":5051100,"open_bench":388.093237,"close_bench":387.78656},{"date":"2021-05-03","open_a":46.18268,"high_a":46.633699,"low_a":46.089072,"close_a":46.361385,"volume_a":10417900,"open_b":121.595315,"high_b":123.062541,"low_b":120.815595,"close_b":122.232506,"volume_b":6109800,"open_bench":389.765946,"close_bench":388.622955},{"date":"2021-05-04","open_a":46.437974,"high_a":46.480522,"low_a":45.901855,"close_a":46.072052,"volume_a":14151000,"open_b":122.391785,"high_b":123.05414,"low_b":120.30414,"close_b":120.698196,"volume_b":6603800,"open_bench":386.643649,"close_bench":386.225464},{"date":"2021-05-05","open_a":46.072052,"high_a":46.157152,"low_a":45.70613,"close_a":45.952915,"volume_a":9665900,"open_b":120.496982,"high_b":121.20964,"low_b":120.31253,"close_b":120.639511,"volume_b":5305200,"open_bench":387.860976,"close_bench":386.346252},{"date":"2021-05-06","open_a":46.055024,"high_a":46.472006,"low_a":45.995456,"close_a":46.412437,"volume_a":11572700,"open_b":121.125796,"high_b":122.375039,"low_b":120.74012,"close_b":122.039665,"volume_b":4472200,"open_bench":386.420616,"close_bench":389.431488},{"date":"2021-05-07","open_a":46.327349,"high_a":46.608174,"low_a":46.089075,"close_a":46.386917,"volume_a":10637500,"open_b":121.762987,"high_b":122.40857,"low_b":121.209643,"close_b":122.039665,"volume_b":4181000,"open_bench":390.193481,"close_bench":392.265747},{"date":"2021-05-10","open_a":46.701767,"high_a":47.229373,"low_a":46.659218,"close_a":46.727295,"volume_a":15545800,"open_b":122.819383,"high_b":124.077005,"low_b":122.500781,"close_b":123.012215,"volume_b":5270600,"open_bench":392.618841,"close_bench":388.381348},{"date":"2021-05-11","open_a":46.829422,"high_a":46.923031,"low_a":45.995462,"close_a":46.225227,"volume_a":12986700,"open_b":123.389499,"high_b":123.783555,"low_b":121.586912,"close_b":122.115105,"volume_b":5266100,"open_bench":383.883649,"close_bench":384.915131},{"date":"2021-05-12","open_a":46.259271,"high_a":46.429468,"low_a":45.850801,"close_a":45.986958,"volume_a":15836500,"open_b":121.469538,"high_b":122.190575,"low_b":120.605972,"close_b":120.924561,"volume_b":5651700,"open_bench":382.146,"close_bench":376.73761},{"date":"2021-05-13","open_a":45.92739,"high_a":46.616682,"low_a":45.808254,"close_a":46.386917,"volume_a":15475800,"open_b":120.932966,"high_b":123.347592,"low_b":120.379609,"close_b":122.718781,"volume_b":4849400,"open_bench":378.280247,"close_bench":381.263214},{"date":"2021-05-14","open_a":46.718792,"high_a":46.888985,"low_a":46.540083,"close_a":46.574123,"volume_a":11725300,"open_b":123.414624,"high_b":124.001512,"low_b":122.760662,"close_b":122.903191,"volume_b":4224700,"open_bench":383.98593,"close_bench":387.117584},{"date":"2021-05-17","open_a":46.66774,"high_a":46.820917,"low_a":46.378407,"close_a":46.497543,"volume_a":12119800,"open_b":123.246957,"high_b":123.582318,"low_b":122.165408,"close_b":122.551071,"volume_b":3747100,"open_bench":386.011695,"close_bench":386.132477},{"date":"2021-05-18","open_a":46.284797,"high_a":46.429465,"low_a":46.038012,"close_a":46.242249,"volume_a":13232500,"open_b":122.945124,"high_b":123.20503,"low_b":121.838423,"close_b":121.997711,"volume_b":4527300,"open_bench":386.392653,"close_bench":382.805664},{"date":"2021-05-19","open_a":46.055035,"high_a":46.106095,"low_a":45.621037,"close_a":46.097584,"volume_a":15126100,"open_b":121.268307,"high_b":122.098342,"low_b":120.798797,"close_b":121.930649,"volume_b":4432700,"open_bench":378.140805,"close_bench":381.802124},{"date":"2021-05-20","open_a":46.055029,"high_a":46.633695,"low_a":46.055029,"close_a":46.50605,"volume_a":10948400,"open_b":121.779769,"high_b":123.859034,"low_b":121.662391,"close_b":123.439827,"volume_b":3964600,"open_bench":382.675717,"close_bench":385.909607},{"date":"2021-05-21","open_a":46.633699,"high_a":46.906012,"low_a":46.140132,"close_a":46.480522,"volume_a":16033200,"open_b":123.976388,"high_b":124.345293,"low_b":122.844536,"close_b":123.330818,"volume_b":5194200,"open_bench":387.387031,"close_bench":385.593536},{"date":"2021-05-24","open_a":46.489029,"high_a":46.744322,"low_a":46.369892,"close_a":46.633694,"volume_a":10326100,"open_b":123.78356,"high_b":124.37884,"low_b":123.255355,"close_b":123.959633,"volume_b":3627800,"open_bench":387.823824,"close_bench":389.524414},{"date":"2021-05-25","open_a":46.650723,"high_a":46.650723,"low_a":46.250761,"close_a":46.625195,"volume_a":11916500,"open_b":123.95963,"high_b":124.395596,"low_b":123.255352,"close_b":124.336914,"volume_b":4649300,"open_bench":390.602308,"close_bench":388.660126},{"date":"2021-05-26","open_a":46.786874,"high_a":46.846442,"low_a":46.454992,"close_a":46.829422,"volume_a":16064300,"open_b":124.009933,"high_b":124.647123,"low_b":123.817088,"close_b":124.336914,"volume_b":4462600,"open_bench":389.245622,"close_bench":389.431488},{"date":"2021-05-27","open_a":46.982594,"high_a":47.288948,"low_a":46.616673,"close_a":47.220871,"volume_a":59109600,"open_b":124.554896,"high_b":124.730969,"low_b":122.777461,"close_b":122.836143,"volume_b":9203600,"open_bench":390.453662,"close_bench":389.635895},{"date":"2021-05-28","open_a":47.101735,"high_a":47.408088,"low_a":46.97409,"close_a":47.050678,"volume_a":17011600,"open_b":123.590721,"high_b":124.50459,"low_b":123.448192,"close_b":124.03508,"volume_b":4913700,"open_bench":391.197136,"close_bench":390.332916},{"date":"2021-06-01","open_a":47.093235,"high_a":47.35704,"low_a":46.888999,"close_a":47.042175,"volume_a":13304000,"open_b":124.739349,"high_b":124.798044,"low_b":123.322426,"close_b":123.775177,"volume_b":5530300,"open_bench":392.683968,"close_bench":389.989075},{"date":"2021-06-02","open_a":47.229385,"high_a":47.280445,"low_a":47.025149,"close_a":47.229385,"volume_a":11328500,"open_b":124.253049,"high_b":124.529728,"low_b":123.439786,"close_b":124.269821,"volume_b":4546900,"open_bench":390.639474,"close_bench":390.602295},{"date":"2021-06-03","open_a":46.957065,"high_a":47.433614,"low_a":46.888985,"close_a":47.348515,"volume_a":17364300,"open_b":123.902452,"high_b":124.789209,"low_b":123.437953,"close_b":124.713203,"volume_b":4651600,"open_bench":388.297701,"close_bench":389.152618},{"date":"2021-06-04","open_a":47.56979,"high_a":48.063357,"low_a":47.46767,"close_a":47.859123,"volume_a":18935100,"open_b":124.991926,"high_b":125.490201,"low_b":124.544321,"close_b":124.856796,"volume_b":4427400,"open_bench":390.992688,"close_bench":392.711853},{"date":"2021-06-07","open_a":47.918676,"high_a":47.944205,"low_a":47.493186,"close_a":47.688911,"volume_a":14010800,"open_b":124.789241,"high_b":125.20307,"low_b":123.902484,"close_b":124.7808,"volume_b":4724100,"open_bench":392.702454,"close_bench":392.33075},{"date":"2021-06-08","open_a":47.867616,"high_a":47.867616,"low_a":47.271929,"close_a":47.357029,"volume_a":10968300,"open_b":124.738573,"high_b":124.882144,"low_b":123.235293,"close_b":123.615334,"volume_b":5997200,"open_bench":393.185684,"close_bench":392.414398},{"date":"2021-06-09","open_a":47.442121,"high_a":47.586786,"low_a":47.212357,"close_a":47.212357,"volume_a":9838800,"open_b":124.358522,"high_b":124.645663,"low_b":123.319741,"close_b":123.387306,"volume_b":4008000,"open_bench":393.2508,"close_bench":391.82901},{"date":"2021-06-10","open_a":47.578281,"high_a":47.756986,"low_a":47.450633,"close_a":47.578281,"volume_a":12444400,"open_b":123.564635,"high_b":125.00035,"low_b":123.412624,"close_b":124.654099,"volume_b":4698200,"open_bench":393.046337,"close_bench":393.65036},{"date":"2021-06-11","open_a":47.646358,"high_a":47.833572,"low_a":47.365533,"close_a":47.791023,"volume_a":11825800,"open_b":125.034145,"high_b":125.093256,"low_b":123.970016,"close_b":124.73011,"volume_b":4364600,"open_bench":394.198697,"close_bench":394.300903},{"date":"2021-06-14","open_a":47.748159,"high_a":47.765307,"low_a":47.328038,"close_a":47.628124,"volume_a":9710800,"open_b":124.544291,"high_b":125.52396,"low_b":123.969997,"close_b":125.473289,"volume_b":3831900,"open_bench":394.41231,"close_bench":395.183624},{"date":"2021-06-15","open_a":47.825333,"high_a":47.833909,"low_a":47.319471,"close_a":47.508099,"volume_a":11154200,"open_b":125.75202,"high_b":125.980049,"low_b":125.051063,"close_b":125.481773,"volume_b":4256500,"open_bench":395.332322,"close_bench":394.458801},{"date":"2021-06-16","open_a":47.533811,"high_a":47.55096,"low_a":46.847898,"close_a":46.873619,"volume_a":15211700,"open_b":125.718236,"high_b":126.0645,"low_b":123.936255,"close_b":124.23185,"volume_b":5492100,"open_bench":394.598262,"close_bench":392.25647},{"date":"2021-06-17","open_a":46.873617,"high_a":47.285168,"low_a":46.856472,"close_a":47.113689,"volume_a":10658400,"open_b":124.358545,"high_b":125.83649,"low_b":124.071404,"close_b":125.431114,"volume_b":5316400,"open_bench":391.847571,"close_bench":392.126343},{"date":"2021-06-18","open_a":46.736443,"high_a":46.830756,"low_a":46.059106,"close_a":46.101974,"volume_a":31445600,"open_b":124.975016,"high_b":124.975016,"low_b":122.652531,"close_b":122.812996,"volume_b":10661700,"open_bench":388.859566,"close_bench":386.836456},{"date":"2021-06-21","open_a":46.487796,"high_a":46.667847,"low_a":46.299168,"close_a":46.60783,"volume_a":14404300,"open_b":123.091697,"high_b":123.927796,"low_b":122.838332,"close_b":123.775772,"volume_b":4904300,"open_bench":388.58916,"close_bench":392.374359},{"date":"2021-06-22","open_a":46.590677,"high_a":47.027946,"low_a":46.522084,"close_a":46.779305,"volume_a":13072800,"open_b":123.919379,"high_b":124.569679,"low_b":123.54778,"close_b":123.961609,"volume_b":3518000,"open_bench":392.365093,"close_bench":394.472107},{"date":"2021-06-23","open_a":46.779308,"high_a":46.813602,"low_a":46.393481,"close_a":46.402054,"volume_a":12339200,"open_b":123.353513,"high_b":123.412637,"low_b":122.314732,"close_b":122.331627,"volume_b":3901700,"open_bench":394.546645,"close_bench":393.996582},{"date":"2021-06-24","open_a":46.522079,"high_a":46.659262,"low_a":46.299159,"close_a":46.633541,"volume_a":11488400,"open_b":122.340087,"high_b":123.252192,"low_b":122.128952,"close_b":123.024162,"volume_b":3818900,"open_bench":396.131553,"close_bench":396.327332},{"date":"2021-06-25","open_a":46.504946,"high_a":46.702146,"low_a":46.307743,"close_a":46.573536,"volume_a":18880300,"open_b":123.100135,"high_b":123.97001,"low_b":122.660983,"close_b":123.649094,"volume_b":6387600,"open_bench":397.073293,"close_bench":397.735229},{"date":"2021-06-28","open_a":46.513515,"high_a":46.616401,"low_a":46.299167,"close_a":46.522087,"volume_a":10556900,"open_b":123.944713,"high_b":124.645683,"low_b":123.826478,"close_b":124.181183,"volume_b":4282800,"open_bench":398.257304,"close_bench":398.536987},{"date":"2021-06-29","open_a":46.410627,"high_a":46.590678,"low_a":46.059096,"close_a":46.179131,"volume_a":12300900,"open_b":124.527398,"high_b":124.738533,"low_b":123.640629,"close_b":124.096687,"volume_b":3197700,"open_bench":398.919122,"close_bench":398.751312},{"date":"2021-06-30","open_a":46.12769,"high_a":46.453499,"low_a":46.084821,"close_a":46.393482,"volume_a":14614200,"open_b":124.510511,"high_b":125.25371,"low_b":124.31627,"close_b":125.135475,"volume_b":4943700,"open_bench":398.294562,"close_bench":399.087036},{"date":"2021-07-01","open_a":46.590683,"high_a":46.710718,"low_a":46.179136,"close_a":46.264874,"volume_a":13214700,"open_b":125.059487,"high_b":125.904027,"low_b":124.941252,"close_b":125.160828,"volume_b":3997600,"open_bench":399.842251,"close_bench":401.296661},{"date":"2021-07-02","open_a":46.299172,"high_a":46.684999,"low_a":46.299172,"close_a":46.453503,"volume_a":10604600,"open_b":125.751976,"high_b":126.495175,"low_b":125.464835,"close_b":125.760429,"volume_b":3835500,"open_bench":402.452717,"close_bench":404.363953},{"date":"2021-07-06","open_a":46.290595,"high_a":46.376333,"low_a":45.913341,"close_a":46.196281,"volume_a":15278200,"open_b":125.895581,"high_b":126.191162,"low_b":124.797676,"close_b":125.946251,"volume_b":4313800,"open_bench":404.419885,"close_bench":403.627411},{"date":"2021-07-07","open_a":46.059102,"high_a":46.624981,"low_a":46.059102,"close_a":46.573536,"volume_a":14377700,"open_b":126.081346,"high_b":127.272151,"low_b":125.836436,"close_b":126.503616,"volume_b":4225000,"open_bench":404.307992,"close_bench":405.053833},{"date":"2021-07-08","open_a":46.307744,"high_a":46.444926,"low_a":46.076251,"close_a":46.410633,"volume_a":11943900,"open_b":126.191198,"high_b":127.246873,"low_b":125.996957,"close_b":126.562798,"volume_b":4216000,"open_bench":399.75834,"close_bench":401.75351},{"date":"2021-07-09","open_a":46.513521,"high_a":46.745018,"low_a":46.453504,"close_a":46.693573,"volume_a":10847000,"open_b":126.182733,"high_b":126.942814,"low_b":125.980039,"close_b":126.241844,"volume_b":4088600,"open_bench":403.254469,"close_bench":406.042084},{"date":"2021-07-12","open_a":46.573534,"high_a":46.813602,"low_a":46.530665,"close_a":46.710716,"volume_a":15107600,"open_b":126.250315,"high_b":126.503667,"low_b":125.650686,"close_b":126.267197,"volume_b":4873700,"open_bench":405.958175,"close_bench":407.49649},{"date":"2021-07-13","open_a":47.113689,"high_a":47.413775,"low_a":46.94221,"close_a":47.173706,"volume_a":15170800,"open_b":127.652204,"high_b":129.983142,"low_b":127.314393,"close_b":129.180832,"volume_b":8750800,"open_bench":406.713391,"close_bench":406.107391},{"date":"2021-07-14","open_a":47.173713,"high_a":48.314043,"low_a":47.122268,"close_a":48.236877,"volume_a":22002700,"open_b":129.121727,"high_b":131.021942,"low_b":129.045722,"close_b":130.515213,"volume_b":6390300,"open_bench":407.794868,"close_bench":406.713379},{"date":"2021-07-15","open_a":48.219725,"high_a":48.416925,"low_a":47.936784,"close_a":48.391201,"volume_a":15068200,"open_b":130.118302,"high_b":131.207753,"low_b":129.434213,"close_b":131.114853,"volume_b":4920600,"open_bench":405.380155,"close_bench":405.324219},{"date":"2021-07-16","open_a":48.408356,"high_a":48.596983,"low_a":48.236877,"close_a":48.356915,"volume_a":14860500,"open_b":131.478004,"high_b":132.457674,"low_b":130.996623,"close_b":131.596252,"volume_b":5852500,"open_bench":406.498915,"close_bench":402.144989},{"date":"2021-07-19","open_a":48.082545,"high_a":48.314038,"low_a":47.293742,"close_a":47.782455,"volume_a":19527000,"open_b":131.739795,"high_b":133.048822,"low_b":130.701002,"close_b":131.57933,"volume_b":5750200,"open_bench":397.343603,"close_bench":396.206177},{"date":"2021-07-20","open_a":47.78245,"high_a":48.296887,"low_a":47.602398,"close_a":47.868191,"volume_a":16257900,"open_b":131.917169,"high_b":132.871503,"low_b":131.376664,"close_b":131.596252,"volume_b":5213500,"open_bench":396.868079,"close_bench":401.883942},{"date":"2021-07-21","open_a":49.154291,"high_a":49.351491,"low_a":48.382637,"close_a":48.485523,"volume_a":20918200,"open_b":131.15707,"high_b":131.849586,"low_b":130.540559,"close_b":131.1064,"volume_b":4905900,"open_bench":403.077377,"close_bench":405.137787},{"date":"2021-07-22","open_a":48.545534,"high_a":48.648424,"low_a":48.056821,"close_a":48.416927,"volume_a":13402700,"open_b":131.064178,"high_b":131.54556,"low_b":130.033839,"close_b":131.064178,"volume_b":3333500,"open_bench":405.314877,"close_bench":405.986145},{"date":"2021-07-23","open_a":48.519814,"high_a":49.154286,"low_a":48.39978,"close_a":48.879917,"volume_a":12144000,"open_b":131.376646,"high_b":133.259967,"low_b":131.1064,"close_b":132.744797,"volume_b":3588400,"open_bench":407.906747,"close_bench":410.162964},{"date":"2021-07-26","open_a":48.777036,"high_a":48.974236,"low_a":48.494098,"close_a":48.922794,"volume_a":8681100,"open_b":132.415433,"high_b":132.989727,"low_b":131.773588,"close_b":132.651917,"volume_b":2999800,"open_bench":409.575609,"close_bench":411.169861},{"date":"2021-07-27","open_a":48.965654,"high_a":49.334333,"low_a":48.802748,"close_a":49.094261,"volume_a":12794400,"open_b":132.854639,"high_b":133.994774,"low_b":132.618169,"close_b":133.386703,"volume_b":4043900,"open_bench":410.134916,"close_bench":409.295837},{"date":"2021-07-28","open_a":48.862782,"high_a":49.008537,"low_a":48.55412,"close_a":48.648434,"volume_a":9858000,"open_b":133.099534,"high_b":133.403569,"low_b":131.672259,"close_b":132.162094,"volume_b":3556800,"open_bench":409.920587,"close_bench":409.128113},{"date":"2021-07-29","open_a":48.93136,"high_a":49.08569,"low_a":48.751308,"close_a":48.914211,"volume_a":9599100,"open_b":132.787017,"high_b":132.955923,"low_b":131.917142,"close_b":132.432312,"volume_b":3055600,"open_bench":410.051111,"close_bench":410.824921},{"date":"2021-07-30","open_a":48.999961,"high_a":49.085699,"low_a":48.785613,"close_a":48.897072,"volume_a":11727000,"open_b":132.905258,"high_b":133.099511,"low_b":132.271853,"close_b":132.550552,"volume_b":3757300,"open_bench":408.270317,"close_bench":408.829712},{"date":"2021-08-02","open_a":49.042817,"high_a":49.077113,"low_a":48.622693,"close_a":48.768452,"volume_a":9778000,"open_b":132.59279,"high_b":132.68569,"low_b":131.334421,"close_b":132.018509,"volume_b":3425900,"open_bench":410.535914,"close_bench":407.972046},{"date":"2021-08-03","open_a":48.871343,"high_a":48.871343,"low_a":48.656995,"close_a":48.80275,"volume_a":8825500,"open_b":132.592741,"high_b":132.719417,"low_b":132.010006,"close_b":132.314041,"volume_b":3359900,"open_bench":408.764418,"close_bench":411.290985},{"date":"2021-08-04","open_a":48.656988,"high_a":48.828467,"low_a":48.022516,"close_a":48.099682,"volume_a":11888400,"open_b":131.90871,"high_b":132.305645,"low_b":129.949371,"close_b":130.101395,"volume_b":4846300,"open_bench":410.013755,"close_bench":409.267914},{"date":"2021-08-05","open_a":48.391205,"high_a":48.519815,"low_a":48.262598,"close_a":48.44265,"volume_a":9806800,"open_b":130.886824,"high_b":131.064183,"low_b":129.949384,"close_b":130.320984,"volume_b":3733500,"open_bench":410.423998,"close_bench":411.859772},{"date":"2021-08-06","open_a":48.528398,"high_a":48.648436,"low_a":48.391215,"close_a":48.562695,"volume_a":10409300,"open_b":129.864965,"high_b":130.701052,"low_b":129.755171,"close_b":130.337906,"volume_b":3430900,"open_bench":412.176725,"close_bench":412.540314},{"date":"2021-08-09","open_a":48.699873,"high_a":48.768466,"low_a":48.382639,"close_a":48.571266,"volume_a":8859900,"open_b":130.599632,"high_b":130.726308,"low_b":129.814216,"close_b":130.354721,"volume_b":2636600,"open_bench":412.512333,"close_bench":412.204681},{"date":"2021-08-10","open_a":48.734152,"high_a":49.01709,"low_a":48.579821,"close_a":48.699856,"volume_a":10906900,"open_b":130.320946,"high_b":131.165486,"low_b":129.898676,"close_b":130.422287,"volume_b":2670200,"open_bench":412.65223,"close_bench":412.717499},{"date":"2021-08-11","open_a":48.777029,"high_a":49.025674,"low_a":48.622699,"close_a":48.639847,"volume_a":8370700,"open_b":130.658798,"high_b":131.765143,"low_b":130.456104,"close_b":130.937485,"volume_b":3058000,"open_bench":413.780282,"close_bench":413.742981},{"date":"2021-08-12","open_a":48.639848,"high_a":48.854196,"low_a":48.639848,"close_a":48.734161,"volume_a":6169200,"open_b":131.097946,"high_b":131.15707,"low_b":130.565895,"close_b":130.96283,"volume_b":1958700,"open_bench":413.593861,"close_bench":414.983002},{"date":"2021-08-13","open_a":48.708436,"high_a":49.128556,"low_a":48.674139,"close_a":49.068539,"volume_a":7714200,"open_b":130.988148,"high_b":132.314069,"low_b":130.81923,"close_b":132.187393,"volume_b":2113800,"open_bench":415.43051,"close_bench":415.73819},{"date":"2021-08-16","open_a":49.042825,"high_a":49.29147,"low_a":48.897067,"close_a":49.282894,"volume_a":7972000,"open_b":132.53366,"high_b":133.538664,"low_b":132.364754,"close_b":133.479553,"volume_b":3224000,"open_bench":414.442341,"close_bench":416.717194},{"date":"2021-08-17","open_a":49.265739,"high_a":49.35148,"low_a":48.879912,"close_a":49.111408,"volume_a":10346400,"open_b":133.361298,"high_b":133.817344,"low_b":132.795458,"close_b":133.352844,"volume_b":3736300,"open_bench":414.17192,"close_bench":413.985474},{"date":"2021-08-18","open_a":49.034249,"high_a":49.068546,"low_a":48.374057,"close_a":48.44265,"volume_a":13873700,"open_b":133.167062,"high_b":133.606226,"low_b":131.917146,"close_b":131.984711,"volume_b":3419200,"open_bench":412.978528,"close_bench":409.454376},{"date":"2021-08-19","open_a":48.065399,"high_a":48.854198,"low_a":47.971086,"close_a":48.751312,"volume_a":9223700,"open_b":131.596246,"high_b":134.636582,"low_b":131.562457,"close_b":134.205872,"volume_b":4745600,"open_bench":406.741399,"close_bench":410.088409},{"date":"2021-08-20","open_a":48.734174,"high_a":48.897077,"low_a":48.511253,"close_a":48.562695,"volume_a":10543600,"open_b":134.222717,"high_b":134.813905,"low_b":133.648436,"close_b":133.732895,"volume_b":3484100,"open_bench":410.433316,"close_bench":413.35144},{"date":"2021-08-23","open_a":48.588401,"high_a":48.656994,"low_a":48.33976,"close_a":48.391201,"volume_a":8912800,"open_b":133.293775,"high_b":133.876509,"low_b":132.863064,"close_b":133.310669,"volume_b":3102100,"open_bench":415.029619,"close_bench":416.987488},{"date":"2021-08-24","open_a":48.562688,"high_a":48.562688,"low_a":47.928219,"close_a":48.02253,"volume_a":12067200,"open_b":133.226195,"high_b":133.226195,"low_b":131.477991,"close_b":131.65535,"volume_b":3760800,"open_bench":417.649445,"close_bench":417.649445},{"date":"2021-08-25","open_a":48.065391,"high_a":48.159701,"low_a":47.78245,"close_a":48.073963,"volume_a":11270700,"open_b":131.207761,"high_b":131.469567,"low_b":130.194316,"close_b":131.072632,"volume_b":3346700,"open_bench":417.835883,"close_bench":418.525787},{"date":"2021-08-26","open_a":47.971079,"high_a":47.9968,"low_a":47.593825,"close_a":47.619549,"volume_a":10331500,"open_b":131.334403,"high_b":131.469533,"low_b":130.135158,"close_b":130.591217,"volume_b":3220700,"open_bench":418.246033,"close_bench":416.055115},{"date":"2021-08-27","open_a":47.72244,"high_a":47.833902,"low_a":47.542388,"close_a":47.713867,"volume_a":8844100,"open_b":130.768553,"high_b":131.249947,"low_b":130.236502,"close_b":130.853012,"volume_b":4166500,"open_bench":416.856963,"close_bench":419.775116},{"date":"2021-08-30","open_a":47.748166,"high_a":48.185437,"low_a":47.705297,"close_a":48.168289,"volume_a":10034700,"open_b":131.055701,"high_b":131.8918,"low_b":130.912131,"close_b":131.74823,"volume_b":2540300,"open_bench":420.446458,"close_bench":421.621185},{"date":"2021-08-31","open_a":48.151137,"high_a":48.459799,"low_a":48.013954,"close_a":48.279747,"volume_a":14185700,"open_b":131.824226,"high_b":132.145156,"low_b":131.190821,"close_b":132.077591,"volume_b":5291200,"open_bench":421.527826,"close_bench":420.996399},{"date":"2021-09-01","open_a":48.339763,"high_a":48.699866,"low_a":48.254022,"close_a":48.605553,"volume_a":9518900,"open_b":132.457664,"high_b":133.597798,"low_b":132.178964,"close_b":133.361328,"volume_b":4231400,"open_bench":421.928753,"close_bench":421.220184},{"date":"2021-09-02","open_a":48.674145,"high_a":48.897065,"low_a":48.365483,"close_a":48.674145,"volume_a":11653200,"open_b":133.323035,"high_b":133.7482,"low_b":132.540735,"close_b":133.459091,"volume_b":2862500,"open_bench":422.637258,"close_bench":422.516052},{"date":"2021-09-03","open_a":48.408351,"high_a":48.674143,"low_a":48.228299,"close_a":48.639847,"volume_a":13220400,"open_b":132.821333,"high_b":133.858735,"low_b":132.566232,"close_b":133.578125,"volume_b":2035100,"open_bench":421.387989,"close_bench":422.413513},{"date":"2021-09-07","open_a":48.528389,"high_a":48.605555,"low_a":47.593832,"close_a":47.731014,"volume_a":20035300,"open_b":133.059429,"high_b":133.29752,"low_b":130.678506,"close_b":131.401291,"volume_b":4609900,"open_bench":422.068623,"close_bench":420.903229},{"date":"2021-09-08","open_a":47.679569,"high_a":48.425499,"low_a":47.679569,"close_a":48.374054,"volume_a":12040100,"open_b":131.511844,"high_b":133.289028,"low_b":131.205725,"close_b":133.263519,"volume_b":3199100,"open_bench":420.371775,"close_bench":420.390411},{"date":"2021-09-09","open_a":48.168278,"high_a":48.374051,"low_a":47.842469,"close_a":47.893913,"volume_a":12545400,"open_b":133.399578,"high_b":133.476104,"low_b":132.141082,"close_b":132.421692,"volume_b":3560400,"open_bench":420.194648,"close_bench":418.591064},{"date":"2021-09-10","open_a":48.022524,"high_a":48.125413,"low_a":47.602404,"close_a":47.679569,"volume_a":10573700,"open_b":132.634285,"high_b":132.863877,"low_b":131.809465,"close_b":132.192123,"volume_b":3851500,"open_bench":420.511684,"close_bench":415.29071},{"date":"2021-09-13","open_a":47.791026,"high_a":48.356904,"low_a":47.791026,"close_a":48.073963,"volume_a":20274300,"open_b":132.600286,"high_b":133.841759,"low_b":132.132602,"close_b":132.44722,"volume_b":3367200,"open_bench":418.274112,"close_bench":416.353516},{"date":"2021-09-14","open_a":48.307205,"high_a":48.359038,"low_a":47.987576,"close_a":48.108517,"volume_a":13918900,"open_b":132.974426,"high_b":133.221016,"low_b":131.647915,"close_b":131.928513,"volume_b":2808600,"open_bench":417.789258,"close_bench":414.106628},{"date":"2021-09-15","open_a":48.099882,"high_a":48.376318,"low_a":47.875276,"close_a":48.272655,"volume_a":15746000,"open_b":131.988039,"high_b":132.974423,"low_b":131.613892,"close_b":132.668304,"volume_b":3702400,"open_bench":414.526103,"close_bench":417.56546},{"date":"2021-09-16","open_a":48.134447,"high_a":48.367691,"low_a":47.356969,"close_a":47.814816,"volume_a":17430400,"open_b":132.132622,"high_b":132.753358,"low_b":130.7891,"close_b":132.277176,"volume_b":4589900,"open_bench":417.043376,"close_bench":416.903534},{"date":"2021-09-17","open_a":47.52974,"high_a":47.676595,"low_a":46.968228,"close_a":47.028698,"volume_a":33370200,"open_b":131.996572,"high_b":132.226165,"low_b":130.806117,"close_b":131.061218,"volume_b":8127400,"open_bench":416.134843,"close_bench":412.84256},{"date":"2021-09-20","open_a":46.743632,"high_a":46.994153,"low_a":46.415363,"close_a":46.700439,"volume_a":27542500,"open_b":130.865639,"high_b":131.758487,"low_b":129.607155,"close_b":130.619049,"volume_b":5484600,"open_bench":406.74436,"close_bench":405.95871},{"date":"2021-09-21","open_a":46.985501,"high_a":47.356963,"low_a":46.596762,"close_a":46.691788,"volume_a":16418200,"open_b":131.469306,"high_b":132.753298,"low_b":130.074768,"close_b":130.559448,"volume_b":4479800,"open_bench":408.287659,"close_bench":405.575287},{"date":"2021-09-22","open_a":47.089168,"high_a":47.132364,"low_a":46.700432,"close_a":46.760902,"volume_a":12719700,"open_b":131.477838,"high_b":131.877494,"low_b":130.482955,"close_b":130.959137,"volume_b":3001000,"open_bench":407.838629,"close_bench":409.531525},{"date":"2021-09-23","open_a":46.873203,"high_a":47.184194,"low_a":46.6486,"close_a":46.683155,"volume_a":13836200,"open_b":130.687071,"high_b":131.741471,"low_b":130.559514,"close_b":131.061218,"volume_b":2712500,"open_bench":411.39284,"close_bench":414.507385},{"date":"2021-09-24","open_a":46.648595,"high_a":46.855924,"low_a":46.527655,"close_a":46.55357,"volume_a":9682200,"open_b":130.738052,"high_b":131.477834,"low_b":130.567988,"close_b":131.120697,"volume_b":2366100,"open_bench":412.879926,"close_bench":415.190125},{"date":"2021-09-27","open_a":46.51902,"high_a":46.864567,"low_a":46.27714,"close_a":46.311695,"volume_a":12429500,"open_b":130.780554,"high_b":131.078162,"low_b":129.581588,"close_b":129.862198,"volume_b":4482000,"open_bench":414.161305,"close_bench":414.002319},{"date":"2021-09-28","open_a":46.182111,"high_a":46.182111,"low_a":45.292333,"close_a":45.473743,"volume_a":19926100,"open_b":129.564581,"high_b":129.675128,"low_b":127.634344,"close_b":128.442154,"volume_b":5910900,"open_bench":411.243181,"close_bench":405.659424},{"date":"2021-09-29","open_a":45.473743,"high_a":45.922953,"low_a":45.404636,"close_a":45.750179,"volume_a":13941500,"open_b":128.688775,"high_b":130.244878,"low_b":128.136066,"close_b":129.505096,"volume_b":3326500,"open_bench":407.034247,"close_bench":406.342133},{"date":"2021-09-30","open_a":45.93159,"high_a":46.069812,"low_a":45.292334,"close_a":45.326889,"volume_a":17672000,"open_b":130.287379,"high_b":130.46594,"low_b":127.863936,"close_b":127.897957,"volume_b":4949200,"open_bench":407.810559,"close_bench":401.375702},{"date":"2021-10-01","open_a":45.594677,"high_a":45.957501,"low_a":45.413267,"close_a":45.802006,"volume_a":16277400,"open_b":128.697271,"high_b":128.918352,"low_b":127.141168,"close_b":128.357132,"volume_b":4274300,"open_bench":403.096625,"close_bench":406.145691},{"date":"2021-10-04","open_a":45.836566,"high_a":46.24258,"low_a":45.266417,"close_a":45.776096,"volume_a":18973400,"open_b":128.297606,"high_b":130.100299,"low_b":127.107138,"close_b":127.719376,"volume_b":5404900,"open_bench":404.985986,"close_bench":400.908081},{"date":"2021-10-05","open_a":45.802007,"high_a":46.190746,"low_a":45.724259,"close_a":45.85384,"volume_a":17173100,"open_b":128.705773,"high_b":129.581611,"low_b":127.608855,"close_b":128.476181,"volume_b":6103100,"open_bench":402.404488,"close_bench":405.079468},{"date":"2021-10-06","open_a":45.586045,"high_a":46.449908,"low_a":45.361439,"close_a":46.398075,"volume_a":21284500,"open_b":128.263589,"high_b":131.937016,"low_b":127.515309,"close_b":131.766953,"volume_b":5499200,"open_bench":401.497304,"close_bench":406.763062},{"date":"2021-10-07","open_a":46.639959,"high_a":46.890476,"low_a":46.484463,"close_a":46.544933,"volume_a":13774400,"open_b":132.243113,"high_b":134.139342,"low_b":132.243113,"close_b":132.982895,"volume_b":4605100,"open_bench":410.027233,"close_bench":410.279755},{"date":"2021-10-08","open_a":46.648596,"high_a":46.985502,"low_a":46.562211,"close_a":46.752258,"volume_a":13786800,"open_b":133.135973,"high_b":133.255019,"low_b":132.217617,"close_b":132.676788,"volume_b":4186000,"open_bench":411.046739,"close_bench":409.531525},{"date":"2021-10-11","open_a":46.830016,"high_a":46.985512,"low_a":46.657243,"close_a":46.847294,"volume_a":12378500,"open_b":132.982937,"high_b":133.69721,"low_b":132.3622,"close_b":132.855392,"volume_b":2738100,"open_bench":408.876818,"close_bench":406.56662},{"date":"2021-10-12","open_a":47.011427,"high_a":47.365614,"low_a":46.838657,"close_a":46.847294,"volume_a":19474400,"open_b":132.838352,"high_b":134.224391,"low_b":132.829853,"close_b":133.442078,"volume_b":3195400,"open_bench":407.483244,"close_bench":405.565857},{"date":"2021-10-13","open_a":46.795453,"high_a":47.020059,"low_a":46.467188,"close_a":46.855927,"volume_a":12143300,"open_b":133.892772,"high_b":134.394462,"low_b":132.668296,"close_b":134.343445,"volume_b":3599900,"open_bench":406.585309,"close_bench":407.024902},{"date":"2021-10-14","open_a":46.985498,"high_a":47.348322,"low_a":46.91639,"close_a":47.175549,"volume_a":13704000,"open_b":134.879115,"high_b":135.618909,"low_b":134.870616,"close_b":135.423325,"volume_b":3677100,"open_bench":410.672561,"close_bench":413.871307},{"date":"2021-10-15","open_a":47.201466,"high_a":47.408794,"low_a":46.899115,"close_a":47.063248,"volume_a":17867000,"open_b":135.76351,"high_b":135.76351,"low_b":133.935296,"close_b":135.040726,"volume_b":4313000,"open_bench":415.975751,"close_bench":417.023285},{"date":"2021-10-18","open_a":46.950943,"high_a":46.985499,"low_a":46.527652,"close_a":46.59676,"volume_a":15861800,"open_b":134.522019,"high_b":134.947183,"low_b":133.331564,"close_b":134.428482,"volume_b":3274200,"open_bench":415.246229,"close_bench":418.257904},{"date":"2021-10-19","open_a":46.683146,"high_a":46.821364,"low_a":46.449903,"close_a":46.778172,"volume_a":11068500,"open_b":134.896134,"high_b":136.188637,"low_b":133.960767,"close_b":136.137619,"volume_b":3125800,"open_bench":419.875959,"close_bench":421.48468},{"date":"2021-10-20","open_a":46.80409,"high_a":47.305129,"low_a":46.74362,"close_a":47.192829,"volume_a":12474800,"open_b":136.545798,"high_b":137.651228,"low_b":135.780494,"close_b":137.166534,"volume_b":3716500,"open_bench":421.943042,"close_bench":423.140228},{"date":"2021-10-21","open_a":47.166916,"high_a":47.227387,"low_a":46.83001,"close_a":46.950951,"volume_a":14388300,"open_b":137.234564,"high_b":137.268571,"low_b":135.185268,"close_b":135.355331,"volume_b":4164700,"open_bench":422.541578,"close_bench":424.243835},{"date":"2021-10-22","open_a":46.985501,"high_a":47.106442,"low_a":46.709066,"close_a":47.037334,"volume_a":12828600,"open_b":135.695487,"high_b":136.452279,"low_b":135.60195,"close_b":136.027115,"volume_b":3240300,"open_bench":423.813592,"close_bench":423.80423},{"date":"2021-10-25","open_a":46.976875,"high_a":47.054622,"low_a":46.665883,"close_a":46.847294,"volume_a":11855000,"open_b":136.01011,"high_b":136.01011,"low_b":135.023726,"close_b":135.389374,"volume_b":3281200,"open_bench":424.889203,"close_bench":426.077026},{"date":"2021-10-26","open_a":46.92503,"high_a":47.149637,"low_a":46.847283,"close_a":47.054611,"volume_a":12086700,"open_b":135.346813,"high_b":137.243042,"low_b":135.287296,"close_b":137.04747,"volume_b":3684700,"open_bench":427.620244,"close_bench":426.460449},{"date":"2021-10-27","open_a":48.350397,"high_a":48.445423,"low_a":47.745693,"close_a":47.961658,"volume_a":24019800,"open_b":137.540694,"high_b":137.61722,"low_b":135.678472,"close_b":136.57132,"volume_b":3751800,"open_bench":426.918846,"close_bench":424.571228},{"date":"2021-10-28","open_a":48.160364,"high_a":48.471355,"low_a":48.048061,"close_a":48.410885,"volume_a":15928000,"open_b":136.775386,"high_b":137.523667,"low_b":136.452257,"close_b":137.43013,"volume_b":3589600,"open_bench":425.992836,"close_bench":428.667816},{"date":"2021-10-29","open_a":48.264018,"high_a":48.782338,"low_a":48.18627,"close_a":48.69595,"volume_a":24463700,"open_b":136.928423,"high_b":137.974324,"low_b":136.826388,"close_b":137.413116,"volume_b":4558200,"open_bench":426.376301,"close_bench":429.537628},{"date":"2021-11-01","open_a":48.713229,"high_a":48.721869,"low_a":48.289938,"close_a":48.523178,"volume_a":11651100,"open_b":137.073021,"high_b":137.328122,"low_b":136.205682,"close_b":137.124039,"volume_b":3017100,"open_bench":430.519679,"close_bench":430.27652},{"date":"2021-11-02","open_a":48.782339,"high_a":48.808254,"low_a":48.350408,"close_a":48.462708,"volume_a":11498900,"open_b":137.370629,"high_b":138.442039,"low_b":136.851928,"close_b":138.382523,"volume_b":3464500,"open_bench":430.444873,"close_bench":432.016174},{"date":"2021-11-03","open_a":48.376314,"high_a":48.66139,"low_a":48.186263,"close_a":48.626835,"volume_a":10788300,"open_b":137.880815,"high_b":139.785557,"low_b":137.880815,"close_b":139.70903,"volume_b":3595000,"open_bench":431.45497,"close_bench":434.653717},{"date":"2021-11-04","open_a":48.471351,"high_a":48.92056,"low_a":48.40224,"close_a":48.894642,"volume_a":10820200,"open_b":139.640972,"high_b":139.930081,"low_b":138.901191,"close_b":139.717499,"volume_b":3310100,"open_bench":435.252412,"close_bench":436.702148},{"date":"2021-11-05","open_a":49.101963,"high_a":49.378399,"low_a":48.885997,"close_a":49.101963,"volume_a":12886700,"open_b":140.771957,"high_b":141.520237,"low_b":139.896119,"close_b":141.154602,"volume_b":3754300,"open_bench":438.918701,"close_bench":438.217224},{"date":"2021-11-08","open_a":49.024221,"high_a":49.145161,"low_a":48.488627,"close_a":48.6614,"volume_a":12828000,"open_b":140.993027,"high_b":141.23963,"low_b":137.030492,"close_b":138.118912,"volume_b":5766400,"open_bench":439.31151,"close_bench":438.591309},{"date":"2021-11-09","open_a":48.7305,"high_a":48.825525,"low_a":48.454064,"close_a":48.79961,"volume_a":10155800,"open_b":138.0169,"high_b":139.334913,"low_b":137.7533,"close_b":139.037292,"volume_b":3778600,"open_bench":438.956152,"close_bench":437.141663},{"date":"2021-11-10","open_a":48.842807,"high_a":49.171072,"low_a":48.626841,"close_a":48.998302,"volume_a":8813600,"open_b":140.083201,"high_b":140.304282,"low_b":138.629133,"close_b":139.487961,"volume_b":2683500,"open_bench":435.458124,"close_bench":433.624939},{"date":"2021-11-11","open_a":49.093327,"high_a":49.136519,"low_a":48.868721,"close_a":49.015579,"volume_a":7257700,"open_b":139.428401,"high_b":139.436899,"low_b":138.220936,"close_b":138.339981,"volume_b":2453000,"open_bench":435.111944,"close_bench":433.765106},{"date":"2021-11-12","open_a":49.024222,"high_a":49.231551,"low_a":48.851449,"close_a":48.903282,"volume_a":10162500,"open_b":138.612119,"high_b":139.39442,"low_b":137.821319,"close_b":138.306,"volume_b":3540900,"open_bench":435.027802,"close_bench":437.038696},{"date":"2021-11-15","open_a":48.834171,"high_a":48.963752,"low_a":48.609568,"close_a":48.911919,"volume_a":10289200,"open_b":138.186943,"high_b":139.351902,"low_b":138.169946,"close_b":139.283875,"volume_b":3357100,"open_bench":438.320152,"close_bench":437.188416},{"date":"2021-11-16","open_a":48.911919,"high_a":49.188355,"low_a":48.540458,"close_a":48.566376,"volume_a":13756500,"open_b":139.683524,"high_b":140.295762,"low_b":138.799175,"close_b":138.867203,"volume_b":2899400,"open_bench":436.926533,"close_bench":438.918732},{"date":"2021-11-17","open_a":48.410876,"high_a":48.419513,"low_a":48.07397,"close_a":48.298573,"volume_a":13288700,"open_b":138.280451,"high_b":139.173299,"low_b":137.821279,"close_b":138.918198,"volume_b":3332200,"open_bench":438.656855,"close_bench":437.852509},{"date":"2021-11-18","open_a":48.229474,"high_a":48.315859,"low_a":47.633406,"close_a":47.86665,"volume_a":15380100,"open_b":138.442038,"high_b":139.045777,"low_b":137.540692,"close_b":138.960739,"volume_b":3498900,"open_bench":438.881282,"close_bench":439.3396},{"date":"2021-11-19","open_a":47.892555,"high_a":47.935751,"low_a":47.426072,"close_a":47.62476,"volume_a":15813700,"open_b":139.879072,"high_b":140.329745,"low_b":138.977726,"close_b":139.292343,"volume_b":5242200,"open_bench":439.227345,"close_bench":438.553955},{"date":"2021-11-22","open_a":47.598841,"high_a":48.393596,"low_a":47.581566,"close_a":47.918472,"volume_a":16905600,"open_b":138.773642,"high_b":141.83483,"low_b":138.756632,"close_b":139.581451,"volume_b":4157200,"open_bench":440.424577,"close_bench":437.319366},{"date":"2021-11-23","open_a":48.082605,"high_a":48.471344,"low_a":47.944387,"close_a":48.272655,"volume_a":13835900,"open_b":139.079773,"high_b":140.916486,"low_b":138.867191,"close_b":140.51683,"volume_b":3878000,"open_bench":436.992017,"close_bench":437.899261},{"date":"2021-11-24","open_a":48.125803,"high_a":48.238106,"low_a":47.538376,"close_a":47.883923,"volume_a":12598900,"open_b":140.380801,"high_b":140.729439,"low_b":138.24648,"close_b":139.232864,"volume_b":3309500,"open_bench":435.907033,"close_bench":439.068359},{"date":"2021-11-26","open_a":47.158267,"high_a":47.296485,"low_a":46.285767,"close_a":46.415344,"volume_a":14754300,"open_b":138.356995,"high_b":139.164805,"low_b":136.80939,"close_b":137.021973,"volume_b":3402900,"open_bench":432.427721,"close_bench":429.275757},{"date":"2021-11-29","open_a":46.691786,"high_a":47.279213,"low_a":46.588123,"close_a":47.149635,"volume_a":22712500,"open_b":138.322998,"high_b":139.94713,"low_b":137.021996,"close_b":139.572983,"volume_b":5305800,"open_bench":434.045795,"close_bench":434.541504},{"date":"2021-11-30","open_a":46.662125,"high_a":46.688244,"low_a":45.652273,"close_a":45.66098,"volume_a":30485200,"open_b":139.309374,"high_b":139.453928,"low_b":134.683583,"close_b":135.86554,"volume_b":7610000,"open_bench":432.109715,"close_bench":426.086365},{"date":"2021-12-01","open_a":46.12237,"high_a":46.592474,"low_a":45.512976,"close_a":45.530388,"volume_a":18719600,"open_b":138.152917,"high_b":138.816172,"low_b":136.095123,"close_b":136.18866,"volume_b":5385600,"open_bench":431.773074,"close_bench":421.35379},{"date":"2021-12-02","open_a":45.791553,"high_a":46.43577,"low_a":45.713202,"close_a":46.200718,"volume_a":17074200,"open_b":137.083248,"high_b":138.418722,"low_b":136.826423,"close_b":137.502716,"volume_b":4619800,"open_bench":421.568891,"close_bench":427.807343},{"date":"2021-12-03","open_a":46.427073,"high_a":46.670829,"low_a":46.122374,"close_a":46.60989,"volume_a":21064400,"open_b":138.230417,"high_b":141.149636,"low_b":138.119123,"close_b":141.004105,"volume_b":6465900,"open_bench":429.462766,"close_bench":424.084778},{"date":"2021-12-06","open_a":47.280227,"high_a":48.098554,"low_a":47.13223,"close_a":47.802563,"volume_a":26624100,"open_b":142.151228,"high_b":143.212753,"low_b":141.92864,"close_b":142.467972,"volume_b":5403100,"open_bench":426.619544,"close_bench":429.107452},{"date":"2021-12-07","open_a":47.898319,"high_a":48.246542,"low_a":47.750322,"close_a":48.063725,"volume_a":23832700,"open_b":142.467969,"high_b":142.930257,"low_b":141.817357,"close_b":142.305313,"volume_b":5460100,"open_bench":434.363753,"close_bench":437.983368},{"date":"2021-12-08","open_a":47.985372,"high_a":48.1856,"low_a":47.323746,"close_a":47.880905,"volume_a":18026300,"open_b":142.228246,"high_b":142.699089,"low_b":140.421928,"close_b":142.553558,"volume_b":3891900,"open_bench":438.376201,"close_bench":439.143127},{"date":"2021-12-09","open_a":47.663278,"high_a":47.994093,"low_a":47.523989,"close_a":47.759041,"volume_a":13846400,"open_b":142.039941,"high_b":142.861779,"low_b":141.123948,"close_b":142.37381,"volume_b":3644500,"open_bench":437.861847,"close_bench":436.178314},{"date":"2021-12-10","open_a":48.09855,"high_a":49.030051,"low_a":47.941849,"close_a":48.995228,"volume_a":23153600,"open_b":143.426761,"high_b":144.779361,"low_b":142.878874,"close_b":144.650955,"volume_b":5025700,"open_bench":438.872045,"close_bench":440.284332},{"date":"2021-12-13","open_a":49.604612,"high_a":50.431647,"low_a":49.587201,"close_a":50.283649,"volume_a":31362800,"open_b":144.616691,"high_b":146.611347,"low_b":143.863354,"close_b":145.370041,"volume_b":5411800,"open_bench":439.769854,"close_bench":436.384064},{"date":"2021-12-14","open_a":49.970262,"high_a":50.640592,"low_a":49.970262,"close_a":50.318485,"volume_a":24806600,"open_b":144.976291,"high_b":145.866603,"low_b":144.07741,"close_b":145.010529,"volume_b":4985000,"open_bench":433.129192,"close_bench":433.381714},{"date":"2021-12-15","open_a":50.431656,"high_a":50.710236,"low_a":50.1879,"close_a":50.54483,"volume_a":24923800,"open_b":145.335802,"high_b":147.056508,"low_b":145.070421,"close_b":146.87674,"volume_b":5507800,"open_bench":433.437916,"close_bench":440.153381},{"date":"2021-12-16","open_a":50.475182,"high_a":51.258689,"low_a":50.405539,"close_a":51.05846,"volume_a":24696900,"open_b":146.876756,"high_b":148.580336,"low_b":146.405912,"close_b":147.090775,"volume_b":6909900,"open_bench":441.995866,"close_bench":436.27182},{"date":"2021-12-17","open_a":50.919178,"high_a":51.293517,"low_a":50.231433,"close_a":50.257549,"volume_a":51874400,"open_b":146.482976,"high_b":147.570195,"low_b":143.863389,"close_b":144.077408,"volume_b":10159600,"open_bench":433.205423,"close_bench":431.628601},{"date":"2021-12-20","open_a":49.796141,"high_a":50.301066,"low_a":49.622028,"close_a":50.092133,"volume_a":20879500,"open_b":143.101439,"high_b":145.104651,"low_b":142.750446,"close_b":145.036163,"volume_b":5263600,"open_bench":426.569585,"close_bench":427.038879},{"date":"2021-12-21","open_a":50.15308,"high_a":50.466483,"low_a":50.083433,"close_a":50.29237,"volume_a":15864900,"open_b":144.873505,"high_b":145.190261,"low_b":143.632199,"close_b":144.608124,"volume_b":3401400,"open_bench":430.445951,"close_bench":434.622681},{"date":"2021-12-22","open_a":50.240131,"high_a":50.658,"low_a":50.048609,"close_a":50.649296,"volume_a":12447400,"open_b":144.642403,"high_b":145.524159,"low_b":144.05171,"close_b":145.507034,"volume_b":4362900,"open_bench":434.369337,"close_bench":438.968414},{"date":"2021-12-23","open_a":50.945296,"high_a":51.023647,"low_a":50.440368,"close_a":50.684128,"volume_a":11027300,"open_b":145.515619,"high_b":146.072075,"low_b":144.890687,"close_b":145.344406,"volume_b":3241700,"open_bench":439.963282,"close_bench":441.699677},{"date":"2021-12-27","open_a":50.492594,"high_a":51.09328,"low_a":50.492594,"close_a":51.05846,"volume_a":9860000,"open_b":145.524151,"high_b":146.868181,"low_b":145.335813,"close_b":146.791138,"volume_b":2868800,"open_bench":443.069983,"close_bench":447.950653},{"date":"2021-12-28","open_a":50.840817,"high_a":51.310918,"low_a":50.753759,"close_a":51.258686,"volume_a":8979900,"open_b":146.782608,"high_b":147.921176,"low_b":146.56002,"close_b":147.55307,"volume_b":2332100,"open_bench":448.382431,"close_bench":447.584625},{"date":"2021-12-29","open_a":51.319626,"high_a":51.450208,"low_a":51.067162,"close_a":51.319626,"volume_a":9996000,"open_b":147.921141,"high_b":148.494722,"low_b":147.184916,"close_b":148.075241,"volume_b":2299500,"open_bench":447.687932,"close_bench":448.157227},{"date":"2021-12-30","open_a":51.415398,"high_a":51.563392,"low_a":51.154227,"close_a":51.171638,"volume_a":7703900,"open_b":148.563247,"high_b":148.631735,"low_b":147.441791,"close_b":147.818466,"volume_b":1988900,"open_bench":448.579535,"close_bench":446.918243},{"date":"2021-12-31","open_a":51.145513,"high_a":51.667849,"low_a":51.101986,"close_a":51.545971,"volume_a":10021300,"open_b":147.638662,"high_b":148.974137,"low_b":147.339031,"close_b":148.708755,"volume_b":2914900,"open_bench":446.430193,"close_bench":445.791931},{"date":"2022-01-03","open_a":51.206455,"high_a":51.633032,"low_a":50.823409,"close_a":51.624325,"volume_a":20187300,"open_b":147.416079,"high_b":148.263598,"low_b":146.020686,"close_b":148.083817,"volume_b":5488900,"open_bench":447.049634,"close_bench":448.373047},{"date":"2022-01-04","open_a":51.998667,"high_a":52.703824,"low_a":51.876789,"close_a":52.486183,"volume_a":26141600,"open_b":147.809865,"high_b":149.342233,"low_b":147.416065,"close_b":148.297821,"volume_b":4717600,"open_bench":449.790294,"close_bench":448.222839},{"date":"2022-01-05","open_a":52.312061,"high_a":53.278386,"low_a":52.277238,"close_a":52.921455,"volume_a":22507300,"open_b":148.409148,"high_b":150.121298,"low_b":148.297854,"close_b":148.802948,"volume_b":6564600,"open_bench":447.856864,"close_bench":439.616058},{"date":"2022-01-06","open_a":52.642891,"high_a":53.208754,"low_a":52.616772,"close_a":52.642891,"volume_a":17902300,"open_b":148.879934,"high_b":150.326702,"low_b":148.297797,"close_b":148.837128,"volume_b":4288700,"open_bench":439.156177,"close_bench":439.203094},{"date":"2022-01-07","open_a":52.477477,"high_a":52.860526,"low_a":52.285955,"close_a":52.521008,"volume_a":12307900,"open_b":149.034053,"high_b":149.547704,"low_b":147.767066,"close_b":149.025497,"volume_b":4916000,"open_bench":439.212525,"close_bench":437.466736},{"date":"2022-01-10","open_a":52.712525,"high_a":52.947578,"low_a":52.390418,"close_a":52.608059,"volume_a":20954300,"open_b":149.333668,"high_b":150.532167,"low_b":148.63168,"close_b":149.102524,"volume_b":5055900,"open_bench":434.28481,"close_bench":436.922241},{"date":"2022-01-11","open_a":52.608058,"high_a":52.660292,"low_a":52.094426,"close_a":52.625469,"volume_a":19369600,"open_b":148.683033,"high_b":149.153877,"low_b":146.919522,"close_b":149.034027,"volume_b":6452300,"open_bench":436.659477,"close_bench":440.901886},{"date":"2022-01-12","open_a":52.695119,"high_a":52.834409,"low_a":52.399128,"close_a":52.703827,"volume_a":15753800,"open_b":148.871429,"high_b":149.239548,"low_b":148.058161,"close_b":148.871429,"volume_b":5268000,"open_bench":442.628966,"close_bench":442.093964},{"date":"2022-01-13","open_a":52.703824,"high_a":53.139105,"low_a":52.573239,"close_a":53.017227,"volume_a":14318000,"open_b":148.708771,"high_b":149.547721,"low_b":148.306415,"close_b":149.111115,"volume_b":4310300,"open_bench":443.192063,"close_bench":436.002472},{"date":"2022-01-14","open_a":53.052045,"high_a":53.496033,"low_a":52.54712,"close_a":53.443798,"volume_a":19729600,"open_b":149.444969,"high_b":150.617787,"low_b":148.44335,"close_b":150.360962,"volume_b":4474500,"open_bench":432.86754,"close_bench":436.180756},{"date":"2022-01-18","open_a":53.156517,"high_a":53.269688,"low_a":52.625474,"close_a":53.017227,"volume_a":21976700,"open_b":148.6317,"high_b":149.607625,"low_b":148.229344,"close_b":148.922775,"volume_b":5789500,"open_bench":431.506558,"close_bench":428.456146},{"date":"2022-01-19","open_a":52.80828,"high_a":53.260972,"low_a":52.721225,"close_a":53.104271,"volume_a":18654700,"open_b":148.922796,"high_b":150.634946,"low_b":148.588927,"close_b":149.992889,"volume_b":6424500,"open_bench":429.995523,"close_bench":424.007324},{"date":"2022-01-20","open_a":53.104283,"high_a":53.487329,"low_a":52.860527,"close_a":52.886642,"volume_a":23002400,"open_b":150.164086,"high_b":151.422517,"low_b":148.820042,"close_b":148.905655,"volume_b":6284200,"open_bench":425.884511,"close_bench":419.314392},{"date":"2022-01-21","open_a":53.173924,"high_a":53.365446,"low_a":52.581942,"close_a":52.625469,"volume_a":26643200,"open_b":150.095583,"high_b":151.730689,"low_b":149.051171,"close_b":149.14534,"volume_b":6753200,"open_bench":418.197496,"close_bench":411.083008},{"date":"2022-01-24","open_a":52.146659,"high_a":52.538409,"low_a":50.98881,"close_a":52.198891,"volume_a":30185100,"open_b":150.232563,"high_b":151.097194,"low_b":144.642396,"close_b":148.383438,"volume_b":9263900,"open_bench":405.498332,"close_bench":412.828705},{"date":"2022-01-25","open_a":51.676557,"high_a":52.277244,"low_a":51.093279,"close_a":52.077015,"volume_a":19027100,"open_b":147.31334,"high_b":147.655765,"low_b":145.310128,"close_b":146.67984,"volume_b":6536900,"open_bench":406.4651,"close_bench":407.788513},{"date":"2022-01-26","open_a":52.050903,"high_a":52.538416,"low_a":51.545976,"close_a":51.885494,"volume_a":19190200,"open_b":144.984855,"high_b":147.082236,"low_b":142.493674,"close_b":145.130386,"volume_b":6841800,"open_bench":413.654708,"close_bench":406.765472},{"date":"2022-01-27","open_a":52.129249,"high_a":53.043338,"low_a":51.798434,"close_a":51.92902,"volume_a":18893800,"open_b":146.200477,"high_b":148.340664,"low_b":144.488327,"close_b":144.993408,"volume_b":6543500,"open_bench":411.345714,"close_bench":404.756805},{"date":"2022-01-28","open_a":51.702677,"high_a":53.00852,"low_a":51.310923,"close_a":52.964993,"volume_a":19101500,"open_b":144.257191,"high_b":147.869829,"low_b":143.144304,"close_b":147.818466,"volume_b":5796000,"open_bench":406.10844,"close_bench":414.809174},{"date":"2022-01-31","open_a":52.494882,"high_a":53.226154,"low_a":52.364299,"close_a":53.11298,"volume_a":22045300,"open_b":146.191876,"high_b":149.051162,"low_b":146.037776,"close_b":148.546082,"volume_b":5908000,"open_bench":414.14276,"close_bench":422.280334},{"date":"2022-02-01","open_a":53.025933,"high_a":53.226161,"low_a":52.373011,"close_a":52.721237,"volume_a":20841700,"open_b":148.229338,"high_b":148.469038,"low_b":145.156032,"close_b":147.535919,"volume_b":5952700,"open_bench":423.002992,"close_bench":425.133606},{"date":"2022-02-02","open_a":52.773463,"high_a":53.339329,"low_a":52.721231,"close_a":53.260979,"volume_a":20225600,"open_b":147.895486,"high_b":150.360985,"low_b":147.732842,"close_b":150.215454,"volume_b":5767000,"open_bench":427.526973,"close_bench":429.263367},{"date":"2022-02-03","open_a":53.052044,"high_a":53.783317,"low_a":52.747349,"close_a":53.635323,"volume_a":19440500,"open_b":150.044237,"high_b":151.234181,"low_b":148.811488,"close_b":150.129837,"volume_b":4632700,"open_bench":423.256449,"close_bench":419.173584},{"date":"2022-02-04","open_a":53.530862,"high_a":53.696268,"low_a":52.729943,"close_a":53.069462,"volume_a":17403700,"open_b":148.186543,"high_b":149.718911,"low_b":146.551437,"close_b":147.664337,"volume_b":4592400,"open_bench":418.938965,"close_bench":421.144653},{"date":"2022-02-07","open_a":53.435096,"high_a":53.731087,"low_a":52.982403,"close_a":53.626617,"volume_a":17846400,"open_b":147.878355,"high_b":148.092374,"low_b":146.568562,"close_b":147.082199,"volume_b":4813300,"open_bench":421.904823,"close_bench":419.792999},{"date":"2022-02-08","open_a":53.826848,"high_a":54.262129,"low_a":53.748497,"close_a":53.974842,"volume_a":18771700,"open_b":147.784184,"high_b":147.79274,"low_b":145.798085,"close_b":147.261978,"volume_b":3957700,"open_bench":419.295621,"close_bench":423.24707},{"date":"2022-02-09","open_a":54.122839,"high_a":54.253425,"low_a":53.017229,"close_a":53.139107,"volume_a":23595600,"open_b":147.672912,"high_b":147.912612,"low_b":146.397356,"close_b":147.193512,"volume_b":6780200,"open_bench":427.264271,"close_bench":429.441803},{"date":"2022-02-10","open_a":52.547123,"high_a":54.236012,"low_a":52.407833,"close_a":53.435097,"volume_a":23561500,"open_b":144.385613,"high_b":146.774057,"low_b":143.221351,"close_b":144.137344,"volume_b":7899800,"open_bench":423.62247,"close_bench":421.726532},{"date":"2022-02-11","open_a":53.774615,"high_a":53.957432,"low_a":52.364305,"close_a":52.486183,"volume_a":22781400,"open_b":143.8719,"high_b":145.798068,"low_b":143.555156,"close_b":144.317062,"volume_b":7299600,"open_bench":421.810991,"close_bench":413.410614},{"date":"2022-02-14","open_a":52.373002,"high_a":52.921457,"low_a":51.606907,"close_a":52.825695,"volume_a":23641300,"open_b":145.036181,"high_b":145.258756,"low_b":141.337945,"close_b":142.707657,"volume_b":6822500,"open_bench":412.903896,"close_bench":412.059143},{"date":"2022-02-15","open_a":53.069464,"high_a":53.783324,"low_a":52.78218,"close_a":53.025936,"volume_a":13408700,"open_b":143.152838,"high_b":144.034594,"low_b":141.474925,"close_b":142.074188,"volume_b":4943800,"open_bench":416.479865,"close_bench":418.704315},{"date":"2022-02-16","open_a":52.982404,"high_a":53.452508,"low_a":52.721236,"close_a":53.017227,"volume_a":15484700,"open_b":141.791666,"high_b":143.135696,"low_b":140.858534,"close_b":142.365234,"volume_b":4161700,"open_bench":416.66754,"close_bench":419.173584},{"date":"2022-02-17","open_a":52.712533,"high_a":54.218605,"low_a":52.590655,"close_a":54.079315,"volume_a":21468700,"open_b":142.116958,"high_b":143.777745,"low_b":141.252327,"close_b":142.750458,"volume_b":5156800,"open_bench":416.00116,"close_bench":410.219452},{"date":"2022-02-18","open_a":54.079305,"high_a":54.688699,"low_a":53.966134,"close_a":54.444942,"volume_a":21975000,"open_b":143.058646,"high_b":144.026015,"low_b":142.29674,"close_b":143.572296,"volume_b":5181600,"open_bench":410.472833,"close_bench":407.563232},{"date":"2022-02-22","open_a":54.349191,"high_a":54.532008,"low_a":53.731092,"close_a":54.218605,"volume_a":21731400,"open_b":144.308531,"high_b":144.625274,"low_b":142.305306,"close_b":144.120193,"volume_b":7733900,"open_bench":405.366955,"close_bench":403.189423},{"date":"2022-02-23","open_a":54.183779,"high_a":54.26213,"low_a":53.400272,"close_a":53.617912,"volume_a":19248500,"open_b":144.779367,"high_b":144.779367,"low_b":142.553581,"close_b":142.699112,"volume_b":5812800,"open_bench":406.089658,"close_bench":396.037384},{"date":"2022-02-24","open_a":52.207595,"high_a":52.773459,"low_a":51.554674,"close_a":52.677696,"volume_a":27207400,"open_b":141.860151,"high_b":142.39947,"low_b":137.220215,"close_b":140.216476,"volume_b":8888800,"open_bench":385.778599,"close_bench":401.997406},{"date":"2022-02-25","open_a":53.052051,"high_a":54.758355,"low_a":53.03464,"close_a":54.714825,"volume_a":21487800,"open_b":140.755797,"high_b":144.214346,"low_b":140.618822,"close_b":144.145859,"volume_b":6535600,"open_bench":403.226972,"close_bench":410.867096},{"date":"2022-02-28","open_a":53.670145,"high_a":54.270832,"low_a":53.461209,"close_a":54.183777,"volume_a":25879100,"open_b":140.430503,"high_b":141.432122,"low_b":138.179035,"close_b":140.173691,"volume_b":7677000,"open_bench":405.498345,"close_bench":409.815857},{"date":"2022-03-01","open_a":54.096724,"high_a":54.627768,"low_a":53.548269,"close_a":53.94873,"volume_a":15871100,"open_b":139.60013,"high_b":140.738698,"low_b":138.084874,"close_b":138.915268,"volume_b":5107900,"open_bench":408.323473,"close_bench":403.574219},{"date":"2022-03-02","open_a":54.175073,"high_a":54.74094,"low_a":53.783319,"close_a":54.349186,"volume_a":17319100,"open_b":139.523068,"high_b":141.954331,"low_b":139.1036,"close_b":140.841431,"volume_b":5452100,"open_bench":405.817478,"close_bench":410.998505},{"date":"2022-03-03","open_a":54.505881,"high_a":54.862811,"low_a":54.227301,"close_a":54.384003,"volume_a":21262800,"open_b":141.293788,"high_b":142.34507,"low_b":140.285595,"close_b":140.690598,"volume_b":5352200,"open_bench":413.420052,"close_bench":408.952362},{"date":"2022-03-04","open_a":53.504745,"high_a":54.488481,"low_a":53.2784,"close_a":54.471069,"volume_a":17163000,"open_b":139.596242,"high_b":143.473916,"low_b":138.665599,"close_b":142.827637,"volume_b":6340300,"open_bench":405.235571,"close_bench":405.629791},{"date":"2022-03-07","open_a":53.983547,"high_a":54.035782,"low_a":52.860524,"close_a":53.173927,"volume_a":29151400,"open_b":141.37998,"high_b":142.31061,"low_b":139.70827,"close_b":139.984009,"volume_b":6319500,"open_bench":405.047869,"close_bench":393.67218},{"date":"2022-03-08","open_a":53.496034,"high_a":53.574385,"low_a":51.006226,"close_a":51.067165,"volume_a":30242100,"open_b":139.923714,"high_b":141.293822,"low_b":135.882314,"close_b":136.037415,"volume_b":6163700,"open_bench":393.850447,"close_bench":390.687408},{"date":"2022-03-09","open_a":52.077023,"high_a":52.094434,"low_a":51.067171,"close_a":51.328339,"volume_a":23022700,"open_b":137.709113,"high_b":137.92454,"low_b":134.770695,"close_b":135.632401,"volume_b":5764900,"open_bench":399.031463,"close_bench":401.162048},{"date":"2022-03-10","open_a":50.910466,"high_a":51.067167,"low_a":50.057316,"close_a":50.38813,"volume_a":20887300,"open_b":134.245102,"high_b":134.641493,"low_b":132.159777,"close_b":133.14212,"volume_b":6103700,"open_bench":396.572344,"close_bench":399.350586},{"date":"2022-03-11","open_a":50.457774,"high_a":51.232574,"low_a":50.370719,"close_a":50.422951,"volume_a":19855600,"open_b":133.95209,"high_b":135.192947,"low_b":132.254529,"close_b":132.469955,"volume_b":4762600,"open_bench":401.828514,"close_bench":394.272888},{"date":"2022-03-14","open_a":50.852795,"high_a":51.791427,"low_a":50.852795,"close_a":51.352814,"volume_a":17799400,"open_b":133.693556,"high_b":135.977083,"low_b":133.133453,"close_b":134.331223,"volume_b":6075600,"open_bench":395.042542,"close_bench":391.391418},{"date":"2022-03-15","open_a":51.82651,"high_a":52.53706,"low_a":51.738785,"close_a":52.300209,"volume_a":18607200,"open_b":135.56348,"high_b":137.441992,"low_b":135.14124,"close_b":137.011139,"volume_b":6203400,"open_bench":393.991151,"close_bench":399.998138},{"date":"2022-03-16","open_a":52.440563,"high_a":52.580919,"low_a":51.41421,"close_a":52.159851,"volume_a":17347300,"open_b":136.50274,"high_b":137.864236,"low_b":135.322195,"close_b":137.614334,"volume_b":6299000,"open_bench":403.489766,"close_bench":408.867859},{"date":"2022-03-17","open_a":52.159856,"high_a":52.75637,"low_a":51.958094,"close_a":52.712509,"volume_a":15586200,"open_b":137.726368,"high_b":139.27744,"low_b":137.269665,"close_b":138.682861,"volume_b":3587600,"open_bench":406.962563,"close_bench":413.983215},{"date":"2022-03-18","open_a":52.896717,"high_a":52.896717,"low_a":52.379154,"close_a":52.721272,"volume_a":30197800,"open_b":139.061968,"high_b":140.3976,"low_b":137.924511,"close_b":140.276962,"volume_b":12123000,"open_bench":412.378859,"close_bench":418.517456},{"date":"2022-03-21","open_a":53.07217,"high_a":53.598504,"low_a":52.852863,"close_a":53.142349,"volume_a":17440100,"open_b":139.828892,"high_b":141.155913,"low_b":138.794849,"close_b":140.078781,"volume_b":5122600,"open_bench":418.347958,"close_bench":418.39505},{"date":"2022-03-22","open_a":53.449375,"high_a":53.46692,"low_a":52.686189,"close_a":53.335335,"volume_a":17559500,"open_b":140.320082,"high_b":141.98318,"low_b":139.794442,"close_b":141.61264,"volume_b":5779200,"open_bench":419.779072,"close_bench":423.290894},{"date":"2022-03-23","open_a":53.291483,"high_a":53.493244,"low_a":52.633563,"close_a":52.984455,"volume_a":15529200,"open_b":141.535083,"high_b":142.422639,"low_b":140.139125,"close_b":140.871567,"volume_b":4533700,"open_bench":420.767662,"close_bench":417.839569},{"date":"2022-03-24","open_a":53.19498,"high_a":53.528326,"low_a":53.037079,"close_a":53.493237,"volume_a":13151300,"open_b":141.371353,"high_b":141.948694,"low_b":140.690612,"close_b":141.724655,"volume_b":3160300,"open_bench":419.854412,"close_bench":424.138245},{"date":"2022-03-25","open_a":53.616049,"high_a":54.045892,"low_a":53.554644,"close_a":53.975712,"volume_a":12259800,"open_b":141.690206,"high_b":142.956914,"low_b":141.690206,"close_b":142.388199,"volume_b":4146800,"open_bench":424.769003,"close_bench":426.209503},{"date":"2022-03-28","open_a":53.975712,"high_a":54.335374,"low_a":53.695,"close_a":54.317829,"volume_a":11485200,"open_b":142.267529,"high_b":142.84487,"low_b":141.069759,"close_b":142.810394,"volume_b":4308000,"open_bench":425.616384,"close_bench":429.24118},{"date":"2022-03-29","open_a":54.572231,"high_a":54.852943,"low_a":54.159937,"close_a":54.52837,"volume_a":14720600,"open_b":144.154664,"high_b":145.266271,"low_b":143.361896,"close_b":144.930206,"volume_b":4440400,"open_bench":433.110769,"close_bench":434.55127},{"date":"2022-03-30","open_a":54.396789,"high_a":54.572235,"low_a":54.00204,"close_a":54.572235,"volume_a":13641000,"open_b":144.852694,"high_b":144.973332,"low_b":143.146508,"close_b":144.732056,"volume_b":3871500,"open_bench":433.412064,"close_bench":431.868011},{"date":"2022-03-31","open_a":54.580995,"high_a":54.949428,"low_a":54.388004,"close_a":54.388004,"volume_a":16822900,"open_b":145.085303,"high_b":145.886696,"low_b":144.137435,"close_b":144.232224,"volume_b":6076200,"open_bench":431.105379,"close_bench":425.220978},{"date":"2022-04-01","open_a":54.6775,"high_a":55.221379,"low_a":54.466964,"close_a":55.151199,"volume_a":12644900,"open_b":144.671712,"high_b":146.429585,"low_b":144.223621,"close_b":146.283096,"volume_b":3239600,"open_bench":426.793289,"close_bench":426.426117},{"date":"2022-04-04","open_a":54.914345,"high_a":55.08102,"low_a":54.142388,"close_a":54.861713,"volume_a":11505800,"open_b":145.722982,"high_b":145.964258,"low_b":144.292548,"close_b":145.903946,"volume_b":3759900,"open_bench":426.623828,"close_bench":430.079132},{"date":"2022-04-05","open_a":54.738895,"high_a":55.475765,"low_a":54.712576,"close_a":54.800301,"volume_a":15721200,"open_b":145.042246,"high_b":147.679062,"low_b":145.042246,"close_b":146.059052,"volume_b":4480700,"open_bench":428.59148,"close_bench":424.646576},{"date":"2022-04-06","open_a":55.063472,"high_a":55.493311,"low_a":54.651175,"close_a":55.352955,"volume_a":19267400,"open_b":146.420979,"high_b":148.816519,"low_b":146.050452,"close_b":148.549393,"volume_b":4294800,"open_bench":420.748832,"close_bench":420.400452},{"date":"2022-04-07","open_a":55.379277,"high_a":55.852976,"low_a":55.01084,"close_a":55.651215,"volume_a":13960800,"open_b":148.23915,"high_b":149.316282,"low_b":147.162019,"close_b":148.687241,"volume_b":5041600,"open_bench":419.524883,"close_bench":422.51886},{"date":"2022-04-08","open_a":55.686301,"high_a":56.177549,"low_a":55.405589,"close_a":55.993332,"volume_a":12703500,"open_b":149.066374,"high_b":149.669564,"low_b":147.859978,"close_b":149.187012,"volume_b":3883100,"open_bench":421.765603,"close_bench":421.389008},{"date":"2022-04-11","open_a":56.063506,"high_a":56.405626,"low_a":55.826654,"close_a":55.975784,"volume_a":12720100,"open_b":149.600634,"high_b":150.772553,"low_b":148.48904,"close_b":148.730316,"volume_b":3825100,"open_bench":418.13143,"close_bench":414.186554},{"date":"2022-04-12","open_a":56.475808,"high_a":56.853015,"low_a":56.045969,"close_a":56.633709,"volume_a":18089900,"open_b":149.747126,"high_b":150.496805,"low_b":148.75617,"close_b":149.333511,"volume_b":3924300,"open_bench":417.16167,"close_bench":412.651886},{"date":"2022-04-13","open_a":56.528452,"high_a":56.905659,"low_a":56.344235,"close_a":56.782848,"volume_a":11883000,"open_b":149.574803,"high_b":149.781617,"low_b":147.980643,"close_b":149.075012,"volume_b":4394000,"open_bench":412.407094,"close_bench":417.378235},{"date":"2022-04-14","open_a":56.984596,"high_a":57.519705,"low_a":56.861785,"close_a":57.037228,"volume_a":16388400,"open_b":149.488617,"high_b":150.160746,"low_b":148.006483,"close_b":148.127121,"volume_b":4834300,"open_bench":417.604135,"close_bench":412.181091},{"date":"2022-04-18","open_a":56.817918,"high_a":57.177584,"low_a":56.300358,"close_a":56.528439,"volume_a":11679400,"open_b":147.541158,"high_b":148.480414,"low_b":146.455414,"close_b":146.851791,"volume_b":3086000,"open_bench":411.258468,"close_bench":412.350616},{"date":"2022-04-19","open_a":56.581071,"high_a":57.23899,"low_a":56.528439,"close_a":57.081089,"volume_a":11993800,"open_b":146.352015,"high_b":149.221495,"low_b":146.352015,"close_b":148.988831,"volume_b":3642000,"open_bench":412.247057,"close_bench":419.00708},{"date":"2022-04-20","open_a":57.405673,"high_a":58.10745,"low_a":57.370583,"close_a":57.861828,"volume_a":14827100,"open_b":149.695461,"high_b":151.634298,"low_b":149.695461,"close_b":151.203445,"volume_b":4197600,"open_bench":420.777126,"close_bench":418.696381},{"date":"2022-04-21","open_a":57.879373,"high_a":58.77414,"low_a":57.651292,"close_a":58.081131,"volume_a":15120400,"open_b":151.45332,"high_b":152.728654,"low_b":150.496827,"close_b":150.660553,"volume_b":3701300,"open_bench":422.302339,"close_bench":412.435364},{"date":"2022-04-22","open_a":57.967098,"high_a":58.195172,"low_a":57.177593,"close_a":57.238998,"volume_a":14902200,"open_b":150.789844,"high_b":151.03112,"low_b":148.16164,"close_b":148.34259,"volume_b":5864000,"open_bench":411.352584,"close_bench":401.118439},{"date":"2022-04-25","open_a":58.774145,"high_a":58.949588,"low_a":56.826702,"close_a":57.844288,"volume_a":25082000,"open_b":149.255959,"high_b":149.867762,"low_b":147.351585,"close_b":149.712662,"volume_b":7369900,"open_bench":398.887123,"close_bench":403.444},{"date":"2022-04-26","open_a":57.475843,"high_a":58.221491,"low_a":57.054779,"close_a":57.063553,"volume_a":21982600,"open_b":149.910852,"high_b":151.10001,"low_b":148.842333,"close_b":149.333511,"volume_b":7386600,"open_bench":400.920767,"close_bench":391.759949},{"date":"2022-04-27","open_a":57.019681,"high_a":58.142529,"low_a":57.019681,"close_a":57.510925,"volume_a":16971000,"open_b":149.436929,"high_b":152.254709,"low_b":149.23874,"close_b":150.669174,"volume_b":5180300,"open_bench":392.833176,"close_bench":392.86142},{"date":"2022-04-28","open_a":56.721433,"high_a":58.142534,"low_a":56.581073,"close_a":58.063587,"volume_a":22010600,"open_b":150.393417,"high_b":153.056083,"low_b":149.781613,"close_b":152.952682,"volume_b":5235800,"open_bench":397.587819,"close_bench":402.784912},{"date":"2022-04-29","open_a":57.782869,"high_a":57.923222,"low_a":56.572293,"close_a":56.677563,"volume_a":20137100,"open_b":152.875142,"high_b":152.875142,"low_b":147.70491,"close_b":147.963425,"volume_b":6505600,"open_bench":398.811812,"close_bench":387.89978},{"date":"2022-05-02","open_a":57.265311,"high_a":57.291626,"low_a":54.896803,"close_a":55.651215,"volume_a":24970300,"open_b":149.075033,"high_b":149.471424,"low_b":142.414054,"close_b":144.559692,"volume_b":7692100,"open_bench":387.965652,"close_bench":390.23468},{"date":"2022-05-03","open_a":55.896843,"high_a":56.08983,"low_a":54.975753,"close_a":55.335419,"volume_a":21686300,"open_b":145.748829,"high_b":145.895318,"low_b":143.542866,"close_b":144.757874,"volume_b":5564900,"open_bench":390.733673,"close_bench":392.023529},{"date":"2022-05-04","open_a":55.265234,"high_a":57.124955,"low_a":55.212599,"close_a":57.046001,"volume_a":19780200,"open_b":144.430398,"high_b":150.040094,"low_b":144.430398,"close_b":149.816055,"volume_b":6231500,"open_bench":392.682561,"close_bench":403.961792},{"date":"2022-05-05","open_a":56.747749,"high_a":57.046002,"low_a":56.151235,"close_a":56.589848,"volume_a":20155200,"open_b":149.161201,"high_b":149.350777,"low_b":145.697142,"close_b":146.825974,"volume_b":6534300,"open_bench":399.715597,"close_bench":389.603851},{"date":"2022-05-06","open_a":56.317904,"high_a":57.291625,"low_a":56.203867,"close_a":56.791607,"volume_a":18984000,"open_b":145.835007,"high_b":147.179265,"low_b":145.386917,"close_b":146.843201,"volume_b":5824600,"open_bench":387.05243,"close_bench":387.278381},{"date":"2022-05-09","open_a":56.554752,"high_a":57.335482,"low_a":56.177544,"close_a":56.677563,"volume_a":20055300,"open_b":146.19693,"high_b":148.980235,"low_b":145.421388,"close_b":147.954803,"volume_b":7269100,"open_bench":381.403402,"close_bench":374.878784},{"date":"2022-05-10","open_a":56.581073,"high_a":57.300398,"low_a":56.010874,"close_a":56.151234,"volume_a":21322200,"open_b":149.178405,"high_b":150.402025,"low_b":147.136168,"close_b":147.773834,"volume_b":5936700,"open_bench":380.829051,"close_bench":375.744934},{"date":"2022-05-11","open_a":56.431946,"high_a":57.195129,"low_a":56.10737,"close_a":56.414398,"volume_a":19648400,"open_b":147.196531,"high_b":148.566639,"low_b":146.420988,"close_b":147.058655,"volume_b":5376900,"open_bench":374.784597,"close_bench":369.775787},{"date":"2022-05-12","open_a":56.28282,"high_a":56.607389,"low_a":55.449451,"close_a":56.589848,"volume_a":22574700,"open_b":147.567065,"high_b":148.204732,"low_b":144.904398,"close_b":146.83461,"volume_b":6473500,"open_bench":366.593471,"close_bench":369.38974},{"date":"2022-05-13","open_a":56.853012,"high_a":57.721471,"low_a":56.247731,"close_a":57.651291,"volume_a":16167000,"open_b":147.093066,"high_b":150.074572,"low_b":146.02456,"close_b":149.695419,"volume_b":4714600,"open_bench":373.504124,"close_bench":378.221069},{"date":"2022-05-16","open_a":57.405673,"high_a":58.230261,"low_a":57.405673,"close_a":57.861828,"volume_a":13223200,"open_b":149.686849,"high_b":151.918662,"low_b":149.436947,"close_b":151.203445,"volume_b":5317200,"open_bench":376.58285,"close_bench":376.686401},{"date":"2022-05-17","open_a":57.984642,"high_a":57.984642,"low_a":57.028469,"close_a":57.70393,"volume_a":21300600,"open_b":151.065556,"high_b":151.384383,"low_b":148.713104,"close_b":150.333115,"volume_b":5012200,"open_bench":382.749697,"close_bench":384.434998},{"date":"2022-05-18","open_a":57.388121,"high_a":57.475842,"low_a":53.475694,"close_a":53.68623,"volume_a":31437300,"open_b":149.669611,"high_b":149.824712,"low_b":140.38904,"close_b":141.018082,"volume_b":8875400,"open_bench":379.896959,"close_bench":368.937836},{"date":"2022-05-19","open_a":53.475696,"high_a":53.475696,"low_a":52.335303,"close_a":52.63356,"volume_a":31493900,"open_b":140.02711,"high_b":140.268387,"low_b":137.588483,"close_b":138.906891,"volume_b":6423800,"open_bench":365.887332,"close_bench":366.678192},{"date":"2022-05-20","open_a":53.063398,"high_a":53.721317,"low_a":52.335299,"close_a":53.493237,"volume_a":29133800,"open_b":139.803072,"high_b":140.509663,"low_b":137.545395,"close_b":139.777222,"volume_b":5768800,"open_bench":370.246559,"close_bench":366.838318},{"date":"2022-05-23","open_a":54.080975,"high_a":55.35295,"low_a":54.028341,"close_a":55.142418,"volume_a":19406900,"open_b":141.500655,"high_b":143.86172,"low_b":140.949164,"close_b":142.698425,"volume_b":4657800,"open_bench":369.851096,"close_bench":373.701874},{"date":"2022-05-24","open_a":55.15997,"high_a":56.309137,"low_a":55.04593,"close_a":56.160004,"volume_a":16154200,"open_b":143.060317,"high_b":144.990528,"low_b":142.31925,"close_b":144.611389,"volume_b":6327600,"open_bench":369.596965,"close_bench":370.849182},{"date":"2022-05-25","open_a":56.14246,"high_a":56.554757,"low_a":55.791568,"close_a":56.203865,"volume_a":15360500,"open_b":144.637235,"high_b":146.050431,"low_b":143.870318,"close_b":145.602341,"volume_b":5781000,"open_bench":369.36151,"close_bench":374.125519},{"date":"2022-05-26","open_a":56.203863,"high_a":57.107405,"low_a":56.107368,"close_a":56.405628,"volume_a":16159800,"open_b":145.59373,"high_b":147.429154,"low_b":145.59373,"close_b":146.584686,"volume_b":5193800,"open_bench":375.349501,"close_bench":381.601074},{"date":"2022-05-27","open_a":56.370554,"high_a":56.870572,"low_a":56.265284,"close_a":56.738987,"volume_a":17709000,"open_b":147.575659,"high_b":148.049599,"low_b":146.662253,"close_b":148.015137,"volume_b":6713500,"open_bench":384.048993,"close_bench":390.969055},{"date":"2022-05-31","open_a":56.212634,"high_a":56.423167,"low_a":55.309089,"close_a":55.598576,"volume_a":43159000,"open_b":147.334345,"high_b":147.334345,"low_b":142.500178,"close_b":144.551041,"volume_b":13610000,"open_bench":389.359088,"close_bench":388.77536},{"date":"2022-06-01","open_a":55.633673,"high_a":55.861753,"low_a":54.502054,"close_a":55.326645,"volume_a":15408300,"open_b":144.921603,"high_b":145.093955,"low_b":141.16458,"close_b":143.465332,"volume_b":5613900,"open_bench":390.884347,"close_bench":385.630737},{"date":"2022-06-02","open_a":55.423139,"high_a":55.914387,"low_a":54.484508,"close_a":55.905613,"volume_a":14586000,"open_b":143.708248,"high_b":144.688751,"low_b":141.131181,"close_b":144.628006,"volume_b":4677300,"open_bench":385.470607,"close_bench":392.974396},{"date":"2022-06-03","open_a":55.765249,"high_a":55.817883,"low_a":55.098558,"close_a":55.238914,"volume_a":13539900,"open_b":143.916548,"high_b":144.549964,"low_b":142.614999,"close_b":143.040176,"volume_b":4222000,"open_bench":388.276331,"close_bench":386.525146},{"date":"2022-06-06","open_a":55.554726,"high_a":55.940704,"low_a":54.966983,"close_a":55.151199,"volume_a":15112800,"open_b":144.43718,"high_b":145.443701,"low_b":142.936051,"close_b":143.638885,"volume_b":3447800,"open_bench":390.517145,"close_bench":387.702057},{"date":"2022-06-07","open_a":54.721357,"high_a":55.589809,"low_a":54.581001,"close_a":55.484543,"volume_a":17886300,"open_b":143.760347,"high_b":144.784238,"low_b":142.493514,"close_b":144.523926,"volume_b":4168300,"open_bench":384.227882,"close_bench":391.420959},{"date":"2022-06-08","open_a":55.256464,"high_a":55.449451,"low_a":54.879256,"close_a":55.168739,"volume_a":10211500,"open_b":143.812396,"high_b":144.740827,"low_b":143.265742,"close_b":143.803711,"volume_b":2646800,"open_bench":389.716782,"close_bench":387.165314},{"date":"2022-06-09","open_a":55.151198,"high_a":55.572266,"low_a":54.195021,"close_a":54.212566,"volume_a":13382200,"open_b":143.335145,"high_b":144.844932,"low_b":141.148541,"close_b":141.243988,"volume_b":3522900,"open_bench":385.395367,"close_bench":377.957489},{"date":"2022-06-10","open_a":53.519557,"high_a":54.291517,"low_a":53.037084,"close_a":53.870449,"volume_a":14967600,"open_b":139.673466,"high_b":142.233175,"low_b":138.354561,"close_b":141.018402,"volume_b":5203300,"open_bench":371.781117,"close_bench":366.99826},{"date":"2022-06-13","open_a":53.29148,"high_a":54.642409,"low_a":53.221301,"close_a":53.809044,"volume_a":23064200,"open_b":139.291677,"high_b":141.322089,"low_b":138.016159,"close_b":138.606201,"volume_b":6300700,"open_bench":357.630388,"close_bench":353.064087},{"date":"2022-06-14","open_a":53.658838,"high_a":53.800208,"low_a":51.829861,"close_a":52.333492,"volume_a":24620000,"open_b":138.831778,"high_b":138.875166,"low_b":134.371808,"close_b":135.465103,"volume_b":6159200,"open_bench":354.805876,"close_bench":352.000183},{"date":"2022-06-15","open_a":53.013833,"high_a":53.579314,"low_a":52.103764,"close_a":52.722256,"volume_a":19684700,"open_b":137.02699,"high_b":138.69297,"low_b":135.300264,"close_b":136.914185,"volume_b":5940100,"open_bench":355.286039,"close_bench":357.018433},{"date":"2022-06-16","open_a":51.812187,"high_a":52.589724,"low_a":51.467597,"close_a":52.19212,"volume_a":15053800,"open_b":135.907656,"high_b":137.495546,"low_b":134.58875,"close_b":136.25473,"volume_b":7207900,"open_bench":348.836767,"close_bench":345.202545},{"date":"2022-06-17","open_a":52.748767,"high_a":53.1287,"low_a":52.245136,"close_a":52.510204,"volume_a":34781900,"open_b":136.254762,"high_b":137.955458,"low_b":135.508541,"close_b":136.280792,"volume_b":9881800,"open_bench":345.615698,"close_bench":345.946625},{"date":"2022-06-21","open_a":52.563213,"high_a":53.791366,"low_a":52.139099,"close_a":53.632324,"volume_a":17705000,"open_b":137.243884,"high_b":139.66477,"low_b":136.532378,"close_b":139.447845,"volume_b":4449600,"open_bench":351.648547,"close_bench":354.655457},{"date":"2022-06-22","open_a":53.543964,"high_a":54.294995,"low_a":53.296567,"close_a":54.029926,"volume_a":15207100,"open_b":139.118146,"high_b":141.426228,"low_b":138.450014,"close_b":140.593231,"volume_b":4706500,"open_bench":350.447629,"close_bench":354.012451},{"date":"2022-06-23","open_a":54.162458,"high_a":54.701433,"low_a":54.100609,"close_a":54.674927,"volume_a":17190900,"open_b":142.111719,"high_b":143.29179,"low_b":141.565065,"close_b":142.389389,"volume_b":5810700,"open_bench":356.13991,"close_bench":357.482605},{"date":"2022-06-24","open_a":54.878153,"high_a":55.717539,"low_a":54.878153,"close_a":55.699867,"volume_a":18147100,"open_b":143.725641,"high_b":144.202878,"low_b":142.250555,"close_b":144.150818,"volume_b":5951500,"open_bench":360.640838,"close_bench":368.848389},{"date":"2022-06-27","open_a":55.646856,"high_a":55.920762,"low_a":55.33761,"close_a":55.585007,"volume_a":12142100,"open_b":143.9252,"high_b":145.018509,"low_b":143.465321,"close_b":144.341705,"volume_b":4595800,"open_bench":369.765581,"close_bench":367.439484},{"date":"2022-06-28","open_a":55.735216,"high_a":56.336037,"low_a":54.878158,"close_a":55.028362,"volume_a":13150100,"open_b":145.035869,"high_b":146.16388,"low_b":142.172473,"close_b":142.311295,"volume_b":4372900,"open_bench":368.990228,"close_bench":359.931641},{"date":"2022-06-29","open_a":55.258085,"high_a":55.761716,"low_a":55.160892,"close_a":55.425961,"volume_a":11795500,"open_b":144.090091,"high_b":145.391639,"low_b":143.474032,"close_b":144.671448,"volume_b":3653600,"open_bench":360.480183,"close_bench":359.638611},{"date":"2022-06-30","open_a":55.063704,"high_a":55.76172,"low_a":54.922334,"close_a":55.585007,"volume_a":15514200,"open_b":145.018507,"high_b":145.408969,"low_b":143.716959,"close_b":144.610687,"volume_b":6295100,"open_bench":355.76173,"close_bench":356.716766},{"date":"2022-07-01","open_a":55.770546,"high_a":56.945684,"low_a":55.505477,"close_a":56.883835,"volume_a":15143200,"open_b":144.680111,"high_b":147.092313,"low_b":143.829756,"close_b":146.979507,"volume_b":3918700,"open_bench":356.064206,"close_bench":360.489471},{"date":"2022-07-05","open_a":56.707131,"high_a":56.707131,"low_a":55.152057,"close_a":55.911922,"volume_a":15921200,"open_b":146.320064,"high_b":146.97951,"low_b":143.456655,"close_b":146.64978,"volume_b":4971600,"open_bench":355.421349,"close_bench":361.17041},{"date":"2022-07-06","open_a":56.194663,"high_a":56.627607,"low_a":55.832402,"close_a":56.026787,"volume_a":13450500,"open_b":147.508779,"high_b":149.200789,"low_b":147.057572,"close_b":148.116165,"volume_b":4209100,"open_bench":361.312202,"close_bench":362.390167},{"date":"2022-07-07","open_a":55.894253,"high_a":56.176996,"low_a":55.399459,"close_a":55.585007,"volume_a":14828400,"open_b":147.092318,"high_b":148.411211,"low_b":146.970841,"close_b":147.821182,"volume_b":5150200,"open_bench":364.158423,"close_bench":367.81778},{"date":"2022-07-08","open_a":55.691027,"high_a":56.035616,"low_a":55.558494,"close_a":55.788219,"volume_a":11311500,"open_b":148.003392,"high_b":149.851582,"low_b":147.986035,"close_b":149.140076,"volume_b":3482900,"open_bench":366.191405,"close_bench":367.515228},{"date":"2022-07-11","open_a":55.655696,"high_a":56.070969,"low_a":55.302269,"close_a":55.611515,"volume_a":12702100,"open_b":148.73225,"high_b":149.93835,"low_b":147.526149,"close_b":147.916611,"volume_b":4921300,"open_bench":364.848654,"close_bench":363.316833},{"date":"2022-07-12","open_a":55.664522,"high_a":56.062126,"low_a":55.196234,"close_a":55.372944,"volume_a":14731100,"open_b":150.111897,"high_b":150.389567,"low_b":146.528297,"close_b":147.074951,"volume_b":5201600,"open_bench":362.768447,"close_bench":360.101929},{"date":"2022-07-13","open_a":55.072537,"high_a":55.664523,"low_a":54.763288,"close_a":55.116714,"volume_a":13145300,"open_b":146.24196,"high_b":148.758294,"low_b":145.75605,"close_b":147.595581,"volume_b":4533100,"open_bench":354.683762,"close_bench":358.210724},{"date":"2022-07-14","open_a":54.188974,"high_a":55.001852,"low_a":54.162465,"close_a":54.940002,"volume_a":14433000,"open_b":146.025034,"high_b":148.515339,"low_b":145.513095,"close_b":148.359146,"volume_b":4218300,"open_bench":353.274877,"close_bench":357.340851},{"date":"2022-07-15","open_a":55.443635,"high_a":55.63802,"low_a":54.842811,"close_a":55.222744,"volume_a":21968400,"open_b":148.671501,"high_b":150.042466,"low_b":147.526145,"close_b":148.480606,"volume_b":4659100,"open_bench":361.728287,"close_bench":364.167877},{"date":"2022-07-18","open_a":55.240418,"high_a":55.275761,"low_a":54.348016,"close_a":54.471718,"volume_a":10780800,"open_b":148.506642,"high_b":148.584745,"low_b":145.643246,"close_b":145.998993,"volume_b":3479700,"open_bench":367.240935,"close_bench":361.160919},{"date":"2022-07-19","open_a":54.860478,"high_a":55.355275,"low_a":54.719108,"close_a":55.249245,"volume_a":10998200,"open_b":147.725747,"high_b":148.159597,"low_b":146.320079,"close_b":147.560883,"volume_b":4201600,"open_bench":365.066072,"close_bench":370.919159},{"date":"2022-07-20","open_a":55.134393,"high_a":55.293435,"low_a":54.029938,"close_a":54.339184,"volume_a":11010500,"open_b":147.552181,"high_b":148.003388,"low_b":145.304844,"close_b":145.998993,"volume_b":3460700,"open_bench":371.10834,"close_bench":373.283142},{"date":"2022-07-21","open_a":54.127122,"high_a":54.674934,"low_a":53.71185,"close_a":54.215481,"volume_a":11959600,"open_b":145.764703,"high_b":146.996833,"low_b":145.391586,"close_b":146.215897,"volume_b":3088300,"open_bench":372.706291,"close_bench":377.08429},{"date":"2022-07-22","open_a":54.233157,"high_a":54.666102,"low_a":54.180142,"close_a":54.418705,"volume_a":11644700,"open_b":146.76256,"high_b":148.003376,"low_b":146.198561,"close_b":147.17038,"volume_b":3757800,"open_bench":377.207339,"close_bench":373.585785},{"date":"2022-07-25","open_a":54.383353,"high_a":55.046026,"low_a":54.1713,"close_a":54.948833,"volume_a":14941700,"open_b":147.777796,"high_b":148.255033,"low_b":146.675815,"close_b":147.378662,"volume_b":3357200,"open_bench":374.209838,"close_bench":374.039642},{"date":"2022-07-26","open_a":55.443636,"high_a":56.371378,"low_a":55.249251,"close_a":55.850075,"volume_a":20118000,"open_b":148.003369,"high_b":149.235499,"low_b":146.849328,"close_b":149.096664,"volume_b":3732100,"open_bench":372.403798,"close_bench":369.61438},{"date":"2022-07-27","open_a":55.558502,"high_a":55.832405,"low_a":54.595417,"close_a":55.673363,"volume_a":12142900,"open_b":147.933952,"high_b":149.200785,"low_b":146.693136,"close_b":148.888412,"volume_b":4479000,"open_bench":372.895476,"close_bench":379.211914},{"date":"2022-07-28","open_a":55.593835,"high_a":56.768976,"low_a":55.567329,"close_a":56.601097,"volume_a":10837000,"open_b":149.131399,"high_b":151.95142,"low_b":148.54137,"close_b":151.708466,"volume_b":3134200,"open_bench":380.015628,"close_bench":383.968109},{"date":"2022-07-29","open_a":56.291865,"high_a":56.804333,"low_a":56.221181,"close_a":56.698303,"volume_a":13734200,"open_b":150.684565,"high_b":152.26377,"low_b":150.450283,"close_b":151.812576,"volume_b":4215500,"open_bench":385.395975,"close_bench":389.565948},{"date":"2022-08-01","open_a":56.627601,"high_a":57.466991,"low_a":56.57459,"close_a":57.007534,"volume_a":12831500,"open_b":151.456835,"high_b":154.181409,"low_b":151.456835,"close_b":153.539307,"volume_b":4034600,"open_bench":386.880472,"close_bench":388.412292},{"date":"2022-08-02","open_a":57.069386,"high_a":57.078224,"low_a":56.044456,"close_a":56.230003,"volume_a":15241900,"open_b":153.530652,"high_b":153.712862,"low_b":152.038209,"close_b":152.272491,"volume_b":3781100,"open_bench":386.852096,"close_bench":385.849792},{"date":"2022-08-03","open_a":56.150488,"high_a":56.636447,"low_a":55.938434,"close_a":56.477406,"volume_a":11317000,"open_b":151.916708,"high_b":153.634748,"low_b":151.187844,"close_b":153.435181,"volume_b":4401300,"open_bench":387.967885,"close_bench":391.892029},{"date":"2022-08-04","open_a":56.23001,"high_a":56.309531,"low_a":55.593843,"close_a":56.256516,"volume_a":11795600,"open_b":153.07941,"high_b":153.35708,"low_b":151.482847,"close_b":152.602173,"volume_b":3822500,"open_bench":391.816325,"close_bench":391.627228},{"date":"2022-08-05","open_a":56.221173,"high_a":56.221173,"low_a":55.372949,"close_a":56.000282,"volume_a":10445400,"open_b":151.318013,"high_b":151.586998,"low_b":149.591288,"close_b":151.456848,"volume_b":3295600,"open_bench":387.362708,"close_bench":390.965332},{"date":"2022-08-08","open_a":56.1063,"high_a":56.380206,"low_a":55.443627,"close_a":55.638012,"volume_a":9165200,"open_b":152.532766,"high_b":152.688959,"low_b":150.389562,"close_b":150.849442,"volume_b":2846100,"open_bench":392.648414,"close_bench":390.511414},{"date":"2022-08-09","open_a":55.823566,"high_a":55.938431,"low_a":55.531989,"close_a":55.708702,"volume_a":7910900,"open_b":151.665048,"high_b":152.046838,"low_b":150.94487,"close_b":151.413422,"volume_b":3228800,"open_bench":389.783427,"close_bench":388.960785},{"date":"2022-08-10","open_a":56.22117,"high_a":56.36254,"low_a":55.752879,"close_a":56.238842,"volume_a":8857300,"open_b":152.550144,"high_b":153.088113,"low_b":151.604355,"close_b":152.662949,"volume_b":4041100,"open_bench":395.98633,"close_bench":397.130463},{"date":"2022-08-11","open_a":56.274186,"high_a":56.539258,"low_a":55.735215,"close_a":55.858913,"volume_a":8507800,"open_b":153.027342,"high_b":153.686788,"low_b":151.560929,"close_b":151.881973,"volume_b":2743800,"open_bench":399.967176,"close_bench":397.130463},{"date":"2022-08-12","open_a":56.044457,"high_a":56.336032,"low_a":55.744046,"close_a":56.28302,"volume_a":9489300,"open_b":152.654248,"high_b":153.912421,"low_b":152.177024,"close_b":153.869034,"volume_b":3756200,"open_bench":399.059402,"close_bench":403.853455},{"date":"2022-08-15","open_a":56.23001,"high_a":57.157755,"low_a":56.168161,"close_a":56.989876,"volume_a":9863300,"open_b":154.25083,"high_b":155.630468,"low_b":153.409161,"close_b":155.56105,"volume_b":3043100,"open_bench":401.650257,"close_bench":405.517639},{"date":"2022-08-16","open_a":57.051714,"high_a":57.63487,"low_a":56.839661,"close_a":57.458153,"volume_a":9569600,"open_b":155.994906,"high_b":156.793187,"low_b":155.005729,"close_b":156.46347,"volume_b":3395700,"open_bench":404.449174,"close_bench":406.311951},{"date":"2022-08-17","open_a":57.281448,"high_a":57.626038,"low_a":57.219599,"close_a":57.325623,"volume_a":8144000,"open_b":156.272544,"high_b":157.114212,"low_b":155.942814,"close_b":156.376663,"volume_b":2666200,"open_bench":402.728265,"close_bench":403.427979},{"date":"2022-08-18","open_a":57.396305,"high_a":57.740895,"low_a":57.175414,"close_a":57.626034,"volume_a":6532000,"open_b":156.897313,"high_b":156.992747,"low_b":156.133733,"close_b":156.532867,"volume_b":3010600,"open_bench":403.626498,"close_bench":404.600464},{"date":"2022-08-19","open_a":57.617207,"high_a":57.846935,"low_a":57.396316,"close_a":57.581863,"volume_a":8921100,"open_b":155.977553,"high_b":157.062177,"low_b":155.595777,"close_b":156.333313,"volume_b":3704000,"open_bench":401.84881,"close_bench":399.163391},{"date":"2022-08-22","open_a":57.29028,"high_a":57.360967,"low_a":56.539256,"close_a":56.7248,"volume_a":9087900,"open_b":156.307264,"high_b":156.593606,"low_b":154.467747,"close_b":154.840851,"volume_b":3385000,"open_bench":394.350464,"close_bench":390.851868},{"date":"2022-08-23","open_a":56.389047,"high_a":56.87501,"low_a":56.265348,"close_a":56.786648,"volume_a":8443800,"open_b":154.294215,"high_b":155.196616,"low_b":153.834322,"close_b":154.771439,"volume_b":2575000,"open_bench":390.426364,"close_bench":389.906311},{"date":"2022-08-24","open_a":56.760139,"high_a":57.060554,"low_a":56.592266,"close_a":56.866169,"volume_a":6844700,"open_b":154.771434,"high_b":155.751938,"low_b":154.545838,"close_b":155.543686,"volume_b":3340600,"open_bench":389.679362,"close_bench":391.15448},{"date":"2022-08-25","open_a":56.866173,"high_a":57.175419,"low_a":56.459734,"close_a":57.140076,"volume_a":7913900,"open_b":155.344131,"high_b":155.621801,"low_b":153.921105,"close_b":155.552383,"volume_b":3401800,"open_bench":392.638967,"close_bench":396.676575},{"date":"2022-08-26","open_a":57.193087,"high_a":57.254936,"low_a":55.646847,"close_a":55.761711,"volume_a":10089500,"open_b":156.151062,"high_b":156.168407,"low_b":151.560929,"close_b":151.881973,"volume_b":5246600,"open_bench":396.563109,"close_bench":383.249451},{"date":"2022-08-29","open_a":55.390617,"high_a":55.691029,"low_a":55.14322,"close_a":55.425961,"volume_a":9115900,"open_b":151.45686,"high_b":152.098949,"low_b":150.294147,"close_b":151.4048,"volume_b":4184500,"open_bench":380.308804,"close_bench":380.715393},{"date":"2022-08-30","open_a":55.584997,"high_a":55.584997,"low_a":54.77212,"close_a":54.91349,"volume_a":8456800,"open_b":151.274627,"high_b":151.578326,"low_b":149.530557,"close_b":150.103241,"volume_b":4026300,"open_bench":381.868924,"close_bench":376.535889},{"date":"2022-08-31","open_a":55.116713,"high_a":55.249245,"low_a":54.515889,"close_a":54.524723,"volume_a":14633600,"open_b":150.81472,"high_b":151.335345,"low_b":149.417725,"close_b":149.47847,"volume_b":5597900,"open_bench":378.162208,"close_bench":373.670746},{"date":"2022-09-01","open_a":54.409861,"high_a":54.8163,"low_a":54.153627,"close_a":54.780956,"volume_a":14939700,"open_b":149.303765,"high_b":151.103235,"low_b":148.709771,"close_b":150.989685,"volume_b":4846100,"open_bench":371.505488,"close_bench":374.843353},{"date":"2022-09-02","open_a":55.125543,"high_a":55.461299,"low_a":53.809035,"close_a":54.029926,"volume_a":11809800,"open_b":151.74966,"high_b":152.841573,"low_b":148.404038,"close_b":149.07666,"volume_b":5801500,"open_bench":378.493301,"close_bench":370.8909},{"date":"2022-09-06","open_a":54.153631,"high_a":54.754455,"low_a":53.888563,"close_a":54.056438,"volume_a":12181800,"open_b":149.356188,"high_b":150.762572,"low_b":147.740165,"close_b":148.072098,"volume_b":4900100,"open_bench":371.732485,"close_bench":369.491486},{"date":"2022-09-07","open_a":54.180141,"high_a":55.213912,"low_a":54.180141,"close_a":55.054871,"volume_a":13290700,"open_b":149.041693,"high_b":151.802051,"low_b":148.438955,"close_b":151.339081,"volume_b":6074600,"open_bench":369.179349,"close_bench":376.129303},{"date":"2022-09-08","open_a":54.85165,"high_a":55.046031,"low_a":54.197811,"close_a":54.88699,"volume_a":17193500,"open_b":150.928517,"high_b":151.496306,"low_b":148.989279,"close_b":150.832428,"volume_b":3888100,"open_bench":373.869438,"close_bench":378.58783},{"date":"2022-09-09","open_a":54.93117,"high_a":55.558502,"low_a":54.657267,"close_a":55.063705,"volume_a":11855900,"open_b":150.814963,"high_b":152.474678,"low_b":149.958907,"close_b":151.312881,"volume_b":3662000,"open_bench":380.819319,"close_bench":384.469238},{"date":"2022-09-12","open_a":55.160895,"high_a":55.49665,"low_a":55.019525,"close_a":55.222744,"volume_a":13026900,"open_b":151.312882,"high_b":152.378589,"low_b":150.902321,"close_b":151.906876,"volume_b":5953300,"open_bench":386.530576,"close_bench":388.601379},{"date":"2022-09-13","open_a":54.957677,"high_a":54.975348,"low_a":53.261232,"close_a":53.429111,"volume_a":18532800,"open_b":150.683904,"high_b":150.972171,"low_b":145.835811,"close_b":146.23764,"volume_b":6926200,"open_bench":379.95889,"close_bench":371.704071},{"date":"2022-09-14","open_a":53.579313,"high_a":53.968084,"low_a":53.4026,"close_a":53.711849,"volume_a":12881600,"open_b":146.857871,"high_b":147.792555,"low_b":146.37744,"close_b":147.347046,"volume_b":5156900,"open_bench":372.999508,"close_bench":373.122437},{"date":"2022-09-15","open_a":53.756348,"high_a":53.756348,"low_a":52.830741,"close_a":52.98204,"volume_a":11451900,"open_b":147.216018,"high_b":147.242223,"low_b":144.429455,"close_b":144.901169,"volume_b":5461000,"open_bench":371.571557,"close_bench":368.886139},{"date":"2022-09-16","open_a":53.097744,"high_a":53.222344,"low_a":52.608241,"close_a":52.990944,"volume_a":20493500,"open_b":145.320448,"high_b":146.028005,"low_b":144.350829,"close_b":145.853302,"volume_b":12012600,"open_bench":364.723872,"close_bench":366.072083},{"date":"2022-09-19","open_a":52.857443,"high_a":53.489347,"low_a":52.715043,"close_a":53.391449,"volume_a":12596200,"open_b":145.931968,"high_b":147.460646,"low_b":145.442794,"close_b":147.390762,"volume_b":3720900,"open_bench":362.938865,"close_bench":368.910919},{"date":"2022-09-20","open_a":53.115538,"high_a":53.427039,"low_a":52.795135,"close_a":53.31134,"volume_a":11984600,"open_b":146.386159,"high_b":147.757595,"low_b":145.757217,"close_b":147.556686,"volume_b":3872000,"open_bench":365.597395,"close_bench":364.676422},{"date":"2022-09-21","open_a":53.524947,"high_a":54.023351,"low_a":52.857442,"close_a":52.866344,"volume_a":14165300,"open_b":148.395322,"high_b":150.273406,"low_b":147.050078,"close_b":147.137436,"volume_b":5134500,"open_bench":366.594226,"close_bench":358.315002},{"date":"2022-09-22","open_a":52.999844,"high_a":53.160045,"low_a":52.688343,"close_a":52.741741,"volume_a":11657600,"open_b":147.102459,"high_b":148.404024,"low_b":146.796732,"close_b":147.277176,"volume_b":4290600,"open_bench":357.545938,"close_bench":355.305237},{"date":"2022-09-23","open_a":52.474741,"high_a":52.599342,"low_a":51.620337,"close_a":52.154339,"volume_a":14824000,"open_b":147.093697,"high_b":147.408168,"low_b":145.468931,"close_b":147.20726,"volume_b":4565100,"open_bench":351.849278,"close_bench":349.352234},{"date":"2022-09-26","open_a":52.056437,"high_a":52.127636,"low_a":51.34443,"close_a":51.504631,"volume_a":18670100,"open_b":147.041314,"high_b":148.508838,"low_b":146.342487,"close_b":147.146133,"volume_b":5326400,"open_bench":347.890011,"close_bench":345.896149},{"date":"2022-09-27","open_a":51.691538,"high_a":51.949637,"low_a":50.116228,"close_a":50.178528,"volume_a":23829300,"open_b":146.962706,"high_b":148.500115,"low_b":144.342116,"close_b":145.014725,"volume_b":5400700,"open_bench":349.418612,"close_bench":345.013153},{"date":"2022-09-28","open_a":50.427727,"high_a":50.95283,"low_a":49.760222,"close_a":50.712528,"volume_a":21480100,"open_b":145.80088,"high_b":148.168138,"low_b":144.342086,"close_b":147.338287,"volume_b":5480900,"open_bench":345.962612,"close_bench":351.801758},{"date":"2022-09-29","open_a":50.810429,"high_a":50.988431,"low_a":49.840323,"close_a":50.347626,"volume_a":14570700,"open_b":147.451882,"high_b":147.748879,"low_b":144.132465,"close_b":145.538849,"volume_b":5167700,"open_bench":348.269804,"close_bench":344.453003},{"date":"2022-09-30","open_a":50.614631,"high_a":50.846035,"low_a":49.840327,"close_a":49.858128,"volume_a":16124600,"open_b":146.482264,"high_b":146.962708,"low_b":142.394136,"close_b":142.612518,"volume_b":5971900,"open_bench":343.513037,"close_bench":339.126556},{"date":"2022-10-03","open_a":50.160736,"high_a":50.596837,"low_a":49.893735,"close_a":50.418839,"volume_a":16277600,"open_b":142.839672,"high_b":144.735231,"low_b":142.402907,"close_b":144.350876,"volume_b":5093400,"open_bench":342.829437,"close_bench":348.079926},{"date":"2022-10-04","open_a":50.685826,"high_a":51.451231,"low_a":50.249721,"close_a":50.534523,"volume_a":18861400,"open_b":146.220204,"high_b":147.390731,"low_b":144.612912,"close_b":145.975616,"volume_b":4596800,"open_bench":353.577248,"close_bench":358.865723},{"date":"2022-10-05","open_a":50.285332,"high_a":50.418834,"low_a":49.404225,"close_a":50.053932,"volume_a":14324900,"open_b":145.853337,"high_b":145.958157,"low_b":143.023095,"close_b":145.006012,"volume_b":3538600,"open_bench":354.517153,"close_bench":358.030121},{"date":"2022-10-06","open_a":49.840315,"high_a":49.929315,"low_a":48.834608,"close_a":48.977009,"volume_a":17192800,"open_b":144.822524,"high_b":145.023446,"low_b":141.948616,"close_b":142.210678,"volume_b":4623400,"open_bench":356.634489,"close_bench":354.336823},{"date":"2022-10-07","open_a":48.941415,"high_a":49.030415,"low_a":48.28281,"close_a":48.51421,"volume_a":11479800,"open_b":141.773942,"high_b":142.39414,"low_b":140.620875,"close_b":141.171204,"volume_b":4779200,"open_bench":350.320632,"close_bench":344.453003},{"date":"2022-10-10","open_a":48.59431,"high_a":48.914713,"low_a":48.078109,"close_a":48.40741,"volume_a":10239100,"open_b":141.503159,"high_b":142.420355,"low_b":140.804332,"close_b":141.35466,"volume_b":4839200,"open_bench":345.563855,"close_bench":341.822998},{"date":"2022-10-11","open_a":48.469718,"high_a":49.146124,"low_a":48.273919,"close_a":48.487518,"volume_a":14303900,"open_b":141.721525,"high_b":143.49479,"low_b":141.127517,"close_b":142.027252,"volume_b":5222600,"open_bench":340.1329,"close_bench":339.658173},{"date":"2022-10-12","open_a":49.155021,"high_a":49.875927,"low_a":48.985921,"close_a":49.074921,"volume_a":18377400,"open_b":147.390722,"high_b":149.111577,"low_b":145.530112,"close_b":147.967255,"volume_b":9795300,"open_bench":340.06648,"close_bench":338.537842},{"date":"2022-10-13","open_a":48.487521,"high_a":49.911532,"low_a":48.291718,"close_a":49.724628,"volume_a":16220700,"open_b":145.966878,"high_b":152.902696,"low_b":145.722291,"close_b":152.527084,"volume_b":8412700,"open_bench":331.559375,"close_bench":347.47226},{"date":"2022-10-14","open_a":50.098423,"high_a":50.258624,"low_a":48.896913,"close_a":48.932514,"volume_a":15997900,"open_b":153.723825,"high_b":153.968413,"low_b":148.491389,"close_b":148.666092,"volume_b":5937900,"open_bench":349.921747,"close_bench":339.553711},{"date":"2022-10-17","open_a":49.511017,"high_a":50.03612,"low_a":49.457615,"close_a":49.564415,"volume_a":13721400,"open_b":150.483041,"high_b":151.636094,"low_b":149.434807,"close_b":150.884857,"volume_b":6401800,"open_bench":345.611329,"close_bench":348.279297},{"date":"2022-10-18","open_a":50.116231,"high_a":50.534532,"low_a":49.929328,"close_a":50.23193,"volume_a":11823100,"open_b":152.710527,"high_b":154.536202,"low_b":151.478847,"close_b":152.920166,"volume_b":4547400,"open_bench":356.169251,"close_bench":352.371429},{"date":"2022-10-19","open_a":50.11623,"high_a":50.320931,"low_a":49.600025,"close_a":49.804726,"volume_a":10182300,"open_b":152.710556,"high_b":153.601547,"low_b":150.535461,"close_b":151.435196,"volume_b":5200200,"open_bench":350.339553,"close_bench":349.874329},{"date":"2022-10-20","open_a":49.635625,"high_a":49.769124,"low_a":48.914719,"close_a":49.021523,"volume_a":16905100,"open_b":152.168931,"high_b":152.308699,"low_b":149.469726,"close_b":149.775467,"volume_b":4314000,"open_bench":349.428121,"close_bench":346.940552},{"date":"2022-10-21","open_a":48.950321,"high_a":49.938228,"low_a":48.941422,"close_a":49.804726,"volume_a":15034100,"open_b":149.548329,"high_b":151.723424,"low_b":149.006745,"close_b":151.173096,"volume_b":5071500,"open_bench":346.665184,"close_bench":355.371704},{"date":"2022-10-24","open_a":50.409923,"high_a":51.38003,"low_a":50.347623,"close_a":51.237629,"volume_a":17416700,"open_b":152.701774,"high_b":155.470863,"low_b":152.579481,"close_b":155.208801,"volume_b":5316500,"open_bench":356.890886,"close_bench":359.720245},{"date":"2022-10-25","open_a":52.545944,"high_a":52.608244,"low_a":51.397836,"close_a":52.465843,"volume_a":28829900,"open_b":155.59316,"high_b":156.248307,"low_b":154.274134,"close_b":155.724197,"volume_b":5220000,"open_bench":359.64422,"close_bench":365.464386},{"date":"2022-10-26","open_a":52.519242,"high_a":53.204546,"low_a":52.385743,"close_a":52.857445,"volume_a":15831400,"open_b":156.588979,"high_b":158.021568,"low_b":155.767872,"close_b":156.423019,"volume_b":5136000,"open_bench":362.331192,"close_bench":362.710968},{"date":"2022-10-27","open_a":53.195643,"high_a":53.257943,"low_a":52.848542,"close_a":52.98204,"volume_a":16026100,"open_b":157.558616,"high_b":158.047791,"low_b":155.986273,"close_b":156.257065,"volume_b":5424100,"open_bench":363.707954,"close_bench":360.774139},{"date":"2022-10-28","open_a":53.017639,"high_a":54.210246,"low_a":52.928639,"close_a":54.076744,"volume_a":12888900,"open_b":156.97335,"high_b":159.794849,"low_b":156.781172,"close_b":159.18338,"volume_b":6715300,"open_bench":360.669725,"close_bench":369.357239},{"date":"2022-10-31","open_a":53.854249,"high_a":53.907651,"low_a":53.106644,"close_a":53.266846,"volume_a":14016400,"open_b":158.266181,"high_b":159.838537,"low_b":157.707122,"close_b":158.615601,"volume_b":4877200,"open_bench":366.907594,"close_bench":366.689209},{"date":"2022-11-01","open_a":53.427046,"high_a":53.65845,"low_a":52.839643,"close_a":53.079945,"volume_a":10659500,"open_b":158.056534,"high_b":159.690031,"low_b":157.139325,"close_b":157.899292,"volume_b":4084500,"open_bench":370.42059,"close_bench":365.084625},{"date":"2022-11-02","open_a":53.079939,"high_a":53.605043,"low_a":52.305635,"close_a":52.341236,"volume_a":14549200,"open_b":157.916743,"high_b":160.36263,"low_b":155.636829,"close_b":155.697983,"volume_b":4970600,"open_bench":364.495976,"close_bench":355.922394},{"date":"2022-11-03","open_a":51.718239,"high_a":52.510343,"low_a":51.709337,"close_a":52.314541,"volume_a":9194000,"open_b":154.422628,"high_b":156.274507,"low_b":154.204245,"close_b":155.296158,"volume_b":3753800,"open_bench":352.694218,"close_bench":352.257477},{"date":"2022-11-04","open_a":52.777342,"high_a":53.035445,"low_a":52.11874,"close_a":52.741741,"volume_a":9725900,"open_b":155.418441,"high_b":157.051938,"low_b":154.204235,"close_b":156.169678,"volume_b":3922400,"open_bench":357.944685,"close_bench":357.327545},{"date":"2022-11-07","open_a":52.741745,"high_a":53.19565,"low_a":52.723948,"close_a":52.946449,"volume_a":8849700,"open_b":156.816116,"high_b":157.776991,"low_b":156.213378,"close_b":157.418854,"volume_b":3246100,"open_bench":358.618866,"close_bench":360.745667},{"date":"2022-11-08","open_a":53.177848,"high_a":53.213449,"low_a":52.456942,"close_a":53.044346,"volume_a":10081600,"open_b":157.619764,"high_b":159.078558,"low_b":156.187176,"close_b":157.855621,"volume_b":3724700,"open_bench":361.846957,"close_bench":362.691986},{"date":"2022-11-09","open_a":53.035442,"high_a":53.079941,"low_a":52.287837,"close_a":52.305637,"volume_a":10115200,"open_b":157.191717,"high_b":158.17008,"low_b":155.837743,"close_b":155.968781,"volume_b":4723600,"open_bench":360.726587,"close_bench":355.219757},{"date":"2022-11-10","open_a":53.622847,"high_a":54.325953,"low_a":53.115544,"close_a":54.183552,"volume_a":16804800,"open_b":159.550249,"high_b":159.786106,"low_b":155.977514,"close_b":158.720398,"volume_b":6704100,"open_bench":368.436202,"close_bench":374.740601},{"date":"2022-11-11","open_a":54.619653,"high_a":54.690855,"low_a":53.587246,"close_a":54.575153,"volume_a":18972100,"open_b":158.222502,"high_b":158.711677,"low_b":153.252114,"close_b":155.532028,"volume_b":7390500,"open_bench":375.59507,"close_bench":378.367493},{"date":"2022-11-14","open_a":54.575152,"high_a":54.931156,"low_a":54.032248,"close_a":54.050049,"volume_a":16361000,"open_b":156.073602,"high_b":156.519098,"low_b":154.178044,"close_b":154.378952,"volume_b":5633000,"open_bench":376.611034,"close_bench":375.148865},{"date":"2022-11-15","open_a":54.406059,"high_a":54.69086,"low_a":53.400351,"close_a":53.961056,"volume_a":13295800,"open_b":155.523285,"high_b":156.099818,"low_b":153.23464,"close_b":154.780792,"volume_b":5153600,"open_bench":380.874066,"close_bench":378.348511},{"date":"2022-11-16","open_a":54.10345,"high_a":54.317049,"low_a":53.658447,"close_a":53.863148,"volume_a":14199500,"open_b":155.750422,"high_b":156.912218,"low_b":155.243773,"close_b":155.776627,"volume_b":4701100,"open_bench":376.724921,"close_bench":375.462158},{"date":"2022-11-17","open_a":53.56945,"high_a":54.058956,"low_a":53.516052,"close_a":54.032253,"volume_a":11212600,"open_b":155.042852,"high_b":157.375175,"low_b":154.990429,"close_b":157.261612,"volume_b":3920300,"open_bench":370.724383,"close_bench":374.313324},{"date":"2022-11-18","open_a":54.281452,"high_a":54.619652,"low_a":54.112349,"close_a":54.414951,"volume_a":10415600,"open_b":157.418883,"high_b":158.842728,"low_b":157.25291,"close_b":158.397232,"volume_b":4498100,"open_bench":377.636378,"close_bench":376.012817},{"date":"2022-11-21","open_a":54.655258,"high_a":55.367262,"low_a":54.468358,"close_a":55.251564,"volume_a":15399400,"open_b":159.218312,"high_b":161.716608,"low_b":159.06107,"close_b":161.445816,"volume_b":4397600,"open_bench":374.69309,"close_bench":374.645599},{"date":"2022-11-22","open_a":55.598663,"high_a":55.625362,"low_a":55.224858,"close_a":55.491859,"volume_a":12319600,"open_b":162.310597,"high_b":163.018154,"low_b":161.052712,"close_b":161.463272,"volume_b":4109600,"open_bench":376.582547,"close_bench":379.687256},{"date":"2022-11-23","open_a":55.61647,"high_a":55.883471,"low_a":55.447366,"close_a":55.74107,"volume_a":8990400,"open_b":161.603074,"high_b":162.546489,"low_b":161.166309,"close_b":161.672958,"volume_b":2511000,"open_bench":379.354933,"close_bench":382.079895},{"date":"2022-11-25","open_a":55.830065,"high_a":55.901267,"low_a":55.518564,"close_a":55.794464,"volume_a":5165500,"open_b":161.236198,"high_b":161.603079,"low_b":160.677139,"close_b":160.825638,"volume_b":1837000,"open_bench":381.519657,"close_bench":381.994385},{"date":"2022-11-28","open_a":55.794466,"high_a":56.159368,"low_a":55.678767,"close_a":55.803368,"volume_a":12267500,"open_b":161.078964,"high_b":162.031122,"low_b":160.301523,"close_b":160.633469,"volume_b":6411400,"open_bench":378.918187,"close_bench":375.898926},{"date":"2022-11-29","open_a":55.643156,"high_a":55.687656,"low_a":55.162553,"close_a":55.607555,"volume_a":10969600,"open_b":160.118107,"high_b":160.222926,"low_b":158.126459,"close_b":159.497894,"volume_b":4375400,"open_bench":376.031797,"close_bench":375.253265},{"date":"2022-11-30","open_a":55.580675,"high_a":57.077528,"low_a":55.347634,"close_a":57.014786,"volume_a":30433900,"open_b":158.895139,"high_b":162.214557,"low_b":157.445089,"close_b":162.048584,"volume_b":7915600,"open_bench":375.50014,"close_bench":387.074005},{"date":"2022-12-01","open_a":57.014779,"high_a":57.462938,"low_a":56.853442,"close_a":57.176117,"volume_a":12001400,"open_b":163.033037,"high_b":163.780174,"low_b":161.872791,"close_b":163.402206,"volume_b":4683300,"open_bench":388.108913,"close_bench":386.789185},{"date":"2022-12-02","open_a":57.014785,"high_a":57.830431,"low_a":56.826559,"close_a":57.678059,"volume_a":13643500,"open_b":161.732129,"high_b":163.762568,"low_b":161.591489,"close_b":163.217606,"volume_b":3547700,"open_bench":381.91846,"close_bench":386.342926},{"date":"2022-12-05","open_a":56.835517,"high_a":57.185081,"low_a":56.701068,"close_a":56.889297,"volume_a":9185100,"open_b":161.23113,"high_b":161.679418,"low_b":160.255476,"close_b":160.958649,"volume_b":4083200,"open_bench":383.532647,"close_bench":379.393005},{"date":"2022-12-06","open_a":56.934111,"high_a":57.256786,"low_a":56.674178,"close_a":56.862404,"volume_a":10099500,"open_b":161.512387,"high_b":161.98703,"low_b":158.831499,"close_b":159.648956,"volume_b":5835100,"open_bench":379.231523,"close_bench":373.924042},{"date":"2022-12-07","open_a":56.871365,"high_a":57.113371,"low_a":56.692101,"close_a":56.952034,"volume_a":12307600,"open_b":159.807152,"high_b":160.369698,"low_b":158.8227,"close_b":160.13237,"volume_b":4241700,"open_bench":373.079052,"close_bench":373.287933},{"date":"2022-12-08","open_a":57.005809,"high_a":57.229889,"low_a":56.78173,"close_a":57.194038,"volume_a":9946200,"open_b":159.807183,"high_b":161.714566,"low_b":159.323742,"close_b":161.538773,"volume_b":3800800,"open_bench":375.167841,"close_bench":376.212219},{"date":"2022-12-09","open_a":57.014776,"high_a":57.220929,"low_a":56.548691,"close_a":56.593506,"volume_a":13914900,"open_b":161.916707,"high_b":161.916707,"low_b":160.897088,"close_b":160.94104,"volume_b":3116100,"open_bench":374.977924,"close_bench":373.401825},{"date":"2022-12-12","open_a":56.736917,"high_a":57.373305,"low_a":56.476984,"close_a":57.337452,"volume_a":11647900,"open_b":161.890343,"high_b":161.890343,"low_b":160.290608,"close_b":161.705765,"volume_b":4691400,"open_bench":374.189949,"close_bench":378.785339},{"date":"2022-12-13","open_a":57.893171,"high_a":57.991766,"low_a":57.185082,"close_a":57.355385,"volume_a":17521600,"open_b":163.911979,"high_b":164.228412,"low_b":160.8004,"close_b":161.617844,"volume_b":5634900,"open_bench":389.485657,"close_bench":381.652649},{"date":"2022-12-14","open_a":57.534649,"high_a":57.946955,"low_a":56.97893,"close_a":57.355385,"volume_a":12826700,"open_b":161.793682,"high_b":163.384632,"low_b":160.088461,"close_b":161.169601,"volume_b":5392500,"open_bench":381.310813,"close_bench":379.212524},{"date":"2022-12-15","open_a":56.996848,"high_a":57.265743,"low_a":56.0109,"close_a":56.566616,"volume_a":12212300,"open_b":159.95662,"high_b":160.598284,"low_b":156.801088,"close_b":158.435989,"volume_b":6127800,"open_bench":374.370312,"close_bench":369.936371},{"date":"2022-12-16","open_a":56.243946,"high_a":56.593509,"low_a":55.867494,"close_a":56.243946,"volume_a":33631500,"open_b":158.286531,"high_b":159.262198,"low_b":156.880159,"close_b":158.51506,"volume_b":15708100,"open_bench":367.390547,"close_bench":365.568756},{"date":"2022-12-19","open_a":56.315654,"high_a":56.835516,"low_a":55.992979,"close_a":56.324615,"volume_a":10919700,"open_b":158.972167,"high_b":161.266303,"low_b":158.523893,"close_b":159.394073,"volume_b":4447800,"open_bench":365.759639,"close_bench":362.468964},{"date":"2022-12-20","open_a":56.306685,"high_a":56.602472,"low_a":56.243943,"close_a":56.279797,"volume_a":10286500,"open_b":159.815986,"high_b":160.193941,"low_b":158.778784,"close_b":159.297379,"volume_b":3249000,"open_bench":361.715409,"close_bench":362.964905},{"date":"2022-12-21","open_a":56.53077,"high_a":57.409164,"low_a":56.53077,"close_a":57.185081,"volume_a":10411300,"open_b":159.578638,"high_b":161.450855,"low_b":159.446784,"close_b":160.413666,"volume_b":4236900,"open_bench":365.54974,"close_bench":368.39212},{"date":"2022-12-22","open_a":56.844477,"high_a":56.996853,"low_a":56.154316,"close_a":56.772774,"volume_a":9666700,"open_b":159.851103,"high_b":160.018112,"low_b":157.266903,"close_b":159.174286,"volume_b":4304700,"open_bench":365.358974,"close_bench":363.136597},{"date":"2022-12-23","open_a":56.916182,"high_a":57.247818,"low_a":56.647287,"close_a":57.203003,"volume_a":6463300,"open_b":159.016115,"high_b":160.45764,"low_b":158.611778,"close_b":160.202728,"volume_b":2197800,"open_bench":362.116077,"close_bench":365.225525},{"date":"2022-12-27","open_a":57.301602,"high_a":57.624277,"low_a":57.104411,"close_a":57.55257,"volume_a":7320700,"open_b":161.099281,"high_b":161.389346,"low_b":160.211517,"close_b":160.914703,"volume_b":3045000,"open_bench":365.111065,"close_bench":363.785248},{"date":"2022-12-28","open_a":57.776646,"high_a":57.946949,"low_a":56.90722,"close_a":56.978924,"volume_a":7159400,"open_b":161.820046,"high_b":162.206786,"low_b":159.657752,"close_b":159.75444,"volume_b":2694300,"open_bench":363.718431,"close_bench":359.26413},{"date":"2022-12-29","open_a":57.185074,"high_a":57.498787,"low_a":57.095443,"close_a":57.319523,"volume_a":7169300,"open_b":159.903867,"high_b":160.73011,"low_b":159.877499,"close_b":159.956604,"volume_b":2549200,"open_bench":362.096953,"close_bench":365.730988},{"date":"2022-12-30","open_a":57.292642,"high_a":57.292642,"low_a":56.620404,"close_a":57.014786,"volume_a":7650200,"open_b":159.429222,"high_b":159.939032,"low_b":157.592146,"close_b":158.796356,"volume_b":3136200,"open_bench":363.060239,"close_bench":364.767548},{"date":"2023-01-03","open_a":56.969968,"high_a":56.969968,"low_a":55.733049,"close_a":56.423214,"volume_a":12180500,"open_b":158.734805,"high_b":158.875445,"low_b":155.20132,"close_b":157.697617,"volume_b":5137400,"open_bench":366.618028,"close_bench":363.231995},{"date":"2023-01-04","open_a":56.60248,"high_a":56.763818,"low_a":56.028834,"close_a":56.396324,"volume_a":13387900,"open_b":157.697651,"high_b":158.690888,"low_b":156.062749,"close_b":157.310898,"volume_b":4771300,"open_bench":365.482995,"close_bench":366.036224},{"date":"2023-01-05","open_a":56.109498,"high_a":56.288762,"low_a":55.643416,"close_a":55.750973,"volume_a":9814700,"open_b":156.423098,"high_b":157.135069,"low_b":155.456216,"close_b":155.667175,"volume_b":3647100,"open_bench":364.090415,"close_bench":361.85849},{"date":"2023-01-06","open_a":56.387356,"high_a":57.068556,"low_a":56.091573,"close_a":56.826553,"volume_a":9990000,"open_b":157.170251,"high_b":159.824772,"low_b":157.161466,"close_b":159.183121,"volume_b":3713200,"open_bench":364.939276,"close_bench":370.156647},{"date":"2023-01-09","open_a":56.736924,"high_a":57.113379,"low_a":56.091577,"close_a":56.118465,"volume_a":9442600,"open_b":158.602941,"high_b":160.650963,"low_b":157.170201,"close_b":157.627274,"volume_b":3319200,"open_bench":372.340863,"close_bench":369.946777},{"date":"2023-01-10","open_a":56.109492,"high_a":56.226014,"low_a":55.499995,"close_a":55.688225,"volume_a":10134600,"open_b":157.152645,"high_b":157.803094,"low_b":155.658383,"close_b":156.326416,"volume_b":3817300,"open_bench":369.365006,"close_bench":372.541199},{"date":"2023-01-11","open_a":55.912306,"high_a":56.073643,"low_a":55.042876,"close_a":55.580666,"volume_a":10823000,"open_b":157.680026,"high_b":157.961305,"low_b":154.419008,"close_b":156.502197,"volume_b":4369900,"open_bench":374.11505,"close_bench":377.253082},{"date":"2023-01-12","open_a":55.670304,"high_a":55.733046,"low_a":54.648503,"close_a":54.863617,"volume_a":13907300,"open_b":156.159397,"high_b":156.783464,"low_b":154.102589,"close_b":154.506912,"volume_b":4246600,"open_bench":378.349971,"close_bench":378.626556},{"date":"2023-01-13","open_a":54.406483,"high_a":55.087683,"low_a":54.307888,"close_a":55.060795,"volume_a":13986400,"open_b":154.172899,"high_b":154.462963,"low_b":153.091758,"close_b":154.032272,"volume_b":4720400,"open_bench":375.440805,"close_bench":380.095428},{"date":"2023-01-17","open_a":55.329701,"high_a":55.912309,"low_a":55.177328,"close_a":55.284885,"volume_a":18327800,"open_b":154.832126,"high_b":156.062705,"low_b":154.25201,"close_b":154.753021,"volume_b":5032600,"open_bench":380.076308,"close_bench":379.399078},{"date":"2023-01-18","open_a":55.132506,"high_a":55.204213,"low_a":53.492246,"close_a":53.608768,"volume_a":19679400,"open_b":154.032285,"high_b":154.471775,"low_b":150.595474,"close_b":150.850372,"volume_b":5032600,"open_bench":380.581828,"close_bench":373.409119},{"date":"2023-01-19","open_a":53.537068,"high_a":54.352718,"low_a":53.519142,"close_a":53.528107,"volume_a":14894500,"open_b":150.551506,"high_b":151.333797,"low_b":148.969341,"close_b":149.101196,"volume_b":4720400,"open_bench":371.377546,"close_bench":370.690826},{"date":"2023-01-20","open_a":53.528105,"high_a":53.859742,"low_a":53.241284,"close_a":53.85078,"volume_a":14237700,"open_b":148.503466,"high_b":149.329708,"low_b":147.193796,"close_b":149.320923,"volume_b":5779100,"open_bench":372.083384,"close_bench":377.596436},{"date":"2023-01-23","open_a":54.030041,"high_a":54.289974,"low_a":53.823888,"close_a":53.985226,"volume_a":12613400,"open_b":149.43523,"high_b":150.094464,"low_b":148.301353,"close_b":148.652939,"volume_b":5440400,"open_bench":378.397653,"close_bench":382.127075},{"date":"2023-01-24","open_a":53.850774,"high_a":55.437254,"low_a":53.599803,"close_a":54.272041,"volume_a":10103300,"open_b":149.083614,"high_b":150.191123,"low_b":148.063996,"close_b":150.032913,"volume_b":4125600,"open_bench":380.457906,"close_bench":381.716949},{"date":"2023-01-25","open_a":53.958333,"high_a":54.621606,"low_a":53.680473,"close_a":54.612644,"volume_a":11265800,"open_b":149.725275,"high_b":151.1668,"low_b":148.846296,"close_b":151.122849,"volume_b":3852800,"open_bench":377.663234,"close_bench":381.860016},{"date":"2023-01-26","open_a":54.505085,"high_a":54.765015,"low_a":54.289967,"close_a":54.505085,"volume_a":11884800,"open_b":150.867976,"high_b":150.894344,"low_b":149.461604,"close_b":149.619827,"volume_b":4558400,"open_bench":384.511678,"close_bench":386.056854},{"date":"2023-01-27","open_a":54.531983,"high_a":54.63954,"low_a":54.110713,"close_a":54.218273,"volume_a":14087000,"open_b":149.224247,"high_b":149.62857,"low_b":148.17826,"close_b":149.092392,"volume_b":5451400,"open_bench":385.017121,"close_bench":386.943817},{"date":"2023-01-30","open_a":54.263084,"high_a":54.827762,"low_a":54.191377,"close_a":54.352715,"volume_a":15632000,"open_b":149.233041,"high_b":150.709733,"low_b":148.670495,"close_b":148.969345,"volume_b":5191500,"open_bench":384.196883,"close_bench":382.088959},{"date":"2023-01-31","open_a":54.657463,"high_a":54.980138,"low_a":54.236193,"close_a":54.962212,"volume_a":14344400,"open_b":149.602214,"high_b":150.366934,"low_b":148.828721,"close_b":150.322983,"volume_b":4665500,"open_bench":382.604021,"close_bench":387.70694},{"date":"2023-02-01","open_a":54.800877,"high_a":55.195259,"low_a":54.334792,"close_a":54.97118,"volume_a":12535400,"open_b":150.199939,"high_b":151.465671,"low_b":148.995728,"close_b":150.797638,"volume_b":4147700,"open_bench":386.495567,"close_bench":391.827393},{"date":"2023-02-02","open_a":54.496126,"high_a":54.693317,"low_a":53.698404,"close_a":54.030041,"volume_a":16947200,"open_b":148.951811,"high_b":149.91868,"low_b":148.11677,"close_b":149.400085,"volume_b":5653800,"open_bench":395.699942,"close_bench":397.531281},{"date":"2023-02-03","open_a":54.09278,"high_a":54.155522,"low_a":53.088905,"close_a":53.626698,"volume_a":15404400,"open_b":150.270271,"high_b":150.560322,"low_b":146.833459,"close_b":148.652939,"volume_b":5308200,"open_bench":392.580877,"close_bench":393.305786},{"date":"2023-02-06","open_a":53.590836,"high_a":54.08381,"low_a":53.528094,"close_a":53.931435,"volume_a":11330900,"open_b":149.197891,"high_b":151.518395,"low_b":149.066036,"close_b":151.026169,"volume_b":5640000,"open_bench":390.864029,"close_bench":390.902161},{"date":"2023-02-07","open_a":53.48328,"high_a":53.994184,"low_a":53.366761,"close_a":53.841808,"volume_a":14922700,"open_b":149.681331,"high_b":151.210759,"low_b":148.775984,"close_b":150.797638,"volume_b":6044700,"open_bench":389.986498,"close_bench":396.014618},{"date":"2023-02-08","open_a":53.707368,"high_a":53.707368,"low_a":53.411584,"close_a":53.528107,"volume_a":11012000,"open_b":150.30542,"high_b":150.595485,"low_b":149.637401,"close_b":150.44606,"volume_b":5687300,"open_bench":394.049799,"close_bench":391.684326},{"date":"2023-02-09","open_a":53.913516,"high_a":54.065888,"low_a":53.420538,"close_a":53.438465,"volume_a":12515300,"open_b":154.102609,"high_b":155.131013,"low_b":151.219546,"close_b":151.869995,"volume_b":9619400,"open_bench":395.270632,"close_bench":388.288696},{"date":"2023-02-10","open_a":53.528099,"high_a":53.689436,"low_a":52.775192,"close_a":53.438465,"volume_a":14420700,"open_b":152.168827,"high_b":155.218885,"low_b":151.755706,"close_b":154.876083,"volume_b":6176000,"open_bench":387.115515,"close_bench":389.194855},{"date":"2023-02-13","open_a":53.689444,"high_a":54.460275,"low_a":53.537068,"close_a":54.316864,"volume_a":16191800,"open_b":154.928852,"high_b":156.660442,"low_b":154.735475,"close_b":156.256119,"volume_b":5205100,"open_bench":389.843353,"close_bench":393.763519},{"date":"2023-02-14","open_a":53.967299,"high_a":54.585761,"low_a":53.384695,"close_a":53.411583,"volume_a":20505700,"open_b":155.667197,"high_b":155.799039,"low_b":153.636745,"close_b":154.498154,"volume_b":5203000,"open_bench":392.247093,"close_bench":393.582458},{"date":"2023-02-15","open_a":53.689443,"high_a":53.716331,"low_a":53.088908,"close_a":53.411583,"volume_a":13171400,"open_b":153.821312,"high_b":154.744243,"low_b":153.460927,"close_b":154.709076,"volume_b":4125500,"open_bench":391.39818,"close_bench":394.860535},{"date":"2023-02-16","open_a":53.088906,"high_a":53.617734,"low_a":52.837938,"close_a":53.079945,"volume_a":16538300,"open_b":153.592773,"high_b":155.491357,"low_b":153.416966,"close_b":154.269577,"volume_b":4973200,"open_bench":389.910192,"close_bench":389.423737},{"date":"2023-02-17","open_a":53.330911,"high_a":53.985223,"low_a":53.223354,"close_a":53.886627,"volume_a":16814700,"open_b":153.961944,"high_b":155.069452,"low_b":153.671879,"close_b":154.946396,"volume_b":3952100,"open_bench":387.306307,"close_bench":388.450897},{"date":"2023-02-21","open_a":53.770114,"high_a":53.985229,"low_a":53.528108,"close_a":53.599812,"volume_a":14278300,"open_b":154.331126,"high_b":155.939659,"low_b":153.531252,"close_b":154.858505,"volume_b":5797300,"open_bench":384.444882,"close_bench":380.658234},{"date":"2023-02-22","open_a":53.644624,"high_a":54.101748,"low_a":53.554994,"close_a":53.761147,"volume_a":12409600,"open_b":155.333194,"high_b":155.957262,"low_b":154.498167,"close_b":154.805801,"volume_b":5299500,"open_bench":381.068357,"close_bench":380.133636},{"date":"2023-02-23","open_a":53.78803,"high_a":54.15552,"low_a":53.60877,"close_a":53.859737,"volume_a":12222800,"open_b":155.280411,"high_b":155.570476,"low_b":153.812518,"close_b":154.972763,"volume_b":3801400,"open_bench":383.014068,"close_bench":382.15564},{"date":"2023-02-24","open_a":53.492251,"high_a":53.698404,"low_a":53.232318,"close_a":53.635662,"volume_a":9736200,"open_b":154.225656,"high_b":154.990364,"low_b":153.197239,"close_b":154.665146,"volume_b":2959100,"open_bench":377.157679,"close_bench":378.073334},{"date":"2023-02-27","open_a":53.940404,"high_a":54.047961,"low_a":53.546026,"close_a":53.617729,"volume_a":8849800,"open_b":155.552885,"high_b":155.798998,"low_b":154.392627,"close_b":154.62117,"volume_b":3208500,"open_bench":381.402142,"close_bench":379.360992},{"date":"2023-02-28","open_a":53.510177,"high_a":53.510177,"low_a":53.008238,"close_a":53.339874,"volume_a":16082200,"open_b":153.715832,"high_b":153.874042,"low_b":152.239141,"close_b":152.529205,"volume_b":4317100,"open_bench":378.884095,"close_bench":377.958893},{"date":"2023-03-01","open_a":52.999281,"high_a":53.044096,"low_a":52.318077,"close_a":52.757275,"volume_a":13275800,"open_b":151.465668,"high_b":151.588725,"low_b":148.652925,"close_b":150.595474,"volume_b":5359100,"open_bench":377.148163,"close_bench":376.509094},{"date":"2023-03-02","open_a":52.739346,"high_a":53.581884,"low_a":52.703492,"close_a":53.528107,"volume_a":11758300,"open_b":150.887482,"high_b":153.763468,"low_b":150.241495,"close_b":153.374115,"volume_b":4476300,"open_bench":374.544179,"close_bench":379.437256},{"date":"2023-03-03","open_a":53.375722,"high_a":53.492244,"low_a":53.070974,"close_a":53.277126,"volume_a":13101800,"open_b":153.090916,"high_b":153.489126,"low_b":151.613103,"close_b":153.223648,"volume_b":3627100,"open_bench":381.249601,"close_bench":385.522705},{"date":"2023-03-06","open_a":53.115798,"high_a":54.119676,"low_a":53.079948,"close_a":54.101749,"volume_a":16282700,"open_b":152.400707,"high_b":153.674992,"low_b":152.400707,"close_b":153.533401,"volume_b":3611400,"open_bench":386.342903,"close_bench":385.789703},{"date":"2023-03-07","open_a":54.218276,"high_a":54.289979,"low_a":53.554999,"close_a":53.78804,"volume_a":13961200,"open_b":153.683819,"high_b":154.205918,"low_b":152.090963,"close_b":152.772354,"volume_b":4094900,"open_bench":385.741994,"close_bench":379.876007},{"date":"2023-03-08","open_a":53.814922,"high_a":54.074855,"low_a":53.537063,"close_a":53.814922,"volume_a":10851600,"open_b":152.807756,"high_b":153.409508,"low_b":152.011335,"close_b":153.188278,"volume_b":2920200,"open_bench":379.990481,"close_bench":380.496002},{"date":"2023-03-09","open_a":54.263084,"high_a":54.263084,"low_a":53.133722,"close_a":53.295059,"volume_a":13900900,"open_b":153.639552,"high_b":154.294396,"low_b":151.49805,"close_b":152.029007,"volume_b":4042100,"open_bench":381.278249,"close_bench":373.476044},{"date":"2023-03-10","open_a":53.510173,"high_a":53.904552,"low_a":52.873789,"close_a":53.070976,"volume_a":13745800,"open_b":152.675049,"high_b":154.188266,"low_b":151.285734,"close_b":152.23259,"volume_b":4129900,"open_bench":372.932284,"close_bench":368.086914},{"date":"2023-03-13","open_a":53.142683,"high_a":54.558863,"low_a":53.106829,"close_a":53.608768,"volume_a":21956000,"open_b":152.878555,"high_b":156.834142,"low_b":152.745822,"close_b":153.719238,"volume_b":8265100,"open_bench":364.176223,"close_bench":367.562256},{"date":"2023-03-14","open_a":53.779073,"high_a":54.191379,"low_a":53.196465,"close_a":53.805962,"volume_a":19496900,"open_b":152.639627,"high_b":153.807713,"low_b":151.825503,"close_b":153.559937,"volume_b":6480800,"open_bench":372.464889,"close_bench":373.638092},{"date":"2023-03-15","open_a":53.268172,"high_a":54.209305,"low_a":53.097873,"close_a":54.16449,"volume_a":21089900,"open_b":153.559924,"high_b":156.329723,"low_b":153.259055,"close_b":156.303177,"volume_b":7099800,"open_bench":368.06785,"close_bench":371.30127},{"date":"2023-03-16","open_a":54.272874,"high_a":54.498672,"low_a":53.920629,"close_a":54.462543,"volume_a":15658300,"open_b":156.285488,"high_b":156.630606,"low_b":155.143949,"close_b":156.196991,"volume_b":4734400,"open_bench":368.954902,"close_bench":377.815826},{"date":"2023-03-17","open_a":54.471583,"high_a":54.507708,"low_a":53.866441,"close_a":54.209656,"volume_a":27351200,"open_b":156.197024,"high_b":156.356317,"low_b":154.17941,"close_b":154.975845,"volume_b":8854600,"open_bench":376.490694,"close_bench":373.398102},{"date":"2023-03-20","open_a":54.336098,"high_a":54.733501,"low_a":54.164492,"close_a":54.733501,"volume_a":12904100,"open_b":156.188152,"high_b":157.232349,"low_b":155.312077,"close_b":157.15271,"volume_b":4367000,"open_bench":374.173664,"close_bench":376.988586},{"date":"2023-03-21","open_a":54.706404,"high_a":54.868979,"low_a":54.083202,"close_a":54.480606,"volume_a":13716500,"open_b":157.21467,"high_b":157.665974,"low_b":155.993478,"close_b":157.524384,"volume_b":3812000,"open_bench":380.339665,"close_bench":381.938629},{"date":"2023-03-22","open_a":54.55286,"high_a":54.977358,"low_a":54.218676,"close_a":54.23674,"volume_a":10494200,"open_b":157.373934,"high_b":158.727859,"low_b":156.117352,"close_b":156.196991,"volume_b":4380800,"open_bench":381.766329,"close_bench":375.427948},{"date":"2023-03-23","open_a":54.272868,"high_a":54.543826,"low_a":53.938688,"close_a":54.119324,"volume_a":15479000,"open_b":155.798794,"high_b":156.825303,"low_b":154.878484,"close_b":155.435974,"volume_b":4655600,"open_bench":378.28117,"close_bench":376.442871},{"date":"2023-03-24","open_a":54.417386,"high_a":55.112845,"low_a":54.1103,"close_a":55.004463,"volume_a":12975600,"open_b":156.214667,"high_b":158.701286,"low_b":156.090779,"close_b":158.480057,"volume_b":3713800,"open_bench":375.169401,"close_bench":378.913055},{"date":"2023-03-27","open_a":55.221223,"high_a":55.528309,"low_a":55.049617,"close_a":55.410892,"volume_a":12777400,"open_b":158.91367,"high_b":159.860525,"low_b":158.674738,"close_b":158.83403,"volume_b":4078600,"open_bench":381.18227,"close_bench":379.621613},{"date":"2023-03-28","open_a":55.447023,"high_a":55.763142,"low_a":55.33864,"close_a":55.474117,"volume_a":10925700,"open_b":158.692492,"high_b":159.692454,"low_b":158.692492,"close_b":158.780975,"volume_b":2850000,"open_bench":378.932191,"close_bench":378.76944},{"date":"2023-03-29","open_a":55.745082,"high_a":55.952816,"low_a":55.627666,"close_a":55.871529,"volume_a":9441000,"open_b":159.860594,"high_b":160.090681,"low_b":159.276558,"close_b":159.878296,"volume_b":4496000,"open_bench":382.915241,"close_bench":384.274841},{"date":"2023-03-30","open_a":55.889586,"high_a":56.079256,"low_a":55.690882,"close_a":55.862488,"volume_a":9870700,"open_b":160.170271,"high_b":160.462296,"low_b":159.356147,"close_b":160.019836,"volume_b":2743000,"open_bench":386.898234,"close_bench":386.524841},{"date":"2023-03-31","open_a":56.052161,"high_a":56.223766,"low_a":55.880552,"close_a":56.025063,"volume_a":14440900,"open_b":161.196806,"high_b":161.488831,"low_b":160.453477,"close_b":161.320694,"volume_b":4479800,"open_bench":387.444033,"close_bench":391.972809},{"date":"2023-04-03","open_a":56.268924,"high_a":56.467627,"low_a":55.87152,"close_a":56.359245,"volume_a":12300700,"open_b":160.966724,"high_b":161.727755,"low_b":159.639346,"close_b":161.497681,"volume_b":3902300,"open_bench":391.455707,"close_bench":393.46637},{"date":"2023-04-04","open_a":56.350216,"high_a":56.666336,"low_a":56.070229,"close_a":56.187641,"volume_a":11433400,"open_b":161.214497,"high_b":162.258708,"low_b":160.258796,"close_b":160.922485,"volume_b":3306400,"open_bench":394.107863,"close_bench":391.283386},{"date":"2023-04-05","open_a":56.476654,"high_a":56.91922,"low_a":56.449559,"close_a":56.720516,"volume_a":12789600,"open_b":162.161367,"high_b":163.674584,"low_b":161.99323,"close_b":162.506485,"volume_b":3826300,"open_bench":390.555781,"close_bench":390.258972},{"date":"2023-04-06","open_a":56.774721,"high_a":56.937296,"low_a":56.341189,"close_a":56.756657,"volume_a":9644200,"open_b":162.205619,"high_b":163.479904,"low_b":162.028638,"close_b":163.143631,"volume_b":3389700,"open_bench":389.464225,"close_bench":391.781281},{"date":"2023-04-10","open_a":56.44053,"high_a":56.621166,"low_a":56.016029,"close_a":56.621166,"volume_a":7952600,"open_b":162.67463,"high_b":162.736567,"low_b":160.780904,"close_b":162.117126,"volume_b":2408100,"open_bench":389.311013,"close_bench":392.18338},{"date":"2023-04-11","open_a":56.585043,"high_a":56.639233,"low_a":56.359245,"close_a":56.52182,"volume_a":9913000,"open_b":162.311798,"high_b":162.718867,"low_b":161.727749,"close_b":161.869339,"volume_b":3586700,"open_bench":392.805761,"close_bench":392.288727},{"date":"2023-04-12","open_a":56.431497,"high_a":56.928252,"low_a":56.277955,"close_a":56.621166,"volume_a":9593700,"open_b":161.665825,"high_b":162.630383,"low_b":161.276459,"close_b":161.550781,"volume_b":3620900,"open_bench":394.347246,"close_bench":390.689758},{"date":"2023-04-13","open_a":56.684392,"high_a":57.045671,"low_a":56.431499,"close_a":57.03664,"volume_a":9871400,"open_b":161.60385,"high_b":163.364843,"low_b":161.152547,"close_b":163.161316,"volume_b":4303600,"open_bench":391.771658,"close_bench":395.87915},{"date":"2023-04-14","open_a":56.946323,"high_a":57.036645,"low_a":56.702461,"close_a":56.946323,"volume_a":8946000,"open_b":162.240999,"high_b":162.842751,"low_b":161.869335,"close_b":162.391434,"volume_b":2572900,"open_bench":395.247225,"close_bench":394.912109},{"date":"2023-04-17","open_a":57.190185,"high_a":57.379855,"low_a":57.02761,"close_a":57.316631,"volume_a":9452400,"open_b":162.993177,"high_b":163.48873,"low_b":162.364879,"close_b":163.223251,"volume_b":3224400,"open_bench":394.825981,"close_bench":396.329193},{"date":"2023-04-18","open_a":57.361789,"high_a":57.443077,"low_a":57.072768,"close_a":57.406952,"volume_a":9035700,"open_b":163.506432,"high_b":163.904642,"low_b":162.594966,"close_b":163.559525,"volume_b":2438600,"open_bench":397.899448,"close_bench":396.587738},{"date":"2023-04-19","open_a":57.560494,"high_a":57.650815,"low_a":57.289537,"close_a":57.515335,"volume_a":9769500,"open_b":163.922335,"high_b":163.99313,"low_b":163.055105,"close_b":163.462173,"volume_b":2209900,"open_bench":394.682394,"close_bench":396.520721},{"date":"2023-04-20","open_a":57.470182,"high_a":57.804363,"low_a":57.298573,"close_a":57.768234,"volume_a":10706400,"open_b":163.568353,"high_b":164.630253,"low_b":163.002006,"close_b":164.001968,"volume_b":3868200,"open_bench":393.715273,"close_bench":394.356781},{"date":"2023-04-21","open_a":57.804355,"high_a":58.048215,"low_a":57.623716,"close_a":57.849518,"volume_a":12931100,"open_b":164.586024,"high_b":164.931141,"low_b":163.824993,"close_b":164.072769,"volume_b":3781800,"open_bench":394.653655,"close_bench":394.663239},{"date":"2023-04-24","open_a":58.481747,"high_a":58.698512,"low_a":57.461138,"close_a":57.759193,"volume_a":15532300,"open_b":165.037339,"high_b":165.09929,"low_b":163.727663,"close_b":164.15242,"volume_b":3540200,"open_bench":394.462135,"close_bench":395.074921},{"date":"2023-04-25","open_a":57.957902,"high_a":57.957902,"low_a":57.47018,"close_a":57.66888,"volume_a":12818900,"open_b":165.090458,"high_b":168.29386,"low_b":164.807291,"close_b":167.87796,"volume_b":6231600,"open_bench":393.112139,"close_bench":388.803589},{"date":"2023-04-26","open_a":57.406953,"high_a":57.596622,"low_a":57.253408,"close_a":57.397919,"volume_a":13179600,"open_b":166.789452,"high_b":167.621277,"low_b":166.249651,"close_b":166.842545,"volume_b":4481700,"open_bench":389.416318,"close_bench":387.156708},{"date":"2023-04-27","open_a":57.379855,"high_a":57.587589,"low_a":57.000515,"close_a":57.515335,"volume_a":13795800,"open_b":166.524024,"high_b":168.090334,"low_b":165.93113,"close_b":167.86026,"volume_b":4574500,"open_bench":389.68445,"close_bench":394.864288},{"date":"2023-04-28","open_a":57.452106,"high_a":57.948865,"low_a":57.33469,"close_a":57.939831,"volume_a":13687600,"open_b":168.04606,"high_b":169.568121,"low_b":167.815986,"close_b":168.922134,"volume_b":4608900,"open_bench":393.983362,"close_bench":398.234467},{"date":"2023-05-01","open_a":57.939833,"high_a":58.427558,"low_a":57.858542,"close_a":58.075314,"volume_a":9091500,"open_b":168.532785,"high_b":170.196436,"low_b":168.38235,"close_b":169.621231,"volume_b":3137800,"open_bench":397.794091,"close_bench":397.832397},{"date":"2023-05-02","open_a":57.930805,"high_a":57.930805,"low_a":57.145028,"close_a":57.813393,"volume_a":10948400,"open_b":169.338048,"high_b":170.346855,"low_b":167.913329,"close_b":170.125626,"volume_b":3712200,"open_bench":397.123878,"close_bench":393.361084},{"date":"2023-05-03","open_a":57.822416,"high_a":57.966931,"low_a":57.425016,"close_a":57.488239,"volume_a":11084600,"open_b":170.417666,"high_b":171.01056,"low_b":169.709728,"close_b":170.06369,"volume_b":3389600,"open_bench":393.85894,"close_bench":390.661041},{"date":"2023-05-04","open_a":57.569532,"high_a":57.741138,"low_a":57.361795,"close_a":57.551468,"volume_a":11938300,"open_b":170.860117,"high_b":171.629992,"low_b":170.045993,"close_b":171.125595,"volume_b":4267600,"open_bench":389.617452,"close_bench":387.894043},{"date":"2023-05-05","open_a":57.479204,"high_a":57.994023,"low_a":57.470173,"close_a":57.822414,"volume_a":9039000,"open_b":170.630033,"high_b":172.098988,"low_b":170.249511,"close_b":171.913162,"volume_b":4146500,"open_bench":391.513185,"close_bench":395.074921},{"date":"2023-05-08","open_a":57.632742,"high_a":57.858536,"low_a":57.452102,"close_a":57.73209,"volume_a":7240700,"open_b":171.143265,"high_b":171.674221,"low_b":170.479576,"close_b":171.09903,"volume_b":2249900,"open_bench":395.400462,"close_bench":395.180237},{"date":"2023-05-09","open_a":57.479205,"high_a":57.578557,"low_a":57.154055,"close_a":57.253407,"volume_a":9067700,"open_b":171.550335,"high_b":172.267117,"low_b":170.532684,"close_b":171.798111,"volume_b":3087600,"open_bench":393.638738,"close_bench":393.447235},{"date":"2023-05-10","open_a":57.190178,"high_a":57.479199,"low_a":56.892126,"close_a":57.352753,"volume_a":10134000,"open_b":171.771572,"high_b":173.072404,"low_b":170.497287,"close_b":171.913162,"volume_b":4352900,"open_bench":396.271762,"close_bench":395.285583},{"date":"2023-05-11","open_a":57.425017,"high_a":57.741133,"low_a":56.910194,"close_a":57.67791,"volume_a":9581500,"open_b":171.922004,"high_b":173.019308,"low_b":170.789309,"close_b":172.860016,"volume_b":3690200,"open_bench":394.423887,"close_bench":394.596222},{"date":"2023-05-12","open_a":57.677904,"high_a":57.975959,"low_a":57.533393,"close_a":57.903702,"volume_a":8740800,"open_b":172.860012,"high_b":173.638744,"low_b":172.514894,"close_b":173.550247,"volume_b":2741000,"open_bench":395.831415,"close_bench":394.079254},{"date":"2023-05-15","open_a":58.00306,"high_a":58.030155,"low_a":57.54243,"close_a":57.750164,"volume_a":8150000,"open_b":173.948478,"high_b":174.222801,"low_b":171.67423,"close_b":171.913162,"volume_b":2946400,"open_bench":394.682353,"close_bench":395.438751},{"date":"2023-05-16","open_a":57.714037,"high_a":57.714037,"low_a":57.045672,"close_a":57.099865,"volume_a":9129600,"open_b":172.099013,"high_b":172.099013,"low_b":170.382268,"close_b":171.169846,"volume_b":5803100,"open_bench":394.337663,"close_bench":392.796173},{"date":"2023-05-17","open_a":57.262438,"high_a":57.280499,"low_a":56.657297,"close_a":57.03664,"volume_a":11583000,"open_b":171.063637,"high_b":171.311414,"low_b":168.877887,"close_b":169.957489,"volume_b":4878800,"open_bench":394.806854,"close_bench":397.564331},{"date":"2023-05-18","open_a":56.910186,"high_a":56.937284,"low_a":56.512782,"close_a":56.720516,"volume_a":13595100,"open_b":169.302673,"high_b":169.647791,"low_b":168.497394,"close_b":169.515045,"volume_b":3709400,"open_bench":397.248305,"close_bench":401.394104},{"date":"2023-05-19","open_a":56.937289,"high_a":56.991479,"low_a":56.548916,"close_a":56.74762,"volume_a":14251400,"open_b":170.311446,"high_b":170.665408,"low_b":169.107956,"close_b":169.762787,"volume_b":3883000,"open_bench":402.294163,"close_bench":400.810089},{"date":"2023-05-22","open_a":56.71149,"high_a":56.783744,"low_a":55.492178,"close_a":55.555401,"volume_a":15055400,"open_b":169.187576,"high_b":169.364557,"low_b":164.957666,"close_b":165.161194,"volume_b":5787000,"open_bench":400.829257,"close_bench":400.97287},{"date":"2023-05-23","open_a":55.555407,"high_a":55.699918,"low_a":55.248321,"close_a":55.456059,"volume_a":13932700,"open_b":165.028509,"high_b":165.152397,"low_b":163.347156,"close_b":164.656845,"volume_b":6382300,"open_bench":399.335592,"close_bench":396.472809},{"date":"2023-05-24","open_a":55.437986,"high_a":55.483145,"low_a":54.841878,"close_a":54.986389,"volume_a":11090100,"open_b":164.833802,"high_b":165.249715,"low_b":163.152448,"close_b":163.61261,"volume_b":3721000,"open_bench":394.873866,"close_bench":393.600433},{"date":"2023-05-25","open_a":54.616086,"high_a":54.850915,"low_a":54.327065,"close_a":54.561893,"volume_a":14463600,"open_b":161.67464,"high_b":163.01972,"low_b":161.010951,"close_b":162.648056,"volume_b":4712700,"open_bench":397.09514,"close_bench":397.008972},{"date":"2023-05-26","open_a":54.59802,"high_a":54.769625,"low_a":54.290934,"close_a":54.426411,"volume_a":12570300,"open_b":162.568398,"high_b":163.046248,"low_b":161.409143,"close_b":162.453354,"volume_b":4449300,"open_bench":397.660043,"close_bench":402.150513},{"date":"2023-05-30","open_a":54.272873,"high_a":54.372225,"low_a":53.758053,"close_a":53.992882,"volume_a":15634200,"open_b":161.10834,"high_b":162.019805,"low_b":159.904849,"close_b":160.710129,"volume_b":4331800,"open_bench":404.074979,"close_bench":402.30368},{"date":"2023-05-31","open_a":54.11932,"high_a":54.44447,"low_a":53.622564,"close_a":53.884491,"volume_a":59432200,"open_b":161.816218,"high_b":163.373669,"low_b":160.577337,"close_b":161.364914,"volume_b":20414600,"open_bench":400.484575,"close_bench":400.072876},{"date":"2023-06-01","open_a":54.137393,"high_a":54.345127,"low_a":53.983852,"close_a":54.191586,"volume_a":13402700,"open_b":162.554574,"high_b":162.795175,"low_b":161.110995,"close_b":162.349625,"volume_b":5277400,"open_bench":400.302551,"close_bench":403.873871},{"date":"2023-06-02","open_a":54.435451,"high_a":55.383803,"low_a":54.263845,"close_a":55.239292,"volume_a":15259800,"open_b":162.198107,"high_b":164.220895,"low_b":161.912948,"close_b":164.015945,"volume_b":4288300,"open_bench":406.439996,"close_bench":409.714508},{"date":"2023-06-05","open_a":55.094767,"high_a":55.492171,"low_a":54.68833,"close_a":54.868969,"volume_a":12915600,"open_b":163.828819,"high_b":165.896165,"low_b":163.61495,"close_b":164.13179,"volume_b":4311700,"open_bench":410.059081,"close_bench":408.929291},{"date":"2023-06-06","open_a":55.04962,"high_a":55.085748,"low_a":54.299968,"close_a":54.471577,"volume_a":10943500,"open_b":164.835755,"high_b":164.960508,"low_b":160.781245,"close_b":161.788193,"volume_b":3975600,"open_bench":408.517607,"close_bench":409.819733},{"date":"2023-06-07","open_a":54.200612,"high_a":54.516732,"low_a":54.029007,"close_a":54.390285,"volume_a":16817900,"open_b":161.030775,"high_b":161.948607,"low_b":160.130755,"close_b":160.496109,"volume_b":5162700,"open_bench":410.212315,"close_bench":408.40271},{"date":"2023-06-08","open_a":54.309,"high_a":54.561893,"low_a":54.17352,"close_a":54.525764,"volume_a":15585900,"open_b":160.13076,"high_b":162.447627,"low_b":159.801057,"close_b":162.269409,"volume_b":4036100,"open_bench":408.469763,"close_bench":410.872986},{"date":"2023-06-09","open_a":54.43545,"high_a":54.796725,"low_a":54.345128,"close_a":54.616089,"volume_a":12581500,"open_b":161.921853,"high_b":162.919881,"low_b":161.289166,"close_b":162.492157,"volume_b":3200100,"open_bench":411.667613,"close_bench":411.610168},{"date":"2023-06-12","open_a":54.72447,"high_a":54.778663,"low_a":54.209651,"close_a":54.381256,"volume_a":14208900,"open_b":162.616946,"high_b":162.830802,"low_b":161.012962,"close_b":162.091187,"volume_b":4207100,"open_bench":412.586791,"close_bench":415.344238},{"date":"2023-06-13","open_a":54.381247,"high_a":54.796715,"low_a":54.227706,"close_a":54.598015,"volume_a":13397800,"open_b":162.29613,"high_b":162.349594,"low_b":161.360472,"close_b":161.77037,"volume_b":2838700,"open_bench":416.799624,"close_bench":418.082611},{"date":"2023-06-14","open_a":54.850911,"high_a":55.16703,"low_a":54.733494,"close_a":54.968327,"volume_a":14099700,"open_b":162.180284,"high_b":163.650594,"low_b":161.68127,"close_b":163.22287,"volume_b":3559300,"open_bench":418.417693,"close_bench":418.580444},{"date":"2023-06-15","open_a":55.159447,"high_a":55.787395,"low_a":54.913727,"close_a":55.72369,"volume_a":15271400,"open_b":164.194161,"high_b":165.896166,"low_b":163.534755,"close_b":165.486267,"volume_b":4326100,"open_bench":417.766662,"close_bench":423.769928},{"date":"2023-06-16","open_a":56.060418,"high_a":56.579158,"low_a":55.95121,"close_a":56.124123,"volume_a":26738000,"open_b":165.869435,"high_b":167.161528,"low_b":165.477348,"close_b":165.780319,"volume_b":8925700,"open_bench":425.747679,"close_bench":422.326477},{"date":"2023-06-20","open_a":56.051306,"high_a":56.388031,"low_a":55.732782,"close_a":55.75098,"volume_a":13971800,"open_b":166.225856,"high_b":167.411021,"low_b":165.067437,"close_b":165.129807,"volume_b":4154800,"open_bench":420.394838,"close_bench":420.135345},{"date":"2023-06-21","open_a":55.796497,"high_a":56.169626,"low_a":55.496172,"close_a":55.905704,"volume_a":13948700,"open_b":165.495182,"high_b":166.528862,"low_b":164.131799,"close_b":165.236755,"volume_b":3832400,"open_bench":419.155041,"close_bench":417.982605},{"date":"2023-06-22","open_a":56.251529,"high_a":56.515447,"low_a":56.051312,"close_a":56.28793,"volume_a":15193400,"open_b":166.395186,"high_b":167.152627,"low_b":165.628839,"close_b":166.947678,"volume_b":3185800,"open_bench":417.031266,"close_bench":419.491455},{"date":"2023-06-23","open_a":56.369839,"high_a":56.397144,"low_a":55.678186,"close_a":55.696388,"volume_a":25112100,"open_b":167.526885,"high_b":167.526885,"low_b":165.700127,"close_b":165.807068,"volume_b":7592600,"open_bench":416.051017,"close_bench":416.320099},{"date":"2023-06-26","open_a":55.705493,"high_a":55.778301,"low_a":54.959235,"close_a":55.714596,"volume_a":12856100,"open_b":165.887256,"high_b":166.127843,"low_b":164.069403,"close_b":164.755554,"volume_b":3706100,"open_bench":415.753097,"close_bench":414.61911},{"date":"2023-06-27","open_a":55.714589,"high_a":55.841999,"low_a":55.496171,"close_a":55.596279,"volume_a":9068500,"open_b":164.559526,"high_b":166.288264,"low_b":163.926853,"close_b":165.940735,"volume_b":5763000,"open_bench":415.493599,"close_bench":419.164673},{"date":"2023-06-28","open_a":55.405168,"high_a":55.550779,"low_a":54.968334,"close_a":55.077541,"volume_a":11173200,"open_b":165.967469,"high_b":166.404113,"low_b":163.410015,"close_b":163.69516,"volume_b":5193900,"open_bench":418.088317,"close_bench":419.376099},{"date":"2023-06-29","open_a":54.804528,"high_a":54.922835,"low_a":54.394995,"close_a":54.622513,"volume_a":17142300,"open_b":163.294163,"high_b":163.864466,"low_b":161.636715,"close_b":163.85556,"volume_b":5770400,"open_bench":418.962913,"close_bench":421.029083},{"date":"2023-06-30","open_a":54.604303,"high_a":54.922826,"low_a":54.485992,"close_a":54.80452,"volume_a":19739200,"open_b":163.998116,"high_b":165.316954,"low_b":163.85555,"close_b":165.049622,"volume_b":7635100,"open_bench":424.229239,"close_bench":425.997498},{"date":"2023-07-03","open_a":54.786321,"high_a":55.177652,"low_a":54.631606,"close_a":55.132149,"volume_a":9694500,"open_b":164.229827,"high_b":165.423898,"low_b":163.490211,"close_b":165.38826,"volume_b":2389000,"open_bench":425.651595,"close_bench":426.487671},{"date":"2023-07-05","open_a":54.859126,"high_a":55.605384,"low_a":54.749918,"close_a":55.541679,"volume_a":14716600,"open_b":164.853595,"high_b":166.359557,"low_b":164.826863,"close_b":166.261536,"volume_b":5277100,"open_bench":424.680896,"close_bench":425.853333},{"date":"2023-07-06","open_a":55.268652,"high_a":55.532573,"low_a":55.059336,"close_a":55.113941,"volume_a":12603900,"open_b":165.575396,"high_b":166.413033,"low_b":164.755585,"close_b":166.279373,"volume_b":5592100,"open_bench":422.287982,"close_bench":422.518616},{"date":"2023-07-07","open_a":54.886422,"high_a":55.059334,"low_a":54.358578,"close_a":54.38588,"volume_a":14574000,"open_b":165.58429,"high_b":165.637754,"low_b":163.098112,"close_b":163.14267,"volume_b":6309900,"open_bench":421.528802,"close_bench":421.451904},{"date":"2023-07-10","open_a":54.577004,"high_a":54.740817,"low_a":53.821643,"close_a":53.976357,"volume_a":15480600,"open_b":164.024863,"high_b":165.664485,"low_b":163.766437,"close_b":164.229813,"volume_b":5508800,"open_bench":421.096307,"close_bench":422.518616},{"date":"2023-07-11","open_a":54.11286,"high_a":54.203869,"low_a":53.548616,"close_a":54.167465,"volume_a":15967700,"open_b":164.301074,"high_b":164.612951,"low_b":162.554512,"close_b":163.944626,"volume_b":3914300,"open_bench":423.277797,"close_bench":425.209412},{"date":"2023-07-12","open_a":54.376784,"high_a":54.649805,"low_a":54.103763,"close_a":54.458691,"volume_a":14241300,"open_b":163.944659,"high_b":164.506057,"low_b":163.053559,"close_b":163.22287,"volume_b":6080100,"open_bench":428.986275,"close_bench":428.630676},{"date":"2023-07-13","open_a":54.713509,"high_a":55.022933,"low_a":54.586099,"close_a":54.922825,"volume_a":11269800,"open_b":165.566478,"high_b":167.58036,"low_b":163.303075,"close_b":167.108078,"volume_b":8223100,"open_bench":430.4374,"close_bench":432.032684},{"date":"2023-07-14","open_a":54.73171,"high_a":55.550776,"low_a":54.686207,"close_a":55.423367,"volume_a":11634200,"open_b":167.40216,"high_b":168.088311,"low_b":166.716008,"close_b":167.71405,"volume_b":4467600,"open_bench":432.916776,"close_bench":431.76355},{"date":"2023-07-17","open_a":55.295955,"high_a":55.605379,"low_a":55.050239,"close_a":55.341461,"volume_a":10014300,"open_b":166.047682,"high_b":166.644718,"low_b":165.227871,"close_b":165.415009,"volume_b":5637400,"open_bench":431.619434,"close_bench":433.262756},{"date":"2023-07-18","open_a":55.47797,"high_a":55.741892,"low_a":54.977432,"close_a":55.123043,"volume_a":11152900,"open_b":165.245674,"high_b":166.475397,"low_b":163.802096,"close_b":164.36348,"volume_b":5684100,"open_bench":432.93607,"close_bench":436.482208},{"date":"2023-07-19","open_a":55.305059,"high_a":56.233333,"low_a":55.223153,"close_a":56.096821,"volume_a":12936000,"open_b":164.853598,"high_b":166.279364,"low_b":164.336758,"close_b":165.976379,"volume_b":4140800,"open_bench":437.270256,"close_bench":437.45285},{"date":"2023-07-20","open_a":56.133219,"high_a":56.797571,"low_a":56.124117,"close_a":56.779369,"volume_a":11563000,"open_b":167.081334,"high_b":169.300179,"low_b":166.466473,"close_b":168.685318,"volume_b":5056500,"open_bench":436.46291,"close_bench":434.550476},{"date":"2023-07-21","open_a":56.843084,"high_a":57.043301,"low_a":56.642871,"close_a":56.824883,"volume_a":12815100,"open_b":169.407102,"high_b":170.619,"low_b":168.685313,"close_b":169.45166,"volume_b":15022100,"open_bench":436.261077,"close_bench":434.550476},{"date":"2023-07-24","open_a":56.806671,"high_a":57.161602,"low_a":56.788473,"close_a":56.843075,"volume_a":10251100,"open_b":169.834841,"high_b":170.583363,"low_b":169.433849,"close_b":170.128906,"volume_b":4225800,"open_bench":435.694134,"close_bench":436.491791},{"date":"2023-07-25","open_a":56.697469,"high_a":56.779376,"low_a":56.469951,"close_a":56.651966,"volume_a":13422700,"open_b":169.674453,"high_b":171.100206,"low_b":169.362563,"close_b":170.520996,"volume_b":3521600,"open_bench":436.222683,"close_bench":437.683411},{"date":"2023-07-26","open_a":56.29704,"high_a":57.489232,"low_a":55.869305,"close_a":57.380024,"volume_a":17145500,"open_b":170.307125,"high_b":171.429921,"low_b":169.362561,"close_b":170.734863,"volume_b":3898900,"open_bench":436.75121,"close_bench":437.750671},{"date":"2023-07-27","open_a":57.380027,"high_a":57.580244,"low_a":56.77028,"close_a":56.824883,"volume_a":11517100,"open_b":171.091265,"high_b":171.349678,"low_b":167.883298,"close_b":167.999146,"volume_b":4141800,"open_bench":441.123766,"close_bench":434.848358},{"date":"2023-07-28","open_a":56.961391,"high_a":57.125204,"low_a":56.624665,"close_a":56.861282,"volume_a":9721800,"open_b":169.433826,"high_b":170.529876,"low_b":169.121936,"close_b":169.585312,"volume_b":5602300,"open_bench":438.106282,"close_bench":439.105743},{"date":"2023-07-31","open_a":56.833982,"high_a":56.833982,"low_a":56.178729,"close_a":56.360744,"volume_a":15577900,"open_b":168.818981,"high_b":168.943735,"low_b":166.502128,"close_b":167.0457,"volume_b":5687700,"open_bench":439.576644,"close_bench":439.941833},{"date":"2023-08-01","open_a":56.506353,"high_a":56.752072,"low_a":56.078618,"close_a":56.21513,"volume_a":9281600,"open_b":167.072454,"high_b":168.061576,"low_b":166.101144,"close_b":166.475418,"volume_b":2853900,"open_bench":438.481119,"close_bench":438.682953},{"date":"2023-08-02","open_a":56.024008,"high_a":56.961381,"low_a":56.024008,"close_a":56.388035,"volume_a":11059400,"open_b":166.270437,"high_b":169.282361,"low_b":166.101126,"close_b":167.803131,"volume_b":4285900,"open_bench":435.578767,"close_bench":432.580414},{"date":"2023-08-03","open_a":56.306137,"high_a":56.533655,"low_a":55.996712,"close_a":56.096821,"volume_a":8037400,"open_b":167.67838,"high_b":168.614038,"low_b":166.609062,"close_b":166.742722,"volume_b":2857400,"open_bench":430.5719,"close_bench":431.340698},{"date":"2023-08-04","open_a":56.133213,"high_a":56.324327,"low_a":55.168536,"close_a":55.250443,"volume_a":9359300,"open_b":166.742725,"high_b":167.535804,"low_b":164.274387,"close_b":164.497162,"volume_b":4651100,"open_bench":433.147423,"close_bench":429.389862},{"date":"2023-08-07","open_a":55.32326,"high_a":55.814699,"low_a":55.277757,"close_a":55.687286,"volume_a":6982600,"open_b":165.103108,"high_b":166.822939,"low_b":164.942717,"close_b":166.537781,"volume_b":3323400,"open_bench":431.215823,"close_bench":433.137848},{"date":"2023-08-08","open_a":55.687286,"high_a":55.832898,"low_a":55.168546,"close_a":55.441566,"volume_a":7956100,"open_b":166.814001,"high_b":167.063508,"low_b":163.989214,"close_b":164.354568,"volume_b":5017900,"open_bench":430.61032,"close_bench":431.254211},{"date":"2023-08-09","open_a":55.432465,"high_a":56.087718,"low_a":55.286854,"close_a":55.568977,"volume_a":7904100,"open_b":163.855524,"high_b":164.853552,"low_b":162.741649,"close_b":163.944626,"volume_b":4413100,"open_bench":431.523273,"close_bench":428.371155},{"date":"2023-08-10","open_a":55.832898,"high_a":56.178726,"low_a":55.32326,"close_a":55.441566,"volume_a":7870300,"open_b":164.158519,"high_b":165.379322,"low_b":163.543657,"close_b":163.7397,"volume_b":3727200,"open_bench":430.716039,"close_bench":428.524933},{"date":"2023-08-11","open_a":55.368765,"high_a":55.696391,"low_a":55.232252,"close_a":55.669086,"volume_a":8483900,"open_b":163.962485,"high_b":164.211992,"low_b":163.320892,"close_b":163.998123,"volume_b":2873800,"open_bench":426.660562,"close_bench":428.275055},{"date":"2023-08-14","open_a":55.805593,"high_a":55.951204,"low_a":55.286853,"close_a":55.405163,"volume_a":7176300,"open_b":164.310037,"high_b":164.728856,"low_b":163.187256,"close_b":163.623886,"volume_b":3385600,"open_bench":427.362195,"close_bench":430.639221},{"date":"2023-08-15","open_a":55.232253,"high_a":55.423371,"low_a":54.94103,"close_a":55.03204,"volume_a":7187400,"open_b":163.20506,"high_b":163.481312,"low_b":161.405046,"close_b":161.601089,"volume_b":3331800,"open_bench":428.870908,"close_bench":425.622711},{"date":"2023-08-16","open_a":55.095743,"high_a":55.350562,"low_a":54.968334,"close_a":55.041138,"volume_a":8372300,"open_b":161.823862,"high_b":163.02684,"low_b":161.244639,"close_b":161.476334,"volume_b":2633700,"open_bench":425.209452,"close_bench":422.49942},{"date":"2023-08-17","open_a":55.132142,"high_a":55.532572,"low_a":55.013832,"close_a":55.159443,"volume_a":12873500,"open_b":161.476338,"high_b":161.654555,"low_b":158.901058,"close_b":159.016891,"volume_b":4503000,"open_bench":423.960185,"close_bench":419.28006},{"date":"2023-08-18","open_a":55.095743,"high_a":55.596282,"low_a":54.941029,"close_a":55.468872,"volume_a":11083800,"open_b":159.025781,"high_b":159.997077,"low_b":157.965369,"close_b":158.77626,"volume_b":6020300,"open_bench":416.473839,"close_bench":419.481812},{"date":"2023-08-21","open_a":55.386963,"high_a":55.450668,"low_a":54.895524,"close_a":54.995632,"volume_a":8648500,"open_b":158.375305,"high_b":158.580268,"low_b":157.038654,"close_b":157.769363,"volume_b":4284100,"open_bench":420.4909,"close_bench":422.211121},{"date":"2023-08-22","open_a":54.913724,"high_a":55.132143,"low_a":54.586098,"close_a":54.658905,"volume_a":9411500,"open_b":156.994085,"high_b":157.564402,"low_b":156.218832,"close_b":156.56636,"volume_b":3489300,"open_bench":423.97934,"close_bench":421.067474},{"date":"2023-08-23","open_a":54.786317,"high_a":55.004732,"low_a":54.749913,"close_a":54.850021,"volume_a":7749500,"open_b":157.947556,"high_b":158.865388,"low_b":157.163383,"close_b":158.722809,"volume_b":4042700,"open_bench":422.124638,"close_bench":425.757263},{"date":"2023-08-24","open_a":54.850022,"high_a":55.405166,"low_a":54.686209,"close_a":54.704411,"volume_a":8955100,"open_b":158.624793,"high_b":160.790174,"low_b":157.813902,"close_b":157.885178,"volume_b":5709800,"open_bench":427.352512,"close_bench":419.856628},{"date":"2023-08-25","open_a":54.97743,"high_a":55.123042,"low_a":54.6134,"close_a":54.959229,"volume_a":7629200,"open_b":158.669359,"high_b":160.139683,"low_b":158.473316,"close_b":159.881256,"volume_b":4710700,"open_bench":421.576784,"close_bench":422.816498},{"date":"2023-08-28","open_a":55.104839,"high_a":55.268652,"low_a":54.941025,"close_a":55.113941,"volume_a":6254300,"open_b":161.128829,"high_b":161.449612,"low_b":159.809991,"close_b":160.620895,"volume_b":3189800,"open_bench":424.998058,"close_bench":425.497803},{"date":"2023-08-29","open_a":55.113943,"high_a":55.177648,"low_a":54.740814,"close_a":55.059338,"volume_a":8189600,"open_b":160.665423,"high_b":161.485233,"low_b":159.524815,"close_b":161.226807,"volume_b":4368600,"open_bench":425.392055,"close_bench":431.648254},{"date":"2023-08-30","open_a":55.18675,"high_a":55.277759,"low_a":54.868226,"close_a":55.03204,"volume_a":9447900,"open_b":161.699096,"high_b":162.411966,"low_b":161.039677,"close_b":161.360474,"volume_b":3153400,"open_bench":431.984599,"close_bench":433.426117},{"date":"2023-08-31","open_a":54.877324,"high_a":55.032038,"low_a":54.431391,"close_a":54.449593,"volume_a":14663500,"open_b":160.934221,"high_b":161.194466,"low_b":159.606118,"close_b":159.659958,"volume_b":5566700,"open_bench":434.041144,"close_bench":432.79184},{"date":"2023-09-01","open_a":54.604305,"high_a":54.722615,"low_a":53.803441,"close_a":53.976357,"volume_a":11262300,"open_b":160.386832,"high_b":160.458623,"low_b":156.716594,"close_b":157.326813,"volume_b":4801400,"open_bench":435.501875,"close_bench":433.59906},{"date":"2023-09-05","open_a":53.92175,"high_a":53.985454,"low_a":53.393906,"close_a":53.530418,"volume_a":12474600,"open_b":157.281923,"high_b":157.829319,"low_b":155.828176,"close_b":156.285843,"volume_b":5081800,"open_bench":433.157056,"close_bench":431.725128},{"date":"2023-09-06","open_a":53.3484,"high_a":53.539517,"low_a":53.166385,"close_a":53.494011,"volume_a":13923500,"open_b":156.96785,"high_b":157.030673,"low_b":155.675636,"close_b":156.797348,"volume_b":5933800,"open_bench":430.917937,"close_bench":428.822937},{"date":"2023-09-07","open_a":53.648724,"high_a":53.839838,"low_a":53.057176,"close_a":53.08448,"volume_a":17628600,"open_b":157.802417,"high_b":158.888227,"low_b":157.228101,"close_b":158.295959,"volume_b":4934300,"open_bench":425.834105,"close_bench":427.506287},{"date":"2023-09-08","open_a":53.057176,"high_a":53.157284,"low_a":52.793254,"close_a":53.08448,"volume_a":14312300,"open_b":157.766495,"high_b":158.968966,"low_b":157.587024,"close_b":158.179291,"volume_b":2995000,"open_bench":427.554354,"close_bench":428.150177},{"date":"2023-09-11","open_a":53.148188,"high_a":53.68513,"low_a":52.993474,"close_a":53.585022,"volume_a":14767400,"open_b":158.681802,"high_b":161.014954,"low_b":158.681802,"close_b":160.566269,"volume_b":4512700,"open_bench":430.764139,"close_bench":430.965973},{"date":"2023-09-12","open_a":53.685126,"high_a":53.694226,"low_a":52.957066,"close_a":53.057175,"volume_a":12292700,"open_b":160.39581,"high_b":160.557329,"low_b":159.55228,"close_b":159.974045,"volume_b":3331100,"open_bench":429.52443,"close_bench":428.601837},{"date":"2023-09-13","open_a":53.148193,"high_a":53.403012,"low_a":53.093587,"close_a":53.184593,"volume_a":12641400,"open_b":160.027878,"high_b":161.652115,"low_b":159.920198,"close_b":161.239319,"volume_b":3781400,"open_bench":428.822861,"close_bench":429.101562},{"date":"2023-09-14","open_a":53.423095,"high_a":53.744146,"low_a":53.377228,"close_a":53.624897,"volume_a":12311600,"open_b":161.472679,"high_b":163.043076,"low_b":160.925283,"close_b":162.63028,"volume_b":4118300,"open_bench":431.561738,"close_bench":432.801422},{"date":"2023-09-15","open_a":53.579024,"high_a":53.790001,"low_a":52.973612,"close_a":53.147896,"volume_a":55727500,"open_b":162.791765,"high_b":163.15969,"low_b":161.176498,"close_b":161.382889,"volume_b":11661200,"open_bench":431.222728,"close_bench":427.586914},{"date":"2023-09-18","open_a":53.184586,"high_a":53.523983,"low_a":52.863535,"close_a":53.478119,"volume_a":13617400,"open_b":162.423857,"high_b":162.423857,"low_b":160.117611,"close_b":160.970123,"volume_b":3182600,"open_bench":427.278307,"close_bench":427.837677},{"date":"2023-09-19","open_a":53.441431,"high_a":53.56068,"low_a":53.129552,"close_a":53.36805,"volume_a":11998900,"open_b":160.503494,"high_b":160.745788,"low_b":159.1036,"close_b":160.018921,"volume_b":3369300,"open_bench":426.921448,"close_bench":426.950378},{"date":"2023-09-20","open_a":53.478124,"high_a":53.964291,"low_a":53.276322,"close_a":53.606544,"volume_a":14766600,"open_b":160.709833,"high_b":161.005966,"low_b":159.336859,"close_b":160.072708,"volume_b":3695800,"open_bench":428.204133,"close_bench":423.025299},{"date":"2023-09-21","open_a":53.542337,"high_a":53.588201,"low_a":52.670909,"close_a":52.780987,"volume_a":15941800,"open_b":159.821492,"high_b":160.027883,"low_b":157.156303,"close_b":157.380646,"volume_b":5351000,"open_bench":420.189956,"close_bench":416.033386},{"date":"2023-09-22","open_a":53.065337,"high_a":53.175414,"low_a":52.744286,"close_a":52.836014,"volume_a":14644100,"open_b":156.985803,"high_b":158.286986,"low_b":156.474296,"close_b":157.281937,"volume_b":4542300,"open_bench":417.055696,"close_bench":415.097961},{"date":"2023-09-25","open_a":52.652563,"high_a":52.735119,"low_a":52.04715,"close_a":52.285645,"volume_a":12358800,"open_b":156.752456,"high_b":157.434465,"low_b":155.406402,"close_b":156.438385,"volume_b":3553600,"open_bench":413.892408,"close_bench":416.843475},{"date":"2023-09-26","open_a":52.037978,"high_a":52.193919,"low_a":51.735273,"close_a":51.854519,"volume_a":11194700,"open_b":155.244912,"high_b":155.891019,"low_b":154.024489,"close_b":154.814178,"volume_b":5213700,"open_bench":413.815266,"close_bench":410.719543},{"date":"2023-09-27","open_a":51.753616,"high_a":51.854518,"low_a":51.074821,"close_a":51.322491,"volume_a":13766300,"open_b":154.62573,"high_b":154.966735,"low_b":151.682359,"close_b":152.166946,"volume_b":5729900,"open_bench":411.88647,"close_bench":410.883484},{"date":"2023-09-28","open_a":51.579326,"high_a":51.661882,"low_a":51.093162,"close_a":51.194065,"volume_a":12484800,"open_b":152.489958,"high_b":152.7143,"low_b":150.919561,"close_b":152.104095,"volume_b":4841600,"open_bench":410.333833,"close_bench":413.265594},{"date":"2023-09-29","open_a":51.496777,"high_a":51.616025,"low_a":51.019784,"close_a":51.35001,"volume_a":12271600,"open_b":153.010441,"high_b":153.306574,"low_b":150.793937,"close_b":152.050262,"volume_b":5194200,"open_bench":416.30342,"close_bench":412.262573},{"date":"2023-10-02","open_a":51.285793,"high_a":51.377521,"low_a":50.524443,"close_a":50.891357,"volume_a":11792500,"open_b":151.727208,"high_b":152.095133,"low_b":150.147843,"close_b":151.807968,"volume_b":3838800,"open_bench":411.433193,"close_bench":412.098633},{"date":"2023-10-03","open_a":50.597829,"high_a":50.845498,"low_a":50.249257,"close_a":50.340988,"volume_a":13604200,"open_b":151.529798,"high_b":152.552798,"low_b":149.88761,"close_b":150.488846,"volume_b":4873400,"open_bench":409.928752,"close_bench":406.582275},{"date":"2023-10-04","open_a":50.350156,"high_a":50.707899,"low_a":50.185043,"close_a":50.487751,"volume_a":12100500,"open_b":151.161835,"high_b":151.772053,"low_b":149.914492,"close_b":151.574631,"volume_b":4134000,"open_bench":407.045171,"close_bench":409.542969},{"date":"2023-10-05","open_a":50.212566,"high_a":50.405196,"low_a":47.919336,"close_a":48.047756,"volume_a":29878100,"open_b":151.449061,"high_b":151.48495,"low_b":143.462464,"close_b":143.668869,"volume_b":10556400,"open_bench":409.253611,"close_bench":409.388641},{"date":"2023-10-06","open_a":47.735872,"high_a":48.882487,"low_a":47.286398,"close_a":48.744892,"volume_a":30298900,"open_b":142.484287,"high_b":144.404657,"low_b":139.837051,"close_b":143.83931,"volume_b":12774700,"open_bench":406.948737,"close_bench":414.249268},{"date":"2023-10-09","open_a":48.579783,"high_a":48.75407,"low_a":47.800086,"close_a":48.506401,"volume_a":15704700,"open_b":143.579092,"high_b":144.853355,"low_b":142.358669,"close_b":144.799515,"volume_b":7244500,"open_bench":412.359012,"close_bench":416.901367},{"date":"2023-10-10","open_a":49.653022,"high_a":49.836481,"low_a":49.130166,"close_a":49.561291,"volume_a":20909700,"open_b":147.392912,"high_b":148.227459,"low_b":145.400751,"close_b":147.527512,"volume_b":11456400,"open_bench":417.528209,"close_bench":419.071259},{"date":"2023-10-11","open_a":49.735571,"high_a":49.790606,"low_a":48.63482,"close_a":49.26775,"volume_a":21138900,"open_b":148.074903,"high_b":148.460779,"low_b":145.014884,"close_b":145.930191,"volume_b":7371500,"open_bench":420.132147,"close_bench":420.787933},{"date":"2023-10-12","open_a":49.313613,"high_a":49.377826,"low_a":48.093616,"close_a":48.442188,"volume_a":19330400,"open_b":145.912269,"high_b":145.957126,"low_b":141.272857,"close_b":141.856155,"volume_b":9024100,"open_bench":421.395517,"close_bench":418.222626},{"date":"2023-10-13","open_a":48.377976,"high_a":49.001731,"low_a":48.286245,"close_a":48.515568,"volume_a":14494600,"open_b":142.340697,"high_b":143.794445,"low_b":141.739461,"close_b":143.579071,"volume_b":6757100,"open_bench":419.717419,"close_bench":416.139496},{"date":"2023-10-16","open_a":48.634824,"high_a":49.130163,"low_a":48.469711,"close_a":49.010914,"volume_a":12816800,"open_b":144.476457,"high_b":145.328955,"low_b":143.399617,"close_b":144.548248,"volume_b":5605000,"open_bench":418.376879,"close_bench":420.517853},{"date":"2023-10-17","open_a":48.900844,"high_a":49.625502,"low_a":48.809113,"close_a":49.597984,"volume_a":16457200,"open_b":143.579111,"high_b":145.041827,"low_b":142.825323,"close_b":143.911133,"volume_b":6411300,"open_bench":417.402874,"close_bench":420.498596},{"date":"2023-10-18","open_a":49.625496,"high_a":50.019932,"low_a":49.460383,"close_a":49.579632,"volume_a":12809500,"open_b":144.494419,"high_b":146.630148,"low_b":144.359805,"close_b":145.400757,"volume_b":6650600,"open_bench":418.733747,"close_bench":414.895416},{"date":"2023-10-19","open_a":49.680532,"high_a":50.230906,"low_a":49.405345,"close_a":49.854816,"volume_a":16202200,"open_b":145.284076,"high_b":146.019925,"low_b":143.947004,"close_b":144.081604,"volume_b":6336800,"open_bench":415.609055,"close_bench":411.249939},{"date":"2023-10-20","open_a":50.019925,"high_a":50.4694,"low_a":49.937369,"close_a":50.056618,"volume_a":15686400,"open_b":144.808462,"high_b":145.580201,"low_b":143.31884,"close_b":143.579071,"volume_b":5486500,"open_bench":410.815997,"close_bench":406.196503},{"date":"2023-10-23","open_a":49.882342,"high_a":50.350159,"low_a":49.579634,"close_a":49.607155,"volume_a":21212700,"open_b":143.220161,"high_b":145.014899,"low_b":142.735574,"close_b":143.650894,"volume_b":5795800,"open_bench":404.672745,"close_bench":405.492493},{"date":"2023-10-24","open_a":51.047307,"high_a":51.524295,"low_a":50.726252,"close_a":51.038132,"volume_a":25290800,"open_b":143.946995,"high_b":146.136578,"low_b":143.291906,"close_b":145.544312,"volume_b":5265200,"open_bench":407.604558,"close_bench":408.549683},{"date":"2023-10-25","open_a":51.148202,"high_a":51.570159,"low_a":50.964747,"close_a":51.478428,"volume_a":16701000,"open_b":145.418698,"high_b":146.719881,"low_b":144.898223,"close_b":145.687912,"volume_b":4534600,"open_bench":406.871566,"close_bench":402.686035},{"date":"2023-10-26","open_a":51.707752,"high_a":51.94625,"low_a":51.111514,"close_a":51.16655,"volume_a":16228400,"open_b":145.98404,"high_b":146.594258,"low_b":144.74568,"close_b":144.844391,"volume_b":5147000,"open_bench":401.625205,"close_bench":397.864014},{"date":"2023-10-27","open_a":50.955573,"high_a":51.496776,"low_a":50.542791,"close_a":50.671215,"volume_a":13626500,"open_b":145.391762,"high_b":145.499442,"low_b":142.582992,"close_b":143.238068,"volume_b":5078000,"open_bench":399.445699,"close_bench":396.060638},{"date":"2023-10-30","open_a":50.845496,"high_a":51.625193,"low_a":50.836321,"close_a":51.505947,"volume_a":15014300,"open_b":144.745704,"high_b":145.894336,"low_b":144.243179,"close_b":145.625122,"volume_b":4625900,"open_bench":398.838133,"close_bench":400.795868},{"date":"2023-10-31","open_a":51.891207,"high_a":52.028802,"low_a":51.505946,"close_a":51.817825,"volume_a":16639400,"open_b":146.226327,"high_b":146.585283,"low_b":145.068727,"close_b":146.522461,"volume_b":5947400,"open_bench":401.364847,"close_bench":403.312958},{"date":"2023-11-01","open_a":52.102191,"high_a":52.175576,"low_a":51.606852,"close_a":51.771965,"volume_a":14110800,"open_b":146.87247,"high_b":148.451835,"low_b":146.05586,"close_b":147.94931,"volume_b":6682500,"open_bench":404.277374,"close_bench":407.614197},{"date":"2023-11-02","open_a":51.826996,"high_a":52.423237,"low_a":51.698576,"close_a":52.368198,"volume_a":13087200,"open_b":147.832661,"high_b":150.004294,"low_b":147.769839,"close_b":149.70816,"volume_b":5413400,"open_bench":411.394589,"close_bench":415.425812},{"date":"2023-11-03","open_a":52.652562,"high_a":52.826846,"low_a":52.010457,"close_a":52.04715,"volume_a":13958700,"open_b":151.368253,"high_b":151.484916,"low_b":149.582497,"close_b":149.672226,"volume_b":4063300,"open_bench":417.721161,"close_bench":419.215973},{"date":"2023-11-06","open_a":52.083841,"high_a":52.542487,"low_a":52.047152,"close_a":52.258129,"volume_a":12673200,"open_b":148.478717,"high_b":150.309344,"low_b":148.254375,"close_b":149.591446,"volume_b":5784600,"open_bench":419.968159,"close_bench":420.180328},{"date":"2023-11-07","open_a":52.212262,"high_a":52.652565,"low_a":52.037978,"close_a":52.45076,"volume_a":10449300,"open_b":149.681241,"high_b":150.497851,"low_b":149.071023,"close_b":150.022232,"volume_b":4687200,"open_bench":420.180341,"close_bench":421.37619},{"date":"2023-11-08","open_a":52.579175,"high_a":52.71677,"low_a":52.212257,"close_a":52.368198,"volume_a":13744000,"open_b":150.910579,"high_b":151.06313,"low_b":149.555557,"close_b":150.210632,"volume_b":4926600,"open_bench":421.974093,"close_bench":421.684784},{"date":"2023-11-09","open_a":52.31316,"high_a":52.414062,"low_a":51.707747,"close_a":51.973763,"volume_a":12021100,"open_b":150.381166,"high_b":150.632428,"low_b":148.792818,"close_b":149.106903,"volume_b":5639300,"open_bench":422.82275,"close_bench":418.396149},{"date":"2023-11-10","open_a":52.1664,"high_a":52.221439,"low_a":51.423396,"close_a":52.028809,"volume_a":11678900,"open_b":150.138883,"high_b":150.560648,"low_b":147.940331,"close_b":149.78891,"volume_b":5552900,"open_bench":420.459954,"close_bench":424.92511},{"date":"2023-11-13","open_a":51.937078,"high_a":52.377377,"low_a":51.918731,"close_a":52.221439,"volume_a":12798200,"open_b":149.860673,"high_b":151.457989,"low_b":149.376099,"close_b":150.551651,"volume_b":5402300,"open_bench":423.594263,"close_bench":424.520081},{"date":"2023-11-14","open_a":52.303987,"high_a":52.505793,"low_a":52.111357,"close_a":52.377369,"volume_a":12697100,"open_b":150.713183,"high_b":151.152899,"low_b":149.761974,"close_b":150.856766,"volume_b":6238900,"open_bench":430.431922,"close_bench":432.756134},{"date":"2023-11-15","open_a":52.560832,"high_a":52.597524,"low_a":52.056322,"close_a":52.478275,"volume_a":10201900,"open_b":151.386189,"high_b":151.466949,"low_b":149.205589,"close_b":150.085007,"volume_b":5611200,"open_bench":434.086988,"close_bench":433.672302},{"date":"2023-11-16","open_a":52.514972,"high_a":52.771813,"low_a":52.111365,"close_a":52.423244,"volume_a":16010400,"open_b":151.108023,"high_b":152.552788,"low_b":149.322253,"close_b":150.497818,"volume_b":5386400,"open_bench":433.228673,"close_bench":434.202728},{"date":"2023-11-17","open_a":52.459933,"high_a":52.597528,"low_a":52.010462,"close_a":52.524143,"volume_a":14475800,"open_b":150.802924,"high_b":150.982395,"low_b":148.756923,"close_b":149.645309,"volume_b":5583100,"open_bench":434.212359,"close_bench":434.742798},{"date":"2023-11-20","open_a":52.331507,"high_a":52.817674,"low_a":52.24895,"close_a":52.652561,"volume_a":13051000,"open_b":148.963311,"high_b":149.654289,"low_b":148.712049,"close_b":149.429947,"volume_b":4886200,"open_bench":434.492034,"close_bench":438.089264},{"date":"2023-11-21","open_a":52.707594,"high_a":53.239625,"low_a":52.588348,"close_a":53.23045,"volume_a":13891600,"open_b":149.959381,"high_b":150.632407,"low_b":148.891509,"close_b":150.578568,"volume_b":3980000,"open_bench":437.047668,"close_bench":437.13446},{"date":"2023-11-22","open_a":53.441429,"high_a":53.698273,"low_a":53.322184,"close_a":53.588196,"volume_a":11320600,"open_b":151.072122,"high_b":152.166914,"low_b":151.027251,"close_b":151.852829,"volume_b":3235600,"open_bench":438.78365,"close_bench":438.822205},{"date":"2023-11-24","open_a":53.624889,"high_a":53.890905,"low_a":53.514815,"close_a":53.725792,"volume_a":4816000,"open_b":151.610579,"high_b":152.570771,"low_b":151.610579,"close_b":151.987473,"volume_b":1751700,"open_bench":438.8704,"close_bench":439.092194},{"date":"2023-11-27","open_a":53.698282,"high_a":53.835874,"low_a":53.450613,"close_a":53.624897,"volume_a":16246500,"open_b":152.283588,"high_b":152.283588,"low_b":150.838822,"close_b":151.045227,"volume_b":3664700,"open_bench":438.465385,"close_bench":438.301453},{"date":"2023-11-28","open_a":53.569853,"high_a":53.964289,"low_a":53.533161,"close_a":53.734966,"volume_a":13739600,"open_b":151.099067,"high_b":152.337427,"low_b":150.776013,"close_b":151.5298,"volume_b":4056100,"open_bench":437.915665,"close_bench":438.735413},{"date":"2023-11-29","open_a":53.734964,"high_a":53.817517,"low_a":53.294661,"close_a":53.41391,"volume_a":11263600,"open_b":151.745156,"high_b":151.763093,"low_b":149.54659,"close_b":150.004257,"volume_b":4722400,"open_bench":440.876347,"close_bench":438.426758},{"date":"2023-11-30","open_a":53.589588,"high_a":54.051886,"low_a":53.256733,"close_a":54.033394,"volume_a":22727500,"open_b":150.19867,"high_b":152.233147,"low_b":149.032242,"close_b":152.169846,"volume_b":7346700,"open_bench":439.265904,"close_bench":440.153137},{"date":"2023-12-01","open_a":53.876212,"high_a":54.26454,"low_a":53.848475,"close_a":54.218311,"volume_a":15371200,"open_b":152.24217,"high_b":152.676202,"low_b":151.347009,"close_b":152.531525,"volume_b":4897400,"open_bench":439.545483,"close_bench":442.756958},{"date":"2023-12-04","open_a":54.17208,"high_a":54.514179,"low_a":54.033389,"close_a":54.153587,"volume_a":14942200,"open_b":152.269324,"high_b":153.64373,"low_b":152.079447,"close_b":152.938446,"volume_b":5323300,"open_bench":439.381606,"close_bench":440.4328},{"date":"2023-12-05","open_a":54.135099,"high_a":54.393988,"low_a":54.0149,"close_a":54.236805,"volume_a":11891500,"open_b":152.947436,"high_b":153.327218,"low_b":151.645383,"close_b":151.853348,"volume_b":4553800,"open_bench":439.053683,"close_bench":440.345978},{"date":"2023-12-06","open_a":54.144347,"high_a":54.301526,"low_a":53.820736,"close_a":54.181328,"volume_a":14868500,"open_b":151.889535,"high_b":152.196977,"low_b":150.75927,"close_b":151.464554,"volume_b":4726600,"open_bench":442.47735,"close_bench":438.571533},{"date":"2023-12-07","open_a":54.246043,"high_a":54.449455,"low_a":53.987157,"close_a":54.310768,"volume_a":11587900,"open_b":151.907626,"high_b":152.178892,"low_b":150.732147,"close_b":151.419357,"volume_b":4934800,"open_bench":440.644921,"close_bench":441.917938},{"date":"2023-12-08","open_a":54.310773,"high_a":54.320018,"low_a":53.96867,"close_a":54.190575,"volume_a":11178700,"open_b":151.039589,"high_b":151.111928,"low_b":149.439144,"close_b":149.80986,"volume_b":5115200,"open_bench":441.175389,"close_bench":443.817871},{"date":"2023-12-11","open_a":54.39399,"high_a":54.634383,"low_a":54.088871,"close_a":54.588154,"volume_a":14524800,"open_b":151.401253,"high_b":152.025161,"low_b":150.515129,"close_b":151.744858,"volume_b":6912600,"open_bench":443.325955,"close_bench":445.544067},{"date":"2023-12-12","open_a":54.736086,"high_a":54.939495,"low_a":54.458708,"close_a":54.939495,"volume_a":12899000,"open_b":153.065021,"high_b":153.065021,"low_b":151.645415,"close_b":152.332611,"volume_b":4515800,"open_bench":445.196905,"close_bench":447.578979},{"date":"2023-12-13","open_a":54.967236,"high_a":55.494255,"low_a":54.79156,"close_a":55.411041,"volume_a":16196800,"open_b":152.395906,"high_b":155.5697,"low_b":152.016139,"close_b":155.533524,"volume_b":6082100,"open_bench":447.955064,"close_bench":453.751129},{"date":"2023-12-14","open_a":55.448028,"high_a":55.51275,"low_a":54.338512,"close_a":54.588154,"volume_a":22478300,"open_b":155.207967,"high_b":155.497321,"low_b":151.853348,"close_b":152.133652,"volume_b":7119300,"open_bench":455.679914,"close_bench":455.207367},{"date":"2023-12-15","open_a":54.393988,"high_a":54.708347,"low_a":53.441652,"close_a":54.181328,"volume_a":65177400,"open_b":151.410311,"high_b":152.368773,"low_b":150.506099,"close_b":151.003418,"volume_b":17734600,"open_bench":454.61285,"close_bench":454.457916},{"date":"2023-12-18","open_a":54.366244,"high_a":55.004218,"low_a":54.199817,"close_a":54.569656,"volume_a":16717700,"open_b":151.961868,"high_b":153.942094,"low_b":151.374135,"close_b":152.730453,"volume_b":5439500,"open_bench":456.055693,"close_bench":457.014313},{"date":"2023-12-19","open_a":54.551169,"high_a":54.736089,"low_a":54.218314,"close_a":54.39399,"volume_a":14612200,"open_b":151.853374,"high_b":152.658123,"low_b":151.283716,"close_b":151.862411,"volume_b":4855900,"open_bench":457.556506,"close_bench":459.793304},{"date":"2023-12-20","open_a":54.08887,"high_a":54.246049,"low_a":53.228995,"close_a":53.26598,"volume_a":17701000,"open_b":150.560355,"high_b":151.428405,"low_b":149.800821,"close_b":149.818909,"volume_b":5157800,"open_bench":458.941206,"close_bench":453.421844},{"date":"2023-12-21","open_a":53.487878,"high_a":53.691291,"low_a":53.136534,"close_a":53.617325,"volume_a":11725100,"open_b":150.551307,"high_b":151.09384,"low_b":149.547631,"close_b":150.976288,"volume_b":4374000,"open_bench":456.394548,"close_bench":457.721161},{"date":"2023-12-22","open_a":53.737516,"high_a":54.051879,"low_a":53.645058,"close_a":53.922436,"volume_a":9029400,"open_b":151.328915,"high_b":152.061324,"low_b":150.876809,"close_b":151.618256,"volume_b":3030400,"open_bench":458.84439,"close_bench":458.641052},{"date":"2023-12-26","open_a":53.68205,"high_a":54.283035,"low_a":53.68205,"close_a":54.144348,"volume_a":6422500,"open_b":151.591154,"high_b":153.191614,"low_b":151.229475,"close_b":152.685257,"volume_b":2779200,"open_bench":459.047717,"close_bench":460.577637},{"date":"2023-12-27","open_a":54.218314,"high_a":54.338512,"low_a":53.996413,"close_a":54.283035,"volume_a":8560100,"open_b":152.721402,"high_b":153.254897,"low_b":152.233133,"close_b":153.173508,"volume_b":3502500,"open_bench":460.374269,"close_bench":461.41037},{"date":"2023-12-28","open_a":54.227561,"high_a":54.430969,"low_a":54.116607,"close_a":54.320019,"volume_a":8400100,"open_b":153.173512,"high_b":153.300101,"low_b":152.540566,"close_b":153.164474,"volume_b":2911100,"open_bench":461.768698,"close_bench":461.584717},{"date":"2023-12-29","open_a":54.310778,"high_a":54.532679,"low_a":54.209072,"close_a":54.48645,"volume_a":9244300,"open_b":153.119259,"high_b":153.779344,"low_b":152.965545,"close_b":153.571365,"volume_b":3291500,"open_bench":461.390984,"close_bench":460.248383},{"date":"2024-01-02","open_a":54.366242,"high_a":55.364807,"low_a":54.246043,"close_a":55.30933,"volume_a":16322600,"open_b":153.282027,"high_b":156.419644,"low_b":152.983635,"close_b":156.347305,"volume_b":5743100,"open_bench":457.198205,"close_bench":457.672668},{"date":"2024-01-03","open_a":55.411037,"high_a":55.660679,"low_a":55.253854,"close_a":55.438774,"volume_a":14830600,"open_b":158.237121,"high_b":158.417961,"low_b":156.157431,"close_b":156.383484,"volume_b":5588200,"open_bench":455.523045,"close_bench":453.935028},{"date":"2024-01-04","open_a":55.52199,"high_a":55.716155,"low_a":55.161398,"close_a":55.253857,"volume_a":12912900,"open_b":155.171831,"high_b":156.907917,"low_b":154.240494,"close_b":155.045242,"volume_b":6283200,"open_bench":453.460568,"close_bench":452.4729},{"date":"2024-01-05","open_a":55.290841,"high_a":55.429532,"low_a":54.634379,"close_a":55.170643,"volume_a":10412800,"open_b":155.135681,"high_b":155.135681,"low_b":151.509782,"close_b":152.757599,"volume_b":5252500,"open_bench":452.676255,"close_bench":453.092651},{"date":"2024-01-08","open_a":55.179889,"high_a":55.642186,"low_a":54.939492,"close_a":55.577465,"volume_a":11554600,"open_b":152.757602,"high_b":153.471923,"low_b":151.717749,"close_b":152.911316,"volume_b":5735600,"open_bench":453.586385,"close_bench":459.560883},{"date":"2024-01-09","open_a":55.466525,"high_a":55.605212,"low_a":55.124422,"close_a":55.475769,"volume_a":10493700,"open_b":151.717727,"high_b":151.808154,"low_b":150.035892,"close_b":151.15712,"volume_b":6268800,"open_bench":456.917422,"close_bench":458.863739},{"date":"2024-01-10","open_a":55.679179,"high_a":55.854851,"low_a":55.457274,"close_a":55.660686,"volume_a":12258000,"open_b":150.6327,"high_b":151.87148,"low_b":150.280057,"close_b":150.931091,"volume_b":4394900,"open_bench":459.134889,"close_bench":461.458832},{"date":"2024-01-11","open_a":55.706912,"high_a":55.716156,"low_a":55.189137,"close_a":55.300091,"volume_a":13754200,"open_b":151.13,"high_b":151.13,"low_b":149.032236,"close_b":150.216751,"volume_b":4800700,"open_bench":462.456191,"close_bench":461.255493},{"date":"2024-01-12","open_a":55.549727,"high_a":55.901071,"low_a":55.355559,"close_a":55.836349,"volume_a":13218000,"open_b":150.406634,"high_b":151.347022,"low_b":149.710401,"close_b":151.247559,"volume_b":3546100,"open_bench":462.698288,"close_bench":461.575043},{"date":"2024-01-16","open_a":55.836359,"high_a":55.873343,"low_a":55.337076,"close_a":55.466522,"volume_a":11826900,"open_b":151.247569,"high_b":151.473622,"low_b":149.366806,"close_b":150.25293,"volume_b":4576600,"open_bench":460.199992,"close_bench":459.880432},{"date":"2024-01-17","open_a":55.346324,"high_a":55.669931,"low_a":55.309339,"close_a":55.466522,"volume_a":8982100,"open_b":150.090132,"high_b":150.75924,"low_b":149.592813,"close_b":150.497025,"volume_b":4392300,"open_bench":456.86905,"close_bench":457.324158},{"date":"2024-01-18","open_a":55.189134,"high_a":55.688417,"low_a":55.041199,"close_a":55.623695,"volume_a":10397700,"open_b":149.619962,"high_b":151.256584,"low_b":149.185944,"close_b":151.15712,"volume_b":4709400,"open_bench":458.989627,"close_bench":461.391022},{"date":"2024-01-19","open_a":55.716158,"high_a":55.73465,"low_a":55.142909,"close_a":55.318584,"volume_a":14392600,"open_b":151.129982,"high_b":151.220409,"low_b":149.710377,"close_b":149.900253,"volume_b":6528600,"open_bench":462.514291,"close_bench":467.142822},{"date":"2024-01-22","open_a":55.253854,"high_a":55.54048,"low_a":54.957984,"close_a":55.078182,"volume_a":14314300,"open_b":150.063004,"high_b":150.741163,"low_b":149.213043,"close_b":149.294418,"volume_b":5697400,"open_bench":468.672807,"close_bench":468.130554},{"date":"2024-01-23","open_a":55.041196,"high_a":55.475757,"low_a":54.976475,"close_a":55.337067,"volume_a":14481600,"open_b":148.797163,"high_b":151.844354,"low_b":148.616323,"close_b":151.582138,"volume_b":4932700,"open_bench":468.672716,"close_bench":469.495758},{"date":"2024-01-24","open_a":55.290839,"high_a":55.300087,"low_a":54.449457,"close_a":54.467949,"volume_a":16370500,"open_b":151.582105,"high_b":151.672532,"low_b":149.66518,"close_b":149.737518,"volume_b":4984700,"open_bench":472.352332,"close_bench":470.009033},{"date":"2024-01-25","open_a":54.560413,"high_a":54.708349,"low_a":54.236806,"close_a":54.699104,"volume_a":14899700,"open_b":149.927401,"high_b":150.641737,"low_b":149.140741,"close_b":150.60556,"volume_b":5373800,"open_bench":472.129616,"close_bench":472.565369},{"date":"2024-01-26","open_a":54.782318,"high_a":55.004223,"low_a":54.671367,"close_a":54.893269,"volume_a":13025100,"open_b":150.641734,"high_b":151.934751,"low_b":150.641734,"close_b":151.781036,"volume_b":4849000,"open_bench":472.139346,"close_bench":471.965057},{"date":"2024-01-29","open_a":54.865524,"high_a":55.281593,"low_a":54.708341,"close_a":55.226116,"volume_a":14547100,"open_b":151.482636,"high_b":152.540563,"low_b":151.356047,"close_b":152.043243,"volume_b":5713600,"open_bench":472.274803,"close_bench":475.702606},{"date":"2024-01-30","open_a":55.337072,"high_a":55.494255,"low_a":54.967236,"close_a":55.383305,"volume_a":22061500,"open_b":151.609211,"high_b":153.44476,"low_b":151.292745,"close_b":153.372421,"volume_b":4804200,"open_bench":475.015188,"close_bench":475.334747},{"date":"2024-01-31","open_a":55.568224,"high_a":55.891834,"low_a":54.819303,"close_a":55.004223,"volume_a":22837000,"open_b":153.679871,"high_b":154.059638,"low_b":151.799109,"close_b":152.386841,"volume_b":5769200,"open_bench":473.136685,"close_bench":467.578583},{"date":"2024-02-01","open_a":55.078179,"high_a":56.409596,"low_a":54.884014,"close_a":56.381859,"volume_a":15870900,"open_b":152.386854,"high_b":155.343631,"low_b":151.157125,"close_b":155.253204,"volume_b":4920800,"open_bench":469.27314,"close_bench":473.698334},{"date":"2024-02-02","open_a":56.23393,"high_a":56.280159,"low_a":55.688418,"close_a":55.975044,"volume_a":17544900,"open_b":154.837265,"high_b":155.488299,"low_b":153.806464,"close_b":154.593124,"volume_b":5307700,"open_bench":474.134041,"close_bench":478.68512},{"date":"2024-02-05","open_a":55.827112,"high_a":55.827112,"low_a":55.364813,"close_a":55.512749,"volume_a":12759600,"open_b":154.213359,"high_b":155.144696,"low_b":152.857041,"close_b":154.547913,"volume_b":5142700,"open_bench":478.055721,"close_bench":476.942139},{"date":"2024-02-06","open_a":55.438765,"high_a":55.521979,"low_a":55.179879,"close_a":55.420273,"volume_a":12775200,"open_b":155.027156,"high_b":155.370761,"low_b":153.661801,"close_b":155.000031,"volume_b":5182700,"open_bench":477.881368,"close_bench":478.326813},{"date":"2024-02-07","open_a":55.540488,"high_a":55.669931,"low_a":55.42029,"close_a":55.466522,"volume_a":10259900,"open_b":156.094132,"high_b":156.338259,"low_b":154.963867,"close_b":155.045242,"volume_b":5769700,"open_bench":480.563642,"close_bench":482.316284},{"date":"2024-02-08","open_a":55.374058,"high_a":55.448027,"low_a":54.930252,"close_a":55.318584,"volume_a":12874800,"open_b":155.27129,"high_b":157.423319,"low_b":154.990986,"close_b":157.197266,"volume_b":9269700,"open_bench":482.316267,"close_bench":482.529297},{"date":"2024-02-09","open_a":54.985722,"high_a":55.087428,"low_a":54.578897,"close_a":55.068935,"volume_a":15235300,"open_b":154.801105,"high_b":154.972908,"low_b":150.976292,"close_b":151.609238,"volume_b":12731700,"open_bench":483.032822,"close_bench":485.318054},{"date":"2024-02-12","open_a":55.161402,"high_a":55.253861,"low_a":54.514184,"close_a":55.198387,"volume_a":13516800,"open_b":151.781056,"high_b":154.683583,"low_b":151.184272,"close_b":154.267639,"volume_b":7867500,"open_bench":485.288962,"close_bench":485.10498},{"date":"2024-02-13","open_a":55.013466,"high_a":56.085997,"low_a":54.357003,"close_a":54.874775,"volume_a":24195700,"open_b":154.819207,"high_b":155.027172,"low_b":151.220446,"close_b":152.703354,"volume_b":7302000,"open_bench":478.859397,"close_bench":478.423645},{"date":"2024-02-14","open_a":54.708348,"high_a":55.09668,"low_a":54.588153,"close_a":54.819302,"volume_a":15872500,"open_b":152.215049,"high_b":152.486315,"low_b":150.153447,"close_b":151.473602,"volume_b":5731900,"open_bench":481.047798,"close_bench":482.771393},{"date":"2024-02-15","open_a":54.93949,"high_a":55.096673,"low_a":54.67136,"close_a":54.921001,"volume_a":13695800,"open_b":151.862412,"high_b":152.24218,"low_b":150.714059,"close_b":151.781036,"volume_b":4451400,"open_bench":483.468576,"close_bench":486.102386},{"date":"2024-02-16","open_a":54.87477,"high_a":55.124411,"low_a":54.514178,"close_a":54.911755,"volume_a":13947100,"open_b":151.193283,"high_b":151.374122,"low_b":149.809826,"close_b":150.388535,"volume_b":5246000,"open_bench":485.802216,"close_bench":483.68161},{"date":"2024-02-20","open_a":55.105928,"high_a":56.252429,"low_a":55.022714,"close_a":56.122986,"volume_a":18339300,"open_b":151.672524,"high_b":153.28202,"low_b":151.175204,"close_b":152.495346,"volume_b":6117900,"open_bench":481.948309,"close_bench":481.018738},{"date":"2024-02-21","open_a":56.39112,"high_a":56.65925,"low_a":56.243184,"close_a":56.622269,"volume_a":14373000,"open_b":153.670855,"high_b":153.87882,"low_b":151.781055,"close_b":152.658142,"volume_b":4192600,"open_bench":479.72118,"close_bench":481.454437},{"date":"2024-02-22","open_a":56.391116,"high_a":56.631509,"low_a":55.938062,"close_a":56.539051,"volume_a":12982800,"open_b":151.582082,"high_b":152.658096,"low_b":150.189602,"close_b":152.133652,"volume_b":7539200,"open_bench":488.038966,"close_bench":491.418365},{"date":"2024-02-23","open_a":56.483571,"high_a":56.973606,"low_a":56.400358,"close_a":56.585278,"volume_a":13612900,"open_b":151.853375,"high_b":153.942102,"low_b":151.726786,"close_b":153.35437,"volume_b":6409800,"open_bench":493.13228,"close_bench":491.757294},{"date":"2024-02-26","open_a":56.622256,"high_a":56.649992,"low_a":56.085988,"close_a":56.132217,"volume_a":10336100,"open_b":153.191589,"high_b":153.191589,"low_b":151.645395,"close_b":152.1427,"volume_b":3837000,"open_bench":492.193036,"close_bench":489.956238},{"date":"2024-02-27","open_a":55.975039,"high_a":56.19694,"low_a":55.586707,"close_a":55.790119,"volume_a":9931100,"open_b":151.744878,"high_b":152.106558,"low_b":151.319898,"close_b":152.052307,"volume_b":3567600,"open_bench":490.643762,"close_bench":490.866455},{"date":"2024-02-28","open_a":55.81786,"high_a":55.928814,"low_a":55.531237,"close_a":55.8456,"volume_a":8076500,"open_b":152.016121,"high_b":152.359726,"low_b":150.506088,"close_b":151.030533,"volume_b":5275300,"open_bench":489.31719,"close_bench":490.217743},{"date":"2024-02-29","open_a":55.79937,"high_a":56.067504,"low_a":55.383305,"close_a":55.494255,"volume_a":18152700,"open_b":151.435988,"high_b":151.864198,"low_b":150.251539,"close_b":150.643311,"volume_b":8489400,"open_bench":491.970287,"close_bench":491.97995},{"date":"2024-03-01","open_a":55.383302,"high_a":55.383302,"low_a":54.865527,"close_a":55.041199,"volume_a":10927600,"open_b":150.205996,"high_b":150.333551,"low_b":148.739104,"close_b":149.959991,"volume_b":6109900,"open_bench":492.851483,"close_bench":496.598816},{"date":"2024-03-04","open_a":54.763823,"high_a":55.429534,"low_a":54.754579,"close_a":55.300091,"volume_a":10150200,"open_b":149.832433,"high_b":150.925765,"low_b":149.057995,"close_b":150.670654,"volume_b":4303700,"open_bench":495.80488,"close_bench":496.066284},{"date":"2024-03-05","open_a":55.392544,"high_a":55.577464,"low_a":54.93949,"close_a":55.031952,"volume_a":12310000,"open_b":150.533971,"high_b":150.789082,"low_b":147.427075,"close_b":147.636627,"volume_b":6631300,"open_bench":494.071522,"close_bench":491.10849},{"date":"2024-03-06","open_a":55.031952,"high_a":55.475757,"low_a":54.902509,"close_a":55.059689,"volume_a":12378800,"open_b":147.654859,"high_b":149.413313,"low_b":147.636633,"close_b":148.511307,"volume_b":5302500,"open_bench":494.371684,"close_bench":493.597046},{"date":"2024-03-07","open_a":55.189134,"high_a":55.263103,"low_a":54.773069,"close_a":54.957985,"volume_a":13686900,"open_b":149.258422,"high_b":149.331299,"low_b":148.055747,"close_b":148.611526,"volume_b":5801600,"open_bench":496.87964,"close_bench":498.496704},{"date":"2024-03-08","open_a":54.82854,"high_a":55.281593,"low_a":54.523425,"close_a":55.031952,"volume_a":13239000,"open_b":147.891731,"high_b":149.604612,"low_b":147.682165,"close_b":148.556839,"volume_b":6065300,"open_bench":499.126207,"close_bench":495.5047},{"date":"2024-03-11","open_a":55.272349,"high_a":55.743895,"low_a":55.087433,"close_a":55.697666,"volume_a":14114300,"open_b":149.349526,"high_b":150.716194,"low_b":148.793747,"close_b":150.087524,"volume_b":5404800,"open_bench":494.303945,"close_bench":495.078583},{"date":"2024-03-12","open_a":55.771633,"high_a":56.16921,"low_a":55.669927,"close_a":55.938061,"volume_a":12684600,"open_b":150.461102,"high_b":150.716212,"low_b":149.385996,"close_b":149.914429,"volume_b":6158500,"open_bench":497.179862,"close_bench":500.404358},{"date":"2024-03-13","open_a":56.289413,"high_a":56.594528,"low_a":56.215444,"close_a":56.511314,"volume_a":13909500,"open_b":150.424653,"high_b":151.372204,"low_b":150.096652,"close_b":150.606873,"volume_b":4585800,"open_bench":500.723824,"close_bench":499.619934},{"date":"2024-03-14","open_a":56.460051,"high_a":56.665087,"low_a":56.292293,"close_a":56.38549,"volume_a":13996600,"open_b":151.48153,"high_b":151.48153,"low_b":149.60464,"close_b":150.133087,"volume_b":5155300,"open_bench":500.588244,"close_bench":498.632294},{"date":"2024-03-15","open_a":55.938137,"high_a":56.338894,"low_a":55.58398,"close_a":55.807659,"volume_a":36849900,"open_b":150.388213,"high_b":150.679775,"low_b":149.094435,"close_b":150.023773,"volume_b":21887300,"open_bench":495.577542,"close_bench":495.208435},{"date":"2024-03-18","open_a":55.816974,"high_a":56.292291,"low_a":55.733094,"close_a":56.040653,"volume_a":15856700,"open_b":152.201319,"high_b":156.875316,"low_b":152.110202,"close_b":156.037094,"volume_b":10478500,"open_bench":499.25887,"close_bench":498.15155},{"date":"2024-03-19","open_a":56.143176,"high_a":56.245692,"low_a":55.975417,"close_a":56.133854,"volume_a":15030600,"open_b":155.891323,"high_b":157.303536,"low_b":155.800206,"close_b":156.529099,"volume_b":7653700,"open_bench":497.461901,"close_bench":500.9198},{"date":"2024-03-20","open_a":56.08725,"high_a":56.674405,"low_a":56.068609,"close_a":56.618484,"volume_a":15258800,"open_b":156.984693,"high_b":157.713586,"low_b":155.909573,"close_b":156.583801,"volume_b":5046700,"open_bench":500.978096,"close_bench":505.552979},{"date":"2024-03-21","open_a":56.44142,"high_a":56.842176,"low_a":56.21774,"close_a":56.35754,"volume_a":13067100,"open_b":156.638449,"high_b":158.005118,"low_b":155.845786,"close_b":157.121338,"volume_b":4343000,"open_bench":508.379568,"close_bench":507.223694},{"date":"2024-03-22","open_a":56.404126,"high_a":56.655764,"low_a":56.320247,"close_a":56.376167,"volume_a":11502800,"open_b":157.640666,"high_b":157.713556,"low_b":156.492655,"close_b":156.729553,"volume_b":4106400,"open_bench":507.136269,"close_bench":506.262115},{"date":"2024-03-25","open_a":56.366851,"high_a":56.581209,"low_a":56.031333,"close_a":56.292294,"volume_a":13144700,"open_b":156.893542,"high_b":157.704432,"low_b":156.811546,"close_b":157.257996,"volume_b":3941900,"open_bench":504.892421,"close_bench":504.863312},{"date":"2024-03-26","open_a":56.385495,"high_a":56.581212,"low_a":56.273657,"close_a":56.422775,"volume_a":13277400,"open_b":156.866229,"high_b":157.595122,"low_b":156.547348,"close_b":157.37645,"volume_b":4593000,"open_bench":506.281457,"close_bench":503.930878},{"date":"2024-03-27","open_a":56.655768,"high_a":57.252242,"low_a":56.571889,"close_a":56.879444,"volume_a":12551200,"open_b":158.533526,"high_b":159.162195,"low_b":157.667972,"close_b":158.141754,"volume_b":4911300,"open_bench":506.747803,"close_bench":508.165894},{"date":"2024-03-28","open_a":56.981961,"high_a":57.093799,"low_a":56.851483,"close_a":57.019241,"volume_a":13683500,"open_b":159.226003,"high_b":160.182674,"low_b":158.87067,"close_b":159.453781,"volume_b":5710400,"open_bench":508.204694,"close_bench":508.068695},{"date":"2024-04-01","open_a":57.019237,"high_a":57.131075,"low_a":56.506643,"close_a":56.553242,"volume_a":11668700,"open_b":159.964009,"high_b":160.565333,"low_b":157.303549,"close_b":157.604218,"volume_b":4815400,"open_bench":508.806902,"close_bench":507.184753},{"date":"2024-04-02","open_a":56.338896,"high_a":56.599853,"low_a":56.022019,"close_a":56.059299,"volume_a":13017300,"open_b":157.567764,"high_b":158.415092,"low_b":156.128205,"close_b":156.201096,"volume_b":5158500,"open_bench":503.377194,"close_bench":503.960022},{"date":"2024-04-03","open_a":55.966095,"high_a":56.282972,"low_a":55.714456,"close_a":55.761059,"volume_a":13672900,"open_b":154.889112,"high_b":155.727334,"low_b":153.704663,"close_b":154.597549,"volume_b":5828000,"open_bench":502.872168,"close_bench":504.513702},{"date":"2024-04-04","open_a":56.003379,"high_a":56.0593,"low_a":55.145945,"close_a":55.267105,"volume_a":14320700,"open_b":155.727353,"high_b":156.51091,"low_b":154.278687,"close_b":154.570236,"volume_b":5422600,"open_bench":508.505812,"close_bench":498.355499},{"date":"2024-04-05","open_a":55.220498,"high_a":55.621255,"low_a":54.903621,"close_a":55.462814,"volume_a":10157800,"open_b":154.342445,"high_b":154.861785,"low_b":153.130677,"close_b":154.10556,"volume_b":4412200,"open_bench":499.705653,"close_bench":503.561768},{"date":"2024-04-08","open_a":55.360303,"high_a":55.425542,"low_a":55.052744,"close_a":55.239143,"volume_a":10284500,"open_b":154.114654,"high_b":154.934663,"low_b":153.741108,"close_b":154.506439,"volume_b":3755700,"open_bench":504.261226,"close_bench":503.843506},{"date":"2024-04-09","open_a":55.434859,"high_a":55.677179,"low_a":55.108664,"close_a":55.658539,"volume_a":10779100,"open_b":155.098643,"high_b":155.545093,"low_b":153.731974,"close_b":155.44487,"volume_b":3773900,"open_bench":505.572329,"close_bench":504.426178},{"date":"2024-04-10","open_a":55.285739,"high_a":55.350978,"low_a":54.698583,"close_a":54.912941,"volume_a":11861900,"open_b":154.497333,"high_b":154.551998,"low_b":152.857328,"close_b":153.932434,"volume_b":4581500,"open_bench":498.753737,"close_bench":499.375397},{"date":"2024-04-11","open_a":55.127309,"high_a":55.332344,"low_a":54.791791,"close_a":55.034107,"volume_a":11109000,"open_b":154.542863,"high_b":154.770642,"low_b":152.857314,"close_b":153.394867,"volume_b":4241500,"open_bench":500.890694,"close_bench":503.144165},{"date":"2024-04-12","open_a":54.940911,"high_a":54.996828,"low_a":54.176678,"close_a":54.316475,"volume_a":12250900,"open_b":152.784415,"high_b":153.53152,"low_b":152.447308,"close_b":153.157974,"volume_b":5453500,"open_bench":499.618196,"close_bench":496.199158},{"date":"2024-04-15","open_a":54.624023,"high_a":54.810422,"low_a":54.092788,"close_a":54.185986,"volume_a":12425000,"open_b":154.169322,"high_b":154.296877,"low_b":151.271977,"close_b":152.110199,"volume_b":5133200,"open_bench":500.356425,"close_bench":489.982727},{"date":"2024-04-16","open_a":54.288512,"high_a":54.409672,"low_a":53.990275,"close_a":54.111435,"volume_a":11236800,"open_b":152.183089,"high_b":153.531531,"low_b":152.055534,"close_b":152.611313,"volume_b":4569700,"open_bench":490.458646,"close_bench":489.089081},{"date":"2024-04-17","open_a":54.316469,"high_a":54.577429,"low_a":54.14871,"close_a":54.530827,"volume_a":13805100,"open_b":154.059965,"high_b":154.615744,"low_b":153.258196,"close_b":154.415298,"volume_b":4069200,"open_bench":491.536875,"close_bench":486.194611},{"date":"2024-04-18","open_a":54.633347,"high_a":54.968865,"low_a":54.55879,"close_a":54.903625,"volume_a":11125900,"open_b":154.980224,"high_b":157.330882,"low_b":154.980224,"close_b":156.957336,"volume_b":5574700,"open_bench":487.583562,"close_bench":485.194092},{"date":"2024-04-19","open_a":55.0807,"high_a":56.255011,"low_a":54.987502,"close_a":56.07793,"volume_a":21207900,"open_b":156.966429,"high_b":158.779548,"low_b":156.529099,"close_b":158.651993,"volume_b":6636700,"open_bench":485.11645,"close_bench":480.959198},{"date":"2024-04-22","open_a":56.049974,"high_a":56.506652,"low_a":55.65854,"close_a":56.432091,"volume_a":15624500,"open_b":158.907104,"high_b":161.449101,"low_b":158.560877,"close_b":160.774887,"volume_b":7288700,"open_bench":483.552556,"close_bench":485.388367},{"date":"2024-04-23","open_a":56.413449,"high_a":56.553249,"low_a":56.040654,"close_a":56.515968,"volume_a":13689100,"open_b":159.171323,"high_b":159.335331,"low_b":155.016653,"close_b":156.000656,"volume_b":9145000,"open_bench":487.389309,"close_bench":491.148315},{"date":"2024-04-24","open_a":56.217729,"high_a":57.45728,"low_a":55.770374,"close_a":57.364079,"volume_a":19548300,"open_b":153.522413,"high_b":162.259976,"low_b":152.747962,"close_b":161.640427,"volume_b":10016700,"open_bench":492.032235,"close_bench":490.915222},{"date":"2024-04-25","open_a":57.662321,"high_a":58.333356,"low_a":57.196325,"close_a":57.541164,"volume_a":19048500,"open_b":162.433127,"high_b":164.574247,"low_b":160.191798,"close_b":160.975342,"volume_b":7441500,"open_bench":484.863872,"close_bench":489.050262},{"date":"2024-04-26","open_a":57.364084,"high_a":57.718242,"low_a":57.177685,"close_a":57.541164,"volume_a":10994300,"open_b":158.934477,"high_b":162.697376,"low_b":158.84336,"close_b":159.973145,"volume_b":4472100,"open_bench":491.828246,"close_bench":493.683472},{"date":"2024-04-29","open_a":57.708921,"high_a":57.830081,"low_a":57.345445,"close_a":57.820763,"volume_a":13409300,"open_b":159.900226,"high_b":160.647345,"low_b":159.462896,"close_b":160.483337,"volume_b":3224500,"open_bench":495.460962,"close_bench":495.431824},{"date":"2024-04-30","open_a":57.913958,"high_a":58.557035,"low_a":57.289526,"close_a":57.569122,"volume_a":19349200,"open_b":160.601774,"high_b":160.783994,"low_b":159.298876,"close_b":160.273773,"volume_b":4465100,"open_bench":493.974834,"close_bench":487.583557},{"date":"2024-05-01","open_a":57.597079,"high_a":58.314713,"low_a":57.047204,"close_a":57.718239,"volume_a":16729800,"open_b":159.863786,"high_b":160.565332,"low_b":158.223781,"close_b":159.581329,"volume_b":4340500,"open_bench":487.000703,"close_bench":486.000244},{"date":"2024-05-02","open_a":57.951236,"high_a":58.156275,"low_a":57.52252,"close_a":57.774158,"volume_a":10991300,"open_b":159.973109,"high_b":160.647337,"low_b":159.034664,"close_b":159.85466,"volume_b":3964800,"open_bench":489.691345,"close_bench":490.546112},{"date":"2024-05-03","open_a":57.774166,"high_a":58.091043,"low_a":57.447967,"close_a":57.941921,"volume_a":11628700,"open_b":160.173566,"high_b":160.802236,"low_b":159.444673,"close_b":160.492447,"volume_b":3491900,"open_bench":496.50028,"close_bench":496.626556},{"date":"2024-05-06","open_a":58.063077,"high_a":58.137639,"low_a":57.690283,"close_a":58.109676,"volume_a":9310700,"open_b":161.048222,"high_b":161.048222,"low_b":159.563118,"close_b":160.191788,"volume_b":3576200,"open_bench":499.016026,"close_bench":501.755157},{"date":"2024-05-07","open_a":58.491787,"high_a":58.547708,"low_a":58.016473,"close_a":58.361309,"volume_a":9807000,"open_b":161.26689,"high_b":162.314663,"low_b":160.820439,"close_b":162.196228,"volume_b":3708000,"open_bench":502.716745,"close_bench":502.308807},{"date":"2024-05-08","open_a":58.575668,"high_a":58.68751,"low_a":58.230832,"close_a":58.575668,"volume_a":10484400,"open_b":162.824875,"high_b":162.979763,"low_b":161.439981,"close_b":161.640427,"volume_b":3400900,"open_bench":500.482719,"close_bench":502.357361},{"date":"2024-05-09","open_a":58.538391,"high_a":58.855268,"low_a":58.48247,"close_a":58.60363,"volume_a":9113700,"open_b":161.503789,"high_b":162.569788,"low_b":161.348901,"close_b":162.232681,"volume_b":2421400,"open_bench":502.541929,"close_bench":505.251892},{"date":"2024-05-10","open_a":58.640909,"high_a":59.050988,"low_a":58.519753,"close_a":58.957787,"volume_a":8371700,"open_b":162.13244,"high_b":164.146005,"low_b":161.822665,"close_b":163.808884,"volume_b":4201600,"open_bench":506.84474,"close_bench":505.902588},{"date":"2024-05-13","open_a":58.939154,"high_a":59.293308,"low_a":58.883233,"close_a":59.256031,"volume_a":10137600,"open_b":164.464888,"high_b":165.230233,"low_b":163.863564,"close_b":164.820221,"volume_b":4408100,"open_bench":507.573284,"close_bench":505.970581},{"date":"2024-05-14","open_a":59.256025,"high_a":59.423781,"low_a":58.529069,"close_a":58.808666,"volume_a":10852200,"open_b":164.956895,"high_b":165.166447,"low_b":162.724672,"close_b":163.881775,"volume_b":3508600,"open_bench":506.164889,"close_bench":508.292084},{"date":"2024-05-15","open_a":58.771385,"high_a":59.07894,"low_a":58.715464,"close_a":58.836624,"volume_a":9838900,"open_b":163.307779,"high_b":164.364658,"low_b":163.089107,"close_b":163.508224,"volume_b":4785100,"open_bench":510.749515,"close_bench":514.586243},{"date":"2024-05-16","open_a":58.939149,"high_a":59.395823,"low_a":58.780708,"close_a":59.013706,"volume_a":10228000,"open_b":163.663104,"high_b":167.107107,"low_b":163.508216,"close_b":166.833771,"volume_b":5984700,"open_bench":514.683401,"close_bench":513.527527},{"date":"2024-05-17","open_a":59.032346,"high_a":59.060305,"low_a":58.659548,"close_a":58.743427,"volume_a":11539000,"open_b":166.359999,"high_b":166.69712,"low_b":165.667559,"close_b":165.99556,"volume_b":4449100,"open_bench":513.644149,"close_bench":514.265808},{"date":"2024-05-20","open_a":58.65023,"high_a":58.724788,"low_a":58.212193,"close_a":58.314713,"volume_a":11000900,"open_b":165.904433,"high_b":165.940885,"low_b":163.918214,"close_b":164.282654,"volume_b":3221100,"open_bench":514.382217,"close_bench":514.858154},{"date":"2024-05-21","open_a":58.547712,"high_a":58.696831,"low_a":58.193555,"close_a":58.631592,"volume_a":9796200,"open_b":165.366903,"high_b":165.439794,"low_b":163.818015,"close_b":164.984238,"volume_b":3288100,"open_bench":514.100606,"close_bench":516.120911},{"date":"2024-05-22","open_a":58.445191,"high_a":58.73411,"low_a":58.398592,"close_a":58.715469,"volume_a":7419400,"open_b":164.410195,"high_b":165.940871,"low_b":163.981972,"close_b":165.904419,"volume_b":4047000,"open_bench":515.431439,"close_bench":514.634949},{"date":"2024-05-23","open_a":58.240161,"high_a":58.603637,"low_a":57.830082,"close_a":57.867363,"volume_a":9834300,"open_b":165.212018,"high_b":165.722239,"low_b":163.125563,"close_b":163.335129,"volume_b":4009000,"open_bench":517.675099,"close_bench":510.875854},{"date":"2024-05-24","open_a":57.988513,"high_a":58.174909,"low_a":57.755515,"close_a":57.783474,"volume_a":8203100,"open_b":163.42621,"high_b":163.471768,"low_b":162.059541,"close_b":162.168884,"volume_b":3447400,"open_bench":512.711495,"close_bench":514.25592},{"date":"2024-05-28","open_a":57.587768,"high_a":57.904645,"low_a":57.224292,"close_a":57.615726,"volume_a":12347000,"open_b":160.56532,"high_b":160.610879,"low_b":156.620202,"close_b":157.968658,"volume_b":6178100,"open_bench":515.062307,"close_bench":514.615479},{"date":"2024-05-29","open_a":57.289522,"high_a":57.671638,"low_a":56.916724,"close_a":57.50388,"volume_a":11861800,"open_b":157.103112,"high_b":157.786447,"low_b":155.545118,"close_b":155.93689,"volume_b":5352100,"open_bench":510.603933,"close_bench":511.011871},{"date":"2024-05-30","open_a":57.643678,"high_a":57.85804,"low_a":57.41068,"close_a":57.75552,"volume_a":9433800,"open_b":156.119108,"high_b":156.447109,"low_b":155.253541,"close_b":155.326431,"volume_b":3903100,"open_bench":509.477112,"close_bench":507.621857},{"date":"2024-05-31","open_a":57.77416,"high_a":58.743428,"low_a":57.494559,"close_a":58.65023,"volume_a":19816600,"open_b":154.788883,"high_b":157.595111,"low_b":154.451776,"close_b":157.531326,"volume_b":8155600,"open_bench":508.573861,"close_bench":512.245422},{"date":"2024-06-03","open_a":58.445191,"high_a":58.799349,"low_a":58.202875,"close_a":58.65023,"volume_a":9369000,"open_b":157.367297,"high_b":157.531291,"low_b":155.827515,"close_b":156.009735,"volume_b":5032400,"open_bench":513.848067,"close_bench":512.663025},{"date":"2024-06-04","open_a":58.901873,"high_a":59.619507,"low_a":58.659553,"close_a":59.591545,"volume_a":11895600,"open_b":155.99153,"high_b":158.88886,"low_b":155.636197,"close_b":158.433304,"volume_b":4619100,"open_bench":511.361625,"close_bench":513.236267},{"date":"2024-06-05","open_a":59.638138,"high_a":59.712695,"low_a":59.04166,"close_a":59.572895,"volume_a":9632500,"open_b":158.387805,"high_b":158.506254,"low_b":156.674923,"close_b":158.068924,"volume_b":5370700,"open_bench":515.547884,"close_bench":519.335999},{"date":"2024-06-06","open_a":59.572897,"high_a":59.982976,"low_a":59.498339,"close_a":59.787258,"volume_a":8658500,"open_b":158.533534,"high_b":158.897974,"low_b":157.330859,"close_b":157.804642,"volume_b":5536700,"open_bench":519.637185,"close_bench":519.326355},{"date":"2024-06-07","open_a":59.731342,"high_a":59.852506,"low_a":59.470389,"close_a":59.563587,"volume_a":9523100,"open_b":157.487868,"high_b":158.415342,"low_b":156.799145,"close_b":157.065445,"volume_b":3521100,"open_bench":518.354887,"close_bench":518.694885},{"date":"2024-06-10","open_a":59.591538,"high_a":59.638141,"low_a":59.060303,"close_a":59.265343,"volume_a":15685300,"open_b":156.211417,"high_b":156.450169,"low_b":152.088261,"close_b":152.345383,"volume_b":9429400,"open_bench":517.888751,"close_bench":520.297607},{"date":"2024-06-11","open_a":59.265342,"high_a":59.321263,"low_a":58.780706,"close_a":59.228062,"volume_a":8396000,"open_b":151.399536,"high_b":151.941341,"low_b":150.150652,"close_b":151.583206,"volume_b":8284500,"open_bench":518.753312,"close_bench":521.55072},{"date":"2024-06-12","open_a":59.358541,"high_a":59.358541,"low_a":58.426549,"close_a":58.60363,"volume_a":9569600,"open_b":151.408718,"high_b":151.821948,"low_b":149.168072,"close_b":150.444504,"volume_b":6281100,"open_bench":526.096443,"close_bench":525.834167},{"date":"2024-06-13","open_a":58.584994,"high_a":58.752753,"low_a":58.202878,"close_a":58.706154,"volume_a":9668900,"open_b":148.837511,"high_b":150.196586,"low_b":148.341623,"close_b":149.985382,"volume_b":5432600,"open_bench":527.572881,"close_bench":526.892944},{"date":"2024-06-14","open_a":58.588751,"high_a":58.879908,"low_a":58.316375,"close_a":58.748417,"volume_a":8179200,"open_b":149.801728,"high_b":150.958776,"low_b":149.250744,"close_b":150.426163,"volume_b":3603300,"open_bench":525.367968,"close_bench":527.213501},{"date":"2024-06-17","open_a":58.579354,"high_a":59.077144,"low_a":58.400902,"close_a":58.814159,"volume_a":10532700,"open_b":150.132285,"high_b":152.813709,"low_b":149.489467,"close_b":152.565765,"volume_b":5412200,"open_bench":526.533556,"close_bench":531.409546},{"date":"2024-06-18","open_a":58.729627,"high_a":59.002002,"low_a":58.645096,"close_a":58.823551,"volume_a":10651700,"open_b":152.529048,"high_b":153.419796,"low_b":151.74849,"close_b":152.877991,"volume_b":3519500,"open_bench":531.467771,"close_bench":532.759644},{"date":"2024-06-20","open_a":58.607528,"high_a":58.926864,"low_a":58.184877,"close_a":58.400898,"volume_a":13306300,"open_b":152.189282,"high_b":153.943232,"low_b":152.189282,"close_b":153.061661,"volume_b":4942400,"open_bench":533.682464,"close_bench":531.312439},{"date":"2024-06-21","open_a":58.400902,"high_a":59.264986,"low_a":58.400902,"close_a":58.955044,"volume_a":28278100,"open_b":153.548367,"high_b":155.256399,"low_b":152.850453,"close_b":153.61264,"volume_b":10899100,"open_bench":530.492922,"close_bench":530.600098},{"date":"2024-06-24","open_a":59.161673,"high_a":60.166638,"low_a":59.086533,"close_a":60.082111,"volume_a":13212500,"open_b":153.814661,"high_b":155.752267,"low_b":153.438156,"close_b":154.347275,"volume_b":4476900,"open_bench":530.424714,"close_bench":528.875305},{"date":"2024-06-25","open_a":60.053935,"high_a":60.176035,"low_a":59.650069,"close_a":59.960014,"volume_a":10235300,"open_b":154.347278,"high_b":155.155384,"low_b":153.58509,"close_b":153.676926,"volume_b":4217300,"open_bench":530.093359,"close_bench":530.911926},{"date":"2024-06-26","open_a":59.54676,"high_a":60.213608,"low_a":59.387091,"close_a":60.157257,"volume_a":9402500,"open_b":152.336187,"high_b":153.364675,"low_b":151.381165,"close_b":153.116745,"volume_b":4778300,"open_bench":529.801071,"close_bench":531.574585},{"date":"2024-06-27","open_a":60.157255,"high_a":60.363878,"low_a":59.753385,"close_a":60.025761,"volume_a":8494100,"open_b":153.015728,"high_b":153.585068,"low_b":151.766845,"close_b":152.675949,"volume_b":4326500,"open_bench":531.438143,"close_bench":532.412598},{"date":"2024-06-28","open_a":60.016365,"high_a":60.166637,"low_a":59.659459,"close_a":59.781559,"volume_a":17358800,"open_b":151.794384,"high_b":152.639215,"low_b":151.169935,"close_b":151.454605,"volume_b":8755900,"open_bench":533.182342,"close_bench":530.317444},{"date":"2024-07-01","open_a":60.138461,"high_a":60.392056,"low_a":59.283768,"close_a":59.434044,"volume_a":10033400,"open_b":151.555661,"high_b":152.492328,"low_b":149.29666,"close_b":149.581329,"volume_b":4508000,"open_bench":531.691506,"close_bench":531.408936},{"date":"2024-07-02","open_a":59.415258,"high_a":59.527968,"low_a":59.039571,"close_a":59.311947,"volume_a":8878200,"open_b":150.141485,"high_b":151.032234,"low_b":149.057902,"close_b":150.214951,"volume_b":4650800,"open_bench":529.810818,"close_bench":534.985168},{"date":"2024-07-03","open_a":59.349516,"high_a":59.800343,"low_a":59.10532,"close_a":59.48101,"volume_a":9154200,"open_b":149.608865,"high_b":150.72001,"low_b":148.727308,"close_b":149.315018,"volume_b":2879600,"open_bench":534.673362,"close_bench":537.37262},{"date":"2024-07-05","open_a":59.48101,"high_a":59.913052,"low_a":59.171064,"close_a":59.884872,"volume_a":11735500,"open_b":149.351737,"high_b":151.078139,"low_b":149.195628,"close_b":150.958755,"volume_b":4208200,"open_bench":537.674635,"close_bench":540.471313},{"date":"2024-07-08","open_a":59.725215,"high_a":59.725215,"low_a":59.020797,"close_a":59.133503,"volume_a":11235100,"open_b":149.737438,"high_b":150.371065,"low_b":148.690581,"close_b":148.874237,"volume_b":5268800,"open_bench":541.250918,"close_bench":541.095032},{"date":"2024-07-09","open_a":59.227417,"high_a":59.387083,"low_a":58.785981,"close_a":58.879902,"volume_a":11657500,"open_b":148.938494,"high_b":149.278259,"low_b":148.304867,"close_b":148.67218,"volume_b":5801200,"open_bench":542.049978,"close_bench":541.621216},{"date":"2024-07-10","open_a":58.908083,"high_a":59.039574,"low_a":58.504217,"close_a":59.011398,"volume_a":11930400,"open_b":148.617102,"high_b":150.389422,"low_b":147.533519,"close_b":150.224121,"volume_b":7372300,"open_bench":542.839282,"close_bench":546.980713},{"date":"2024-07-11","open_a":58.832942,"high_a":59.603105,"low_a":58.607531,"close_a":59.264984,"volume_a":11431600,"open_b":146.100958,"high_b":151.417888,"low_b":145.118375,"close_b":150.554688,"volume_b":10258600,"open_bench":547.097658,"close_bench":542.264343},{"date":"2024-07-12","open_a":59.603103,"high_a":60.213599,"low_a":59.537357,"close_a":59.828518,"volume_a":9201500,"open_b":151.482229,"high_b":154.301393,"low_b":151.482229,"close_b":152.786209,"volume_b":6677000,"open_bench":543.384994,"close_bench":545.684692},{"date":"2024-07-15","open_a":59.913051,"high_a":60.053932,"low_a":59.49979,"close_a":59.556145,"volume_a":8207500,"open_b":152.446378,"high_b":152.712691,"low_b":150.279197,"close_b":150.472046,"volume_b":4551800,"open_bench":547.672469,"close_bench":547.185242},{"date":"2024-07-16","open_a":59.574936,"high_a":60.40145,"low_a":59.537366,"close_a":60.36388,"volume_a":9025100,"open_b":150.058846,"high_b":151.55566,"low_b":149.53541,"close_b":151.298538,"volume_b":4461700,"open_bench":548.491082,"close_bench":550.430237},{"date":"2024-07-17","open_a":60.551727,"high_a":61.41581,"low_a":60.495376,"close_a":61.24675,"volume_a":12508200,"open_b":152.473937,"high_b":156.174685,"low_b":152.336198,"close_b":156.009384,"volume_b":7671000,"open_bench":544.525056,"close_bench":542.712585},{"date":"2024-07-18","open_a":60.861674,"high_a":61.810284,"low_a":60.786534,"close_a":61.22797,"volume_a":11128200,"open_b":154.925818,"high_b":158.562265,"low_b":154.631957,"close_b":156.45018,"volume_b":6228600,"open_bench":544.242402,"close_bench":538.541809},{"date":"2024-07-19","open_a":61.650607,"high_a":61.650607,"low_a":60.880444,"close_a":61.32188,"volume_a":13775700,"open_b":156.367506,"high_b":156.560354,"low_b":154.705391,"close_b":155.522675,"volume_b":5332800,"open_bench":538.30807,"close_bench":534.965698},{"date":"2024-07-22","open_a":61.284311,"high_a":61.321881,"low_a":60.410833,"close_a":60.833481,"volume_a":11794200,"open_b":155.43086,"high_b":155.844104,"low_b":153.676924,"close_b":153.961594,"volume_b":5308400,"open_bench":538.873267,"close_bench":540.48114},{"date":"2024-07-23","open_a":61.45338,"high_a":62.026307,"low_a":60.908629,"close_a":61.011944,"volume_a":11810300,"open_b":154.080974,"high_b":154.60441,"low_b":152.602516,"close_b":152.694351,"volume_b":3344300,"open_bench":540.373897,"close_bench":539.633362},{"date":"2024-07-24","open_a":61.134038,"high_a":61.960559,"low_a":60.889845,"close_a":61.81028,"volume_a":13244600,"open_b":152.694367,"high_b":154.889095,"low_b":151.473031,"close_b":154.429947,"volume_b":5867000,"open_bench":534.838961,"close_bench":527.40387},{"date":"2024-07-25","open_a":62.026295,"high_a":62.824633,"low_a":61.857234,"close_a":62.05447,"volume_a":16328600,"open_b":155.642081,"high_b":159.315267,"low_b":155.06355,"close_b":157.047073,"volume_b":6270900,"open_bench":527.520843,"close_bench":524.655945},{"date":"2024-07-26","open_a":62.054473,"high_a":63.031266,"low_a":61.998122,"close_a":62.974915,"volume_a":13530100,"open_b":156.780734,"high_b":158.644874,"low_b":156.780734,"close_b":158.635696,"volume_b":5746300,"open_bench":528.427065,"close_bench":530.53186},{"date":"2024-07-29","open_a":62.815252,"high_a":62.956137,"low_a":62.355035,"close_a":62.768295,"volume_a":12140200,"open_b":158.213306,"high_b":159.315273,"low_b":156.890971,"close_b":159.058151,"volume_b":4152700,"open_bench":532.071572,"close_bench":530.84375},{"date":"2024-07-30","open_a":62.43957,"high_a":63.70752,"low_a":62.326861,"close_a":63.566635,"volume_a":13258000,"open_b":158.580622,"high_b":159.278523,"low_b":157.533765,"close_b":159.030579,"volume_b":4994600,"open_bench":532.305298,"close_bench":528.154114},{"date":"2024-07-31","open_a":63.397569,"high_a":63.500885,"low_a":62.561661,"close_a":62.683758,"volume_a":15576800,"open_b":159.710126,"high_b":160.417219,"low_b":158.185751,"close_b":158.562256,"volume_b":6245600,"open_bench":534.955929,"close_bench":536.739197},{"date":"2024-08-01","open_a":62.927944,"high_a":63.970482,"low_a":62.646174,"close_a":63.829597,"volume_a":16495900,"open_b":158.433693,"high_b":161.216131,"low_b":158.406146,"close_b":160.665161,"volume_b":5994500,"open_bench":538.454208,"close_bench":529.138428},{"date":"2024-08-02","open_a":64.364959,"high_a":65.435673,"low_a":63.576015,"close_a":65.116341,"volume_a":22514800,"open_b":164.228109,"high_b":165.247419,"low_b":159.636628,"close_b":163.493469,"volume_b":6856100,"open_bench":522.063976,"close_bench":519.286804},{"date":"2024-08-05","open_a":64.797021,"high_a":65.322991,"low_a":63.613605,"close_a":63.961113,"volume_a":25484300,"open_b":164.852603,"high_b":166.129035,"low_b":159.012236,"close_b":159.820328,"volume_b":7502300,"open_bench":498.569759,"close_bench":504.163116},{"date":"2024-08-06","open_a":64.00806,"high_a":64.656119,"low_a":63.65115,"close_a":63.914139,"volume_a":12969100,"open_b":159.122421,"high_b":161.813037,"low_b":158.27759,"close_b":158.396973,"volume_b":7287100,"open_bench":505.956198,"close_bench":508.811401},{"date":"2024-08-07","open_a":63.914147,"high_a":64.975467,"low_a":63.707517,"close_a":64.299225,"volume_a":13365300,"open_b":157.598042,"high_b":160.141727,"low_b":157.331742,"close_b":157.75415,"volume_b":5428900,"open_bench":514.969887,"close_bench":505.410492},{"date":"2024-08-08","open_a":63.867182,"high_a":64.674915,"low_a":63.726297,"close_a":64.552818,"volume_a":15240900,"open_b":156.303249,"high_b":158.86529,"low_b":155.935922,"close_b":158.286758,"volume_b":5112500,"open_bench":510.526306,"close_bench":517.094177},{"date":"2024-08-09","open_a":64.524633,"high_a":64.609167,"low_a":64.101985,"close_a":64.505852,"volume_a":12080400,"open_b":158.029653,"high_b":158.66328,"low_b":156.468552,"close_b":158.305145,"volume_b":3223000,"open_bench":516.275571,"close_bench":519.374329},{"date":"2024-08-12","open_a":64.289829,"high_a":64.374363,"low_a":63.707515,"close_a":64.026848,"volume_a":11012300,"open_b":158.167393,"high_b":158.74591,"low_b":156.7624,"close_b":157.414383,"volume_b":3802300,"open_bench":520.563267,"close_bench":519.647278},{"date":"2024-08-13","open_a":63.942323,"high_a":64.44011,"low_a":63.876577,"close_a":64.299225,"volume_a":9877600,"open_b":157.598036,"high_b":158.332676,"low_b":156.312427,"close_b":158.286758,"volume_b":3894700,"open_bench":522.823983,"close_bench":528.193176},{"date":"2024-08-14","open_a":64.083207,"high_a":64.703091,"low_a":63.989279,"close_a":64.411934,"volume_a":14328000,"open_b":158.158214,"high_b":160.095821,"low_b":157.882723,"close_b":159.517303,"volume_b":3555800,"open_bench":528.982525,"close_bench":529.859558},{"date":"2024-08-15","open_a":64.449501,"high_a":64.731264,"low_a":64.336791,"close_a":64.477676,"volume_a":11285100,"open_b":159.517274,"high_b":160.059066,"low_b":157.790873,"close_b":158.4245,"volume_b":4472100,"open_bench":535.462598,"close_bench":538.941406},{"date":"2024-08-16","open_a":64.656124,"high_a":65.088173,"low_a":64.468282,"close_a":64.975464,"volume_a":12651200,"open_b":158.424506,"high_b":158.663258,"low_b":157.129704,"close_b":158.516327,"volume_b":4614600,"open_bench":537.333654,"close_bench":540.149841},{"date":"2024-08-19","open_a":65.135136,"high_a":65.200882,"low_a":64.665525,"close_a":64.787628,"volume_a":9843900,"open_b":158.553089,"high_b":159.765233,"low_b":158.351063,"close_b":159.618317,"volume_b":3521700,"open_bench":540.55899,"close_bench":545.314331},{"date":"2024-08-20","open_a":64.797018,"high_a":65.172709,"low_a":64.590388,"close_a":65.163315,"volume_a":11957400,"open_b":159.351998,"high_b":161.647739,"low_b":158.406153,"close_b":161.482452,"volume_b":5330100,"open_bench":544.86619,"close_bench":544.427673},{"date":"2024-08-21","open_a":65.25724,"high_a":65.867729,"low_a":65.116355,"close_a":65.341766,"volume_a":8792400,"open_b":161.565082,"high_b":162.373187,"low_b":160.352924,"close_b":160.89473,"volume_b":4020700,"open_bench":545.470322,"close_bench":546.298584},{"date":"2024-08-22","open_a":65.670479,"high_a":65.670479,"low_a":64.609159,"close_a":65.116341,"volume_a":12509200,"open_b":161.464067,"high_b":161.693641,"low_b":160.426388,"close_b":161.381424,"volume_b":5265900,"open_bench":548.189053,"close_bench":542.010986},{"date":"2024-08-23","open_a":65.182104,"high_a":65.604752,"low_a":64.750062,"close_a":65.548401,"volume_a":11834700,"open_b":162.198708,"high_b":162.244611,"low_b":159.480544,"close_b":161.500793,"volume_b":5283600,"open_bench":545.236462,"close_bench":547.77002},{"date":"2024-08-26","open_a":65.585966,"high_a":66.609716,"low_a":65.492045,"close_a":66.534576,"volume_a":12912200,"open_b":161.914045,"high_b":162.39155,"low_b":161.37224,"close_b":161.592636,"volume_b":4146700,"open_bench":548.793071,"close_bench":546.464111},{"date":"2024-08-27","open_a":66.985404,"high_a":67.18264,"low_a":66.731817,"close_a":67.154465,"volume_a":16988400,"open_b":161.904846,"high_b":162.428281,"low_b":160.261101,"close_b":160.977371,"volume_b":3520600,"open_bench":545.197411,"close_bench":547.214539},{"date":"2024-08-28","open_a":67.257784,"high_a":67.445626,"low_a":66.966627,"close_a":67.389275,"volume_a":13272600,"open_b":161.206957,"high_b":161.381435,"low_b":159.186707,"close_b":159.710129,"volume_b":3915500,"open_bench":546.873476,"close_bench":544.037781},{"date":"2024-08-29","open_a":67.530151,"high_a":67.830702,"low_a":66.900873,"close_a":67.671036,"volume_a":17201700,"open_b":159.563201,"high_b":159.746872,"low_b":157.864361,"close_b":158.029648,"volume_b":4654300,"open_bench":545.9965,"close_bench":544.086548},{"date":"2024-08-30","open_a":67.811912,"high_a":68.159427,"low_a":67.41744,"close_a":68.065506,"volume_a":18624200,"open_b":158.002102,"high_b":159.379547,"low_b":157.258284,"close_b":158.755112,"volume_b":8677300,"open_bench":546.44476,"close_bench":549.280396},{"date":"2024-09-03","open_a":68.037328,"high_a":68.835667,"low_a":67.858874,"close_a":68.572685,"volume_a":14755800,"open_b":158.295963,"high_b":163.355772,"low_b":157.836815,"close_b":163.034363,"volume_b":7032100,"open_bench":546.152434,"close_bench":537.976807},{"date":"2024-09-04","open_a":68.84507,"high_a":69.061087,"low_a":67.53955,"close_a":68.15004,"volume_a":18140800,"open_b":163.364957,"high_b":164.971977,"low_b":161.886499,"close_b":163.934311,"volume_b":6188500,"open_bench":536.144709,"close_bench":536.875549},{"date":"2024-09-05","open_a":68.206391,"high_a":68.262742,"low_a":66.666065,"close_a":66.844513,"volume_a":23866100,"open_b":164.127138,"high_b":165.045435,"low_b":163.016007,"close_b":164.650574,"volume_b":5743100,"open_bench":536.817093,"close_bench":535.569763},{"date":"2024-09-06","open_a":66.675445,"high_a":67.314117,"low_a":66.581524,"close_a":66.81633,"volume_a":17485700,"open_b":164.183279,"high_b":165.238107,"low_b":163.776149,"close_b":164.090744,"volume_b":6395300,"open_bench":535.891442,"close_bench":526.556152},{"date":"2024-09-09","open_a":66.900871,"high_a":67.915227,"low_a":66.666065,"close_a":67.483185,"volume_a":12939300,"open_b":163.822429,"high_b":165.432423,"low_b":162.721333,"close_b":164.877258,"volume_b":4666200,"open_bench":530.736549,"close_bench":532.451538},{"date":"2024-09-10","open_a":67.595896,"high_a":68.018543,"low_a":67.173241,"close_a":67.201424,"volume_a":10277600,"open_b":165.302874,"high_b":165.719265,"low_b":164.368337,"close_b":164.525635,"volume_b":4315800,"open_bench":534.351744,"close_bench":534.770752},{"date":"2024-09-11","open_a":67.022968,"high_a":67.032362,"low_a":66.280981,"close_a":66.759987,"volume_a":11418200,"open_b":164.183292,"high_b":164.470129,"low_b":161.398162,"close_b":162.647308,"volume_b":4036000,"open_bench":534.683047,"close_bench":540.256897},{"date":"2024-09-12","open_a":66.684847,"high_a":67.041749,"low_a":66.365514,"close_a":66.900871,"volume_a":12337800,"open_b":162.647301,"high_b":164.414606,"low_b":160.95402,"close_b":164.007477,"volume_b":4222800,"open_bench":540.831888,"close_bench":544.807678},{"date":"2024-09-13","open_a":66.990697,"high_a":67.586469,"low_a":66.650266,"close_a":67.529732,"volume_a":11073800,"open_b":164.192526,"high_b":164.664432,"low_b":163.193212,"close_b":164.109253,"volume_b":3290400,"open_bench":545.411782,"close_bench":547.653015},{"date":"2024-09-16","open_a":67.870165,"high_a":68.314628,"low_a":67.728322,"close_a":68.182236,"volume_a":10786500,"open_b":165.626766,"high_b":166.024634,"low_b":163.174755,"close_b":163.970505,"volume_b":4940100,"open_bench":547.389979,"close_bench":548.461914},{"date":"2024-09-17","open_a":67.993117,"high_a":68.418662,"low_a":67.822897,"close_a":67.898552,"volume_a":12974400,"open_b":163.776145,"high_b":164.784707,"low_b":163.128446,"close_b":163.655853,"volume_b":6305100,"open_bench":550.664024,"close_bench":548.685913},{"date":"2024-09-18","open_a":67.983657,"high_a":68.44703,"low_a":67.397344,"close_a":67.851265,"volume_a":10719700,"open_b":163.646625,"high_b":164.756969,"low_b":161.398165,"close_b":162.119904,"volume_b":5307900,"open_bench":549.338906,"close_bench":547.058716},{"date":"2024-09-19","open_a":67.671579,"high_a":68.030935,"low_a":66.763744,"close_a":66.782661,"volume_a":13635600,"open_b":162.619539,"high_b":163.054426,"low_b":161.139071,"close_b":161.610977,"volume_b":5504000,"open_bench":556.423122,"close_bench":556.39386},{"date":"2024-09-20","open_a":66.584078,"high_a":67.898548,"low_a":66.555702,"close_a":67.747238,"volume_a":48578500,"open_b":159.677157,"high_b":160.787501,"low_b":157.502722,"close_b":158.390991,"volume_b":12198400,"open_bench":555.031342,"close_bench":555.432068},{"date":"2024-09-23","open_a":67.662125,"high_a":68.276799,"low_a":67.312227,"close_a":67.832344,"volume_a":11613000,"open_b":158.104147,"high_b":160.944791,"low_b":157.974607,"close_b":159.251511,"volume_b":5350300,"open_bench":556.497616,"close_bench":556.820129},{"date":"2024-09-24","open_a":67.350071,"high_a":67.917466,"low_a":66.763757,"close_a":67.454094,"volume_a":13330600,"open_b":158.215179,"high_b":158.779606,"low_b":156.836493,"close_b":157.225113,"volume_b":7837900,"open_bench":557.611758,"close_bench":558.413269},{"date":"2024-09-25","open_a":67.81344,"high_a":68.012028,"low_a":67.435174,"close_a":67.567566,"volume_a":12329100,"open_b":157.669247,"high_b":158.011613,"low_b":156.151773,"close_b":156.743958,"volume_b":4505100,"open_bench":558.256925,"close_bench":557.181702},{"date":"2024-09-26","open_a":67.009621,"high_a":67.718867,"low_a":66.990703,"close_a":67.520279,"volume_a":10815200,"open_b":156.577412,"high_b":158.224425,"low_b":156.077748,"close_b":157.012299,"volume_b":5673600,"open_bench":561.423846,"close_bench":559.390747},{"date":"2024-09-27","open_a":67.652676,"high_a":68.276809,"low_a":67.577021,"close_a":67.889091,"volume_a":11175700,"open_b":157.262158,"high_b":158.825885,"low_b":157.003064,"close_b":157.299164,"volume_b":7545800,"open_bench":560.456141,"close_bench":558.579407},{"date":"2024-09-30","open_a":68.087684,"high_a":68.276813,"low_a":67.794531,"close_a":67.955292,"volume_a":18015300,"open_b":157.835815,"high_b":158.909154,"low_b":156.910526,"close_b":157.345413,"volume_b":5054300,"open_bench":557.553184,"close_bench":560.817871},{"date":"2024-10-01","open_a":68.182238,"high_a":68.79692,"low_a":67.737776,"close_a":67.813431,"volume_a":13294500,"open_b":158.492748,"high_b":159.094194,"low_b":157.086319,"close_b":158.576035,"volume_b":7095900,"open_bench":560.465969,"close_bench":555.793762},{"date":"2024-10-02","open_a":67.510838,"high_a":67.548666,"low_a":66.584093,"close_a":67.151489,"volume_a":11155500,"open_b":158.687078,"high_b":158.909152,"low_b":155.92972,"close_b":156.882767,"volume_b":5786000,"open_bench":554.904357,"close_bench":556.028381},{"date":"2024-10-03","open_a":67.009616,"high_a":67.13255,"low_a":66.385476,"close_a":66.688087,"volume_a":9374800,"open_b":155.578103,"high_b":156.475634,"low_b":154.680571,"close_b":155.938965,"volume_b":5090300,"open_bench":554.562256,"close_bench":555.011902},{"date":"2024-10-04","open_a":66.158521,"high_a":66.461132,"low_a":65.931564,"close_a":66.357109,"volume_a":12684800,"open_b":154.662063,"high_b":155.911209,"low_b":154.35673,"close_b":155.420807,"volume_b":4390100,"open_bench":559.439565,"close_bench":560.055359},{"date":"2024-10-07","open_a":66.243637,"high_a":66.290915,"low_a":65.127755,"close_a":65.260147,"volume_a":14512000,"open_b":155.189498,"high_b":155.383801,"low_b":154.023639,"close_b":154.717606,"volume_b":5459500,"open_bench":558.413361,"close_bench":554.99231},{"date":"2024-10-08","open_a":65.288514,"high_a":65.66678,"low_a":65.118294,"close_a":65.420906,"volume_a":13549700,"open_b":155.189482,"high_b":157.900572,"low_b":154.384485,"close_b":157.687759,"volume_b":8297800,"open_bench":557.55312,"close_bench":560.241089},{"date":"2024-10-09","open_a":65.43037,"high_a":65.950479,"low_a":65.316888,"close_a":65.789719,"volume_a":9037700,"open_b":156.725496,"high_b":160.334118,"low_b":156.466416,"close_b":159.649399,"volume_b":7177500,"open_bench":560.231257,"close_bench":564.121521},{"date":"2024-10-10","open_a":66.0356,"high_a":66.111255,"low_a":65.24124,"close_a":65.487114,"volume_a":8635200,"open_b":159.214516,"high_b":160.130544,"low_b":158.428015,"close_b":160.065781,"volume_b":5292400,"open_bench":562.782412,"close_bench":563.134277},{"date":"2024-10-11","open_a":65.685695,"high_a":65.808629,"low_a":65.250691,"close_a":65.789719,"volume_a":8786300,"open_b":161.000289,"high_b":162.092124,"low_b":159.954708,"close_b":161.749771,"volume_b":4720300,"open_bench":563.056067,"close_bench":566.50647},{"date":"2024-10-14","open_a":65.789711,"high_a":66.640809,"low_a":65.695147,"close_a":66.517868,"volume_a":7659300,"open_b":161.888576,"high_b":163.100703,"low_b":161.407422,"close_b":162.943405,"volume_b":4535900,"open_bench":568.109539,"close_bench":571.139648},{"date":"2024-10-15","open_a":66.63135,"high_a":67.340597,"low_a":66.347656,"close_a":66.517868,"volume_a":10358700,"open_b":163.183986,"high_b":164.137032,"low_b":162.036622,"close_b":162.758347,"volume_b":5802300,"open_bench":571.403587,"close_bench":566.702087},{"date":"2024-10-16","open_a":66.281469,"high_a":66.858323,"low_a":66.167994,"close_a":66.73539,"volume_a":7937300,"open_b":161.934845,"high_b":162.230944,"low_b":160.657955,"close_b":161.444443,"volume_b":4400900,"open_bench":566.702045,"close_bench":569.165161},{"date":"2024-10-17","open_a":66.792113,"high_a":66.848858,"low_a":65.799172,"close_a":66.101784,"volume_a":14674600,"open_b":161.351912,"high_b":162.360473,"low_b":160.722708,"close_b":161.620239,"volume_b":4125900,"open_bench":572.693746,"close_bench":569.21405},{"date":"2024-10-18","open_a":66.073422,"high_a":66.716473,"low_a":65.827548,"close_a":66.61245,"volume_a":15087500,"open_b":159.427317,"high_b":162.230942,"low_b":159.260757,"close_b":161.98111,"volume_b":5097000,"open_bench":570.895275,"close_bench":571.403564},{"date":"2024-10-21","open_a":66.196356,"high_a":66.508426,"low_a":65.543847,"close_a":65.676239,"volume_a":11981200,"open_b":161.425917,"high_b":164.238789,"low_b":161.324134,"close_b":161.934814,"volume_b":6436300,"open_bench":570.680219,"close_bench":570.46521},{"date":"2024-10-22","open_a":65.250694,"high_a":65.959941,"low_a":64.948082,"close_a":65.676239,"volume_a":18603300,"open_b":161.472176,"high_b":161.879305,"low_b":160.546887,"close_b":161.342636,"volume_b":3790500,"open_bench":567.943355,"close_bench":570.16217},{"date":"2024-10-23","open_a":63.349912,"high_a":64.966993,"low_a":62.962194,"close_a":64.314491,"volume_a":24655200,"open_b":160.722714,"high_b":160.852254,"low_b":159.584613,"close_b":160.028748,"volume_b":3800700,"open_bench":568.14859,"close_bench":564.952332},{"date":"2024-10-24","open_a":63.974043,"high_a":64.342851,"low_a":63.312075,"close_a":63.643063,"volume_a":17568800,"open_b":160.00099,"high_b":160.574665,"low_b":158.872136,"close_b":159.297775,"volume_b":4244900,"open_bench":566.897554,"close_bench":566.174255},{"date":"2024-10-25","open_a":63.425565,"high_a":64.02133,"low_a":63.160781,"close_a":63.283714,"volume_a":11138100,"open_b":159.584587,"high_b":161.000278,"low_b":158.77959,"close_b":158.955383,"volume_b":4442600,"open_bench":568.392953,"close_bench":565.978638},{"date":"2024-10-28","open_a":63.321538,"high_a":63.737631,"low_a":62.981099,"close_a":63.047295,"volume_a":10761400,"open_b":160.056484,"high_b":160.056484,"low_b":157.049295,"close_b":157.151077,"volume_b":5186400,"open_bench":569.438797,"close_bench":567.728271},{"date":"2024-10-29","open_a":62.687945,"high_a":62.735224,"low_a":61.959781,"close_a":61.997608,"volume_a":16525900,"open_b":156.558889,"high_b":156.558889,"low_b":154.495499,"close_b":154.985901,"volume_b":5127900,"open_bench":566.770483,"close_bench":568.647217},{"date":"2024-10-30","open_a":61.950322,"high_a":62.924352,"low_a":61.770644,"close_a":62.338039,"volume_a":14177800,"open_b":154.523285,"high_b":155.115469,"low_b":153.672021,"close_b":153.792313,"volume_b":4801500,"open_bench":568.177976,"close_bench":566.92688},{"date":"2024-10-31","open_a":62.234027,"high_a":62.404247,"low_a":61.713917,"close_a":61.761196,"volume_a":13383700,"open_b":153.690537,"high_b":154.985937,"low_b":153.422196,"close_b":153.672028,"volume_b":5582400,"open_bench":562.577183,"close_bench":555.813293},{"date":"2024-11-01","open_a":61.912512,"high_a":62.09219,"low_a":61.364026,"close_a":61.477509,"volume_a":12163100,"open_b":153.681273,"high_b":153.829322,"low_b":152.867028,"close_b":153.218628,"volume_b":6339000,"open_bench":558.432892,"close_bench":558.15918},{"date":"2024-11-04","open_a":61.543695,"high_a":61.865216,"low_a":61.13706,"close_a":61.572063,"volume_a":10984500,"open_b":153.218631,"high_b":154.199435,"low_b":152.848521,"close_b":153.912598,"volume_b":4885500,"open_bench":558.296002,"close_bench":556.956909},{"date":"2024-11-05","open_a":61.127603,"high_a":61.912505,"low_a":61.127603,"close_a":61.81794,"volume_a":8780200,"open_b":153.551711,"high_b":155.411551,"low_b":153.292631,"close_b":155.309769,"volume_b":3678300,"open_bench":557.865945,"close_bench":563.691528},{"date":"2024-11-06","open_a":60.919556,"high_a":61.33565,"low_a":59.746937,"close_a":60.238682,"volume_a":22899300,"open_b":154.865662,"high_b":154.902682,"low_b":151.377332,"close_b":152.404404,"volume_b":7849300,"open_bench":575.909546,"close_bench":577.708008},{"date":"2024-11-07","open_a":60.79662,"high_a":60.891184,"low_a":60.14411,"close_a":60.200851,"volume_a":14856100,"open_b":152.210051,"high_b":152.580161,"low_b":151.257004,"close_b":151.747406,"volume_b":5333500,"open_bench":579.701966,"close_bench":582.174866},{"date":"2024-11-08","open_a":60.437269,"high_a":60.749336,"low_a":60.276508,"close_a":60.446724,"volume_a":14719300,"open_b":151.599354,"high_b":153.431437,"low_b":151.423547,"close_b":152.774475,"volume_b":4618300,"open_bench":582.722323,"close_bench":584.696777},{"date":"2024-11-11","open_a":60.512932,"high_a":60.635869,"low_a":59.81314,"close_a":59.917164,"volume_a":13980900,"open_b":152.126793,"high_b":153.736787,"low_b":151.728911,"close_b":151.987991,"volume_b":3762000,"open_bench":586.28021,"close_bench":585.253906},{"date":"2024-11-12","open_a":60.219767,"high_a":60.285964,"low_a":59.690195,"close_a":59.76585,"volume_a":15021000,"open_b":151.830674,"high_b":152.672691,"low_b":150.9609,"close_b":152.061996,"volume_b":5386600,"open_bench":585.17567,"close_bench":583.435852},{"date":"2024-11-13","open_a":59.623992,"high_a":59.82258,"low_a":59.094424,"close_a":59.57671,"volume_a":13177600,"open_b":151.275502,"high_b":152.487629,"low_b":150.239169,"close_b":152.432114,"volume_b":4145000,"open_bench":583.895232,"close_bench":583.719299},{"date":"2024-11-14","open_a":59.680745,"high_a":59.680745,"low_a":59.037694,"close_a":59.151173,"volume_a":15910800,"open_b":152.432135,"high_b":154.421501,"low_b":151.590119,"close_b":152.811493,"volume_b":5422700,"open_bench":583.846362,"close_bench":579.965881},{"date":"2024-11-15","open_a":59.094432,"high_a":59.226824,"low_a":58.054201,"close_a":58.385185,"volume_a":26284300,"open_b":152.099022,"high_b":152.099022,"low_b":146.241941,"close_b":146.769348,"volume_b":9108900,"open_bench":576.417836,"close_bench":572.537415},{"date":"2024-11-18","open_a":58.489206,"high_a":58.763449,"low_a":58.214967,"close_a":58.498665,"volume_a":16062100,"open_b":147.500336,"high_b":147.907466,"low_b":145.973601,"close_b":146.501022,"volume_b":6991300,"open_bench":572.996661,"close_bench":574.883179},{"date":"2024-11-19","open_a":58.536494,"high_a":59.217368,"low_a":58.281164,"close_a":59.188999,"volume_a":16254800,"open_b":145.964353,"high_b":146.075384,"low_b":144.20631,"close_b":145.011307,"volume_b":8248900,"open_bench":571.520806,"close_bench":576.98468},{"date":"2024-11-20","open_a":59.293022,"high_a":59.586175,"low_a":58.649971,"close_a":59.567265,"volume_a":18904900,"open_b":144.761482,"high_b":147.222754,"low_b":144.530159,"close_b":146.880402,"volume_b":5067300,"open_bench":577.062948,"close_bench":577.180237},{"date":"2024-11-21","open_a":59.718567,"high_a":60.342704,"low_a":59.529434,"close_a":60.295418,"volume_a":17292300,"open_b":146.556544,"high_b":148.508905,"low_b":146.149415,"close_b":148.360855,"volume_b":5977000,"open_bench":580.014817,"close_bench":580.278687},{"date":"2024-11-22","open_a":60.749336,"high_a":60.9763,"low_a":60.191398,"close_a":60.446724,"volume_a":19562200,"open_b":148.879012,"high_b":149.95235,"low_b":147.944475,"close_b":149.896835,"volume_b":5846300,"open_bench":580.268965,"close_bench":582.077271},{"date":"2024-11-25","open_a":60.824996,"high_a":61.193803,"low_a":60.371078,"close_a":60.881733,"volume_a":36805300,"open_b":150.544521,"high_b":151.368028,"low_b":149.896821,"close_b":150.868378,"volume_b":8580400,"open_bench":585.996799,"close_bench":584.051697},{"date":"2024-11-26","open_a":60.815539,"high_a":61.127609,"low_a":60.579124,"close_a":61.042503,"volume_a":14473300,"open_b":151.071932,"high_b":151.45129,"low_b":149.804277,"close_b":150.044861,"volume_b":4653600,"open_bench":585.292952,"close_bench":587.101257},{"date":"2024-11-27","open_a":61.43022,"high_a":61.562612,"low_a":60.7588,"close_a":60.92902,"volume_a":13263400,"open_b":149.841325,"high_b":152.793,"low_b":149.600741,"close_b":150.563049,"volume_b":3622900,"open_bench":586.915564,"close_bench":585.322327},{"date":"2024-11-29","open_a":61.000476,"high_a":61.219632,"low_a":60.762271,"close_a":61.057652,"volume_a":10648400,"open_b":149.369407,"high_b":151.368035,"low_b":148.925272,"close_b":151.238495,"volume_b":2799300,"open_bench":586.13361,"close_bench":588.958435},{"date":"2024-12-02","open_a":61.029065,"high_a":61.162462,"low_a":60.362077,"close_a":60.64793,"volume_a":16500100,"open_b":151.11821,"high_b":151.423543,"low_b":149.674747,"close_b":150.868378,"volume_b":4967400,"open_bench":589.368892,"close_bench":590.014038},{"date":"2024-12-03","open_a":60.714627,"high_a":61.057648,"low_a":60.057169,"close_a":60.524059,"volume_a":24874500,"open_b":150.674076,"high_b":150.868393,"low_b":149.360168,"close_b":149.619247,"volume_b":4277400,"open_bench":589.779429,"close_bench":590.287659},{"date":"2024-12-04","open_a":60.295377,"high_a":60.362076,"low_a":58.980468,"close_a":59.275845,"volume_a":28137700,"open_b":148.601448,"high_b":148.999316,"low_b":147.500352,"close_b":148.212814,"volume_b":6964600,"open_bench":591.968946,"close_bench":593.953125},{"date":"2024-12-05","open_a":59.523586,"high_a":60.142926,"low_a":59.171034,"close_a":59.952358,"volume_a":21040400,"open_b":148.906764,"high_b":148.980789,"low_b":146.963666,"close_b":148.499649,"volume_b":5229600,"open_bench":593.95309,"close_bench":592.975647},{"date":"2024-12-06","open_a":59.866607,"high_a":60.371608,"low_a":59.561696,"close_a":59.580753,"volume_a":15581100,"open_b":148.700276,"high_b":150.053366,"low_b":147.151214,"close_b":147.244522,"volume_b":8359200,"open_bench":593.738082,"close_bench":594.099731},{"date":"2024-12-09","open_a":59.733209,"high_a":59.961891,"low_a":58.980472,"close_a":59.647453,"volume_a":22348300,"open_b":147.356486,"high_b":150.706554,"low_b":146.637943,"close_b":148.812225,"volume_b":8380400,"open_bench":593.982519,"close_bench":591.040405},{"date":"2024-12-10","open_a":59.742735,"high_a":60.066699,"low_a":58.875655,"close_a":59.942829,"volume_a":14949400,"open_b":148.373659,"high_b":149.698755,"low_b":146.833938,"close_b":149.157516,"volume_b":5329400,"open_bench":591.714795,"close_bench":589.202759},{"date":"2024-12-11","open_a":60.028591,"high_a":60.628878,"low_a":59.456891,"close_a":59.68557,"volume_a":32819000,"open_b":149.502809,"high_b":149.782748,"low_b":145.816785,"close_b":146.292709,"volume_b":7925900,"open_bench":592.115532,"close_bench":593.757629},{"date":"2024-12-12","open_a":60.552649,"high_a":60.905197,"low_a":60.238215,"close_a":60.828972,"volume_a":17542100,"open_b":148.457639,"high_b":148.746918,"low_b":147.412495,"close_b":148.280334,"volume_b":6849500,"open_bench":592.897489,"close_bench":590.698242},{"date":"2024-12-13","open_a":60.571702,"high_a":60.828967,"low_a":60.133399,"close_a":60.142925,"volume_a":13429000,"open_b":147.00191,"high_b":148.90558,"low_b":146.56332,"close_b":147.412506,"volume_b":4353900,"open_bench":592.721597,"close_bench":590.580994},{"date":"2024-12-16","open_a":60.247739,"high_a":60.848026,"low_a":59.409244,"close_a":59.599812,"volume_a":24559700,"open_b":147.300518,"high_b":148.205691,"low_b":145.462176,"close_b":145.592819,"volume_b":6696500,"open_bench":592.330624,"close_bench":593.102783},{"date":"2024-12-17","open_a":59.504528,"high_a":60.552644,"low_a":59.380658,"close_a":60.409721,"volume_a":21902400,"open_b":145.331557,"high_b":147.907095,"low_b":144.864973,"close_b":145.704819,"volume_b":6572600,"open_bench":590.561337,"close_bench":590.659058},{"date":"2024-12-18","open_a":59.847555,"high_a":60.638406,"low_a":59.828498,"close_a":59.885666,"volume_a":20946100,"open_b":144.669007,"high_b":145.415544,"low_b":143.810494,"close_b":144.1091,"volume_b":7620900,"open_bench":590.356242,"close_bench":573.055542},{"date":"2024-12-19","open_a":59.742733,"high_a":60.190563,"low_a":59.456883,"close_a":59.504524,"volume_a":21300600,"open_b":142.858636,"high_b":143.782477,"low_b":141.290921,"close_b":141.346909,"volume_b":9829500,"open_bench":578.020755,"close_bench":572.879395},{"date":"2024-12-20","open_a":59.523587,"high_a":60.085759,"low_a":59.342546,"close_a":59.599812,"volume_a":42458100,"open_b":140.777698,"high_b":143.231934,"low_b":140.451083,"close_b":142.578705,"volume_b":19205200,"open_bench":570.561035,"close_bench":579.760315},{"date":"2024-12-23","open_a":59.552167,"high_a":59.799903,"low_a":58.627916,"close_a":59.437828,"volume_a":13010800,"open_b":142.224097,"high_b":142.224097,"low_b":139.704547,"close_b":141.17894,"volume_b":6822200,"open_bench":579.505283,"close_bench":583.232056},{"date":"2024-12-24","open_a":59.266323,"high_a":59.876137,"low_a":59.085282,"close_a":59.876137,"volume_a":5019100,"open_b":140.441751,"high_b":142.616034,"low_b":140.357769,"close_b":142.597366,"volume_b":3399800,"open_bench":584.575691,"close_bench":589.714722},{"date":"2024-12-26","open_a":59.666507,"high_a":59.78085,"low_a":59.456886,"close_a":59.618866,"volume_a":7943800,"open_b":142.336057,"high_b":142.94261,"low_b":142.065445,"close_b":142.252075,"volume_b":5528500,"open_bench":587.949392,"close_bench":589.753967},{"date":"2024-12-27","open_a":59.504524,"high_a":59.980942,"low_a":59.275842,"close_a":59.504524,"volume_a":8542800,"open_b":141.178953,"high_b":143.213267,"low_b":141.150959,"close_b":142.672028,"volume_b":4287000,"open_bench":586.027122,"close_bench":583.545898},{"date":"2024-12-30","open_a":59.399716,"high_a":59.399716,"low_a":58.770845,"close_a":59.104336,"volume_a":8972200,"open_b":142.457395,"high_b":142.858651,"low_b":140.992315,"close_b":141.580215,"volume_b":5397000,"open_bench":576.563059,"close_bench":576.886658},{"date":"2024-12-31","open_a":59.094812,"high_a":59.485474,"low_a":58.885187,"close_a":59.32349,"volume_a":9256700,"open_b":141.654858,"high_b":142.914626,"low_b":140.852333,"close_b":141.897476,"volume_b":4274800,"open_bench":578.544251,"close_bench":574.788086},{"date":"2025-01-02","open_a":59.409242,"high_a":59.790377,"low_a":58.713673,"close_a":58.923298,"volume_a":12991000,"open_b":142.662683,"high_b":143.390553,"low_b":139.872507,"close_b":140.171127,"volume_b":4970200,"open_bench":578.034214,"close_bench":573.375732},{"date":"2025-01-03","open_a":58.980466,"high_a":59.151977,"low_a":58.71367,"close_a":58.83754,"volume_a":10403200,"open_b":140.871006,"high_b":140.871006,"low_b":139.564563,"close_b":139.648544,"volume_b":5218100,"open_bench":576.210038,"close_bench":580.544861},{"date":"2025-01-06","open_a":58.599338,"high_a":58.666036,"low_a":57.798956,"close_a":57.941883,"volume_a":17924200,"open_b":139.601896,"high_b":139.928496,"low_b":136.335809,"close_b":136.494446,"volume_b":9470600,"open_bench":584.7816,"close_bench":583.889099},{"date":"2025-01-07","open_a":58.227732,"high_a":58.808962,"low_a":57.760842,"close_a":57.970467,"volume_a":17799600,"open_b":136.671741,"high_b":138.080819,"low_b":135.626584,"close_b":135.682571,"volume_b":7473300,"open_bench":585.909439,"close_bench":577.288818},{"date":"2025-01-08","open_a":58.113392,"high_a":58.856602,"low_a":58.056221,"close_a":58.799431,"volume_a":14412400,"open_b":135.682573,"high_b":136.8677,"low_b":134.898716,"close_b":136.746384,"volume_b":6791800,"open_bench":577.357506,"close_bench":578.132263},{"date":"2025-01-10","open_a":58.675561,"high_a":58.742259,"low_a":57.865649,"close_a":58.189613,"volume_a":22425000,"open_b":135.309317,"high_b":135.841229,"low_b":132.052556,"close_b":133.10704,"volume_b":9632700,"open_bench":574.591767,"close_bench":569.305603},{"date":"2025-01-13","open_a":58.246789,"high_a":58.932828,"low_a":57.846597,"close_a":58.742264,"volume_a":18016200,"open_b":133.247005,"high_b":135.439954,"low_b":132.640451,"close_b":134.842728,"volume_b":8101400,"open_bench":564.67658,"close_bench":570.188293},{"date":"2025-01-14","open_a":58.75179,"high_a":59.142453,"low_a":58.408769,"close_a":59.113869,"volume_a":16658300,"open_b":134.553462,"high_b":135.439969,"low_b":133.965562,"close_b":135.262665,"volume_b":6996400,"open_bench":573.101074,"close_bench":570.9729},{"date":"2025-01-15","open_a":59.39019,"high_a":59.66651,"low_a":58.770846,"close_a":58.856602,"volume_a":16187200,"open_b":135.766575,"high_b":136.429116,"low_b":134.534786,"close_b":135.104019,"volume_b":7141600,"open_bench":578.956079,"close_bench":581.358887},{"date":"2025-01-16","open_a":58.523106,"high_a":59.361602,"low_a":58.427824,"close_a":59.313961,"volume_a":12585500,"open_b":134.422805,"high_b":136.979676,"low_b":134.086864,"close_b":136.746384,"volume_b":5068000,"open_bench":582.722068,"close_bench":580.240845},{"date":"2025-01-17","open_a":59.37113,"high_a":59.809434,"low_a":59.237731,"close_a":59.752262,"volume_a":17948900,"open_b":137.847555,"high_b":138.6874,"low_b":137.33431,"close_b":138.342133,"volume_b":7338900,"open_bench":585.458357,"close_bench":586.066406},{"date":"2025-01-21","open_a":59.933305,"high_a":60.247738,"low_a":59.361602,"close_a":59.475941,"volume_a":19817100,"open_b":138.416772,"high_b":139.769863,"low_b":137.894201,"close_b":138.62207,"volume_b":7456200,"open_bench":589.09682,"close_bench":591.430969},{"date":"2025-01-22","open_a":59.190091,"high_a":59.352075,"low_a":58.751788,"close_a":58.866127,"volume_a":18191900,"open_b":138.323453,"high_b":139.461919,"low_b":137.912857,"close_b":138.19281,"volume_b":5905100,"open_bench":594.245737,"close_bench":594.755737},{"date":"2025-01-23","open_a":58.88518,"high_a":59.056691,"low_a":58.475461,"close_a":58.627914,"volume_a":16528400,"open_b":138.566076,"high_b":138.985998,"low_b":136.811716,"close_b":138.687378,"volume_b":4759100,"open_bench":594.127985,"close_bench":598.001892},{"date":"2025-01-24","open_a":58.808963,"high_a":59.085282,"low_a":58.542167,"close_a":58.999527,"volume_a":19244600,"open_b":138.874022,"high_b":139.331265,"low_b":138.230134,"close_b":139.153961,"volume_b":4509600,"open_bench":598.060739,"close_bench":596.256165},{"date":"2025-01-27","open_a":59.904717,"high_a":61.152932,"low_a":59.523585,"close_a":60.857552,"volume_a":20742900,"open_b":140.833678,"high_b":145.079594,"low_b":140.81501,"close_b":144.277069,"volume_b":9820600,"open_bench":583.34976,"close_bench":587.821899},{"date":"2025-01-28","open_a":60.514528,"high_a":61.019532,"low_a":59.342541,"close_a":59.41877,"volume_a":14956000,"open_b":143.847783,"high_b":144.277046,"low_b":140.087119,"close_b":140.208435,"volume_b":6349700,"open_bench":589.047798,"close_bench":592.872681},{"date":"2025-01-29","open_a":59.609342,"high_a":60.371609,"low_a":59.599811,"close_a":59.866608,"volume_a":11325600,"open_b":141.029651,"high_b":141.841503,"low_b":140.189792,"close_b":140.320435,"volume_b":4874100,"open_bench":592.088017,"close_bench":590.214844},{"date":"2025-01-30","open_a":60.838493,"high_a":61.257742,"low_a":60.247736,"close_a":61.029064,"volume_a":13201000,"open_b":141.169625,"high_b":142.84933,"low_b":140.563071,"close_b":141.748184,"volume_b":4671300,"open_bench":592.323481,"close_bench":593.382629},{"date":"2025-01-31","open_a":60.695571,"high_a":60.914723,"low_a":60.400191,"close_a":60.485947,"volume_a":12096800,"open_b":140.684363,"high_b":141.552217,"low_b":140.22712,"close_b":140.619049,"volume_b":4905500,"open_bench":595.795226,"close_bench":590.22467},{"date":"2025-02-03","open_a":60.200099,"high_a":60.628876,"low_a":59.485473,"close_a":60.36208,"volume_a":12067900,"open_b":139.751172,"high_b":141.029621,"low_b":138.883333,"close_b":140.227097,"volume_b":7196200,"open_bench":581.251033,"close_bench":586.252808},{"date":"2025-02-04","open_a":60.609816,"high_a":60.628874,"low_a":59.409243,"close_a":59.714149,"volume_a":15678000,"open_b":136.335806,"high_b":137.875527,"low_b":133.442981,"close_b":133.900238,"volume_b":12804900,"open_bench":586.311579,"close_bench":590.185486},{"date":"2025-02-05","open_a":59.761794,"high_a":60.304909,"low_a":59.637924,"close_a":60.142925,"volume_a":10686800,"open_b":133.806922,"high_b":136.046519,"low_b":133.163034,"close_b":135.925217,"volume_b":9248100,"open_bench":589.067389,"close_bench":592.578369},{"date":"2025-02-06","open_a":60.619344,"high_a":60.733687,"low_a":59.942832,"close_a":60.371609,"volume_a":13423400,"open_b":136.485097,"high_b":137.082324,"low_b":134.338823,"close_b":135.635925,"volume_b":5604900,"open_bench":594.314403,"close_bench":594.638062},{"date":"2025-02-07","open_a":60.43831,"high_a":60.905197,"low_a":60.285856,"close_a":60.828972,"volume_a":13184700,"open_b":135.318629,"high_b":135.449272,"low_b":134.282812,"close_b":134.917374,"volume_b":5872300,"open_bench":595.197027,"close_bench":589.194946},{"date":"2025-02-10","open_a":61.076706,"high_a":61.600768,"low_a":60.65746,"close_a":61.505486,"volume_a":16154300,"open_b":134.702759,"high_b":134.861396,"low_b":133.452303,"close_b":134.702759,"volume_b":8715400,"open_bench":592.392141,"close_bench":593.196289},{"date":"2025-02-11","open_a":64.316339,"high_a":64.478319,"low_a":63.277753,"close_a":64.411621,"volume_a":30589100,"open_b":134.721413,"high_b":135.981195,"low_b":133.816239,"close_b":135.90654,"volume_b":6354000,"open_bench":590.940628,"close_bench":593.647461},{"date":"2025-02-12","open_a":63.716049,"high_a":65.574074,"low_a":63.716049,"close_a":65.469261,"volume_a":26033800,"open_b":134.413468,"high_b":135.104018,"low_b":133.470974,"close_b":134.366821,"volume_b":6658200,"open_bench":587.655163,"close_bench":591.734985},{"date":"2025-02-13","open_a":65.74559,"high_a":66.32682,"low_a":65.516912,"close_a":66.222008,"volume_a":24393100,"open_b":134.590759,"high_b":135.253315,"low_b":134.002873,"close_b":134.917374,"volume_b":7175300,"open_bench":592.833452,"close_bench":597.9823},{"date":"2025-02-14","open_a":66.088614,"high_a":66.727011,"low_a":65.516915,"close_a":65.621727,"volume_a":20779100,"open_b":135.001353,"high_b":136.401105,"low_b":133.750912,"close_b":133.8069,"volume_b":5001800,"open_bench":598.188186,"close_bench":597.95282},{"date":"2025-02-18","open_a":65.316818,"high_a":66.098143,"low_a":65.192945,"close_a":65.793236,"volume_a":15569100,"open_b":133.349655,"high_b":134.516114,"low_b":132.136548,"close_b":134.469467,"volume_b":8738700,"open_bench":599.110141,"close_bench":599.708374},{"date":"2025-02-19","open_a":65.945683,"high_a":66.822289,"low_a":65.907569,"close_a":66.765121,"volume_a":16032300,"open_b":134.590779,"high_b":136.177161,"low_b":134.516124,"close_b":136.065186,"volume_b":5777300,"open_bench":598.325541,"close_bench":601.120605},{"date":"2025-02-20","open_a":66.422112,"high_a":66.765133,"low_a":66.145789,"close_a":66.736549,"volume_a":13398000,"open_b":135.411956,"high_b":139.219281,"low_b":135.309307,"close_b":139.1353,"volume_b":6789800,"open_bench":599.757497,"close_bench":598.619873},{"date":"2025-02-21","open_a":67.003338,"high_a":68.222965,"low_a":66.822297,"close_a":67.984756,"volume_a":21662300,"open_b":139.965828,"high_b":144.22107,"low_b":138.948664,"close_b":143.241241,"volume_b":10604400,"open_bench":598.404044,"close_bench":588.380981},{"date":"2025-02-24","open_a":67.498806,"high_a":68.346835,"low_a":67.012865,"close_a":67.260597,"volume_a":21418400,"open_b":142.158748,"high_b":145.872766,"low_b":141.95345,"close_b":143.567841,"volume_b":7187000,"open_bench":590.420752,"close_bench":585.70343},{"date":"2025-02-25","open_a":67.517873,"high_a":68.384949,"low_a":67.394007,"close_a":68.118156,"volume_a":21555600,"open_b":144.445012,"high_b":147.86975,"low_b":143.707816,"close_b":145.96608,"volume_b":9474600,"open_bench":585.644615,"close_bench":582.790649},{"date":"2025-02-26","open_a":67.908527,"high_a":67.918058,"low_a":67.203432,"close_a":67.460701,"volume_a":14531200,"open_b":145.172904,"high_b":145.210225,"low_b":141.038964,"close_b":141.384232,"volume_b":9076500,"open_bench":584.448194,"close_bench":583.084961},{"date":"2025-02-27","open_a":67.384478,"high_a":68.165802,"low_a":67.098625,"close_a":67.527405,"volume_a":16789500,"open_b":140.814981,"high_b":142.485359,"low_b":139.909807,"close_b":141.860138,"volume_b":6731000,"open_bench":585.35047,"close_bench":573.777832},{"date":"2025-02-28","open_a":68.080027,"high_a":68.19437,"low_a":67.232005,"close_a":67.851349,"volume_a":22167400,"open_b":143.427866,"high_b":144.734309,"low_b":142.168098,"close_b":143.213242,"volume_b":8455500,"open_bench":574.27802,"close_bench":582.731934},{"date":"2025-03-03","open_a":66.927114,"high_a":68.947126,"low_a":66.441173,"close_a":68.909012,"volume_a":21717600,"open_b":141.234922,"high_b":145.714115,"low_b":140.441724,"close_b":145.564819,"volume_b":8321700,"open_bench":584.69333,"close_bench":572.522461},{"date":"2025-03-04","open_a":68.670782,"high_a":69.223428,"low_a":66.584072,"close_a":66.879456,"volume_a":25411900,"open_b":146.824589,"high_b":149.446788,"low_b":143.670492,"close_b":143.885117,"volume_b":10274200,"open_bench":568.54073,"close_bench":565.745605},{"date":"2025-03-05","open_a":66.603155,"high_a":67.403533,"low_a":66.54598,"close_a":66.774666,"volume_a":23933600,"open_b":142.70932,"high_b":144.958257,"low_b":142.522689,"close_b":143.59584,"volume_b":5578700,"open_bench":565.578847,"close_bench":571.826111},{"date":"2025-03-06","open_a":67.089097,"high_a":67.203432,"low_a":66.174369,"close_a":67.136734,"volume_a":21694000,"open_b":143.866473,"high_b":144.61301,"low_b":142.336078,"close_b":144.277069,"volume_b":8421000,"open_bench":564.392188,"close_bench":561.675598},{"date":"2025-03-07","open_a":66.364937,"high_a":68.432587,"low_a":66.364937,"close_a":68.060982,"volume_a":28393300,"open_b":144.272349,"high_b":148.358101,"low_b":143.820472,"close_b":145.392639,"volume_b":8997700,"open_bench":559.900492,"close_bench":564.82373},{"date":"2025-03-10","open_a":68.565988,"high_a":69.766562,"low_a":67.956174,"close_a":68.08004,"volume_a":25251200,"open_b":146.748268,"high_b":150.702221,"low_b":146.268136,"close_b":147.096588,"volume_b":11156900,"open_bench":556.654121,"close_bench":549.779175},{"date":"2025-03-11","open_a":68.48977,"high_a":68.642227,"low_a":67.279673,"close_a":67.689392,"volume_a":28076400,"open_b":146.663534,"high_b":146.908312,"low_b":142.916693,"close_b":143.406235,"volume_b":7981600,"open_bench":548.622037,"close_bench":545.209045},{"date":"2025-03-12","open_a":67.05097,"high_a":67.393991,"low_a":66.450679,"close_a":66.641251,"volume_a":18923300,"open_b":138.896831,"high_b":140.770258,"low_b":137.927172,"close_b":139.489929,"volume_b":8330200,"open_bench":551.338641,"close_bench":548.102234},{"date":"2025-03-13","open_a":66.936633,"high_a":66.936633,"low_a":66.012382,"close_a":66.336349,"volume_a":15746600,"open_b":139.67822,"high_b":141.664613,"low_b":139.386383,"close_b":139.649979,"volume_b":5351100,"open_bench":547.729504,"close_bench":540.795715},{"date":"2025-03-14","open_a":65.760424,"high_a":66.480324,"low_a":65.635637,"close_a":66.384338,"volume_a":14205100,"open_b":138.510868,"high_b":140.421945,"low_b":138.463794,"close_b":139.88533,"volume_b":5277500,"open_bench":545.395386,"close_bench":551.966309},{"date":"2025-03-17","open_a":66.461137,"high_a":67.353813,"low_a":66.307562,"close_a":67.305824,"volume_a":15634800,"open_b":140.48785,"high_b":142.784913,"low_b":140.393717,"close_b":142.474243,"volume_b":5449500,"open_bench":551.946737,"close_bench":556.222778},{"date":"2025-03-18","open_a":67.296218,"high_a":67.373009,"low_a":66.518729,"close_a":66.595512,"volume_a":12657600,"open_b":141.212729,"high_b":142.436576,"low_b":140.035956,"close_b":140.261902,"volume_b":8158600,"open_bench":553.918016,"close_bench":550.210876},{"date":"2025-03-19","open_a":65.798829,"high_a":66.595512,"low_a":65.789228,"close_a":66.499527,"volume_a":14780000,"open_b":139.48053,"high_b":140.158352,"low_b":138.190777,"close_b":139.433456,"volume_b":6080100,"open_bench":551.985864,"close_bench":556.203003},{"date":"2025-03-20","open_a":66.528322,"high_a":67.008255,"low_a":66.31715,"close_a":66.91227,"volume_a":16295300,"open_b":138.915679,"high_b":139.395811,"low_b":138.049575,"close_b":138.529694,"volume_b":4536400,"open_bench":552.476369,"close_bench":554.594727},{"date":"2025-03-21","open_a":66.912271,"high_a":67.161838,"low_a":65.443677,"close_a":65.914009,"volume_a":47479500,"open_b":138.388492,"high_b":139.047482,"low_b":136.270301,"close_b":136.929291,"volume_b":11574800,"open_bench":550.154386,"close_bench":554.777649},{"date":"2025-03-24","open_a":66.038803,"high_a":66.307565,"low_a":65.587665,"close_a":66.182777,"volume_a":12575400,"open_b":136.966928,"high_b":138.124869,"low_b":136.345589,"close_b":137.870682,"volume_b":5968900,"open_bench":561.48632,"close_bench":564.71283},{"date":"2025-03-25","open_a":66.182774,"high_a":66.307561,"low_a":65.846822,"close_a":66.048393,"volume_a":12513900,"open_b":137.588273,"high_b":137.729495,"low_b":136.345609,"close_b":137.014008,"volume_b":4763300,"open_bench":565.912951,"close_bench":566.070374},{"date":"2025-03-26","open_a":66.153972,"high_a":67.526581,"low_a":66.134778,"close_a":67.209824,"volume_a":14478200,"open_b":137.447045,"high_b":140.581969,"low_b":137.211691,"close_b":139.932388,"volume_b":5882200,"open_bench":565.804724,"close_bench":559.312439},{"date":"2025-03-27","open_a":67.36341,"high_a":68.140899,"low_a":67.257824,"close_a":67.900932,"volume_a":11663400,"open_b":140.826754,"high_b":141.598724,"low_b":140.356045,"close_b":140.902069,"volume_b":5397500,"open_bench":557.925371,"close_bench":557.827026},{"date":"2025-03-28","open_a":67.96812,"high_a":68.457654,"low_a":66.739493,"close_a":67.545784,"volume_a":17412900,"open_b":141.626955,"high_b":142.51189,"low_b":138.492031,"close_b":140.525497,"volume_b":8550900,"open_bench":556.302268,"close_bench":546.593262},{"date":"2025-03-31","open_a":67.891333,"high_a":69.05277,"low_a":67.66096,"close_a":68.745613,"volume_a":24016900,"open_b":141.401029,"high_b":143.651018,"low_b":140.977393,"close_b":141.156265,"volume_b":9985700,"open_bench":540.858564,"close_bench":550.262573},{"date":"2025-04-01","open_a":68.803211,"high_a":69.023984,"low_a":68.332879,"close_a":68.985588,"volume_a":15309100,"open_b":141.674039,"high_b":141.956453,"low_b":139.960651,"close_b":140.902069,"volume_b":6072800,"open_bench":548.354248,"close_bench":551.816772},{"date":"2025-04-02","open_a":69.071968,"high_a":69.455916,"low_a":68.198493,"close_a":68.467255,"volume_a":14606500,"open_b":141.429267,"high_b":141.683454,"low_b":139.395815,"close_b":140.384293,"volume_b":5866200,"open_bench":545.993327,"close_bench":555.308838},{"date":"2025-04-03","open_a":69.571108,"high_a":70.982105,"low_a":69.417526,"close_a":70.243011,"volume_a":24959600,"open_b":142.803733,"high_b":145.02548,"low_b":142.022354,"close_b":142.502472,"volume_b":8216200,"open_bench":536.21556,"close_bench":527.94281},{"date":"2025-04-04","open_a":69.513508,"high_a":70.079833,"low_a":66.989063,"close_a":67.123444,"volume_a":26517300,"open_b":144.027569,"high_b":145.844499,"low_b":137.861277,"close_b":138.021317,"volume_b":12911400,"open_bench":515.125349,"close_bench":497.035431},{"date":"2025-04-07","open_a":63.571925,"high_a":66.595497,"low_a":63.399149,"close_a":65.626038,"volume_a":34258400,"open_b":133.041249,"high_b":137.541228,"low_b":132.429318,"close_b":134.801697,"volume_b":14423200,"open_bench":481.207996,"close_bench":496.150146},{"date":"2025-04-08","open_a":66.192378,"high_a":67.171446,"low_a":65.040541,"close_a":65.674049,"volume_a":22774400,"open_b":135.517149,"high_b":136.882203,"low_b":130.998339,"close_b":132.080978,"volume_b":9488900,"open_bench":513.344914,"close_bench":488.379059},{"date":"2025-04-09","open_a":65.194104,"high_a":67.641766,"low_a":64.579791,"close_a":67.142632,"volume_a":27918700,"open_b":132.033896,"high_b":137.399977,"low_b":130.226375,"close_b":137.061066,"volume_b":14067000,"open_bench":485.388642,"close_bench":539.668274},{"date":"2025-04-10","open_a":67.373004,"high_a":68.352065,"low_a":66.211567,"close_a":67.920128,"volume_a":22450200,"open_b":136.901044,"high_b":138.520284,"low_b":134.547498,"close_b":135.69603,"volume_b":10980300,"open_bench":523.486691,"close_bench":516.020569},{"date":"2025-04-11","open_a":68.064113,"high_a":68.966382,"low_a":67.70896,"close_a":68.56324,"volume_a":19988200,"open_b":136.646847,"high_b":136.778645,"low_b":133.794351,"close_b":135.969025,"volume_b":8563300,"open_bench":514.476193,"close_bench":525.227844},{"date":"2025-04-14","open_a":68.438457,"high_a":69.743876,"low_a":68.083304,"close_a":69.542297,"volume_a":13930200,"open_b":135.714852,"high_b":138.557925,"low_b":135.093513,"close_b":138.153122,"volume_b":5393800,"open_bench":535.172854,"close_bench":530.323303},{"date":"2025-04-15","open_a":69.705491,"high_a":69.715084,"low_a":68.84161,"close_a":68.97599,"volume_a":14965500,"open_b":137.296438,"high_b":137.428236,"low_b":134.274493,"close_b":134.472183,"volume_b":5459600,"open_bench":530.864276,"close_bench":528.837891},{"date":"2025-04-16","open_a":69.455909,"high_a":69.753465,"low_a":68.524838,"close_a":68.8032,"volume_a":15276200,"open_b":135.667775,"high_b":135.94078,"low_b":131.582024,"close_b":131.88327,"volume_b":7878600,"open_bench":523.004678,"close_bench":517.082886},{"date":"2025-04-17","open_a":69.014377,"high_a":70.482971,"low_a":68.841601,"close_a":70.070229,"volume_a":20754500,"open_b":132.118638,"high_b":135.159429,"low_b":131.864465,"close_b":134.472183,"volume_b":6131400,"open_bench":519.03065,"close_bench":517.820679},{"date":"2025-04-21","open_a":70.434976,"high_a":70.521367,"low_a":68.937587,"close_a":69.849457,"volume_a":16037200,"open_b":134.152096,"high_b":134.462766,"low_b":132.335166,"close_b":133.4272,"volume_b":7412800,"open_bench":512.6563,"close_bench":505.495117},{"date":"2025-04-22","open_a":70.070232,"high_a":71.394844,"low_a":69.99344,"close_a":70.934113,"volume_a":15613500,"open_b":133.944997,"high_b":135.827833,"low_b":133.634327,"close_b":135.055878,"volume_b":7144700,"open_bench":511.652999,"close_bench":518.646973},{"date":"2025-04-23","open_a":70.348607,"high_a":70.982115,"low_a":69.475125,"close_a":70.358208,"volume_a":16353900,"open_b":134.877004,"high_b":135.573645,"low_b":132.598774,"close_b":133.926163,"volume_b":7760000,"open_bench":531.611917,"close_bench":526.683655},{"date":"2025-04-24","open_a":70.08943,"high_a":70.482979,"low_a":69.398333,"close_a":69.609497,"volume_a":16893900,"open_b":133.163616,"high_b":133.540192,"low_b":126.649003,"close_b":127.383308,"volume_b":15577300,"open_bench":527.962423,"close_bench":537.769775},{"date":"2025-04-25","open_a":69.734286,"high_a":69.993446,"low_a":68.275285,"close_a":69.023987,"volume_a":16316000,"open_b":127.533929,"high_b":127.665727,"low_b":124.078926,"close_b":125.566368,"volume_b":11544700,"open_bench":537.730448,"close_bench":541.655334},{"date":"2025-04-28","open_a":69.18716,"high_a":69.359937,"low_a":68.448059,"close_a":68.908798,"volume_a":15997500,"open_b":126.055908,"high_b":127.477459,"low_b":125.302785,"close_b":125.92411,"volume_b":8498600,"open_bench":542.393105,"close_bench":541.861877},{"date":"2025-04-29","open_a":68.332871,"high_a":69.791864,"low_a":67.459389,"close_a":69.446312,"volume_a":16244100,"open_b":124.71909,"high_b":126.583108,"low_b":123.815336,"close_b":126.441887,"volume_b":7010700,"open_bench":539.953569,"close_bench":545.27533},{"date":"2025-04-30","open_a":70.406179,"high_a":70.530966,"low_a":68.812805,"close_a":69.63829,"volume_a":21074700,"open_b":127.684563,"high_b":128.202332,"low_b":125.302776,"close_b":127.637489,"volume_b":7923500,"open_bench":538.635456,"close_bench":545.491699},{"date":"2025-05-01","open_a":69.110368,"high_a":69.148763,"low_a":68.371274,"close_a":68.428864,"volume_a":18034200,"open_b":126.592498,"high_b":126.912576,"low_b":125.434557,"close_b":125.726395,"volume_b":6807000,"open_bench":551.226505,"close_bench":549.357483},{"date":"2025-05-02","open_a":68.63044,"high_a":68.937597,"low_a":68.246492,"close_a":68.774422,"volume_a":12629600,"open_b":126.291234,"high_b":126.526589,"low_b":125.349816,"close_b":125.914673,"volume_b":7169700,"open_bench":555.515362,"close_bench":557.512268},{"date":"2025-05-05","open_a":68.793608,"high_a":68.918395,"low_a":68.1601,"close_a":68.822403,"volume_a":11932700,"open_b":126.187693,"high_b":126.187693,"low_b":123.984778,"close_b":124.257797,"volume_b":8972200,"open_bench":553.390704,"close_bench":554.315369},{"date":"2025-05-06","open_a":68.803211,"high_a":69.014382,"low_a":68.304083,"close_a":68.841606,"volume_a":12705500,"open_b":124.173066,"high_b":124.173066,"low_b":122.535007,"close_b":123.081032,"volume_b":6889900,"open_bench":548.826329,"close_bench":549.682129},{"date":"2025-05-07","open_a":68.841607,"high_a":69.772678,"low_a":68.793611,"close_a":69.494316,"volume_a":15018200,"open_b":123.278725,"high_b":124.897966,"low_b":122.638568,"close_b":124.191902,"volume_b":6313200,"open_bench":551.010152,"close_bench":551.993835},{"date":"2025-05-08","open_a":69.302336,"high_a":69.561504,"low_a":68.275279,"close_a":68.313675,"volume_a":14371100,"open_b":124.163657,"high_b":125.1898,"low_b":123.570559,"close_b":123.730598,"volume_b":6392400,"open_bench":556.017083,"close_bench":555.840027},{"date":"2025-05-09","open_a":68.140897,"high_a":68.294479,"low_a":67.632169,"close_a":67.689758,"volume_a":11228900,"open_b":123.467006,"high_b":123.937715,"low_b":122.553829,"close_b":122.798607,"volume_b":4770500,"open_bench":557.23675,"close_bench":555.131714},{"date":"2025-05-12","open_a":67.334622,"high_a":67.507398,"low_a":65.942811,"close_a":66.739502,"volume_a":21539000,"open_b":123.175155,"high_b":124.577873,"low_b":122.440849,"close_b":123.965942,"volume_b":8392700,"open_bench":571.982199,"close_bench":573.477417},{"date":"2025-05-13","open_a":66.797081,"high_a":66.893073,"low_a":66.105983,"close_a":66.192368,"volume_a":14389100,"open_b":123.485836,"high_b":123.928304,"low_b":122.149024,"close_b":122.676216,"volume_b":8650200,"open_bench":573.890623,"close_bench":577.264709},{"date":"2025-05-14","open_a":66.11557,"high_a":66.557107,"low_a":65.827614,"close_a":66.384338,"volume_a":16286800,"open_b":122.450278,"high_b":122.544411,"low_b":120.379154,"close_b":120.925171,"volume_b":10463000,"open_bench":578.218823,"close_bench":578.002441},{"date":"2025-05-15","open_a":66.902667,"high_a":68.851202,"low_a":66.806682,"close_a":68.736015,"volume_a":19817100,"open_b":121.621842,"high_b":123.84359,"low_b":121.264099,"close_b":123.796516,"volume_b":9165100,"open_bench":576.00549,"close_bench":580.825562},{"date":"2025-05-16","open_a":68.975971,"high_a":69.359919,"low_a":68.563228,"close_a":69.110352,"volume_a":18441200,"open_b":124.351932,"high_b":124.511971,"low_b":122.996287,"close_b":124.248375,"volume_b":8896400,"open_bench":581.6027,"close_bench":584.504578},{"date":"2025-05-19","open_a":69.139152,"high_a":69.273533,"low_a":68.764805,"close_a":69.043167,"volume_a":13282800,"open_b":124.389605,"high_b":124.511979,"low_b":123.429354,"close_b":124.069511,"volume_b":9406200,"open_bench":578.503999,"close_bench":585.14386},{"date":"2025-05-20","open_a":68.918391,"high_a":69.235141,"low_a":68.563238,"close_a":68.812805,"volume_a":12137800,"open_b":124.03186,"high_b":124.427254,"low_b":123.655298,"close_b":124.078934,"volume_b":7624200,"open_bench":583.412709,"close_bench":583.176575},{"date":"2025-05-21","open_a":68.81281,"high_a":69.187157,"low_a":68.707224,"close_a":68.966385,"volume_a":12094400,"open_b":124.078938,"high_b":124.173071,"low_b":122.440865,"close_b":122.525589,"volume_b":9243700,"open_bench":578.838517,"close_bench":573.349548},{"date":"2025-05-22","open_a":68.956777,"high_a":69.100759,"low_a":68.112098,"close_a":68.284874,"volume_a":11101800,"open_b":122.403204,"high_b":123.043362,"low_b":121.273505,"close_b":122.497337,"volume_b":7140200,"open_bench":573.15283,"close_bench":573.575867},{"date":"2025-05-23","open_a":68.66883,"high_a":69.091173,"low_a":67.852945,"close_a":68.889595,"volume_a":12822100,"open_b":121.970144,"high_b":122.158425,"low_b":120.266179,"close_b":121.763031,"volume_b":8122500,"open_bench":566.581833,"close_bench":569.660767},{"date":"2025-05-27","open_a":68.860797,"high_a":69.321536,"low_a":68.688021,"close_a":68.899193,"volume_a":10666100,"open_b":122.459695,"high_b":123.758857,"low_b":121.56535,"close_b":123.674118,"volume_b":13974800,"open_bench":576.507267,"close_bench":581.504395},{"date":"2025-05-28","open_a":68.822394,"high_a":69.014371,"low_a":68.198487,"close_a":68.294472,"volume_a":8710900,"open_b":123.598789,"high_b":124.446074,"low_b":122.826833,"close_b":123.015114,"volume_b":9431500,"open_bench":581.907599,"close_bench":578.140076},{"date":"2025-05-29","open_a":68.150494,"high_a":68.78401,"low_a":67.632166,"close_a":68.620827,"volume_a":10291100,"open_b":123.194001,"high_b":124.333124,"low_b":122.619735,"close_b":124.191902,"volume_b":6656900,"open_bench":583.383137,"close_bench":580.422241},{"date":"2025-05-30","open_a":68.73602,"high_a":69.561505,"low_a":68.73602,"close_a":69.206352,"volume_a":28019500,"open_b":124.408418,"high_b":124.945033,"low_b":123.598797,"close_b":123.749428,"volume_b":12992100,"open_bench":579.320494,"close_bench":579.77301},{"date":"2025-06-02","open_a":69.139146,"high_a":69.148747,"low_a":68.496038,"close_a":69.110352,"volume_a":10202700,"open_b":123.231635,"high_b":123.354024,"low_b":121.508852,"close_b":123.241058,"volume_b":6723700,"open_bench":578.169696,"close_bench":583.03894},{"date":"2025-06-03","open_a":68.831999,"high_a":68.899189,"low_a":67.641767,"close_a":68.304077,"volume_a":20139800,"open_b":122.977468,"high_b":124.464911,"low_b":122.205498,"close_b":124.125999,"volume_b":7339500,"open_bench":582.674957,"close_bench":586.36377},{"date":"2025-06-04","open_a":68.400075,"high_a":68.966393,"low_a":67.843351,"close_a":68.505661,"volume_a":13565900,"open_b":124.493147,"high_b":124.963856,"low_b":123.419931,"close_b":124.022438,"volume_b":6519700,"open_bench":587.219582,"close_bench":586.20636},{"date":"2025-06-05","open_a":68.515256,"high_a":68.592047,"low_a":67.987334,"close_a":68.064125,"volume_a":13531400,"open_b":123.514076,"high_b":123.71178,"low_b":122.582081,"close_b":123.429352,"volume_b":9560800,"open_bench":587.878515,"close_bench":583.37323},{"date":"2025-06-06","open_a":68.22729,"high_a":68.611237,"low_a":68.092909,"close_a":68.48645,"volume_a":9524000,"open_b":123.965184,"high_b":124.364917,"low_b":123.09909,"close_b":123.755798,"volume_b":7628200,"open_bench":588.891744,"close_bench":589.363953},{"date":"2025-06-09","open_a":68.486453,"high_a":68.947192,"low_a":68.236886,"close_a":68.889595,"volume_a":12281300,"open_b":123.565447,"high_b":124.345872,"low_b":122.946802,"close_b":123.689178,"volume_b":8453100,"open_bench":589.934406,"close_bench":589.895081},{"date":"2025-06-10","open_a":69.014375,"high_a":69.77267,"low_a":68.879994,"close_a":69.446312,"volume_a":12265000,"open_b":123.917579,"high_b":125.744934,"low_b":123.213295,"close_b":125.468933,"volume_b":11852900,"open_bench":590.426245,"close_bench":593.239624},{"date":"2025-06-11","open_a":69.369522,"high_a":69.455913,"low_a":68.879995,"close_a":69.177551,"volume_a":12718500,"open_b":125.573639,"high_b":125.60219,"low_b":123.527372,"close_b":123.632065,"volume_b":9168400,"open_bench":594.331509,"close_bench":591.547668},{"date":"2025-06-12","open_a":69.350329,"high_a":69.494311,"low_a":68.726415,"close_a":69.340729,"volume_a":12806900,"open_b":123.622544,"high_b":125.944811,"low_b":123.451236,"close_b":125.91626,"volume_b":11444900,"open_bench":590.219716,"close_bench":593.898682},{"date":"2025-06-13","open_a":69.379391,"high_a":69.485728,"low_a":68.374039,"close_a":68.654373,"volume_a":17467200,"open_b":125.402312,"high_b":126.639587,"low_b":124.288768,"close_b":124.536232,"volume_b":9581000,"open_bench":588.734436,"close_bench":587.258911},{"date":"2025-06-16","open_a":68.828386,"high_a":69.234402,"low_a":67.900364,"close_a":68.171036,"volume_a":15045900,"open_b":124.974033,"high_b":125.716396,"low_b":123.946144,"close_b":125.069214,"volume_b":11183700,"open_bench":590.603486,"close_bench":592.846252},{"date":"2025-06-17","open_a":68.035697,"high_a":68.113035,"low_a":67.262349,"close_a":67.301018,"volume_a":15437000,"open_b":124.831269,"high_b":125.335695,"low_b":122.899222,"close_b":123.051491,"volume_b":10235200,"open_bench":590.416537,"close_bench":587.780273},{"date":"2025-06-18","open_a":67.668365,"high_a":67.668365,"low_a":66.788673,"close_a":66.904678,"volume_a":16227600,"open_b":123.375102,"high_b":123.422677,"low_b":122.385275,"close_b":122.842125,"volume_b":8785800,"open_bench":588.675334,"close_bench":587.69165},{"date":"2025-06-20","open_a":66.788662,"high_a":67.243008,"low_a":66.48899,"close_a":66.546989,"volume_a":40859400,"open_b":123.279921,"high_b":124.126976,"low_b":122.680315,"close_b":122.842125,"volume_b":23001000,"open_bench":590.356462,"close_bench":586.311462},{"date":"2025-06-23","open_a":66.837002,"high_a":67.484683,"low_a":66.672666,"close_a":67.417015,"volume_a":21822800,"open_b":123.051492,"high_b":123.251365,"low_b":121.652428,"close_b":122.861145,"volume_b":12840400,"open_bench":587.061195,"close_bench":592.102722},{"date":"2025-06-24","open_a":67.610358,"high_a":68.006697,"low_a":67.368685,"close_a":67.871361,"volume_a":15216100,"open_b":122.908728,"high_b":124.878853,"low_b":122.823074,"close_b":124.72657,"volume_b":9226800,"open_bench":596.22676,"close_bench":598.643921},{"date":"2025-06-25","open_a":67.600687,"high_a":67.687685,"low_a":67.117341,"close_a":67.310677,"volume_a":12540400,"open_b":123.974695,"high_b":124.288774,"low_b":121.4716,"close_b":121.842789,"volume_b":10922600,"open_bench":599.758695,"close_bench":598.979309},{"date":"2025-06-26","open_a":67.378356,"high_a":67.591029,"low_a":66.84668,"close_a":67.156021,"volume_a":14454100,"open_b":121.547733,"high_b":122.927772,"low_b":121.44304,"close_b":122.033127,"volume_b":11538100,"open_bench":600.824201,"close_bench":603.665588},{"date":"2025-06-27","open_a":67.272009,"high_a":68.2097,"low_a":67.156011,"close_a":67.987366,"volume_a":23470700,"open_b":122.099727,"high_b":125.002561,"low_b":121.928419,"close_b":124.717033,"volume_b":26536500,"open_bench":604.662046,"close_bench":606.664795},{"date":"2025-06-30","open_a":67.794021,"high_a":68.43204,"low_a":67.755352,"close_a":68.393372,"volume_a":15429000,"open_b":124.869328,"high_b":125.973359,"low_b":124.288761,"close_b":125.668793,"volume_b":11402400,"open_bench":609.1017,"close_bench":609.565369},{"date":"2025-07-01","open_a":68.62538,"high_a":70.036748,"low_a":68.538381,"close_a":69.28273,"volume_a":21729300,"open_b":126.192279,"high_b":129.808924,"low_b":126.078073,"close_b":128.733444,"volume_b":12846700,"open_bench":608.095423,"close_bench":609.368164},{"date":"2025-07-02","open_a":69.302068,"high_a":69.437404,"low_a":68.383707,"close_a":68.54805,"volume_a":13050200,"open_b":128.723916,"high_b":130.389475,"low_b":128.371774,"close_b":129.894562,"volume_b":9388700,"open_bench":608.963575,"close_bench":612.130554},{"date":"2025-07-03","open_a":68.673709,"high_a":69.021719,"low_a":68.103358,"close_a":68.973381,"volume_a":9440500,"open_b":130.103972,"high_b":130.227703,"low_b":128.714427,"close_b":128.847672,"volume_b":4941400,"open_bench":614.103693,"close_bench":616.954956},{"date":"2025-07-07","open_a":68.992724,"high_a":68.992724,"low_a":68.44171,"close_a":68.644714,"volume_a":16984700,"open_b":128.990399,"high_b":129.171233,"low_b":127.229659,"close_b":127.962509,"volume_b":7773700,"open_bench":615.001479,"close_bench":612.357422},{"date":"2025-07-08","open_a":68.325706,"high_a":68.325706,"low_a":67.552351,"close_a":67.90036,"volume_a":17115500,"open_b":126.972693,"high_b":129.123638,"low_b":126.848961,"close_b":128.524033,"volume_b":7718300,"open_bench":613.01844,"close_bench":612.022034},{"date":"2025-07-09","open_a":67.82303,"high_a":67.910029,"low_a":66.672666,"close_a":67.16568,"volume_a":33704100,"open_b":128.390806,"high_b":128.847656,"low_b":126.544427,"close_b":127.991074,"volume_b":7762600,"open_bench":614.419458,"close_bench":615.692139},{"date":"2025-07-10","open_a":66.982007,"high_a":67.523351,"low_a":66.85634,"close_a":67.446014,"volume_a":14608400,"open_b":127.943492,"high_b":130.084926,"low_b":127.182106,"close_b":129.513885,"volume_b":8869700,"open_bench":615.830316,"close_bench":617.428589},{"date":"2025-07-11","open_a":67.252684,"high_a":67.823035,"low_a":66.943343,"close_a":67.542694,"volume_a":12916700,"open_b":128.333712,"high_b":128.857176,"low_b":127.353412,"close_b":128.733444,"volume_b":7963600,"open_bench":614.389791,"close_bench":615.257996},{"date":"2025-07-14","open_a":67.475031,"high_a":67.745703,"low_a":66.962679,"close_a":67.156021,"volume_a":13983200,"open_b":128.248048,"high_b":129.323528,"low_b":127.400993,"close_b":129.028488,"volume_b":8480300,"open_bench":614.804168,"close_bench":616.432068},{"date":"2025-07-15","open_a":67.194672,"high_a":67.233341,"low_a":66.846663,"close_a":67.049667,"volume_a":12639800,"open_b":128.895228,"high_b":128.97137,"low_b":127.296299,"close_b":127.353401,"volume_b":8933500,"open_bench":619.105778,"close_bench":613.797913},{"date":"2025-07-16","open_a":67.040006,"high_a":67.185012,"low_a":66.498662,"close_a":66.962669,"volume_a":13178300,"open_b":127.486651,"high_b":128.971376,"low_b":127.296304,"close_b":128.819107,"volume_b":10941600,"open_bench":615.376431,"close_bench":615.849976},{"date":"2025-07-17","open_a":67.484683,"high_a":68.335376,"low_a":67.19468,"close_a":68.238701,"volume_a":20464700,"open_b":134.862683,"high_b":138.917139,"low_b":134.805581,"close_b":138.422226,"volume_b":26161000,"open_bench":616.027567,"close_bench":619.618713},{"date":"2025-07-18","open_a":68.093713,"high_a":68.345047,"low_a":67.504031,"close_a":67.523361,"volume_a":14928000,"open_b":139.202668,"high_b":139.288337,"low_b":136.042872,"close_b":136.3284,"volume_b":13752800,"open_bench":620.861886,"close_bench":619.164978},{"date":"2025-07-21","open_a":67.852036,"high_a":68.267714,"low_a":67.600694,"close_a":67.736031,"volume_a":17811700,"open_b":136.195142,"high_b":136.661489,"low_b":133.682528,"close_b":134.862686,"volume_b":9025600,"open_bench":620.338989,"close_bench":620.338989},{"date":"2025-07-22","open_a":67.136672,"high_a":67.919697,"low_a":66.401992,"close_a":67.339684,"volume_a":22339900,"open_b":134.415365,"high_b":139.335901,"low_b":134.415365,"close_b":138.993271,"volume_b":8281500,"open_bench":620.664568,"close_bench":620.427795},{"date":"2025-07-23","open_a":67.339676,"high_a":67.417006,"low_a":66.314979,"close_a":66.856331,"volume_a":21030000,"open_b":138.821969,"high_b":139.935512,"low_b":138.089119,"close_b":138.631622,"volume_b":6166000,"open_bench":623.081627,"close_bench":625.705994},{"date":"2025-07-24","open_a":66.953006,"high_a":67.233347,"low_a":66.691996,"close_a":66.798332,"volume_a":12511400,"open_b":138.783896,"high_b":139.402527,"low_b":137.032669,"close_b":137.537094,"volume_b":9749400,"open_bench":626.090848,"close_bench":625.913269},{"date":"2025-07-25","open_a":66.827336,"high_a":66.924011,"low_a":66.459997,"close_a":66.866005,"volume_a":12574800,"open_b":137.822637,"high_b":137.832164,"low_b":135.814448,"close_b":136.528259,"volume_b":7021600,"open_bench":626.574253,"close_bench":628.557251},{"date":"2025-07-28","open_a":66.450334,"high_a":66.605008,"low_a":65.792984,"close_a":65.821983,"volume_a":13668800,"open_b":135.814449,"high_b":136.195157,"low_b":134.63429,"close_b":134.738983,"volume_b":10639100,"open_bench":628.932091,"close_bench":628.399353},{"date":"2025-07-29","open_a":66.063649,"high_a":67.107676,"low_a":65.879982,"close_a":67.069008,"volume_a":16223400,"open_b":134.738974,"high_b":137.042202,"low_b":134.320202,"close_b":136.947021,"volume_b":6282900,"open_bench":629.790476,"close_bench":626.741943},{"date":"2025-07-30","open_a":66.808001,"high_a":67.020674,"low_a":66.247318,"close_a":66.459991,"volume_a":10448200,"open_b":136.832815,"high_b":137.432406,"low_b":135.424217,"close_b":135.966721,"volume_b":9738400,"open_bench":627.393082,"close_bench":625.952698},{"date":"2025-07-31","open_a":65.879981,"high_a":66.363326,"low_a":65.483634,"close_a":65.628639,"volume_a":17193400,"open_b":134.177422,"high_b":134.738964,"low_b":130.998587,"close_b":131.265076,"volume_b":11689700,"open_bench":630.885601,"close_bench":623.604553},{"date":"2025-08-01","open_a":66.314988,"high_a":66.740335,"low_a":65.995979,"close_a":66.56633,"volume_a":16107500,"open_b":132.388156,"high_b":133.063889,"low_b":131.617228,"close_b":132.559464,"volume_b":7630500,"open_bench":617.902027,"close_bench":613.383423},{"date":"2025-08-04","open_a":66.430991,"high_a":66.779,"low_a":66.034644,"close_a":66.662994,"volume_a":10701500,"open_b":132.568967,"high_b":133.549281,"low_b":132.055029,"close_b":132.825943,"volume_b":8384700,"open_bench":617.280475,"close_bench":622.706726},{"date":"2025-08-05","open_a":66.604995,"high_a":67.175339,"low_a":66.45999,"close_a":66.75,"volume_a":11993400,"open_b":132.864008,"high_b":133.577819,"low_b":132.540403,"close_b":132.721237,"volume_b":5535100,"open_bench":623.318468,"close_bench":619.549683},{"date":"2025-08-06","open_a":66.81767,"high_a":67.47502,"low_a":66.343987,"close_a":67.194679,"volume_a":10783000,"open_b":133.149537,"high_b":134.748481,"low_b":132.635599,"close_b":134.006119,"volume_b":6747400,"open_bench":620.615201,"close_bench":624.295227},{"date":"2025-08-07","open_a":67.146346,"high_a":68.364379,"low_a":67.020679,"close_a":68.084038,"volume_a":15790800,"open_b":134.424908,"high_b":137.346767,"low_b":133.70157,"close_b":136.975586,"volume_b":7674600,"open_bench":627.708828,"close_bench":623.772339},{"date":"2025-08-08","open_a":68.016362,"high_a":68.509377,"low_a":67.74569,"close_a":67.997025,"volume_a":9232700,"open_b":136.97556,"high_b":138.355592,"low_b":136.861355,"close_b":138.203323,"volume_b":5315100,"open_bench":625.55807,"close_bench":628.63623},{"date":"2025-08-11","open_a":67.958364,"high_a":68.499708,"low_a":67.629685,"close_a":68.393372,"volume_a":10909400,"open_b":138.308028,"high_b":138.574517,"low_b":136.813777,"close_b":137.87973,"volume_b":5773400,"open_bench":628.912394,"close_bench":627.393005},{"date":"2025-08-12","open_a":68.46105,"high_a":68.693054,"low_a":67.881037,"close_a":68.354713,"volume_a":11061100,"open_b":138.526949,"high_b":139.916493,"low_b":137.736996,"close_b":139.783249,"volume_b":7668700,"open_bench":629.731301,"close_bench":634.072327},{"date":"2025-08-13","open_a":68.335379,"high_a":68.412709,"low_a":67.890702,"close_a":68.113037,"volume_a":10356500,"open_b":139.802278,"high_b":142.324405,"low_b":139.764215,"close_b":142.143585,"volume_b":6147000,"open_bench":636.262489,"close_bench":636.242798},{"date":"2025-08-14","open_a":68.035693,"high_a":68.11303,"low_a":67.078671,"close_a":67.233345,"volume_a":10722400,"open_b":141.848524,"high_b":142.286334,"low_b":140.906303,"close_b":141.448792,"volume_b":6892500,"open_bench":634.170992,"close_bench":636.302063},{"date":"2025-08-15","open_a":67.417022,"high_a":68.045372,"low_a":67.252686,"close_a":67.591026,"volume_a":12381100,"open_b":142.181638,"high_b":143.809134,"low_b":141.325071,"close_b":143.142899,"volume_b":9162200,"open_bench":637.328051,"close_bench":634.812256},{"date":"2025-08-18","open_a":67.649032,"high_a":67.7167,"low_a":66.75001,"close_a":66.827339,"volume_a":14094900,"open_b":143.285687,"high_b":143.285687,"low_b":141.734333,"close_b":142.333939,"volume_b":9901000,"open_bench":634.240031,"close_bench":634.674133},{"date":"2025-08-19","open_a":66.991674,"high_a":67.881028,"low_a":66.933667,"close_a":67.794022,"volume_a":13862200,"open_b":142.762199,"high_b":144.970261,"low_b":142.628955,"close_b":144.817978,"volume_b":8596900,"open_bench":634.496572,"close_bench":631.230957},{"date":"2025-08-20","open_a":68.3837,"high_a":69.321391,"low_a":68.064698,"close_a":68.345032,"volume_a":17530800,"open_b":145.817331,"high_b":147.149773,"low_b":141.515426,"close_b":143.456985,"volume_b":10023500,"open_bench":630.826452,"close_bench":629.553711},{"date":"2025-08-21","open_a":68.248367,"high_a":68.548046,"low_a":67.987364,"close_a":68.306374,"volume_a":8840400,"open_b":143.066766,"high_b":143.066766,"low_b":141.429758,"close_b":141.791412,"volume_b":8153600,"open_bench":627.748329,"close_bench":627.028076},{"date":"2025-08-22","open_a":68.596377,"high_a":68.654376,"low_a":67.591017,"close_a":67.794022,"volume_a":12999600,"open_b":142.429098,"high_b":143.885272,"low_b":141.896122,"close_b":142.419586,"volume_b":5218000,"open_bench":629.208414,"close_bench":636.657166},{"date":"2025-08-25","open_a":67.774698,"high_a":67.890704,"low_a":66.430998,"close_a":66.634003,"volume_a":13634600,"open_b":141.267924,"high_b":141.63912,"low_b":140.458947,"close_b":141.049026,"volume_b":6624300,"open_bench":635.40418,"close_bench":633.855225},{"date":"2025-08-26","open_a":66.74033,"high_a":66.778999,"low_a":66.295653,"close_a":66.440659,"volume_a":17964200,"open_b":140.382826,"high_b":141.096637,"low_b":138.774369,"close_b":139.906952,"volume_b":11208100,"open_bench":633.588884,"close_bench":636.509155},{"date":"2025-08-27","open_a":66.527662,"high_a":66.575999,"low_a":66.189321,"close_a":66.537331,"volume_a":11562700,"open_b":139.906949,"high_b":140.658824,"low_b":139.250241,"close_b":140.516068,"volume_b":8092900,"open_bench":635.927097,"close_bench":637.959473},{"date":"2025-08-28","open_a":66.585653,"high_a":66.604991,"low_a":66.00564,"close_a":66.082977,"volume_a":16209900,"open_b":140.820648,"high_b":140.896775,"low_b":139.174113,"close_b":139.887924,"volume_b":9265100,"open_bench":638.561284,"close_bench":640.21875},{"date":"2025-08-29","open_a":66.382661,"high_a":66.837006,"low_a":66.150657,"close_a":66.692001,"volume_a":14601100,"open_b":140.002158,"high_b":142.019859,"low_b":139.783245,"close_b":141.477356,"volume_b":7760700,"open_bench":638.78823,"close_bench":636.400696},{"date":"2025-09-02","open_a":66.662999,"high_a":67.030339,"low_a":65.986318,"close_a":66.759666,"volume_a":24770100,"open_b":149.843206,"high_b":150.185835,"low_b":142.962072,"close_b":143.028687,"volume_b":17348600,"open_bench":628.951877,"close_bench":631.684753},{"date":"2025-09-03","open_a":66.372999,"high_a":66.798338,"low_a":65.125967,"close_a":66.692001,"volume_a":33581500,"open_b":142.714607,"high_b":143.200008,"low_b":140.011646,"close_b":141.467819,"volume_b":11259200,"open_bench":634.052556,"close_bench":635.108215},{"date":"2025-09-04","open_a":66.808001,"high_a":67.146341,"low_a":65.84131,"close_a":65.976646,"volume_a":16736000,"open_b":141.829489,"high_b":142.257773,"low_b":139.38349,"close_b":139.802261,"volume_b":6342200,"open_bench":635.779086,"close_bench":640.416077},{"date":"2025-09-05","open_a":65.744642,"high_a":66.295655,"low_a":65.609306,"close_a":65.696304,"volume_a":23978000,"open_b":140.314519,"high_b":142.121313,"low_b":139.622556,"close_b":140.689331,"volume_b":7186300,"open_bench":642.744416,"close_bench":638.561279},{"date":"2025-09-08","open_a":65.599637,"high_a":65.618968,"low_a":64.922957,"close_a":65.18396,"volume_a":21671700,"open_b":140.045412,"high_b":140.045412,"low_b":134.884534,"close_b":136.191574,"volume_b":11127600,"open_bench":639.92286,"close_bench":640.130066},{"date":"2025-09-09","open_a":65.038957,"high_a":65.841313,"low_a":65.038957,"close_a":65.59964,"volume_a":14120600,"open_b":135.682206,"high_b":138.03681,"low_b":135.605334,"close_b":137.527451,"volume_b":5866400,"open_bench":640.268114,"close_bench":641.609924},{"date":"2025-09-10","open_a":65.531975,"high_a":65.821977,"low_a":64.971292,"close_a":65.560974,"volume_a":16041900,"open_b":136.691321,"high_b":137.537057,"low_b":135.912866,"close_b":137.104584,"volume_b":6015100,"open_bench":644.855789,"close_bench":643.464722},{"date":"2025-09-11","open_a":65.657648,"high_a":65.899321,"low_a":65.328977,"close_a":65.367645,"volume_a":15128700,"open_b":137.325619,"high_b":139.113188,"low_b":136.950807,"close_b":138.613434,"volume_b":6144400,"open_bench":645.408199,"close_bench":648.811951},{"date":"2025-09-12","open_a":65.348306,"high_a":65.464311,"low_a":64.710294,"close_a":64.777962,"volume_a":18712400,"open_b":138.459684,"high_b":138.632668,"low_b":137.335233,"close_b":137.940704,"volume_b":5713700,"open_bench":648.782364,"close_bench":648.59491},{"date":"2025-09-15","open_a":64.953291,"high_a":65.0507,"low_a":64.388308,"close_a":64.495461,"volume_a":15986100,"open_b":137.815754,"high_b":137.940696,"low_b":135.057522,"close_b":135.163239,"volume_b":7279600,"open_bench":650.795044,"close_bench":652.047974},{"date":"2025-09-16","open_a":64.631833,"high_a":64.787693,"low_a":64.349342,"close_a":64.524681,"volume_a":18555900,"open_b":135.211298,"high_b":135.807149,"low_b":134.461674,"close_b":134.576996,"volume_b":9283500,"open_bench":652.600468,"close_bench":651.150208},{"date":"2025-09-17","open_a":64.690287,"high_a":65.391636,"low_a":64.690287,"close_a":65.30397,"volume_a":17372300,"open_b":134.711521,"high_b":136.508695,"low_b":134.692295,"close_b":135.73024,"volume_b":6092900,"open_bench":651.160135,"close_bench":650.341248},{"date":"2025-09-18","open_a":65.070183,"high_a":65.303966,"low_a":64.631831,"close_a":64.738983,"volume_a":17722900,"open_b":135.163249,"high_b":136.114702,"low_b":134.922984,"close_b":135.249741,"volume_b":6106600,"open_bench":653.014849,"close_bench":653.379883},{"date":"2025-09-19","open_a":64.768206,"high_a":65.001997,"low_a":64.339606,"close_a":64.709763,"volume_a":31192900,"open_b":135.951303,"high_b":136.547154,"low_b":134.990245,"close_b":136.239609,"volume_b":12365500,"open_bench":655.260655,"close_bench":656.616028},{"date":"2025-09-22","open_a":64.44676,"high_a":64.865618,"low_a":64.36883,"close_a":64.495461,"volume_a":13298700,"open_b":135.586111,"high_b":136.008979,"low_b":134.490505,"close_b":135.538055,"volume_b":5797600,"open_bench":655.132105,"close_bench":659.722595},{"date":"2025-09-23","open_a":64.631835,"high_a":65.031222,"low_a":64.193491,"close_a":64.982513,"volume_a":15072800,"open_b":135.547635,"high_b":136.691296,"low_b":134.509705,"close_b":136.374146,"volume_b":6356500,"open_bench":659.603711,"close_bench":656.131226},{"date":"2025-09-24","open_a":64.69029,"high_a":65.050704,"low_a":64.251939,"close_a":64.729248,"volume_a":14759300,"open_b":136.28767,"high_b":136.79703,"low_b":135.518821,"close_b":136.633652,"volume_b":5949100,"open_bench":657.417338,"close_bench":654.043701},{"date":"2025-09-25","open_a":65.021481,"high_a":65.187077,"low_a":64.242194,"close_a":64.271423,"volume_a":14395000,"open_b":137.431329,"high_b":137.834972,"low_b":134.413607,"close_b":134.471268,"volume_b":7101700,"open_bench":650.917495,"close_bench":651.026306},{"date":"2025-09-26","open_a":64.35908,"high_a":64.612349,"low_a":63.803841,"close_a":63.969437,"volume_a":17821600,"open_b":134.49051,"high_b":135.499624,"low_b":134.404018,"close_b":134.971039,"volume_b":6224700,"open_bench":652.470701,"close_bench":654.756042},{"date":"2025-09-29","open_a":63.969443,"high_a":64.388308,"low_a":63.65773,"close_a":64.329865,"volume_a":14818500,"open_b":134.971038,"high_b":135.192077,"low_b":133.702434,"close_b":134.711548,"volume_b":11930200,"open_bench":657.268926,"close_bench":656.596191},{"date":"2025-09-30","open_a":64.359082,"high_a":64.914321,"low_a":64.25193,"close_a":64.602608,"volume_a":13815000,"open_b":134.711549,"high_b":135.53806,"low_b":134.384793,"close_b":134.971039,"volume_b":7583400,"open_bench":655.854147,"close_bench":659.069458},{"date":"2025-10-01","open_a":64.7877,"high_a":65.099413,"low_a":64.573396,"close_a":65.050705,"volume_a":14877500,"open_b":135.326605,"high_b":137.786911,"low_b":135.009454,"close_b":137.565872,"volume_b":5880800,"open_bench":656.091635,"close_bench":661.315308},{"date":"2025-10-02","open_a":64.807171,"high_a":64.924066,"low_a":64.154523,"close_a":64.388306,"volume_a":14350700,"open_b":136.364561,"high_b":137.671601,"low_b":135.864807,"close_b":136.768204,"volume_b":4497300,"open_bench":663.293939,"close_bench":662.077026},{"date":"2025-10-03","open_a":64.388312,"high_a":65.255264,"low_a":64.329868,"close_a":64.924072,"volume_a":12249200,"open_b":136.720145,"high_b":137.777315,"low_b":136.335728,"close_b":136.45105,"volume_b":5425700,"open_bench":662.838843,"close_bench":662.0672},{"date":"2025-10-06","open_a":64.768206,"high_a":64.777949,"low_a":64.135036,"close_a":64.388306,"volume_a":14333300,"open_b":136.249224,"high_b":136.326111,"low_b":133.471767,"close_b":134.259827,"volume_b":6750700,"open_bench":664.451431,"close_bench":664.441528},{"date":"2025-10-07","open_a":64.505201,"high_a":65.51827,"low_a":64.164266,"close_a":65.06044,"volume_a":13328300,"open_b":134.903761,"high_b":136.316518,"low_b":134.250248,"close_b":135.307404,"volume_b":6110300,"open_bench":665.361629,"close_bench":661.978149},{"date":"2025-10-08","open_a":64.943552,"high_a":65.138377,"low_a":64.349347,"close_a":64.407799,"volume_a":12606700,"open_b":135.30738,"high_b":135.489983,"low_b":133.289166,"close_b":133.433319,"volume_b":8256700,"open_bench":663.096078,"close_bench":665.925537},{"date":"2025-10-09","open_a":64.573391,"high_a":65.070187,"low_a":64.407795,"close_a":64.651321,"volume_a":12705200,"open_b":134.99027,"high_b":139.103599,"low_b":133.193081,"close_b":139.074768,"volume_b":13706600,"open_bench":666.341158,"close_bench":663.996399},{"date":"2025-10-10","open_a":64.797432,"high_a":65.654649,"low_a":64.661058,"close_a":65.30397,"volume_a":17991700,"open_b":139.95892,"high_b":144.946815,"low_b":139.353448,"close_b":144.235626,"volume_b":17545200,"open_bench":664.956002,"close_bench":646.049988},{"date":"2025-10-13","open_a":64.514951,"high_a":65.07019,"low_a":64.290904,"close_a":65.07019,"volume_a":13690000,"open_b":142.861305,"high_b":144.139515,"low_b":141.602321,"close_b":143.091965,"volume_b":7272000,"open_bench":653.598568,"close_bench":655.963013},{"date":"2025-10-14","open_a":65.138369,"high_a":65.820239,"low_a":65.021474,"close_a":65.761795,"volume_a":14245600,"open_b":142.899771,"high_b":145.715665,"low_b":142.717168,"close_b":145.638779,"volume_b":8371100,"open_bench":650.155692,"close_bench":655.161682},{"date":"2025-10-15","open_a":65.528016,"high_a":65.966368,"low_a":65.187081,"close_a":65.342941,"volume_a":11193700,"open_b":145.436953,"high_b":146.455672,"low_b":143.361064,"close_b":145.273575,"volume_b":6101100,"open_bench":659.702725,"close_bench":658.070312},{"date":"2025-10-16","open_a":65.440343,"high_a":66.34626,"low_a":65.420864,"close_a":65.839722,"volume_a":15707200,"open_b":146.446077,"high_b":148.368193,"low_b":146.35957,"close_b":146.724777,"volume_b":8532900,"open_bench":659.702715,"close_bench":653.588684},{"date":"2025-10-17","open_a":66.239109,"high_a":66.726161,"low_a":66.044291,"close_a":66.667717,"volume_a":15541800,"open_b":147.781935,"high_b":147.916482,"low_b":145.830988,"close_b":147.724274,"volume_b":6710100,"open_bench":652.460824,"close_bench":657.298645},{"date":"2025-10-20","open_a":66.657974,"high_a":66.862535,"low_a":66.161178,"close_a":66.667717,"volume_a":17786800,"open_b":148.406622,"high_b":148.406622,"low_b":146.772826,"close_b":147.656998,"volume_b":4810600,"open_bench":660.197387,"close_bench":664.134888},{"date":"2025-10-21","open_a":68.742555,"high_a":69.541321,"low_a":68.411363,"close_a":69.375725,"volume_a":33651300,"open_b":148.675733,"high_b":149.146657,"low_b":146.888164,"close_b":147.21492,"volume_b":5614300,"open_bench":664.273362,"close_bench":664.124939},{"date":"2025-10-22","open_a":69.512094,"high_a":69.765363,"low_a":68.684107,"close_a":68.976334,"volume_a":18293000,"open_b":147.186062,"high_b":149.137009,"low_b":146.763209,"close_b":147.07074,"volume_b":6486000,"open_bench":664.827425,"close_bench":660.672241},{"date":"2025-10-23","open_a":68.976342,"high_a":69.093237,"low_a":67.914572,"close_a":68.128876,"volume_a":13989400,"open_b":147.2053,"high_b":147.426339,"low_b":145.148636,"close_b":145.60994,"volume_b":6091800,"open_bench":660.988803,"close_bench":664.589966},{"date":"2025-10-24","open_a":68.099656,"high_a":68.23603,"low_a":67.846387,"close_a":67.904831,"volume_a":15688200,"open_b":146.119301,"high_b":146.484508,"low_b":145.379297,"close_b":145.648392,"volume_b":3580200,"open_bench":669.239755,"close_bench":670.021301},{"date":"2025-10-27","open_a":67.97301,"high_a":68.304209,"low_a":67.456736,"close_a":68.245758,"volume_a":13800000,"open_b":145.129416,"high_b":146.916985,"low_b":145.081375,"close_b":146.68634,"volume_b":4744600,"open_bench":675.442867,"close_bench":677.926086},{"date":"2025-10-28","open_a":68.236026,"high_a":69.064013,"low_a":68.138609,"close_a":68.343178,"volume_a":12730500,"open_b":145.513852,"high_b":147.24376,"low_b":144.08187,"close_b":144.274078,"volume_b":5626500,"open_bench":679.716721,"close_bench":679.726624},{"date":"2025-10-29","open_a":68.109384,"high_a":68.216536,"low_a":66.394965,"close_a":66.58004,"volume_a":19789600,"open_b":142.284676,"high_b":142.870923,"low_b":138.075237,"close_b":140.468277,"volume_b":7767600,"open_bench":681.368861,"close_bench":680.053101},{"date":"2025-10-30","open_a":66.677452,"high_a":67.495703,"low_a":66.619008,"close_a":67.193733,"volume_a":14751700,"open_b":140.641266,"high_b":142.323118,"low_b":140.554775,"close_b":141.804153,"volume_b":5752700,"open_bench":676.60048,"close_bench":672.573914},{"date":"2025-10-31","open_a":66.774871,"high_a":67.281403,"low_a":66.141702,"close_a":67.115807,"volume_a":16742700,"open_b":140.506733,"high_b":141.275583,"low_b":139.641787,"close_b":140.401016,"volume_b":6313600,"open_bench":677.728203,"close_bench":674.780029},{"date":"2025-11-03","open_a":67.018393,"high_a":67.106059,"low_a":65.907915,"close_a":66.209885,"volume_a":15660400,"open_b":140.179982,"high_b":140.228038,"low_b":137.671619,"close_b":138.007996,"volume_b":8331500,"open_bench":678.351474,"close_bench":676.046387},{"date":"2025-11-04","open_a":66.696937,"high_a":66.950206,"low_a":66.394967,"close_a":66.882019,"volume_a":12743800,"open_b":139.065144,"high_b":139.286183,"low_b":136.52795,"close_b":137.335236,"volume_b":8302900,"open_bench":668.893485,"close_bench":668.032776},{"date":"2025-11-05","open_a":66.882018,"high_a":67.047614,"low_a":66.414445,"close_a":66.735901,"volume_a":12850100,"open_b":137.335219,"high_b":137.700426,"low_b":136.06663,"close_b":137.296783,"volume_b":7136400,"open_bench":667.775554,"close_bench":670.347839},{"date":"2025-11-06","open_a":66.531346,"high_a":67.34959,"low_a":66.161181,"close_a":67.27166,"volume_a":15906700,"open_b":136.8643,"high_b":137.633149,"low_b":135.797525,"close_b":136.06662,"volume_b":6449400,"open_bench":669.249681,"close_bench":663.155457},{"date":"2025-11-07","open_a":68.002233,"high_a":69.034789,"low_a":67.612597,"close_a":68.723076,"volume_a":25108300,"open_b":137.364045,"high_b":138.44044,"low_b":136.133892,"close_b":137.38327,"volume_b":7592300,"open_bench":660.781013,"close_bench":663.80835},{"date":"2025-11-10","open_a":68.19706,"high_a":68.830229,"low_a":67.64182,"close_a":68.693848,"volume_a":15318000,"open_b":136.941204,"high_b":137.450564,"low_b":135.759092,"close_b":137.066132,"volume_b":6617500,"open_bench":670.011443,"close_bench":674.166626},{"date":"2025-11-11","open_a":69.14194,"high_a":69.891997,"low_a":68.830227,"close_a":69.755623,"volume_a":16160200,"open_b":137.700421,"high_b":140.083841,"low_b":137.12378,"close_b":139.430328,"volume_b":7683600,"open_bench":672.692527,"close_bench":675.709961},{"date":"2025-11-12","open_a":69.638732,"high_a":70.028375,"low_a":69.492615,"close_a":69.658218,"volume_a":13612600,"open_b":138.93059,"high_b":139.833987,"low_b":138.450061,"close_b":138.767212,"volume_b":5790100,"open_bench":677.480861,"close_bench":676.085938},{"date":"2025-11-13","open_a":69.570557,"high_a":69.814083,"low_a":69.161435,"close_a":69.229622,"volume_a":14002800,"open_b":138.488509,"high_b":140.689326,"low_b":138.056021,"close_b":139.372681,"volume_b":6663300,"open_bench":673.236729,"close_bench":664.867004},{"date":"2025-11-14","open_a":69.56081,"high_a":69.78485,"low_a":68.77178,"close_a":69.317284,"volume_a":13172000,"open_b":140.2088,"high_b":141.121803,"low_b":138.325119,"close_b":140.170364,"volume_b":5770500,"open_bench":658.278041,"close_bench":664.758118},{"date":"2025-11-17","open_a":69.35624,"high_a":69.677696,"low_a":68.791265,"close_a":68.849709,"volume_a":14772300,"open_b":140.429841,"high_b":142.342353,"low_b":139.564895,"close_b":142.073257,"volume_b":7320200,"open_bench":662.551956,"close_bench":658.564941},{"date":"2025-11-18","open_a":69.151677,"high_a":69.492613,"low_a":68.742555,"close_a":69.375725,"volume_a":12838900,"open_b":142.524925,"high_b":143.995343,"low_b":141.70802,"close_b":142.976624,"volume_b":10821500,"open_bench":655.033066,"close_bench":653.034668},{"date":"2025-11-19","open_a":69.356243,"high_a":69.59977,"low_a":68.976343,"close_a":69.268578,"volume_a":12401500,"open_b":142.524948,"high_b":143.005477,"low_b":140.977644,"close_b":141.381287,"volume_b":8123200,"open_bench":653.727143,"close_bench":655.557373},{"date":"2025-11-20","open_a":69.112705,"high_a":69.628987,"low_a":68.781513,"close_a":69.365974,"volume_a":15396500,"open_b":140.891143,"high_b":141.352447,"low_b":139.257332,"close_b":140.410614,"volume_b":6878900,"open_bench":665.727665,"close_bench":645.565247},{"date":"2025-11-21","open_a":69.823819,"high_a":71.294719,"low_a":69.638737,"close_a":71.060928,"volume_a":26492500,"open_b":140.583612,"high_b":143.140031,"low_b":140.295292,"close_b":140.631668,"volume_b":7911800,"open_bench":648.058289,"close_bench":651.99585},{"date":"2025-11-24","open_a":71.26549,"high_a":71.26549,"low_a":69.71666,"close_a":70.710243,"volume_a":39818600,"open_b":140.151151,"high_b":140.641285,"low_b":139.122811,"close_b":139.834,"volume_b":9391400,"open_bench":655.616775,"close_bench":661.592285},{"date":"2025-11-25","open_a":70.914803,"high_a":71.041434,"low_a":70.271891,"close_a":70.729721,"volume_a":17451200,"open_b":140.247229,"high_b":141.390891,"low_b":139.901262,"close_b":140.420227,"volume_b":8954000,"open_bench":661.493314,"close_bench":667.815125},{"date":"2025-11-26","open_a":70.719989,"high_a":71.197298,"low_a":70.515428,"close_a":70.992737,"volume_a":11795100,"open_b":140.622062,"high_b":142.688331,"low_b":140.535555,"close_b":142.217407,"volume_b":5180600,"open_bench":670.397308,"close_bench":672.425415},{"date":"2025-11-28","open_a":71.109637,"high_a":71.333684,"low_a":70.875854,"close_a":71.226532,"volume_a":7423000,"open_b":141.496592,"high_b":143.024686,"low_b":141.496592,"close_b":142.9478,"volume_b":2969300,"open_bench":673.592861,"close_bench":676.095886},{"date":"2025-12-01","open_a":71.216711,"high_a":71.520809,"low_a":70.559478,"close_a":70.579094,"volume_a":14388100,"open_b":142.842114,"high_b":143.985776,"low_b":142.726777,"close_b":143.687836,"volume_b":7623300,"open_bench":671.564717,"close_bench":673.009155},{"date":"2025-12-02","open_a":70.431961,"high_a":70.431961,"low_a":69.088061,"close_a":69.323486,"volume_a":19871300,"open_b":143.639781,"high_b":143.687823,"low_b":140.823873,"close_b":142.822876,"volume_b":6830300,"open_bench":674.641468,"close_bench":674.255676},{"date":"2025-12-03","open_a":69.529474,"high_a":70.03957,"low_a":69.264622,"close_a":69.460808,"volume_a":17001700,"open_b":142.899752,"high_b":144.831474,"low_b":142.150128,"close_b":142.294281,"volume_b":5109200,"open_bench":673.305944,"close_bench":676.590515},{"date":"2025-12-04","open_a":69.480438,"high_a":69.980715,"low_a":69.029204,"close_a":69.107674,"volume_a":17710000,"open_b":141.996372,"high_b":142.476902,"low_b":140.439463,"close_b":141.189087,"volume_b":6803000,"open_bench":677.985405,"close_bench":677.085144},{"date":"2025-12-05","open_a":68.980155,"high_a":69.362723,"low_a":68.538732,"close_a":68.666252,"volume_a":13978200,"open_b":141.318136,"high_b":141.492827,"low_b":140.240927,"close_b":140.73587,"volume_b":5971300,"open_bench":678.153594,"close_bench":678.371277},{"date":"2025-12-08","open_a":68.5093,"high_a":69.039012,"low_a":68.16597,"close_a":68.911491,"volume_a":17775100,"open_b":140.726171,"high_b":142.230385,"low_b":139.940102,"close_b":141.327866,"volume_b":6327400,"open_bench":679.261741,"close_bench":676.333313},{"date":"2025-12-09","open_a":69.029203,"high_a":69.284244,"low_a":68.499491,"close_a":68.754532,"volume_a":13010600,"open_b":142.026586,"high_b":143.200833,"low_b":139.658665,"close_b":140.367096,"volume_b":10035200,"open_bench":675.858445,"close_bench":675.749573},{"date":"2025-12-10","open_a":69.048816,"high_a":69.186148,"low_a":68.450439,"close_a":68.872246,"volume_a":29789400,"open_b":143.404641,"high_b":145.355257,"low_b":142.657384,"close_b":145.277618,"volume_b":18526400,"open_bench":675.274666,"close_bench":680.231201},{"date":"2025-12-11","open_a":69.205778,"high_a":69.34311,"low_a":67.47931,"close_a":67.793213,"volume_a":23124200,"open_b":145.71431,"high_b":146.432454,"low_b":143.753994,"close_b":144.6371,"volume_b":7854000,"open_bench":677.827193,"close_bench":681.814148},{"date":"2025-12-12","open_a":67.793208,"high_a":69.274439,"low_a":67.724542,"close_a":69.176338,"volume_a":18444500,"open_b":145.326133,"high_b":146.218952,"low_b":144.345975,"close_b":146.199539,"volume_b":8143100,"open_bench":680.824833,"close_bench":674.483276},{"date":"2025-12-15","open_a":69.088063,"high_a":69.961105,"low_a":68.980158,"close_a":69.617775,"volume_a":18660100,"open_b":146.480989,"high_b":147.218532,"low_b":145.840483,"close_b":146.752716,"volume_b":8409600,"open_bench":678.420708,"close_bench":673.464172},{"date":"2025-12-16","open_a":69.941486,"high_a":69.941486,"low_a":68.980155,"close_a":69.029205,"volume_a":17370000,"open_b":147.499979,"high_b":147.849344,"low_b":145.559062,"close_b":145.927826,"volume_b":7520300,"open_bench":671.980228,"close_bench":671.624084},{"date":"2025-12-17","open_a":69.225393,"high_a":69.568723,"low_a":68.970345,"close_a":69.323486,"volume_a":13672700,"open_b":145.937533,"high_b":146.772128,"low_b":145.064112,"close_b":145.646393,"volume_b":8067100,"open_bench":672.633198,"close_bench":664.233826},{"date":"2025-12-18","open_a":68.931105,"high_a":69.451013,"low_a":68.833012,"close_a":69.019394,"volume_a":15524800,"open_b":145.374672,"high_b":145.927825,"low_b":144.840917,"close_b":144.957367,"volume_b":6643200,"open_bench":670.367639,"close_bench":669.249695},{"date":"2025-12-19","open_a":68.725105,"high_a":69.421576,"low_a":68.695678,"close_a":68.725105,"volume_a":36911100,"open_b":144.743856,"high_b":145.462,"low_b":143.530783,"close_b":143.783112,"volume_b":17543500,"open_bench":671.346369,"close_bench":675.315369},{"date":"2025-12-22","open_a":68.597582,"high_a":68.882057,"low_a":68.33273,"close_a":68.872246,"volume_a":12828500,"open_b":143.171733,"high_b":144.578895,"low_b":142.084825,"close_b":142.705917,"volume_b":8818700,"open_bench":678.639344,"close_bench":679.522461},{"date":"2025-12-23","open_a":68.862441,"high_a":68.921303,"low_a":68.362164,"close_a":68.538734,"volume_a":9471000,"open_b":142.841774,"high_b":142.841774,"low_b":139.319005,"close_b":139.435455,"volume_b":7254800,"open_bench":678.619507,"close_bench":682.628235},{"date":"2025-12-24","open_a":68.568157,"high_a":68.823205,"low_a":68.421013,"close_a":68.774155,"volume_a":4515300,"open_b":139.542209,"high_b":139.736313,"low_b":138.727028,"close_b":139.493698,"volume_b":2618700,"open_bench":682.61832,"close_bench":685.02948},{"date":"2025-12-26","open_a":68.617204,"high_a":68.82321,"low_a":68.401403,"close_a":68.538734,"volume_a":5711400,"open_b":139.318999,"high_b":139.687778,"low_b":138.862883,"close_b":139.532501,"volume_b":4975500,"open_bench":685.287481,"close_bench":684.960022},{"date":"2025-12-29","open_a":68.666249,"high_a":69.078245,"low_a":68.519106,"close_a":68.823204,"volume_a":9686900,"open_b":139.493698,"high_b":140.764997,"low_b":139.357835,"close_b":139.978928,"volume_b":7201900,"open_bench":682.211448,"close_bench":682.519043},{"date":"2025-12-30","open_a":68.666258,"high_a":68.901683,"low_a":68.470072,"close_a":68.734924,"volume_a":8610900,"open_b":139.716901,"high_b":140.9785,"low_b":139.668376,"close_b":139.901291,"volume_b":4713800,"open_bench":682.122139,"close_bench":681.685547},{"date":"2025-12-31","open_a":68.73492,"high_a":68.823209,"low_a":68.548538,"close_a":68.577972,"volume_a":8127800,"open_b":139.581037,"high_b":139.775126,"low_b":138.930832,"close_b":139.280197,"volume_b":4380300,"open_bench":681.814586,"close_bench":676.63501}]')
df = pd.DataFrame(DF_RECORDS)
csv_bytes = df.to_csv(index=False).encode('utf-8')
slice_sha256 = hashlib.sha256(csv_bytes).hexdigest()
assert slice_sha256 == EXPECTED_SNAPSHOT['aligned_slice_sha256'], (
    f'Data slice SHA-256 mismatch: {slice_sha256} != {EXPECTED_SNAPSHOT["aligned_slice_sha256"]}'
)
assert df['date'].is_monotonic_increasing, 'Dates must be strictly monotonic increasing!'
assert not df.isna().any().any(), 'Aligned DataFrame contains unexpected NaN values!'

n_bars = len(df)
split_idx = max(25, min(n_bars - 25, int(round(n_bars * spec.execution.train_split_ratio))))
split_date = str(df['date'].iloc[split_idx])
assert split_date == EXPECTED_SNAPSHOT['split_date'], f'Split date mismatch: {split_date}'

print(f'Verified {n_bars} aligned trading days ({df["date"].iloc[0]} to {df["date"].iloc[-1]})')
print(f'Formation Split [0:{split_idx}]: {df["date"].iloc[0]} to {df["date"].iloc[split_idx - 1]} ({split_idx} days)')
print(f'Out-of-Sample Split [{split_idx}:{n_bars}]: {split_date} to {df["date"].iloc[-1]} ({n_bars - split_idx} days)')
display(df.head(5))

Verified 2011 aligned trading days (2018-01-02 to 2025-12-31)
Formation Split [0:1006]: 2018-01-02 to 2021-12-29 (1006 days)
Out-of-Sample Split [1006:2011]: 2021-12-30 to 2025-12-31 (1005 days)


,date,open_a,high_a,low_a,close_a,volume_a,open_b,high_b,low_b,close_b,volume_b,open_bench,close_bench
0,2018-01-02,35.099345,35.122279,34.793533,34.816471,10872200,91.111547,91.301474,89.592135,89.690895,4165800,234.555390,235.369812
1,2018-01-03,34.778238,34.931141,34.663558,34.740009,12635600,89.576947,90.002380,89.371823,89.455391,4224000,235.536252,236.858612
2,2018-01-04,34.831763,35.336350,34.747665,35.229317,12709400,89.493391,90.488604,89.341446,89.896034,3147000,237.497832,237.856857
3,2018-01-05,35.183443,35.321057,35.007602,35.221668,13113100,89.956831,90.473426,89.835275,90.154350,2714200,238.645069,239.441986
4,2018-01-08,35.129919,35.244596,35.076402,35.168144,7068600,89.804870,90.207513,89.394628,89.637733,5730400,239.345582,239.879791


## Cell 4 — Strictly Causal Signal Construction & Prefix-Invariance Proof
Computes the signal series at bar $t$ using only prices $[0..t]$. In `ols_log` pairs mode, OLS parameters $(\hat{\alpha}_t, \hat{\beta}_t)$ are estimated on expanding windows $[0..t]$ inside the Formation split and frozen at `split_idx` for Out-of-Sample evaluation. We prove zero look-ahead bias by mutating future prices both inside Formation ($m = \lfloor k/2 \rfloor$) and at the split boundary ($m = k$) and asserting that all prior signals $[0..m)$ are 100% invariant.

In [4]:
# Cell 4: Compute Causal Signals & Prove Prefix Invariance
precomputed = compute_causal_signals(df, spec, split_idx)
signal_values, indicators, pairs_diag, ols_alpha, ols_beta = precomputed

# Prefix-Invariance Test 1: Mutate prices in second half of Formation [m_form : split_idx)
m_form = max(35, split_idx // 2)
df_mut_form = df.copy(deep=True)
df_mut_form.loc[m_form:, 'close_a'] *= np.linspace(1.0, 1.45, n_bars - m_form)
sig_mut_form, _, _, _, _ = compute_causal_signals(df_mut_form, spec, split_idx)
mask_form = np.isfinite(signal_values[:m_form])
assert np.allclose(signal_values[:m_form][mask_form], sig_mut_form[:m_form][mask_form], atol=1e-12), (
    'Look-ahead violation detected inside Formation window!'
)

# Prefix-Invariance Test 2: Mutate Out-of-Sample prices [split_idx : n_bars)
df_mut_oos = df.copy(deep=True)
df_mut_oos.loc[split_idx:, 'close_a'] *= 1.75
sig_mut_oos, _, _, _, _ = compute_causal_signals(df_mut_oos, spec, split_idx)
mask_oos = np.isfinite(signal_values[:split_idx])
assert np.allclose(signal_values[:split_idx][mask_oos], sig_mut_oos[:split_idx][mask_oos], atol=1e-12), (
    'Look-ahead violation detected across split boundary!'
)
print(f'Causal prefix-invariance verified at m_form={m_form} ({df["date"].iloc[m_form]}) and split_idx={split_idx} ({split_date}).')

Causal prefix-invariance verified at m_form=503 (2020-01-02) and split_idx=1006 (2021-12-30).


## Cell 5 — Walk-Forward Trade Simulation & Exact Compounding Reconciliation
Executes signals triggered at the close of bar $t$ on bar $t+1$ (`next_open` or `next_close`) with unit gross portfolio exposure ($|w_A| + |w_B| = 1.0$) and explicit commission + slippage fees. Verifies that compounded per-trade net returns reconcile with terminal portfolio net equity.

In [5]:
# Cell 5: Run Bar-by-Bar Walk-Forward Simulation & Verify Trade Ledger
sim = simulate_strategy_core(df, spec, split_idx, precomputed_signals=precomputed)
trades = sim['trades']
equity_net = sim['equity_net']
equity_gross = sim['equity_gross']
daily_net_ret = sim['daily_net_ret']
daily_gross_ret = sim['daily_gross_ret']
daily_bench_ret = sim['daily_bench_ret']
daily_pos = sim['daily_position']

# Reconcile compounded daily returns vs terminal equity
initial_cap = spec.execution.initial_capital
assert math.isclose(initial_cap * float(np.prod(1.0 + daily_net_ret)), float(equity_net[-1]), rel_tol=1e-9)

# Reconcile compounded trade net returns vs terminal equity
if trades:
    compounded_trades_eq = initial_cap * float(np.prod([1.0 + t.net_return_pct / 100.0 for t in trades]))
    assert math.isclose(compounded_trades_eq, float(equity_net[-1]), rel_tol=1e-3), (
        f'Trade ledger vs daily equity mismatch: {compounded_trades_eq:.2f} vs {equity_net[-1]:.2f}'
    )
    trades_df = pd.DataFrame([t.model_dump() for t in trades])
    print(f'Executed {len(trades)} trades. Terminal Net Equity: ${equity_net[-1]:,.2f} (Compounded Trade Ledger: ${compounded_trades_eq:,.2f})')
    display(trades_df[['trade_id', 'split', 'direction', 'signal_date', 'entry_date', 'exit_date', 'holding_days', 'leg_weight_a', 'leg_weight_b', 'gross_return_pct', 'cost_pct', 'net_return_pct', 'exit_reason', 'cumulative_equity_after']].head(15))
else:
    print('Zero trades triggered across sample.')

Executed 66 trades. Terminal Net Equity: $76,050.35 (Compounded Trade Ledger: $76,050.30)


,trade_id,split,direction,signal_date,entry_date,exit_date,holding_days,leg_weight_a,leg_weight_b,gross_return_pct,cost_pct,net_return_pct,exit_reason,cumulative_equity_after
0,1,in_sample,short_spread,2018-02-14,2018-02-15,2018-02-23,5,-0.5039,0.4961,0.6872,0.2,0.4872,spread_convergence (z=-0.14 <= +0.25),100487.18
1,2,in_sample,short_spread,2018-04-04,2018-04-05,2018-04-27,16,-0.5395,0.4605,-1.6153,0.2,-1.8153,spread_convergence (z=-0.13 <= +0.25),98663.07
2,3,in_sample,short_spread,2018-06-04,2018-06-05,2018-06-15,8,-0.6774,0.3226,0.7553,0.2,0.5553,spread_convergence (z=-0.04 <= +0.25),99210.93
3,4,in_sample,long_spread,2018-06-21,2018-06-22,2018-07-05,8,0.6837,-0.3163,1.0986,0.2,0.8986,spread_convergence (z=-0.07 >= -0.25),100102.41
4,5,in_sample,short_spread,2018-07-25,2018-07-26,2018-08-23,20,-0.6797,0.3203,-0.4203,0.2,-0.6203,holding_period_reached (20d),99481.44
5,6,in_sample,long_spread,2018-08-29,2018-08-30,2018-09-11,7,0.6530,-0.3470,0.8884,0.2,0.6884,spread_convergence (z=0.04 >= -0.25),100166.30
6,7,in_sample,short_spread,2018-10-01,2018-10-02,2018-10-12,8,-0.6400,0.3600,0.7966,0.2,0.5966,spread_convergence (z=-0.11 <= +0.25),100763.93
7,8,in_sample,short_spread,2018-10-31,2018-11-01,2018-11-23,15,-0.6364,0.3636,-0.3548,0.2,-0.5548,spread_convergence (z=-0.07 <= +0.25),100204.86
8,9,in_sample,long_spread,2018-11-29,2018-11-30,2018-12-06,3,0.5890,-0.4110,0.9199,0.2,0.7199,spread_convergence (z=0.57 >= -0.25),100926.23
9,10,in_sample,short_spread,2018-12-14,2018-12-17,2018-12-26,6,-0.5705,0.4295,0.9541,0.2,0.7541,spread_convergence (z=0.03 <= +0.25),101687.31


## Cell 6 — Running High-Water Mark ($E_0$ Included) & Maximum Drawdown Verification
Verifies the exact drawdown equations:
$$E_0 = C_0, \quad H_t = \max\!\left(E_0, \max_{1 \le s \le t} E_s\right), \quad D_t = 1 - \frac{E_t}{H_t}, \quad \mathrm{MDD} = \max_{1 \le t \le T} D_t$$
where $D_t \in [0, 1]$ is reported as a positive loss percentage in tables and $U_t = -D_t \le 0\%$ is plotted on underwater charts.

In [6]:
# Cell 6: Verify Running High-Water Mark Including Initial Capital E_0
running_peak = np.maximum(initial_cap, np.maximum.accumulate(equity_net))
drawdown_loss_pct = ((running_peak - equity_net) / running_peak) * 100.0
underwater_series_pct = -drawdown_loss_pct
max_dd_full = float(np.max(drawdown_loss_pct))

print(f'Full-Sample Max Drawdown (positive loss magnitude): {max_dd_full:.2f}%')
print(f'Underwater Chart Minimum (signed trough): {float(np.min(underwater_series_pct)):.2f}%')
if sim['circuit_breaker_tripped']:
    print(f'Portfolio Circuit Breaker Triggered on: {sim["circuit_breaker_date"]} (Limit: {spec.risk_controls.max_drawdown_circuit_breaker_pct}%)')

Full-Sample Max Drawdown (positive loss magnitude): 25.83%
Underwater Chart Minimum (signed trough): -25.83%
Portfolio Circuit Breaker Triggered on: 2025-07-17 (Limit: 25.0%)


## Cell 7 — Split Performance & Risk Metrics Table
Computes `PeriodMetrics` across In-Sample Formation, Held-Out Out-of-Sample, and Full Sample, explicitly flagging zero-exposure / prior-halt splits.

In [7]:
# Cell 7: Compute PeriodMetrics for Full Sample, In-Sample, and Out-of-Sample
dates = sim['dates']
is_trades = [t for t in trades if t.split == 'in_sample']
oos_trades = [t for t in trades if t.split == 'out_of_sample']
cb_tripped = bool(sim['circuit_breaker_tripped'])
cb_date = sim['circuit_breaker_date']
cb_in_is = bool(cb_tripped and cb_date is not None and cb_date < split_date)
cb_in_oos = bool(cb_tripped and cb_date is not None and cb_date >= split_date)

full_metrics = _compute_period_metrics('Full Sample', dates, daily_gross_ret, daily_net_ret, daily_bench_ret, daily_pos, trades, cb_tripped, cb_date)
is_metrics = _compute_period_metrics('In-Sample Formation', dates[:split_idx], daily_gross_ret[:split_idx], daily_net_ret[:split_idx], daily_bench_ret[:split_idx], daily_pos[:split_idx], is_trades, cb_in_is, cb_date if cb_in_is else None, halted_prior_to_period=False)
oos_metrics = _compute_period_metrics('Out-of-Sample (Held-Out)', dates[split_idx:], daily_gross_ret[split_idx:], daily_net_ret[split_idx:], daily_bench_ret[split_idx:], daily_pos[split_idx:], oos_trades, cb_in_oos, cb_date if cb_tripped else None, halted_prior_to_period=cb_in_is)

def _fmt_sr(m):
    if m.is_zero_exposure:
        return 'N/A (0 trades — Halted in Formation)' if m.halted_prior_to_period else 'N/A (0 trades)'
    return f'{m.sharpe_ratio:.3f} (PSR {m.probabilistic_sharpe_psr_pct:.1f}%)'

period_summary_df = pd.DataFrame([
    {
        'Period': m.period_label,
        'Date Range': f'{m.start_date} -> {m.end_date}',
        'Days': m.trading_days,
        'Trades (N)': m.trade_count,
        'Exposure (%)': f'{m.exposure_pct:.1f}%',
        'Net Return (%)': f'{m.total_net_return_pct:+.2f}%',
        'Benchmark (%)': f'{m.benchmark_return_pct:+.2f}%',
        'Excess (pp)': f'{m.excess_return_pct:+.2f} pp',
        'Max DD (%)': f'{m.max_drawdown_pct:.2f}%',
        'Sharpe / PSR': _fmt_sr(m),
        'Circuit Breaker': m.circuit_breaker_date or ('Halted Prior' if m.halted_prior_to_period else 'Active'),
    }
    for m in (oos_metrics, is_metrics, full_metrics)
])
display(period_summary_df)

,Period,Date Range,Days,Trades (N),Exposure (%),Net Return (%),Benchmark (%),Excess (pp),Max DD (%),Sharpe / PSR,Circuit Breaker
0,Out-of-Sample (Held-Out),2021-12-30 -> 2025-12-31,1005,30,40.1%,-23.59%,+50.98%,-74.58 pp,23.96%,-1.445 (PSR 0.2%),2025-07-17
1,In-Sample Formation,2018-01-02 -> 2021-12-29,1006,36,40.5%,-0.47%,+90.41%,-90.87 pp,9.15%,0.011 (PSR 50.9%),Active
2,Full Sample,2018-01-02 -> 2025-12-31,2011,66,40.3%,-23.95%,+187.48%,-211.43 pp,25.83%,-0.614 (PSR 4.0%),2025-07-17


## Cell 8 — Econometric & Non-IID Statistical Inference
Computes Lo (2002) / Mertens (2002) non-normality-adjusted Sharpe standard errors, Newey-West (1987) HAC factor regression against the benchmark, Politis & Romano (1994) Stationary Bootstrap confidence intervals, and (for pairs) Engle-Granger / ADF cointegration and discrete Ornstein-Uhlenbeck half-life $t_{1/2} = -\ln(2)/\ln(1 + \hat{\lambda})$.

In [8]:
# Cell 8: Econometric Diagnostics (Newey-West HAC, Stationary Bootstrap, Pairs Cointegration)
factor_reg = newey_west_factor_regression(daily_net_ret, daily_bench_ret)
boot_cis = stationary_bootstrap_cis(daily_net_ret, daily_bench_ret, [t.net_return_pct for t in trades], replications=1000, seed=BOOTSTRAP_SEED)

print(f'Newey-West HAC Annualized Alpha: {factor_reg.annualized_alpha_pct:+.2f}% (t_HAC = {factor_reg.alpha_t_stat_hac:.2f}, p = {factor_reg.alpha_p_value_hac:.4f}), Beta = {factor_reg.beta:.3f}')
boot_df = pd.DataFrame([b.model_dump() for b in boot_cis])
display(boot_df[['metric_name', 'point_estimate', 'ci_lower_95', 'ci_upper_95', 'bootstrap_std_error', 'block_length', 'replications']])

if pairs_diag is not None:
    print('\nPairs Cointegration & Co-Movement Diagnostics (Formation Window):')
    pairs_df = pd.DataFrame([{
        'Daily Return Corr (r)': round(pairs_diag.return_correlation, 4),
        'Log-Level Corr': round(pairs_diag.level_correlation, 4),
        'OLS Beta': round(pairs_diag.ols_hedge_ratio_beta, 4),
        'OLS Alpha': round(pairs_diag.ols_alpha, 4),
        'ADF Stat': round(pairs_diag.adf_statistic, 3),
        'ADF p-value': round(pairs_diag.adf_p_value, 4),
        'OU Lambda': round(pairs_diag.ou_lambda, 5),
        'Half-Life (days)': pairs_diag.half_life_days,
        'Hurst H': round(pairs_diag.hurst_exponent, 3),
    }])
    display(pairs_df)

Newey-West HAC Annualized Alpha: -3.15% (t_HAC = -1.78, p = 0.0755), Beta = -0.009


,metric_name,point_estimate,ci_lower_95,ci_upper_95,bootstrap_std_error,block_length,replications
0,Annualized Return (Mean),-3.290,-6.960,0.090,1.810,13,1000
1,Annualized Sharpe Ratio,-0.614,-1.246,0.020,0.320,13,1000
2,Annualized Excess Return vs. Benchmark,-18.420,-29.330,-7.250,5.640,13,1000
3,Mean Net Return per Trade,-0.398,-0.865,0.026,0.227,1,1000



Pairs Cointegration & Co-Movement Diagnostics (Formation Window):


,Daily Return Corr (r),Log-Level Corr,OLS Beta,OLS Alpha,ADF Stat,ADF p-value,OU Lambda,Half-Life (days),Hurst H
0,0.7392,0.9169,0.6866,0.5088,-3.039,0.0314,-0.02056,33.4,0.467


## Cell 9 — Standard Literature Threshold Bracket & 25-Trial Parameter Surface (DSR)
Evaluates Conservative, Baseline, and Aggressive literature profiles alongside the $5 \times 5$ parameter sensitivity grid ($K=25$ trials) and Bailey & López de Prado (2014) Deflated Sharpe Ratio.

In [9]:
# Cell 9: Compute Standard Bracket Profiles & 5x5 Parameter Sensitivity Surface
bracket_profiles = compute_standard_bracket_profiles(df, spec, split_idx)
param_surface = compute_parameter_sensitivity_surface(df, spec, split_idx) if 'parameter_sensitivity_grid' in spec.selected_diagnostics else None

bracket_df = pd.DataFrame([
    {
        'Profile': bp.profile_label,
        'Entry': bp.entry_threshold,
        'Exit': bp.exit_threshold,
        'IS Net (%)': bp.in_sample_metrics.total_net_return_pct,
        'OOS Net (%)': bp.out_of_sample_metrics.total_net_return_pct,
        'Full Net (%)': bp.full_sample_metrics.total_net_return_pct,
        'Full Sharpe': bp.full_sample_metrics.sharpe_ratio,
        'Max DD (%)': bp.full_sample_metrics.max_drawdown_pct,
    }
    for bp in bracket_profiles
])
display(bracket_df)
if param_surface is not None:
    print(f'25-Cell Grid Expected Max SR under Null: {param_surface.expected_max_sr_under_null:.3f} | Selected Spec DSR: {param_surface.selected_spec_deflated_sharpe_psr_pct:.1f}%')

,Profile,Entry,Exit,IS Net (%),OOS Net (%),Full Net (%),Full Sharpe,Max DD (%)
0,Aggressive (1.5σ Entry),1.5,0.50,-6.32,-15.88,-21.20,-0.510,25.27
1,Baseline Literature (2.0σ Entry),2.0,0.25,-0.47,-23.59,-23.95,-0.614,25.83
2,Conservative Tail (2.5σ Entry),2.5,0.00,7.43,-4.98,2.07,0.078,11.65


25-Cell Grid Expected Max SR under Null: 0.497 | Selected Spec DSR: 0.1%


## Cell 10 — Five Adversarial Stress Tests & Non-Overlapping Monte Carlo Null
Runs all 5 adversarial stress tests: (1) Right-Tail Outlier Trade Removal, (2) $t+2$ Execution Latency & $2\times/3\times$ Slippage Shock, (3) Realized Volatility & Bull/Bear Macro Regime Split, (4) 500-Run Matched Non-Overlapping Monte Carlo Random-Entry Benchmark, and (5) Out-of-Sample Cointegration Structural Break / Parameter Cliff-Edge.

In [10]:
# Cell 10: Execute Adversarial Stress Test Suite
def _stress_sim_cb(d_frame, s_spec, s_idx, cost_override, delay_bars):
    return simulate_strategy_core(
        d_frame, s_spec, s_idx,
        override_cost_bps_per_leg=cost_override,
        entry_delay_bars=delay_bars,
        precomputed_signals=precomputed,
    )

stress_suite = run_adversarial_stress_suite(
    df=df, spec=spec, split_idx=split_idx, base_sim=sim,
    full_metrics=full_metrics, oos_metrics=oos_metrics,
    pairs_diag=pairs_diag, parameter_surface=param_surface,
    simulate_fn=_stress_sim_cb,
)
stress_df = pd.DataFrame([
    {
        'Test ID': t.test_id,
        'Title': t.title,
        'Outcome': t.outcome_badge,
        'Baseline': f'{t.baseline_label}: {t.baseline_value}',
        'Stressed': f'{t.stressed_label}: {t.stressed_value}',
    }
    for t in stress_suite.tests
])
display(stress_df)

,Test ID,Title,Outcome,Baseline,Stressed
0,outlier_removal,Right-Tail Outlier Concentration Test (Excl. T...,HIGH_VULNERABILITY,All Trades Compounded: -23.95% (N=66),Excl. Top 3 Trades: -29.53% (N=63)
1,execution_latency_shock,Execution Latency (t+2 Fill Delay) & Liquidity...,HIGH_VULNERABILITY,"Baseline (t+1, 10 bps): -23.95% (SR -0.61)",t+2 Delay + 2× Cost: -24.82% (SR -0.67)
2,regime_vulnerability,Realized Volatility Terciles & Bull/Bear Macro...,HIGH_VULNERABILITY,Positive Regimes: 0 / 5 Regimes,High-Vol vs Low-Vol SR: -0.45 vs -0.77
3,monte_carlo_random_entry,Monte Carlo Random-Entry Null Benchmark (500 N...,HIGH_VULNERABILITY,Actual Signal Return: -23.95% (20th pct),Random-Entry Median: -14.44% (p=0.800±0.018)
4,pairs_structural_break,Out-of-Sample Hedge Ratio Drift & Cointegratio...,HIGH_VULNERABILITY,In-Sample β / ADF p: β=0.687 (p=0.031),OOS β Drift / ADF p: Δβ=181.8% (p=0.897)


## Cell 11 — Publication Figures (Equity Curve, Underwater Drawdown & Causal Signal Trajectory)
Generates the primary visual diagnostics with explicit Out-of-Sample split and circuit-breaker annotations.

In [11]:
# Cell 11: Plot Equity Curve, Underwater Drawdown, and Causal Signal Trajectory
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

bench_equity = initial_cap * np.cumprod(1.0 + daily_bench_ret)
x_idx = np.arange(n_bars)
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(x_idx, equity_net, color='#059669', lw=1.8, label='Confirmed Rule (Net)')
axes[0].plot(x_idx, bench_equity, color='#78716c', lw=1.3, ls='--', label=f'Benchmark ({spec.benchmark_ticker})')
axes[0].axvline(split_idx, color='#d97706', ls=':', lw=1.2, label=f'OOS Split ({split_date})')
axes[0].set_ylabel('Portfolio Equity ($)')
axes[0].set_title(f'{spec.title} — Walk-Forward Equity, Underwater Drawdown & Causal Signal')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].grid(True, alpha=0.25)

axes[1].fill_between(x_idx, underwater_series_pct, 0.0, color='#e11d48', alpha=0.22)
axes[1].plot(x_idx, underwater_series_pct, color='#e11d48', lw=1.2, label='Underwater Drawdown (%)')
axes[1].axvline(split_idx, color='#d97706', ls=':', lw=1.2)
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend(loc='lower left', fontsize=8)
axes[1].grid(True, alpha=0.25)

axes[2].plot(x_idx, signal_values, color='#0284c7', lw=1.1, label=f'Causal Signal ({spec.signal_variant})')
axes[2].axvline(split_idx, color='#d97706', ls=':', lw=1.2)
axes[2].set_ylabel('Signal Value')
axes[2].set_xlabel('Trading Bar Index')
axes[2].legend(loc='upper left', fontsize=8)
axes[2].grid(True, alpha=0.25)

plt.tight_layout()
plt.close(fig)
print('Generated 3-panel Matplotlib diagnostic figure successfully.')

Generated 3-panel Matplotlib diagnostic figure successfully.


## Cell 12 — Empirical Hypothesis Scorecard, Multi-Question Breakdown & Formula Ledger
Synthesizes the 3-layer hypothesis comparison (separating return co-movement, cointegration stationarity, and trading rule P&L), the 5 scorecard dimensions, and the step-by-step substituted formula ledger.

In [12]:
# Cell 12: Build Empirical Hypothesis Scorecard & Formula Calculation Ledger
if len(trades) > 0 and full_metrics.total_gross_return_pct > 0:
    avg_gross_bps = float(np.mean([t.gross_return_pct for t in trades])) * 100.0
    break_even_bps = round(max(0.0, avg_gross_bps / 2.0), 1)
else:
    break_even_bps = 0.0

scorecard = build_hypothesis_scorecard_and_ledger(
    spec=spec,
    full_metrics=full_metrics,
    in_metrics=is_metrics,
    oos_metrics=oos_metrics,
    factor_regression=factor_reg,
    bootstrap_cis=boot_cis,
    break_even_cost_bps=break_even_bps,
    parameter_surface=param_surface,
    pairs_diag=pairs_diag,
    stress_suite=stress_suite,
)

print(f'Historical Verdict: {scorecard.historical_verdict_label} ({scorecard.supported_checks_count}/{scorecard.evaluated_checks_count} evaluated checks supported)')
print(scorecard.executive_summary)

if scorecard.hypothesis_comparison_breakdown:
    display(pd.DataFrame(scorecard.hypothesis_comparison_breakdown))

display(pd.DataFrame([d.model_dump() for d in scorecard.dimensions]))
display(pd.DataFrame([s.model_dump() for s in scorecard.calculation_steps]))

Historical Verdict: CONTRADICTED BY HISTORICAL DATA (0/5 evaluated checks supported)
Historical Audit Summary for 'I think Coca-Cola and Pepsi prices tend to converge after unusually large divergences.': Across 2011 trading days (2018-01-02 to 2025-12-31), the specified rule triggered 66 trades (In-Sample N = 36, Out-of-Sample N = 30), generating -23.95% net return (Sharpe -0.61, Max Drawdown 25.83%) versus +187.48% (Sharpe 0.78) for passive SPY Buy-and-Hold (-211.43 percentage points excess return). The 25.0% portfolio drawdown circuit breaker tripped on 2025-07-17. Econometrically, KO and PEP exhibit daily return correlation r = +0.739, while the log-price spread ADF stationarity test yields p = 0.0314 (stationary cointegrated spread). The trading rule satisfied 0 of 5 evaluated empirical checks (CONTRADICTED BY HISTORICAL DATA). Note: This scorecard summarizes historical sample evidence using explicit heuristic decision thresholds and does not constitute investment advice or a predi

,aspect,lens_title,question_answered,evidence,primary_metric,verdict,status,explanation,interpretation
0,1. Return Co-Movement (Daily Correlation),1. Return Co-Movement (Daily Correlation),Do KO and PEP daily returns move in the same d...,Daily Return r = +0.739 | Log-Price R² = 0.844...,Daily Return r = +0.739 | Log-Price R² = 0.844...,POSITIVE CO-MOVEMENT,SUPPORTED,KO and PEP daily returns exhibit r = +0.739 co...,KO and PEP daily returns exhibit r = +0.739 co...
1,2. Long-Run Equilibrium Cointegration (Station...,2. Long-Run Equilibrium Cointegration (Station...,Is the log-price spread between KO and PEP sta...,ADF Stat = -3.039 (p = 0.0314) | Half-Life = 3...,ADF Stat = -3.039 (p = 0.0314) | Half-Life = 3...,COINTEGRATED (p < 0.05),SUPPORTED,Formation spread does NOT reject the unit-root...,Formation spread does NOT reject the unit-root...
2,3. Active Spread Mean-Reversion Trading Rule,3. Active Spread Mean-Reversion Trading Rule,Did trading ±2.00σ spread divergences generate...,"Net Return: -23.95% (N = 66 trades, IS N = 36,...","Net Return: -23.95% (N = 66 trades, IS N = 36,...",NEGATIVE NET RETURN,CONTRADICTED,Trading ±2.00σ log-spread divergences produced...,Trading ±2.00σ log-spread divergences produced...


,dimension_id,name,question,status,evidence_metric,explanation
0,post_signal_direction,1. Post-Signal Direction & Net Trade Expectancy,Did historical occurrences of the signal produ...,CONTRADICTED,Mean Net Trade: -0.398% | Win Rate: 53.0% (N =...,Across 66 historical signals from 2018-01-02 t...
1,statistical_significance,2. Statistical Significance vs. Noise (Lo 2002...,Is the observed return/Sharpe ratio statistica...,CONTRADICTED,Sharpe: -0.61 ± 0.35 (PSR 4.0%) | HAC Alpha t ...,After adjusting for sample skewness (-0.69) an...
2,out_of_sample_persistence,3. Held-Out Out-of-Sample Persistence,Did the hypothesis continue to hold in the uns...,CONTRADICTED,"In-Sample SR: 0.01 (-0.5%, N=36) → Out-of-Samp...",In the held-out validation sample (2021-12-30 ...
3,transaction_cost_survival,4. Transaction Cost & Microstructure Friction ...,Do historical returns survive realistic commis...,CONTRADICTED,Break-Even Cost: 0.0 bps/notional (Configured:...,Deducting 10.0 bps per unit of portfolio notio...
4,stress_test_resilience,5. Adversarial Stress-Test & Null-Benchmark Re...,"Did the signal survive outlier removal, regime...",CONTRADICTED,0/5 Resilient | 0/5 Moderate | 5/5 High Vulner...,"Out of 5 adversarial stress tests in Step 5, t..."


,step_id,title,formula,substituted_inputs,output_value,interpretation
0,calc_net_return,"Step A — Compounded Net Trade Return, Unit Gro...","R_{net,i} = R_{gross,i} − 2·c, |w_A| + |w_B|...","Initial Capital E_0 = $100,000.00, Unit gross ...",Gross = -13.18% → Net = -23.95% (Mean Net/Trad...,Every trade deducts round-trip commission + sl...
1,calc_lo_sharpe,Step B — Annualized Sharpe Ratio & Lo (2002) /...,"SR = √252 × (μ_d / σ_d), SE(SR_d) = √[(1 − γ...","T = 2011 days, Ann. Vol = 5.36%, Skewness γ₃ =...","Annualized SR = -0.614, Lo (2002) SE = 0.351, ...",Adjusts standard Sharpe inference for non-norm...
2,calc_dsr,Step C — Bailey & López de Prado (2014) Multip...,E[max SR₀] ≈ √V[{SR_k}] × [(1 − γ)·Φ⁻¹(1 − 1/K...,"K = 25 parameter grid trials, Null Hurdle E[ma...",Selected Spec DSR = 0.1% (Grid Best DSR = 37.5%),Penalizes the observed Sharpe ratio for the nu...
3,calc_newey_west,Step D — Newey-West (1987) HAC Benchmark Facto...,"R_strat,t = α + β · R_bench,t + ε_t, Var_HAC...","Benchmark = SPY, Bartlett kernel lags L = 7, S...","Annualized α = -3.15% (t_HAC = -1.78, p = 0.07...",Separates passive correlation with the benchma...
4,calc_pairs_coint,Step E — Cointegrated Spread & Exact Discrete ...,"S_t = log(P_A,t) − (hat(α) + hat(β)·log(P_B,t)...",Formation window: 2018-01-02 to 2021-12-29 | h...,"ADF Stat = -3.039 (p = 0.0314), Half-Life t_{1...",Formation spread does NOT reject the unit-root...


## Cell 13 — Automated End-to-End Reconciliation Assertions
Programmatically asserts that every metric, trade count, drawdown, econometric statistic, and verdict recomputed in this notebook matches the API `AuditReport` payload.

In [13]:
# Cell 13: Assert Exact Reconciliation Between Notebook Execution and AuditReport
assert len(trades) == EXPECTED_SNAPSHOT['trade_count'], f'Trade count mismatch: {len(trades)} != {EXPECTED_SNAPSHOT["trade_count"]}'

for split_key, comp_metrics in [
    ('full_sample', full_metrics),
    ('in_sample', is_metrics),
    ('out_of_sample', oos_metrics),
]:
    exp_m = EXPECTED_SNAPSHOT[split_key]
    for field_name in (
        'trading_days', 'trade_count', 'exposure_pct', 'total_gross_return_pct',
        'total_net_return_pct', 'cagr_net_pct', 'benchmark_return_pct',
        'max_drawdown_pct', 'benchmark_max_drawdown_pct', 'sharpe_ratio',
        'probabilistic_sharpe_psr_pct', 'win_rate_pct', 'is_zero_exposure',
        'circuit_breaker_triggered', 'halted_prior_to_period'
    ):
        actual_val = getattr(comp_metrics, field_name)
        expected_val = exp_m[field_name]
        if isinstance(actual_val, float):
            assert math.isclose(actual_val, float(expected_val), abs_tol=1e-6), (
                f'Mismatch in {split_key}.{field_name}: {actual_val} != {expected_val}'
            )
        else:
            assert actual_val == expected_val, (
                f'Mismatch in {split_key}.{field_name}: {actual_val} != {expected_val}'
            )

for exp_b, act_b in zip(EXPECTED_SNAPSHOT['bootstrap_cis'], boot_cis):
    assert exp_b['metric_name'] == act_b.metric_name, f'Bootstrap metric name mismatch: {act_b.metric_name}'
    assert math.isclose(float(exp_b['point_estimate']), act_b.point_estimate, abs_tol=1e-6), f'Bootstrap point estimate mismatch for {act_b.metric_name}'
    assert math.isclose(float(exp_b['ci_lower_95']), act_b.ci_lower_95, abs_tol=1e-6), f'Bootstrap CI lower mismatch for {act_b.metric_name}'
    assert math.isclose(float(exp_b['ci_upper_95']), act_b.ci_upper_95, abs_tol=1e-6), f'Bootstrap CI upper mismatch for {act_b.metric_name}'

for exp_st, act_st in zip(EXPECTED_SNAPSHOT['stress_test_outcomes'], stress_suite.tests):
    assert exp_st['test_id'] == act_st.test_id, f'Stress test ID mismatch: {act_st.test_id}'
    assert exp_st['outcome_badge'] == act_st.outcome_badge, f'Stress test outcome mismatch for {act_st.test_id}'
    assert exp_st['baseline_value'] == act_st.baseline_value, f'Stress test baseline mismatch for {act_st.test_id}'
    assert exp_st['stressed_value'] == act_st.stressed_value, f'Stress test stressed mismatch for {act_st.test_id}'

assert scorecard.historical_verdict_label == EXPECTED_SNAPSHOT['verdict_label'], 'Verdict label mismatch!'
assert scorecard.supported_checks_count == EXPECTED_SNAPSHOT['supported_checks_count'], 'Supported checks count mismatch!'
assert scorecard.evaluated_checks_count == EXPECTED_SNAPSHOT['evaluated_checks_count'], 'Evaluated checks count mismatch!'
for d in scorecard.dimensions:
    assert d.status == EXPECTED_SNAPSHOT['dimension_statuses'][d.dimension_id], f'Dimension {d.dimension_id} status mismatch!'

print('ALL AUTOMATED RECONCILIATION ASSERTIONS PASSED (100% match with API AuditReport).')

ALL AUTOMATED RECONCILIATION ASSERTIONS PASSED (100% match with API AuditReport).


## Cell 14 — Complete UI-to-Notebook-to-Engine Coverage Map
Maps every metric, table, chart, and badge in the 6-Step Alpha Auditor UI and PDF report to the exact production Python function and notebook cell that verifies it.

| UI Step & Component | Displayed Outputs | Production Function | Notebook Verification Cell |
| :--- | :--- | :--- | :--- |
| **Step 1 & 2: Intake & Spec Builder** | `StrategySpec`, User vs. Software Defaults, Ambiguities | `engine.hypothesis_parser.interpret_hypothesis` | Cell 2 |
| **Step 3: Literature & Rigor Checklist** | Curated Papers, Causal & Split Safeguards | `engine.literature.build_literature_review` | Cell 3 & Cell 4 |
| **Step 4: KPI Strip & Split Table** | OOS / IS / Full Net Return, Excess pp, Sharpe, PSR, MDD, N | `engine.backtester._compute_period_metrics` | Cell 6 & Cell 7 |
| **Step 4: Equity, Drawdown & Signal Charts** | `equity_net`, `equity_benchmark`, `drawdown_net_pct`, `signal_values` | `engine.backtester.simulate_strategy_core` & `compute_causal_signals` | Cell 4, Cell 5, Cell 6, Cell 11 |
| **Step 4: Econometrics & 25-Cell DSR Grid** | Newey-West HAC $\alpha/\beta$, Stationary Bootstrap CI, Pairs ADF/OU, 5×5 DSR | `engine.econometrics.*` & `compute_parameter_sensitivity_surface` | Cell 8 & Cell 9 |
| **Step 5: Adversarial Stress Suite** | Outlier Removal, $t+2$ Latency, Regimes, Non-Overlapping MC Null, Drift | `engine.stress_tests.run_adversarial_stress_suite` | Cell 10 |
| **Step 6: Verdict, Multi-Question Breakdown & Blotter** | Co-Movement vs. Cointegration vs. Rule P&L, 5 Dimensions, Formula Ledger, Trades | `engine.stress_tests.build_hypothesis_scorecard_and_ledger` | Cell 5, Cell 12, Cell 13 |